In [1]:
# ============================================================
# 0. Setup and raw data loading
# ============================================================

from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd

RANDOM_STATE = 9890
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# Change this only if your CSV files are stored in another folder.
DATA_DIR = Path(".")

def find_file(candidate_names, data_dir=DATA_DIR):
    """
    Looks for a file using a short list of possible names.
    This makes the notebook robust to files like school_covariates.csv
    versus school_covariates(2).csv.
    """
    for name in candidate_names:
        path = data_dir / name
        if path.exists():
            return path
    
    raise FileNotFoundError(
        "Could not find any of these files in "
        f"{data_dir.resolve()}:\n" + "\n".join(candidate_names)
    )

PATHS = {
    "school": find_file(["school_covariates.csv", "school_covariates(2).csv"]),
    "district": find_file(["district_covariates.csv", "district_covariates(2).csv"]),
    "train": find_file(["scores_training.csv", "scores_training(2).csv"]),
    "test": find_file(["scores_test.csv", "scores_test(2).csv"]),
}

school_covariates = pd.read_csv(PATHS["school"])
district_covariates = pd.read_csv(PATHS["district"])
scores_training = pd.read_csv(PATHS["train"])
scores_test = pd.read_csv(PATHS["test"])

# Preserve ID and categorical columns as strings.
STRING_COLS = [
    "ASSESSMENT_ID", "SCHOOL", "DISTRICT", "COUNTY",
    "SUBGROUP_NAME", "ASSESSMENT_NAME", "DISTRICT_TYPE", "REGION"
]

for df in [school_covariates, district_covariates, scores_training, scores_test]:
    for col in STRING_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string")

print("Loaded files:")
for key, path in PATHS.items():
    print(f"  {key:8s}: {path}")

def raw_summary(name, df):
    return {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "object_or_string_cols": int(
            df.select_dtypes(include=["object", "string"]).shape[1]
        ),
        "numeric_cols": int(df.select_dtypes(include=[np.number]).shape[1]),
    }

summary = pd.DataFrame([
    raw_summary("school_covariates", school_covariates),
    raw_summary("district_covariates", district_covariates),
    raw_summary("scores_training", scores_training),
    raw_summary("scores_test", scores_test),
])

print("\nRaw table summary:")
print(summary.to_string(index=False))

print("\nKey checks:")
print("school_covariates['SCHOOL'] unique:      ", school_covariates["SCHOOL"].is_unique)
print("district_covariates['DISTRICT'] unique:  ", district_covariates["DISTRICT"].is_unique)
print("scores_training['ASSESSMENT_ID'] unique: ", scores_training["ASSESSMENT_ID"].is_unique)
print("scores_test['ASSESSMENT_ID'] unique:     ", scores_test["ASSESSMENT_ID"].is_unique)

print("\nTarget checks:")
print("'PERCENT_PROFICIENT' in training:", "PERCENT_PROFICIENT" in scores_training.columns)
print("'PERCENT_PROFICIENT' in test:    ", "PERCENT_PROFICIENT" in scores_test.columns)

train_school_coverage = scores_training["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()
test_school_coverage = scores_test["SCHOOL"].isin(school_covariates["SCHOOL"]).mean()

school_districts = school_covariates[["SCHOOL", "DISTRICT"]].drop_duplicates()

train_district_coverage = (
    scores_training[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

test_district_coverage = (
    scores_test[["SCHOOL"]]
    .drop_duplicates()
    .merge(school_districts, on="SCHOOL", how="left")["DISTRICT"]
    .isin(district_covariates["DISTRICT"])
    .mean()
)

print("\nJoin coverage:")
print(f"training rows with SCHOOL in school_covariates: {train_school_coverage:.4f}")
print(f"test rows with SCHOOL in school_covariates:     {test_school_coverage:.4f}")
print(f"training unique schools with DISTRICT data:     {train_district_coverage:.4f}")
print(f"test unique schools with DISTRICT data:         {test_district_coverage:.4f}")

print("\nTarget summary:")
print(scores_training["PERCENT_PROFICIENT"].describe().to_string())

print("\nSoftware:")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Random state:", RANDOM_STATE)

Loaded files:
  school  : school_covariates.csv
  district: district_covariates.csv
  train   : scores_training.csv
  test    : scores_test.csv

Raw table summary:
              table   rows  cols  duplicate_rows  missing_cells  object_or_string_cols  numeric_cols
  school_covariates   4754    52               0          26460                      5            47
district_covariates    674     6               0              0                      1             5
    scores_training 144921     6               0              0                      4             2
        scores_test  48307     5               0              0                      4             1

Key checks:
school_covariates['SCHOOL'] unique:       True
district_covariates['DISTRICT'] unique:   True
scores_training['ASSESSMENT_ID'] unique:  True
scores_test['ASSESSMENT_ID'] unique:      True

Target checks:
'PERCENT_PROFICIENT' in training: True
'PERCENT_PROFICIENT' in test:     False

Join coverage:
training rows with 

In [2]:
# ============================================================
# 1. Merge datasets
# ============================================================

# Merge school covariates
train_full = scores_training.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

test_full = scores_test.merge(
    school_covariates,
    on="SCHOOL",
    how="left",
    validate="many_to_one"
)

# Merge district covariates
train_full = train_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

test_full = test_full.merge(
    district_covariates,
    on="DISTRICT",
    how="left",
    validate="many_to_one"
)

print("train_full shape:", train_full.shape)
print("test_full shape:", test_full.shape)

train_full shape: (144921, 62)
test_full shape: (48307, 61)


In [3]:
# ============================================================
# 2. Missingness overview
# ============================================================

def missing_report(df):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    
    report = pd.DataFrame({
        "missing_count": miss,
        "missing_pct": (miss / len(df)) * 100
    })
    
    return report

train_missing = missing_report(train_full)
test_missing = missing_report(test_full)

print("Train missing columns:", train_missing.shape[0])
print("Test missing columns:", test_missing.shape[0])

print("\nTop 15 missing (train):")
print(train_missing.head(15))

print("\nTop 15 missing (test):")
print(test_missing.head(15))

Train missing columns: 52
Test missing columns: 52

Top 15 missing (train):
                                                    missing_count  missing_pct
TEACHER_TURNOVER_RATE                                      134136    92.558014
KINDERGARTEN_AVERAGE_CLASS_SIZE                            103467    71.395450
GRADE_1_AVERAGE_CLASS_SIZE                                 102899    71.003512
GRADE_2_AVERAGE_CLASS_SIZE                                 102779    70.920709
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_...          89314    61.629439
PERCENT_DROPOUT                                             16061    11.082590
PERCENT_GED                                                 16061    11.082590
PERCENT_STILL_ENROLLED                                      16061    11.082590
PERCENT_NON_DIPLOMA                                         16061    11.082590
PERCENT_DIPLOMA                                             16061    11.082590
SCIENCE_AVERAGE_CLASS_SIZE                             

In [4]:
# ============================================================
# 3. School-level missingness check
# ============================================================

# Example column: ATTENDANCE_RATE (you can change later)
col = "ATTENDANCE_RATE"

train_missing_schools = train_full[train_full[col].isna()]["SCHOOL"].nunique()
test_missing_schools = test_full[test_full[col].isna()]["SCHOOL"].nunique()

train_missing_rows = train_full[col].isna().sum()
test_missing_rows = test_full[col].isna().sum()

print("Using column:", col)

print("\nTrain rows missing:", train_missing_rows)
print("Train unique schools missing:", train_missing_schools)

print("\nTest rows missing:", test_missing_rows)
print("Test unique schools missing:", test_missing_schools)

Using column: ATTENDANCE_RATE

Train rows missing: 3036
Train unique schools missing: 94

Test rows missing: 983
Test unique schools missing: 93


In [5]:
# ============================================================
# 4. Build X and y + preserve IDs
# ============================================================

TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# Save IDs separately (needed for submission later)
train_ids = train_full[ID_COL].copy()
test_ids = test_full[ID_COL].copy()

# Target
y_train = train_full[TARGET].copy()

# Drop target from features
X_train = train_full.drop(columns=[TARGET])
X_test = test_full.copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (144921, 61)
X_test shape: (48307, 61)
y_train shape: (144921,)


In [6]:
# ============================================================
# 5. Column typing
# ============================================================

ID_COL = "ASSESSMENT_ID"

# Identify column types
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

# Remove ID from categorical if present
if ID_COL in categorical_cols:
    categorical_cols.remove(ID_COL)

# High-cardinality (simple rule: > 50 unique values)
high_cardinality_cols = [
    col for col in categorical_cols
    if X_train[col].nunique() > 50
]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))
print("High-cardinality cols:", high_cardinality_cols)
print("Low-cardinality cols:", low_cardinality_cols)

Numeric cols: 53
Categorical cols: 7
High-cardinality cols: ['SCHOOL', 'DISTRICT', 'COUNTY']
Low-cardinality cols: ['SUBGROUP_NAME', 'ASSESSMENT_NAME', 'DISTRICT_TYPE', 'REGION']


In [7]:
# ============================================================
# 6A. Frequency encoding (safe, no leakage)
# ============================================================

X_train_proc = X_train.copy()
X_test_proc = X_test.copy()

freq_encoding_cols = []

for col in high_cardinality_cols:
    freq_map = X_train_proc[col].value_counts(dropna=False)
    
    new_col = col + "_freq"
    X_train_proc[new_col] = X_train_proc[col].map(freq_map).astype(float)
    X_test_proc[new_col] = X_test_proc[col].map(freq_map).fillna(0).astype(float)
    
    freq_encoding_cols.append(new_col)

print("Added frequency columns:", freq_encoding_cols)
print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)

Added frequency columns: ['SCHOOL_freq', 'DISTRICT_freq', 'COUNTY_freq']
X_train_proc shape: (144921, 64)
X_test_proc shape: (48307, 64)


In [8]:
# ============================================================
# 6B. Missing indicators + median imputation (more robust than mean)
# ============================================================

numeric_cols_extended = numeric_cols + freq_encoding_cols

missing_indicator_cols = []

for col in numeric_cols_extended:
    if X_train_proc[col].isna().sum() > 0:
        new_col = col + "_missing"
        
        X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
        X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
        
        median_val = X_train_proc[col].median()
        
        X_train_proc[col] = X_train_proc[col].fillna(median_val)
        X_test_proc[col] = X_test_proc[col].fillna(median_val)
        
        missing_indicator_cols.append(new_col)

print("Missing indicators added:", len(missing_indicator_cols))
print("New shape:", X_train_proc.shape)

Missing indicators added: 52
New shape: (144921, 116)


/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_28395/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_proc[new_col] = X_train_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_28395/1325974498.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_proc[new_col] = X_test_proc[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_28395/1325974498.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

In [9]:
# ============================================================
# 6C. Drop raw high-cardinality categorical columns
# ============================================================

X_train_proc = X_train_proc.drop(columns=high_cardinality_cols)
X_test_proc = X_test_proc.drop(columns=high_cardinality_cols)

print("Shape after dropping high-cardinality cols:", X_train_proc.shape)

Shape after dropping high-cardinality cols: (144921, 113)


### Feature Encoding: High-Cardinality Variables

The raw high-cardinality categorical variables `SCHOOL`, `DISTRICT`, and `COUNTY` were removed from the modeling matrix after frequency encodings were created for them.

This prevents the model from directly using raw ID-like categorical labels while still preserving useful information about how frequently each school, district, or county appears in the training data.

After dropping these raw columns, the feature matrix has 115 columns.

In [10]:
# ============================================================
# 6D. One-hot encode low-cardinality categorical variables
# ============================================================

X_train_proc = pd.get_dummies(
    X_train_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

X_test_proc = pd.get_dummies(
    X_test_proc,
    columns=low_cardinality_cols,
    drop_first=False
)

# Align columns (important!)
X_train_proc, X_test_proc = X_train_proc.align(X_test_proc, join="left", axis=1, fill_value=0)

print("Final X_train shape:", X_train_proc.shape)
print("Final X_test shape:", X_test_proc.shape)

Final X_train shape: (144921, 163)
Final X_test shape: (48307, 163)


### Final Feature Matrix

After full preprocessing:

- High-cardinality variables (`SCHOOL`, `DISTRICT`, `COUNTY`) were replaced with frequency encodings
- Missing values were handled via:
  - median imputation
  - explicit missingness indicator variables
- Low-cardinality categorical variables were one-hot encoded:
  - `SUBGROUP_NAME`, `ASSESSMENT_NAME`, `DISTRICT_TYPE`, `REGION`

Final dimensions:
- Training set: 144,921 rows × 165 features
- Test set: 48,307 rows × 165 features

The feature space is now fully numeric, aligned between train and test, and ready for modeling.

In [11]:
# ============================================================
# Create modeling copy WITHOUT ID (non-destructive)
# ============================================================

X_train_proc_model = X_train_proc.drop(columns=["ASSESSMENT_ID"])
X_test_proc_model = X_test_proc.drop(columns=["ASSESSMENT_ID"])

print("Modeling shape:", X_train_proc_model.shape)

Modeling shape: (144921, 162)


### Modeling Dataset

A separate modeling dataset was created by removing the identifier column `ASSESSMENT_ID`.

Final modeling dimensions:
- Training set: 144,921 rows × 164 features

This ensures that all features used for modeling are numeric or boolean, and that no identifier-based leakage occurs.

In [12]:
# ============================================================
# 7A. Simple Linear Regression (one feature at a time)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

simple_lr_results = []

for col in X_train_proc_model.columns:
    model = LinearRegression()
    model.fit(X_tr[[col]], y_tr)
    
    train_pred = model.predict(X_tr[[col]])
    val_pred = model.predict(X_val[[col]])
    
    simple_lr_results.append({
        "feature": col,
        "train_mse": mean_squared_error(y_tr, train_pred),
        "val_mse": mean_squared_error(y_val, val_pred)
    })

simple_lr_results = pd.DataFrame(simple_lr_results).sort_values("val_mse")

print(simple_lr_results.head(30).to_string(index=False))

                                                    feature  train_mse    val_mse
                         PERCENT_ECONOMICALLY_DISADVANTAGED 570.735244 581.066288
                                         PERCENT_FREE_LUNCH 575.082361 584.868424
                                            PERCENT_DIPLOMA 621.742550 626.029451
                                     PERCENT_STILL_ENROLLED 637.471784 641.000299
                                           PERCENT_HOMELESS 640.796399 644.070716
                                              PERCENT_BLACK 648.963234 652.648495
                                  PERCENT_WITH_DISABILITIES 650.316873 653.054139
                           PERCENT_ENGLISH_LANGUAGE_LEANERS 648.460937 655.348885
                                            ATTENDANCE_RATE 650.513825 657.793054
                                              PERCENT_WHITE 653.127789 658.601548
                                            PERCENT_DROPOUT 653.769449 661.927501
                

In [13]:
# ============================================================
# Extract best simple linear regression feature properly
# ============================================================

best_feature_row = simple_lr_results.loc[simple_lr_results["val_mse"].idxmin()]

print("Best single-feature model:")
print(best_feature_row)

Best single-feature model:
feature      PERCENT_ECONOMICALLY_DISADVANTAGED
train_mse                            570.735244
val_mse                              581.066288
Name: 43, dtype: object


### Simple Linear Regression Baseline

Each feature was tested individually in a simple linear regression model. This creates a baseline ranking of single predictors before fitting larger multiple regression models.

The goal is not to select the final model from one feature, but to identify which variables have the strongest individual linear relationship with `PERCENT_PROFICIENT`.

### Best Simple Linear Regression Feature

The best single-feature model was:

- Feature: `PERCENT_ECONOMICALLY_DISADVANTAGED`
- Train MSE: 570.74  
- Validation MSE: 581.07  

Interpretation:
- Socioeconomic disadvantage is the strongest standalone predictor of `PERCENT_PROFICIENT`.
- However, the error (~581) is substantially higher than the multiple linear regression model (~312), indicating that no single variable explains the outcome well.
- This confirms that predictive power in the dataset is distributed across multiple correlated features rather than dominated by a single factor.

Conclusion:
Simple linear regression provides insight into marginal relationships but is insufficient for accurate prediction on its own.

### Simple Linear Regression Insights

The strongest individual predictors of `PERCENT_PROFICIENT` are:

- Socioeconomic indicators:
  - `PERCENT_ECONOMICALLY_DISADVANTAGED`
  - `PERCENT_FREE_LUNCH`
- Academic outcomes:
  - `PERCENT_DIPLOMA`
  - `PERCENT_STILL_ENROLLED`
- Demographics:
  - `PERCENT_BLACK`, `PERCENT_WHITE`, `PERCENT_HISPANIC`
- Vulnerability indicators:
  - `PERCENT_HOMELESS`, `PERCENT_WITH_DISABILITIES`
- Attendance:
  - `ATTENDANCE_RATE`

Key observations:
- Socioeconomic disadvantage is the strongest single predictor.
- Many top features are highly correlated (e.g., free lunch vs economic disadvantage).
- Missingness indicators appear, confirming that missing data carries signal.
- Frequency-encoded variables are not dominant individually, suggesting entity effects are weaker in isolation.

Conclusion:
Simple linear regression highlights strong marginal relationships but does not account for interactions or multicollinearity.

In [14]:
# ============================================================
# 7B. Multiple Linear Regression (clean baseline)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Fresh split (consistent with 7A)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_proc_model, y_train, test_size=0.2, random_state=9890
)

model = LinearRegression()
model.fit(X_tr, y_tr)

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_mse = mean_squared_error(y_tr, train_pred)
val_mse = mean_squared_error(y_val, val_pred)

print("Train MSE:", train_mse)
print("Validation MSE:", val_mse)

Train MSE: 305.12409327017724
Validation MSE: 312.6016702920729


### Multiple Linear Regression Baseline

A multiple linear regression model was fitted using all available features.

Results:
- Train MSE: 305.12  
- Validation MSE: 312.60  

Interpretation:
- The gap between training and validation error is small, indicating minimal overfitting.
- The model performs substantially better than the best simple linear regression (~581 MSE), showing that predictive power is distributed across multiple features.
- The relatively low validation error suggests that linear relationships capture a large portion of the underlying structure in the data.

Conclusion:
Multiple linear regression provides a strong and stable baseline for evaluating more complex models.

In [15]:
# Select top 10 features from simple LR
top_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()

print(top_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']


In [16]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)

X_tr_poly = poly.fit_transform(X_tr[top_features])
X_val_poly = poly.transform(X_val[top_features])

print("Poly feature shape:", X_tr_poly.shape)

Poly feature shape: (115936, 65)


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_poly, y_tr)

train_pred = model.predict(X_tr_poly)
val_pred = model.predict(X_val_poly)

print("Polynomial Regression:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Polynomial Regression:
Train MSE: 507.61481933233904
Validation MSE: 515.1911286593601


### Polynomial Regression

Polynomial regression (degree 2) was applied to the top 10 features identified from simple linear regression.

Results:
- Train MSE: 507.61  
- Validation MSE: 515.19  

Interpretation:
- Performance is significantly worse than multiple linear regression (~312 MSE).
- This indicates that restricting the model to a small subset of features removes important predictive information.
- The polynomial expansion does not compensate for the loss of breadth in the feature space.

Conclusion:
The dataset appears to benefit more from combining many features linearly rather than modeling nonlinear relationships among a small subset of variables. Polynomial regression is not effective in this setting.

In [18]:
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()
print(top5_features)

['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']


In [19]:
from itertools import combinations

interaction_cols = []

X_tr_int = X_tr.copy()
X_val_int = X_val.copy()

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    
    X_tr_int[new_col] = X_tr[f1] * X_tr[f2]
    X_val_int[new_col] = X_val[f1] * X_val[f2]
    
    interaction_cols.append(new_col)

print("Number of interaction features added:", len(interaction_cols))

Number of interaction features added: 10


In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_tr_int, y_tr)

train_pred = model.predict(X_tr_int)
val_pred = model.predict(X_val_int)

print("Interaction Model:")
print("Train MSE:", mean_squared_error(y_tr, train_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Interaction Model:
Train MSE: 304.3250676749579
Validation MSE: 311.955526746157


### Interaction Model

Pairwise interaction terms were added between the top 5 features identified from simple linear regression.

Results:
- Train MSE: 304.33  
- Validation MSE: 311.96  

Interpretation:
- The interaction model slightly improves performance compared to multiple linear regression (~312.60 → ~311.96).
- The improvement is marginal, suggesting that most of the predictive structure is already captured by additive linear effects.
- Interactions contribute some additional signal but are not a dominant factor in this dataset.

Conclusion:
While interaction terms provide a small improvement, the dataset is largely driven by additive relationships rather than strong nonlinear interactions.

In [21]:
# ============================================================
# 8A. Build controlled candidate feature spaces
# ============================================================

from itertools import combinations

# Use strongest marginal predictors as candidates for engineered terms
top10_features = simple_lr_results.nsmallest(10, "val_mse")["feature"].tolist()
top5_features = simple_lr_results.nsmallest(5, "val_mse")["feature"].tolist()

# Base space
X_tr_base = X_tr.copy()
X_val_base = X_val.copy()

# Base + polynomial squares for top10
X_tr_polyspace = X_tr.copy()
X_val_polyspace = X_val.copy()

poly_cols = []

for col in top10_features:
    new_col = col + "_squared"
    X_tr_polyspace[new_col] = X_tr[col] ** 2
    X_val_polyspace[new_col] = X_val[col] ** 2
    poly_cols.append(new_col)

# Base + pairwise interactions for top5
X_tr_intspace = X_tr.copy()
X_val_intspace = X_val.copy()

interaction_cols = []

for f1, f2 in combinations(top5_features, 2):
    new_col = f"{f1}_x_{f2}"
    X_tr_intspace[new_col] = X_tr[f1] * X_tr[f2]
    X_val_intspace[new_col] = X_val[f1] * X_val[f2]
    interaction_cols.append(new_col)

# Base + polynomial + interactions
X_tr_combined = X_tr_polyspace.copy()
X_val_combined = X_val_polyspace.copy()

for col in interaction_cols:
    X_tr_combined[col] = X_tr_intspace[col]
    X_val_combined[col] = X_val_intspace[col]

feature_spaces = {
    "base": (X_tr_base, X_val_base),
    "base_plus_poly": (X_tr_polyspace, X_val_polyspace),
    "base_plus_interactions": (X_tr_intspace, X_val_intspace),
    "base_plus_poly_interactions": (X_tr_combined, X_val_combined)
}

print("Top 10 features used for squares:")
print(top10_features)

print("\nTop 5 features used for interactions:")
print(top5_features)

print("\nPolynomial columns added:", len(poly_cols))
print("Interaction columns added:", len(interaction_cols))

print("\nFeature space shapes:")
for name, (X_train_space, X_val_space) in feature_spaces.items():
    print(name, X_train_space.shape, X_val_space.shape)

Top 10 features used for squares:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 5 features used for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

Polynomial columns added: 10
Interaction columns added: 10

Feature space shapes:
base (115936, 162) (28985, 162)
base_plus_poly (115936, 172) (28985, 172)
base_plus_interactions (115936, 172) (28985, 172)
base_plus_poly_interactions (115936, 182) (28985, 182)


In [22]:
# ============================================================
# 8B. Forward stepwise selection across feature spaces
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def forward_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    remaining = list(X_train_space.columns)
    selected = []
    rows = []
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            val_mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, val_mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
            
            rows.append({
                "num_features": len(selected),
                "feature_added": best_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(rows), selected

forward_summary = {}

for space_name, (X_train_space, X_val_space) in feature_spaces.items():
    results, selected = forward_stepwise(
        X_train_space, X_val_space, y_tr, y_val, max_features=30
    )
    
    forward_summary[space_name] = {
        "results": results,
        "selected_features": selected,
        "best_val_mse": results["val_mse"].min() if len(results) > 0 else None,
        "best_num_features": results.loc[results["val_mse"].idxmin(), "num_features"] if len(results) > 0 else None
    }
    
    print("\n" + "=" * 80)
    print(space_name)
    print("=" * 80)
    print(results.tail(10).to_string(index=False))
    print("Best validation MSE:", forward_summary[space_name]["best_val_mse"])

KeyboardInterrupt: 

### Forward Stepwise Design Choice

Forward stepwise selection was used to evaluate whether a smaller subset of predictors can approach the performance of the full multiple linear regression model.

Because the full feature space contains 164–184 predictors depending on the feature set, unrestricted stepwise selection would be computationally expensive and would gradually reconstruct the full model. To keep the procedure tractable and focused on model parsimony, the search was capped at 30 selected features.

This cap is not intended to imply that 30 is theoretically optimal. Instead, it provides a practical stopping limit that allows us to examine whether most predictive gains occur early in the selection path. If validation MSE is still improving near 30 features, the cap can be increased later.

### Forward Stepwise Selection

Forward stepwise selection was applied across multiple feature spaces, including:
- Base feature space
- Base + polynomial terms
- Base + interaction terms
- Base + polynomial + interaction terms

Results:
- Best validation MSE (base): 332.31  
- Best validation MSE (poly): 329.66  

Interpretation:
- All stepwise models perform significantly worse than the full multiple linear regression model (~312.60).
- This indicates that predictive performance relies on combining a large number of features rather than selecting a small subset.
- Polynomial and interaction features provide only marginal improvements within the stepwise framework.

Conclusion:
Subset selection via forward stepwise is not effective for this dataset. The data exhibits a high-dimensional additive structure where many weak predictors contribute jointly. Methods that retain all features while controlling complexity (e.g., Ridge or Lasso) are more appropriate.

In [23]:
# ============================================================
# Backward Stepwise (controlled)
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd

def backward_stepwise(X_train_space, X_val_space, y_train, y_val, max_removals=30):
    
    selected = list(X_train_space.columns)
    results = []
    
    # Initial model (full)
    model = LinearRegression()
    model.fit(X_train_space[selected], y_train)
    val_pred = model.predict(X_val_space[selected])
    best_mse = mean_squared_error(y_val, val_pred)
    
    results.append({
        "num_features": len(selected),
        "removed_feature": None,
        "val_mse": best_mse
    })
    
    for _ in range(max_removals):
        candidates = []
        
        for feature in selected:
            trial_features = [f for f in selected if f != feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        worst_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse <= best_mse:
            selected.remove(worst_feature)
            best_mse = candidate_mse
            
            results.append({
                "num_features": len(selected),
                "removed_feature": worst_feature,
                "val_mse": best_mse
            })
        else:
            break
    
    return pd.DataFrame(results)


backward_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = backward_stepwise(X_train_space, X_val_space, y_tr, y_val, max_removals=30)
    
    backward_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features                               removed_feature    val_mse
          154                               PERCENT_DIPLOMA 312.396761
          153                                      GRADE_01 312.387188
          152                                      GRADE_12 312.378367
          151                                 PERCENT_ASIAN 312.378267
          150             REGION_Capital District, New York 312.378267
          149               REGION_North Country (New York) 312.377630
          148                       PERCENT_DIPLOMA_missing 312.377630
          147 ASSESSMENT_NAME_Regents Common Core Algebra I 312.377630
          146                          ASSESSMENT_NAME_ELA4 312.377146
          145                 DISTRICT_TYPE_High-Need Rural 312.377146
Best val MSE: 312.3771461462088

base_plus_poly
 num_features                                      removed_feature    val_mse
          164                                             GRADE_07 308.801547
         

In [24]:
# ============================================================
# Hybrid Stepwise (forward + backward cleanup)
# ============================================================

def hybrid_stepwise(X_train_space, X_val_space, y_train, y_val, max_features=30):
    
    remaining = list(X_train_space.columns)
    selected = []
    results = []
    
    best_mse = float("inf")
    
    while remaining and len(selected) < max_features:
        
        # Forward step
        candidates = []
        
        for feature in remaining:
            trial_features = selected + [feature]
            
            model = LinearRegression()
            model.fit(X_train_space[trial_features], y_train)
            
            val_pred = model.predict(X_val_space[trial_features])
            mse = mean_squared_error(y_val, val_pred)
            
            candidates.append((feature, mse))
        
        best_feature, candidate_mse = min(candidates, key=lambda x: x[1])
        
        if candidate_mse < best_mse:
            selected.append(best_feature)
            remaining.remove(best_feature)
            best_mse = candidate_mse
        else:
            break
        
        # Backward cleanup step
        improved = True
        while improved and len(selected) > 1:
            improved = False
            
            for feature in selected:
                trial_features = [f for f in selected if f != feature]
                
                model = LinearRegression()
                model.fit(X_train_space[trial_features], y_train)
                
                val_pred = model.predict(X_val_space[trial_features])
                mse = mean_squared_error(y_val, val_pred)
                
                if mse < best_mse:
                    selected.remove(feature)
                    best_mse = mse
                    improved = True
                    break
        
        results.append({
            "num_features": len(selected),
            "val_mse": best_mse
        })
    
    return pd.DataFrame(results)


hybrid_summary = {}

for name, (X_train_space, X_val_space) in feature_spaces.items():
    print("\n" + "="*80)
    print(name)
    print("="*80)
    
    results = hybrid_stepwise(X_train_space, X_val_space, y_tr, y_val, max_features=30)
    
    hybrid_summary[name] = results
    
    print(results.tail(10).to_string(index=False))
    print("Best val MSE:", results["val_mse"].min())


base
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly
 num_features    val_mse
           21 352.226262
           22 348.874109
           23 346.326338
           24 343.776219
           25 341.160388
           26 338.657725
           27 336.137043
           28 333.717583
           29 331.600432
           30 329.660042
Best val MSE: 329.6600419738662

base_plus_interactions
 num_features    val_mse
           21 356.639984
           22 352.619203
           23 348.915113
           24 346.330326
           25 344.123928
           26 341.579203
           27 338.762894
           28 336.466988
           29 334.174799
           30 332.314849
Best val MSE: 332.3148493948422

base_plus_poly_interactions
 num

### Backward and Hybrid Stepwise Selection

Backward and hybrid stepwise selection were evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Backward stepwise performed better than forward stepwise because it began with the full model and removed only features whose exclusion improved or preserved validation performance.

Best backward stepwise validation MSE values:

- Base: 312.35
- Base + polynomial: 308.77
- Base + interactions: 311.70
- Base + polynomial + interactions: 308.10

The best backward stepwise model was the base + polynomial + interaction model, with validation MSE of 308.10. This improves on the full multiple linear regression baseline of approximately 312.60.

The hybrid stepwise results matched the forward stepwise results exactly, suggesting that the backward cleanup phase did not remove any features after forward additions. Therefore, in this implementation, hybrid stepwise effectively behaved like forward stepwise.

Interpretation:
- Forward stepwise performed worse because it was limited to 30 selected features and could not capture the distributed signal across many predictors.
- Backward stepwise performed better because it retained most of the full feature space while pruning redundant or harmful variables.
- Polynomial terms provided meaningful improvement when added to the full feature space and pruned through backward selection.
- Interaction terms alone provided only modest improvement.

Conclusion:
The strongest linear-model-family result so far is backward stepwise on the combined polynomial + interaction feature space. This suggests that the dataset is mostly additive and high-dimensional, but selected nonlinear terms can improve performance when incorporated carefully.

In [23]:
# ============================================================
# Validation protocol note
# ============================================================

VALIDATION_PROTOCOL = {
    "current_stage": "development_holdout",
    "split_method": "train_test_split",
    "test_size": 0.20,
    "random_state": 9890,
    "final_stage": "cross_validation_on_shortlist",
    "note": (
        "Current MSE values are development validation scores. "
        "Cross-validation will be run later on shortlisted models."
    )
}

VALIDATION_PROTOCOL

{'current_stage': 'development_holdout',
 'split_method': 'train_test_split',
 'test_size': 0.2,
 'random_state': 9890,
 'final_stage': 'cross_validation_on_shortlist',
 'note': 'Current MSE values are development validation scores. Cross-validation will be run later on shortlisted models.'}

### Cross-Validation Strategy

At this stage, models are being evaluated using a fixed train/validation split. These results are useful for rapid model screening, but they should not be treated as final estimates of generalization performance.

Cross-validation will be applied later after the main model families have been tested. This avoids excessive computation during the exploratory phase while still allowing rigorous comparison among serious finalist models.

Current terminology:
- `val_mse`: development holdout validation MSE
- `cv_mse`: cross-validated MSE, to be computed later for shortlisted models

Planned approach:
1. Use the current validation split to screen many model classes quickly.
2. Record all results in a model ledger.
3. Shortlist the strongest models.
4. Run cross-validation on the shortlist.
5. Tune/refine the strongest cross-validated candidates.
6. Select the final model for submission.

For models involving target encoding or feature selection, special care will be needed to avoid leakage. Target encoding should be performed out-of-fold, and feature selection may need to be repeated inside folds if the selected model becomes a serious finalist.

In [24]:
# ============================================================
# 9A. Ridge Regression: two-stage alpha search + scaler comparison
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# We compare StandardScaler and RobustScaler instead of assuming one.
# We use a two-stage alpha search:
#   1. Broad search across many orders of magnitude
#   2. Fine search around the best alpha from the broad search
#
# Cross-validation is intentionally postponed until later finalist screening.
# These are still development holdout validation results.

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

broad_alphas = np.logspace(-4, 8, 49)  # 0.0001 to 100,000,000

def evaluate_ridge_grid(X_train_space, X_val_space, y_train, y_val, alphas, scaler_name, scaler_factory, stage):
    rows = []
    
    for alpha in alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_train_space, y_train)
        
        train_pred = model.predict(X_train_space)
        val_pred = model.predict(X_val_space)
        
        rows.append({
            "model_class": "Ridge",
            "stage": stage,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

ridge_broad_rows = []

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        result = evaluate_ridge_grid(
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            stage="broad"
        )
        
        result["feature_space"] = feature_space_name
        ridge_broad_rows.append(result)

ridge_broad_results = pd.concat(ridge_broad_rows, ignore_index=True)

best_broad_by_space_scaler = (
    ridge_broad_results
    .loc[ridge_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("Best broad Ridge result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

ridge_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]
    
    # Search within +/- 0.75 log10 units around the broad winner.
    # This is about a 5.6x range on either side.
    log_alpha = np.log10(best_alpha)
    fine_low = max(np.log10(broad_alphas.min()), log_alpha - 0.75)
    fine_high = min(np.log10(broad_alphas.max()), log_alpha + 0.75)
    
    fine_alphas = np.logspace(fine_low, fine_high, 31)
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = evaluate_ridge_grid(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        stage="fine"
    )
    
    result["feature_space"] = feature_space_name
    ridge_fine_rows.append(result)

ridge_fine_results = pd.concat(ridge_fine_rows, ignore_index=True)


# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

ridge_results = pd.concat(
    [ridge_broad_results, ridge_fine_results],
    ignore_index=True
)

best_ridge_by_space = (
    ridge_results
    .loc[ridge_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_ridge_overall = ridge_results.loc[ridge_results["val_mse"].idxmin()]

print("\nBest Ridge result by feature space:")
print(best_ridge_by_space.to_string(index=False))

print("\nOverall best Ridge result:")
print(best_ridge_overall)

# ------------------------------------------------------------
# Optional warning: if best alpha is at broad grid boundary
# ------------------------------------------------------------

if best_ridge_overall["alpha"] == broad_alphas.min():
    print("\nWARNING: Best alpha is at the minimum broad-grid boundary. Consider expanding lower.")
elif best_ridge_overall["alpha"] == broad_alphas.max():
    print("\nWARNING: Best alpha is at the maximum broad-grid boundary. Consider expanding higher.")

Best broad Ridge result by feature space and scaler:
model_class stage   scaler    alpha  train_mse    val_mse  aic  bic               feature_space
      Ridge broad standard 1.778279 300.906794 308.357271  NaN  NaN base_plus_poly_interactions
      Ridge broad   robust 0.316228 300.891402 308.358944  NaN  NaN base_plus_poly_interactions
      Ridge broad standard 0.000100 301.622160 308.925244  NaN  NaN              base_plus_poly
      Ridge broad   robust 0.000100 301.622160 308.925245  NaN  NaN              base_plus_poly
      Ridge broad standard 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad   robust 0.000100 304.325068 311.955527  NaN  NaN      base_plus_interactions
      Ridge broad standard 1.778279 305.124297 312.601534  NaN  NaN                        base
      Ridge broad   robust 0.000100 305.124093 312.601671  NaN  NaN                        base

Best Ridge result by feature space:
model_class stage   scaler    alpha  train_mse

### Ridge Regression Results

Ridge regression was evaluated across four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and alpha was tuned using a two-stage holdout-validation search.

Best Ridge model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 1.41
- Train MSE: 300.90
- Validation MSE: 308.36

Interpretation:
- Ridge performs best on the combined polynomial + interaction feature space.
- StandardScaler slightly outperforms RobustScaler.
- Ridge improves over the base multiple linear regression model but does not quite outperform the best backward stepwise model.
- The improvement appears to come mainly from the engineered feature space rather than from shrinkage alone.

Conclusion:
Ridge is a strong regularized linear model, but the current best linear-family model remains backward stepwise on the combined polynomial + interaction feature space, with validation MSE around 308.10.

In [27]:
# ============================================================
# 10A. Lasso Regression: data-driven two-stage alpha search
# ============================================================

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled spaces
# - Scalers: StandardScaler and RobustScaler
# - Alpha search:
#     Stage 1: broad data-driven search from alpha_max downward
#     Stage 2: fine search around the best broad alpha
# - Metric: development holdout validation MSE
# - Extra output: number of nonzero coefficients
# - CV is intentionally postponed until finalist screening

scaler_factories = {  #This means every feature space will be tested twice: once with standard scaling and once with robust scaling.
    "standard": StandardScaler,
    "robust": RobustScaler
}
# The constants below control the search over Lasso’s tuning parameter,
LASSO_BROAD_GRID_SIZE = 25 # means the first search tries 25 alpha values per feature-space/scaler combination.
LASSO_FINE_GRID_SIZE = 21 # means the second search tries 21 more alpha values near the best one from the broad search.
LASSO_MIN_ALPHA_RATIO = 1e-4 # means the broad search goes from alpha_max down to alpha_max * 0.0001
LASSO_FINE_WIDTH_LOG10 = 0.5 # means the fine search checks values about 3.16 times above and below the best broad alpha, because 10^0.5 ≈ 3.16.

LASSO_MAX_ITER = 30000 # gives the solver many iterations to converge. Lasso can be harder to optimize than Ridge, especially with correlated predictors.
LASSO_TOL = 1e-4 # controls how precise the optimization has to be before it stops.

def compute_lasso_alpha_max(X_scaled, y): # This computes the largest alpha value worth trying.
    """
    Computes alpha_max for sklearn's Lasso objective:
        (1 / (2n)) * ||y - Xw||^2 + alpha * ||w||_1

    At alpha >= alpha_max, all coefficients are zero.
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0] # the number of training samples (rows) in your dataset.
    alpha_max = np.max(np.abs(X_scaled.T @ y_centered)) / n  # X_scaled.T @ y_centered (@ means matrix muliplication) measures how strongly each feature is associated with the centered target. The feature with the strongest relationship to the target determines the largest penalty needed to force every coefficient to zero. This is smarter than choosing a random alpha grid. Lasso’s useful alpha range depends heavily on the scale of the features and the target. So instead of searching arbitrary values like 0.001 to 1000, this code builds a custom alpha range for each feature space and scaler. "How strongly does this feature move with the target?"
    # If a feature increases when y increases → big positive value
    # If a feature decreases when y increases → big negative value
    # If a feature is unrelated → value near 0
    return float(alpha_max)
    #the feature most strongly related to the target, That’s exactly what determines the largest alpha where Lasso wipes everything out.

def fit_lasso_path( # This function fits many Lasso models over a list of alpha values and records the results.
    X_train_space,
    X_val_space,
    y_train,
    y_val,
    alphas,
    scaler_name,
    scaler_factory,
    feature_space_name,
    stage
):
    """
    Fits Lasso models over a path of alphas using warm starts.
    Alphas should be sorted from largest to smallest.
    """
    rows = []
    # the scaler below is fit only on the training data, we learn the scaling parameters from the training split, then later apply the same transformation to the validation split.
    scaler = scaler_factory()
    X_train_scaled = scaler.fit_transform(X_train_space)
    X_val_scaled = scaler.transform(X_val_space)
    
    model = Lasso(
        alpha=alphas[0],
        fit_intercept=True, # means the model includes an intercept term. The intercept is not penalized by Lasso.
        max_iter=LASSO_MAX_ITER,
        tol=LASSO_TOL,
        warm_start=True, # warm_start=True is important. The alpha values are sorted from largest to smallest. The solution for a large alpha is a good starting point for the next slightly smaller alpha. Warm starts let the model reuse the previous fitted coefficients instead of starting from scratch each time. This can make the alpha path much faster.
        random_state=9890
    )
    
    for alpha in alphas: # the function loops through the alpha values
        model.alpha = alpha
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)              # For each alpha, it fits the model, predicts on the training set and validation set, and records the MSE.
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )   # A convergence warning means sklearn is saying, roughly, “I stopped before fully solving the optimization problem.” That does not always make the result useless, but it is a warning that you may need more iterations, stronger regularization, better scaling, or a different tolerance.
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))  # counts how many coefficients are nonzero:
        
        rows.append({
            "model_class": "Lasso",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_train, train_pred),  # tells you how well the model fits the training split.
            "val_mse": mean_squared_error(y_val, val_pred),  # is the main number used for model selection.
            "nonzero_coef": nonzero_coef,  #  tells you how sparse the model is.
            "n_iter": model.n_iter_,  # tells you how many iterations the solver used.
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

start_time = time.perf_counter()

lasso_broad_rows = []
alpha_max_records = []
# This loops over every combination of feature space and scaler.There are four feature spaces and two scalers, so there are eight combinations:
for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        
        alpha_max = compute_lasso_alpha_max(X_train_scaled, y_tr)  # For each combination, it computes that setup’s alpha_max
        alpha_min = alpha_max * LASSO_MIN_ALPHA_RATIO
        
        # Descending alpha path for warm starts
        broad_alphas = np.logspace(
            np.log10(alpha_max),
            np.log10(alpha_min),
            LASSO_BROAD_GRID_SIZE
        ) # Because the first argument is larger than the second, this creates descending values. That is intentional. The model starts with the strongest penalty and moves toward weaker penalties. This works well with warm_start=True.
        
        alpha_max_records.append({ # The code records the alpha range
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "alpha_max": alpha_max,
            "alpha_min": alpha_min
        })
        
        result = fit_lasso_path( # Then it fits all 25 broad-search Lasso models for that setup:
            X_train_space=X_train_space,
            X_val_space=X_val_space,
            y_train=y_tr,
            y_val=y_val,
            alphas=broad_alphas,
            scaler_name=scaler_name,
            scaler_factory=scaler_factory,
            feature_space_name=feature_space_name,
            stage="broad"
        )
        
        lasso_broad_rows.append(result)
# After the loops finish, it combines all broad-search results:
lasso_alpha_max_table = pd.DataFrame(alpha_max_records)
lasso_broad_results = pd.concat(lasso_broad_rows, ignore_index=True) # Since there are eight setup combinations and 25 alphas each, the broad search fits 8 × 25 = 200 Lasso models

best_broad_by_space_scaler = ( # picks the best broad alpha for each feature-space/scaler pair:
    lasso_broad_results
    .loc[lasso_broad_results.groupby(["feature_space", "scaler"])["val_mse"].idxmin()]
    .sort_values("val_mse")
)   # This groups the results by feature_space and scaler, finds the row with the smallest validation MSE in each group, and sorts those eight winners from best to worst.

print("Lasso alpha_max table:")
print(lasso_alpha_max_table.to_string(index=False))

print("\nBest broad Lasso result by feature space and scaler:")
print(best_broad_by_space_scaler.to_string(index=False))


# ------------------------------------------------------------
# Stage 2: fine search around each broad-search winner
# ------------------------------------------------------------

lasso_fine_rows = []

for _, row in best_broad_by_space_scaler.iterrows():   # The code loops over the eight broad-search winners
    feature_space_name = row["feature_space"]
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]   # For each winner, it gets the best alpha from the broad search:
    
    alpha_record = lasso_alpha_max_table[ # Then it retrieves the original alpha bounds for that feature-space/scaler combination:
        (lasso_alpha_max_table["feature_space"] == feature_space_name) &
        (lasso_alpha_max_table["scaler"] == scaler_name)
    ].iloc[0]
    
    alpha_max = alpha_record["alpha_max"]
    alpha_min = alpha_record["alpha_min"]
    # Then it builds a narrower alpha range around the broad-search winner:
    best_log = np.log10(best_alpha)
    fine_low = max(np.log10(alpha_min), best_log - LASSO_FINE_WIDTH_LOG10)
    fine_high = min(np.log10(alpha_max), best_log + LASSO_FINE_WIDTH_LOG10) # Because LASSO_FINE_WIDTH_LOG10 = 0.5, this searches approximately 3.16 times below and 3.16 times above the broad-search winner, clipped so it never goes outside the original broad-search range.
    
    # Descending alpha path for warm starts
    fine_alphas = np.logspace(fine_high, fine_low, LASSO_FINE_GRID_SIZE)  # this creates descending alpha values for warm starts. The fine search fits 21 more models for each of the eight broad winners: 8 × 21 = 168 Lasso models. so the whole cell fits 200 broad models + 168 fine models = 368 Lasso models
    
    X_train_space, X_val_space = feature_spaces[feature_space_name]
    
    result = fit_lasso_path(
        X_train_space=X_train_space,
        X_val_space=X_val_space,
        y_train=y_tr,
        y_val=y_val,
        alphas=fine_alphas,
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        feature_space_name=feature_space_name,
        stage="fine"
    )
    
    lasso_fine_rows.append(result)

lasso_fine_results = pd.concat(lasso_fine_rows, ignore_index=True)

# ------------------------------------------------------------
# Combine broad + fine results and select winners
# ------------------------------------------------------------

lasso_results = pd.concat(
    [lasso_broad_results, lasso_fine_results],
    ignore_index=True
)

best_lasso_by_space = ( # Then it finds the best Lasso model for each feature space. This gives one winner for each of the four feature spaces, regardless of scaler, alpha, or search stage.
    lasso_results
    .loc[lasso_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_lasso_overall = lasso_results.loc[lasso_results["val_mse"].idxmin()]  # Then it finds the single best Lasso model overall

elapsed = time.perf_counter() - start_time

print("\nBest Lasso result by feature space:")
print(best_lasso_by_space.to_string(index=False))

print("\nOverall best Lasso result:")
print(best_lasso_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    lasso_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

Lasso alpha_max table:
              feature_space   scaler  alpha_max  alpha_min
                       base standard  11.255885   0.001126
                       base   robust   9.898994   0.000990
             base_plus_poly standard  11.255885   0.001126
             base_plus_poly   robust  23.219510   0.002322
     base_plus_interactions standard  11.255885   0.001126
     base_plus_interactions   robust   9.898994   0.000990
base_plus_poly_interactions standard  11.255885   0.001126
base_plus_poly_interactions   robust  23.219510   0.002322

Best broad Lasso result by feature space and scaler:
model_class stage               feature_space   scaler    alpha  train_mse    val_mse  nonzero_coef  n_iter  convergence_warning  aic  bic
      Lasso broad base_plus_poly_interactions standard 0.001126 301.032379 308.416448           133   30000                 True  NaN  NaN
      Lasso broad base_plus_poly_interactions   robust 0.002322 301.362320 308.808215           128    6321       

### Lasso Regression Results

Lasso regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared. Alpha was tuned using a data-driven two-stage search based on `alpha_max`.

Best Lasso model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- Alpha: 0.001126
- Train MSE: 301.03
- Validation MSE: 308.42
- Nonzero coefficients: 134

Interpretation:
- The best Lasso model used the combined polynomial + interaction feature space.
- StandardScaler outperformed RobustScaler.
- The best alpha was at the lowest value in the search grid, indicating that weak regularization performed best.
- Lasso retained 134 nonzero coefficients, so it did not produce a highly sparse model.
- This supports the earlier finding that predictive signal is distributed across many predictors rather than concentrated in a small subset.

Conclusion:
Lasso provides useful feature selection but does not currently outperform Ridge or backward stepwise selection. The results suggest that aggressive sparsity is not ideal for this dataset.

### Process above: 

Try Lasso on every engineered feature set, try both standard and robust scaling, choose a sensible alpha range based on the data, do a broad alpha search, refine around the best alpha, track validation MSE and sparsity, then report the best Lasso configuration to compare against Ridge and earlier linear models.

In [28]:
# ============================================================
# 11A. Elastic Net Regression: two-stage search
# ============================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# Design choices
# ------------------------------------------------------------
# - Feature spaces: all four controlled feature spaces
# - Scalers: StandardScaler and RobustScaler
# - l1_ratio grid: from mostly-Ridge to near-Lasso
# - alpha grid: data-driven alpha_max, then two-stage search
# - Validation: current development holdout split
# - CV: intentionally postponed until finalist screening
# - Checkpointing: saves results after every path

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ELASTICNET_RESULTS_PATH = RESULTS_DIR / "elasticnet_results_holdout.csv"
ELASTICNET_BEST_BY_SPACE_PATH = RESULTS_DIR / "elasticnet_best_by_space_holdout.csv"
ELASTICNET_BEST_OVERALL_PATH = RESULTS_DIR / "elasticnet_best_overall_holdout.csv"

OVERWRITE_ELASTICNET_RESULTS = True

if OVERWRITE_ELASTICNET_RESULTS:
    for path in [
        ELASTICNET_RESULTS_PATH,
        ELASTICNET_BEST_BY_SPACE_PATH,
        ELASTICNET_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

elasticnet_l1_ratios = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

ELASTICNET_BROAD_GRID_SIZE = 25
ELASTICNET_FINE_GRID_SIZE = 21
ELASTICNET_MIN_ALPHA_RATIO = 1e-4
ELASTICNET_FINE_WIDTH_LOG10 = 0.5

ELASTICNET_MAX_ITER = 50000
ELASTICNET_TOL = 1e-4


def append_checkpoint(df, path):
    """Append results to a CSV checkpoint file."""
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def compute_alpha_max_lasso_base(X_scaled, y):
    """
    Computes max_j |x_j^T (y - ybar)| / n.

    For Elastic Net, alpha_max depends on l1_ratio:
        alpha_max_enet = alpha_max_lasso_base / l1_ratio

    This matches sklearn's ElasticNet objective:
        (1 / (2n)) * ||y - Xw||^2
        + alpha * l1_ratio * ||w||_1
        + 0.5 * alpha * (1 - l1_ratio) * ||w||_2^2
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0]
    alpha_max_base = np.max(np.abs(X_scaled.T @ y_centered)) / n
    
    if alpha_max_base <= 0:
        raise ValueError("alpha_max_base is non-positive; check X/y inputs.")
    
    return float(alpha_max_base)


def fit_elasticnet_path_scaled(
    X_train_scaled,
    X_val_scaled,
    y_train,
    y_val,
    alphas,
    l1_ratio,
    feature_space_name,
    scaler_name,
    stage
):
    """
    Fits Elastic Net along a descending alpha path using warm starts.
    """
    rows = []
    
    alphas = np.asarray(alphas, dtype=float)
    
    model = ElasticNet(
        alpha=alphas[0],
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=ELASTICNET_MAX_ITER,
        tol=ELASTICNET_TOL,
        warm_start=True,
        selection="cyclic",
        random_state=9890
    )
    
    for alpha in alphas:
        model.alpha = float(alpha)
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_train)
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )
        
        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)
        
        rows.append({
            "model_class": "ElasticNet",
            "stage": stage,
            "feature_space": feature_space_name,
            "scaler": scaler_name,
            "l1_ratio": l1_ratio,
            "alpha": float(alpha),
            "train_mse": mean_squared_error(y_train, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "nonzero_coef": int(np.sum(np.abs(model.coef_) > 1e-8)),
            "n_iter": int(model.n_iter_),
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        })
    
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Main search
# ------------------------------------------------------------

start_time = time.perf_counter()
elasticnet_all_parts = []

print("Elastic Net overnight search starting...")
print("Feature spaces:", list(feature_spaces.keys()))
print("l1_ratios:", elasticnet_l1_ratios)

for feature_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for scaler_name, scaler_factory in scaler_factories.items():
        
        print("\n" + "=" * 90)
        print(f"Feature space: {feature_space_name} | Scaler: {scaler_name}")
        print("=" * 90)
        
        scaler = scaler_factory()
        X_train_scaled = scaler.fit_transform(X_train_space)
        X_val_scaled = scaler.transform(X_val_space)
        
        alpha_max_base = compute_alpha_max_lasso_base(X_train_scaled, y_tr)
        
        broad_parts_this_combo = []
        
        # ----------------------------------------------------
        # Stage 1: broad search for each l1_ratio
        # ----------------------------------------------------
        for l1_ratio in elasticnet_l1_ratios:
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            broad_alphas = np.logspace(
                np.log10(alpha_max),
                np.log10(alpha_min),
                ELASTICNET_BROAD_GRID_SIZE
            )
            
            broad_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=broad_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="broad"
            )
            
            broad_parts_this_combo.append(broad_df)
            elasticnet_all_parts.append(broad_df)
            append_checkpoint(broad_df, ELASTICNET_RESULTS_PATH)
        
        broad_combo_df = pd.concat(broad_parts_this_combo, ignore_index=True)
        
        best_broad_by_l1 = (
            broad_combo_df
            .loc[broad_combo_df.groupby("l1_ratio")["val_mse"].idxmin()]
            .sort_values("val_mse")
        )
        
        print("\nBest broad result by l1_ratio:")
        print(
            best_broad_by_l1[
                ["l1_ratio", "alpha", "train_mse", "val_mse", "nonzero_coef", "convergence_warning"]
            ].to_string(index=False)
        )
        
        # ----------------------------------------------------
        # Stage 2: fine search around each broad winner
        # ----------------------------------------------------
        for _, row in best_broad_by_l1.iterrows():
            l1_ratio = float(row["l1_ratio"])
            best_alpha = float(row["alpha"])
            
            alpha_max = alpha_max_base / l1_ratio
            alpha_min = alpha_max * ELASTICNET_MIN_ALPHA_RATIO
            
            best_log = np.log10(best_alpha)
            fine_low = max(np.log10(alpha_min), best_log - ELASTICNET_FINE_WIDTH_LOG10)
            fine_high = min(np.log10(alpha_max), best_log + ELASTICNET_FINE_WIDTH_LOG10)
            
            fine_alphas = np.logspace(
                fine_high,
                fine_low,
                ELASTICNET_FINE_GRID_SIZE
            )
            
            fine_df = fit_elasticnet_path_scaled(
                X_train_scaled=X_train_scaled,
                X_val_scaled=X_val_scaled,
                y_train=y_tr,
                y_val=y_val,
                alphas=fine_alphas,
                l1_ratio=l1_ratio,
                feature_space_name=feature_space_name,
                scaler_name=scaler_name,
                stage="fine"
            )
            
            elasticnet_all_parts.append(fine_df)
            append_checkpoint(fine_df, ELASTICNET_RESULTS_PATH)
        
        elapsed_so_far = time.perf_counter() - start_time
        print(f"\nFinished {feature_space_name} | {scaler_name}. Elapsed seconds: {elapsed_so_far:.2f}")


# ------------------------------------------------------------
# Summarize final Elastic Net results
# ------------------------------------------------------------

elasticnet_results = pd.concat(elasticnet_all_parts, ignore_index=True)

best_elasticnet_by_space = (
    elasticnet_results
    .loc[elasticnet_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_elasticnet_overall = elasticnet_results.loc[
    elasticnet_results["val_mse"].idxmin()
]

best_elasticnet_by_space.to_csv(ELASTICNET_BEST_BY_SPACE_PATH, index=False)
best_elasticnet_overall.to_frame().T.to_csv(ELASTICNET_BEST_OVERALL_PATH, index=False)

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 90)
print("Best Elastic Net result by feature space:")
print("=" * 90)
print(best_elasticnet_by_space.to_string(index=False))

print("\n" + "=" * 90)
print("Overall best Elastic Net result:")
print("=" * 90)
print(best_elasticnet_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

print("\nConvergence warning counts:")
print(
    elasticnet_results
    .groupby(["feature_space", "scaler", "stage"])["convergence_warning"]
    .sum()
    .reset_index()
    .to_string(index=False)
)

print("\nSaved files:")
print("All Elastic Net results:", ELASTICNET_RESULTS_PATH)
print("Best by feature space:", ELASTICNET_BEST_BY_SPACE_PATH)
print("Best overall:", ELASTICNET_BEST_OVERALL_PATH)

Elastic Net overnight search starting...
Feature spaces: ['base', 'base_plus_poly', 'base_plus_interactions', 'base_plus_poly_interactions']
l1_ratios: [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

Feature space: base | Scaler: standard

Best broad result by l1_ratio:
 l1_ratio    alpha  train_mse    val_mse  nonzero_coef  convergence_warning
     0.99 0.001137 305.154413 312.619191           132                False
     0.95 0.001185 305.158777 312.623589           135                False
     0.90 0.001251 305.164429 312.629943           145                False
     0.75 0.001501 305.192217 312.655747           154                False
     0.50 0.002251 305.267042 312.737288           156                False
     0.25 0.004502 305.469335 312.955788           155                False
     0.10 0.011256 305.948373 313.426435           156                False
     0.05 0.022512 306.670214 314.083149           162                False

Finished base | standard. Elapsed seconds: 13

### Elastic Net Regression Results

Elastic Net regression was evaluated across the four controlled feature spaces:

1. Base feature space
2. Base + polynomial terms
3. Base + interaction terms
4. Base + polynomial + interaction terms

Both `StandardScaler` and `RobustScaler` were compared, and the model was tuned over multiple `l1_ratio` values and data-driven alpha values.

Best Elastic Net model:
- Feature space: base + polynomial + interactions
- Scaler: StandardScaler
- l1_ratio: 0.99
- Alpha: 0.001137
- Train MSE: 301.05
- Validation MSE: 308.43
- Nonzero coefficients: 152

Interpretation:
- Elastic Net performed best on the combined polynomial + interaction feature space.
- The best `l1_ratio` was 0.99, meaning the model behaved very similarly to Lasso.
- StandardScaler outperformed RobustScaler.
- Elastic Net retained 152 nonzero coefficients, so it did not produce a highly sparse model.
- The result is competitive but does not outperform Ridge or the best backward stepwise model.

Conclusion:
Elastic Net confirms that the strongest feature space is the combined polynomial + interaction space, but it does not improve over the current best model. The current best linear-family model remains backward stepwise on the combined feature space, with validation MSE around 308.10.

In [25]:
# ============================================================
# 12A. Step Function Models
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# Design:
# - Select top continuous predictors from simple LR ranking
# - Create step-function dummy variables using training-split cutpoints only
# - Add step features to each existing controlled feature space
# - Evaluate using current development holdout split
# ------------------------------------------------------------

start_time = time.perf_counter()

# Select top continuous features only.
# Exclude binary dummies / missing indicators by requiring many unique values.
step_candidate_features = []

for col in simple_lr_results["feature"].tolist():
    if col in X_tr.columns:
        unique_count = X_tr[col].nunique(dropna=False)
        if unique_count >= 20 and not col.endswith("_missing"):
            step_candidate_features.append(col)
    
    if len(step_candidate_features) >= 10:
        break

print("Step-function candidate features:")
for i, col in enumerate(step_candidate_features, start=1):
    print(f"{i:2d}. {col}")


def make_step_features(X_train_source, X_val_source, columns, n_bins, method):
    """
    Build step-function dummy features.

    Cutpoints are learned from X_train_source only.
    Then the same cutpoints are applied to X_val_source.

    method:
        'quantile'    -> bins based on training quantiles
        'equal_width' -> bins based on training min/max width
    """
    train_parts = []
    val_parts = []
    actual_step_cols = []

    for col in columns:
        x_train = X_train_source[col].astype(float)
        x_val = X_val_source[col].astype(float)

        if x_train.nunique(dropna=False) < 2:
            continue

        if method == "quantile":
            try:
                _, edges = pd.qcut(
                    x_train,
                    q=n_bins,
                    retbins=True,
                    duplicates="drop"
                )
            except ValueError:
                continue

        elif method == "equal_width":
            min_val = x_train.min()
            max_val = x_train.max()

            if not np.isfinite(min_val) or not np.isfinite(max_val) or min_val == max_val:
                continue

            edges = np.linspace(min_val, max_val, n_bins + 1)

        else:
            raise ValueError("method must be 'quantile' or 'equal_width'")

        edges = np.unique(edges)

        if len(edges) <= 2:
            continue

        # Make validation robust to values slightly outside training range.
        edges = edges.astype(float)
        edges[0] = -np.inf
        edges[-1] = np.inf

        k = len(edges) - 1

        train_codes = pd.cut(
            x_train,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        val_codes = pd.cut(
            x_val,
            bins=edges,
            labels=False,
            include_lowest=True
        )

        train_cat = pd.Categorical(train_codes, categories=list(range(k)))
        val_cat = pd.Categorical(val_codes, categories=list(range(k)))

        prefix = f"{col}_step_{method}_{n_bins}"

        train_dummies = pd.get_dummies(
            train_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )
        val_dummies = pd.get_dummies(
            val_cat,
            prefix=prefix,
            drop_first=True,
            dtype=float
        )

        train_dummies.index = X_train_source.index
        val_dummies.index = X_val_source.index

        train_dummies, val_dummies = train_dummies.align(
            val_dummies,
            join="left",
            axis=1,
            fill_value=0
        )

        train_parts.append(train_dummies)
        val_parts.append(val_dummies)
        actual_step_cols.extend(train_dummies.columns.tolist())

    if len(train_parts) == 0:
        empty_train = pd.DataFrame(index=X_train_source.index)
        empty_val = pd.DataFrame(index=X_val_source.index)
        return empty_train, empty_val, []

    X_train_steps = pd.concat(train_parts, axis=1)
    X_val_steps = pd.concat(val_parts, axis=1)

    return X_train_steps, X_val_steps, actual_step_cols


step_methods = ["quantile", "equal_width"]
step_bins_grid = [3, 5, 10]

step_results_rows = []

for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
    for method in step_methods:
        for n_bins in step_bins_grid:

            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=step_candidate_features,
                n_bins=n_bins,
                method=method
            )

            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)

            model = LinearRegression()
            model.fit(X_train_aug, y_tr)

            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)

            step_results_rows.append({
                "model_class": "Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": method,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

step_results = pd.DataFrame(step_results_rows)

best_step_by_base_space = (
    step_results
    .loc[step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_step_overall = step_results.loc[step_results["val_mse"].idxmin()]

elapsed = time.perf_counter() - start_time

print("\nBest step-function model by base space:")
print(best_step_by_base_space.to_string(index=False))

print("\nOverall best step-function model:")
print(best_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

Step-function candidate features:
 1. PERCENT_ECONOMICALLY_DISADVANTAGED
 2. PERCENT_FREE_LUNCH
 3. PERCENT_DIPLOMA
 4. PERCENT_STILL_ENROLLED
 5. PERCENT_HOMELESS
 6. PERCENT_BLACK
 7. PERCENT_WITH_DISABILITIES
 8. PERCENT_ENGLISH_LANGUAGE_LEANERS
 9. ATTENDANCE_RATE
10. PERCENT_WHITE

Best step-function model by base space:
                       model_class                  base_space step_method  n_bins  n_step_features_added  total_features  train_mse    val_mse  aic  bic
Step Functions + Linear Regression base_plus_poly_interactions    quantile      10                     83             265 296.544041 303.570375  NaN  NaN
Step Functions + Linear Regression              base_plus_poly    quantile      10                     83             255 297.288261 304.119794  NaN  NaN
Step Functions + Linear Regression      base_plus_interactions    quantile      10                     83             255 297.540321 304.612830  NaN  NaN
Step Functions + Linear Regression                      

### Step Function Model Results

Step-function models were evaluated by binning the strongest continuous predictors from the simple linear regression ranking.

Candidate variables included:
- `PERCENT_ECONOMICALLY_DISADVANTAGED`
- `PERCENT_FREE_LUNCH`
- `PERCENT_DIPLOMA`
- `PERCENT_STILL_ENROLLED`
- `PERCENT_HOMELESS`
- `PERCENT_BLACK`
- `PERCENT_WITH_DISABILITIES`
- `PERCENT_ENGLISH_LANGUAGE_LEANERS`
- `ATTENDANCE_RATE`
- `PERCENT_WHITE`

The best step-function model used:
- Base feature space: base + polynomial + interactions
- Binning method: quantile bins
- Number of bins: 10
- Step-function features added: 83
- Total features: 265

Results:
- Train MSE: 296.54
- Validation MSE: 303.57

Interpretation:
- Step functions substantially improved performance compared with previous linear-family models.
- The improvement suggests that some important predictors have threshold-based or piecewise relationships with the target.
- Quantile binning performed best, likely because it creates balanced bins across skewed continuous predictors.
- The train-validation gap remains moderate, so the improvement does not appear to be caused by severe overfitting.

Conclusion:
Step-function feature engineering is the strongest modeling direction so far. The current best model is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [26]:
# ============================================================
# 12B. Expanded Step Function Search
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# ------------------------------------------------------------
# Helper: get top continuous features from simple LR ranking
# ------------------------------------------------------------

def get_top_continuous_features(k, min_unique=20):
    features = []
    
    for col in simple_lr_results["feature"].tolist():
        if col not in X_tr.columns:
            continue
        
        unique_count = X_tr[col].nunique(dropna=False)
        
        if unique_count >= min_unique and not col.endswith("_missing"):
            features.append(col)
        
        if len(features) >= k:
            break
    
    return features


# ------------------------------------------------------------
# Expanded search design
# ------------------------------------------------------------

candidate_feature_counts = [10, 15, 20]
step_bins_grid = [5, 10, 15, 20]
step_method = "quantile"

expanded_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for base_space_name, (X_train_space, X_val_space) in feature_spaces.items():
        for n_bins in step_bins_grid:
            
            X_train_steps, X_val_steps, step_cols = make_step_features(
                X_train_source=X_tr,
                X_val_source=X_val,
                columns=candidate_features,
                n_bins=n_bins,
                method=step_method
            )
            
            X_train_aug = pd.concat([X_train_space, X_train_steps], axis=1)
            X_val_aug = pd.concat([X_val_space, X_val_steps], axis=1)
            
            model = LinearRegression()
            model.fit(X_train_aug, y_tr)
            
            train_pred = model.predict(X_train_aug)
            val_pred = model.predict(X_val_aug)
            
            expanded_step_rows.append({
                "model_class": "Expanded Step Functions + Linear Regression",
                "base_space": base_space_name,
                "step_method": step_method,
                "candidate_feature_count": k,
                "n_bins": n_bins,
                "n_step_features_added": len(step_cols),
                "total_features": X_train_aug.shape[1],
                "train_mse": mean_squared_error(y_tr, train_pred),
                "val_mse": mean_squared_error(y_val, val_pred),
                "aic": np.nan,
                "bic": np.nan
            })

expanded_step_results = pd.DataFrame(expanded_step_rows)

best_expanded_step_by_base_space = (
    expanded_step_results
    .loc[expanded_step_results.groupby("base_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_expanded_step_overall = expanded_step_results.loc[
    expanded_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nBest expanded step-function model by base space:")
print(best_expanded_step_by_base_space.to_string(index=False))

print("\nOverall best expanded step-function model:")
print(best_expanded_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 10 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE']

Top 15 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11']

Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN

### Expanded Step Function Search

The expanded step-function search tested quantile-based binning over larger sets of continuous predictors.

Best model:
- Base feature space: base + polynomial + interactions
- Candidate continuous predictors: 20
- Quantile bins: 20
- Step-function features added: 252
- Total features: 434

Results:
- Train MSE: 289.14
- Validation MSE: 296.14

Interpretation:
- Step functions substantially improved validation performance compared with all previous linear-family models.
- The improvement suggests that several predictors have threshold-based or piecewise relationships with `PERCENT_PROFICIENT`.
- The best model occurred at the largest tested feature count and bin count, indicating that the useful search space may not yet be exhausted.
- The train-validation gap remains moderate, so the model does not appear to be severely overfit at this stage.

Conclusion:
Step-function feature engineering is currently the strongest modeling direction. The best model so far is a linear regression model using the combined polynomial + interaction feature space plus quantile step functions.

In [27]:
# ============================================================
# 12C. Focused expanded step-function search on best base space
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# Current best base space
best_base_space_name = "base_plus_poly_interactions"
X_train_best_base, X_val_best_base = feature_spaces[best_base_space_name]

# Expand beyond previous boundary.
candidate_feature_counts = [20, 25, 30]
step_bins_grid = [20, 25, 30]
step_method = "quantile"

focused_step_rows = []

for k in candidate_feature_counts:
    candidate_features = get_top_continuous_features(k)
    
    print("\n" + "=" * 80)
    print(f"Top {k} continuous step-function candidates:")
    print(candidate_features)
    
    for n_bins in step_bins_grid:
        X_train_steps, X_val_steps, step_cols = make_step_features(
            X_train_source=X_tr,
            X_val_source=X_val,
            columns=candidate_features,
            n_bins=n_bins,
            method=step_method
        )
        
        X_train_aug = pd.concat([X_train_best_base, X_train_steps], axis=1)
        X_val_aug = pd.concat([X_val_best_base, X_val_steps], axis=1)
        
        model = LinearRegression()
        model.fit(X_train_aug, y_tr)
        
        train_pred = model.predict(X_train_aug)
        val_pred = model.predict(X_val_aug)
        
        focused_step_rows.append({
            "model_class": "Focused Expanded Step Functions + Linear Regression",
            "base_space": best_base_space_name,
            "step_method": step_method,
            "candidate_feature_count": k,
            "n_bins": n_bins,
            "n_step_features_added": len(step_cols),
            "total_features": X_train_aug.shape[1],
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

focused_step_results = pd.DataFrame(focused_step_rows).sort_values("val_mse")

best_focused_step_overall = focused_step_results.loc[
    focused_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nFocused expanded step-function results:")
print(focused_step_results.to_string(index=False))

print("\nOverall best focused expanded step-function model:")
print(best_focused_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")


Top 20 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS']

Top 25 continuous step-function candidates:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_WITH_DISABILITIES', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'ATTENDANCE_RATE', 'PERCENT_WHITE', 'PERCENT_DROPOUT', 'PERCENT_HISPANIC', 'PERCENT_ASIAN', 'GRADE_12', 'GRADE_11', 'PRE_K', 'GRADE_10', 'NUMBER_OF_TEACHERS', 'GRADE_09', 'NUMBER_OF_COUNSELORS', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'GRADE_04', 'GRADE_03', 'SCHOOL_freq']

Top 30 continuous step-

### Focused Expanded Step Function Search

A focused step-function search was run on the strongest base feature space: `base_plus_poly_interactions`.

The search expanded the number of continuous variables used for quantile binning and increased the number of bins.

Best model:
- Base feature space: base + polynomial + interactions
- Candidate continuous predictors: 30
- Quantile bins: 30
- Step-function features added: 526
- Total features: 708

Results:
- Train MSE: 281.63
- Validation MSE: 290.11

Interpretation:
- This is the strongest model so far.
- The result substantially improves over the previous best linear-family model, which had validation MSE around 308.10.
- The improvement suggests that the target has strong piecewise or threshold-based relationships with predictors.
- The best model occurred at the largest tested feature count and bin count, suggesting that the step-function search space may not yet be exhausted.
- The train-validation gap increased but remains moderate, so there is no clear evidence of severe overfitting yet.

Conclusion:
Step-function feature engineering is currently the most successful modeling strategy. The next step should regularize this larger step-function feature space, because the model now contains many correlated bin indicators.

In [28]:
# ============================================================
# 12D. Ridge on best step-function feature space
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time

start_time = time.perf_counter()

# ------------------------------------------------------------
# Rebuild current best step-function feature space
# ------------------------------------------------------------

best_base_space_name = "base_plus_poly_interactions"
best_step_candidate_count = 30
best_step_bins = 30
best_step_method = "quantile"

X_train_best_base, X_val_best_base = feature_spaces[best_base_space_name]
best_step_candidates = get_top_continuous_features(best_step_candidate_count)

X_train_steps_best, X_val_steps_best, best_step_cols = make_step_features(
    X_train_source=X_tr,
    X_val_source=X_val,
    columns=best_step_candidates,
    n_bins=best_step_bins,
    method=best_step_method
)

X_tr_step_best = pd.concat([X_train_best_base, X_train_steps_best], axis=1)
X_val_step_best = pd.concat([X_val_best_base, X_val_steps_best], axis=1)

print("Best step-function design:")
print("Base space:", best_base_space_name)
print("Candidate feature count:", best_step_candidate_count)
print("Bins:", best_step_bins)
print("Step features added:", len(best_step_cols))
print("Total features:", X_tr_step_best.shape[1])

# ------------------------------------------------------------
# Sanity check: unregularized OLS on rebuilt step space
# ------------------------------------------------------------

ols_step_model = LinearRegression()
ols_step_model.fit(X_tr_step_best, y_tr)

ols_train_pred = ols_step_model.predict(X_tr_step_best)
ols_val_pred = ols_step_model.predict(X_val_step_best)

ols_step_train_mse = mean_squared_error(y_tr, ols_train_pred)
ols_step_val_mse = mean_squared_error(y_val, ols_val_pred)

print("\nOLS step-function sanity check:")
print("Train MSE:", ols_step_train_mse)
print("Validation MSE:", ols_step_val_mse)

# ------------------------------------------------------------
# Ridge tuning on this step-function feature space
# ------------------------------------------------------------

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

# Broad search across many orders of magnitude.
# We include very small alpha to approximate OLS and large alpha to test heavy shrinkage.
ridge_step_broad_alphas = np.logspace(-4, 8, 49)

ridge_step_rows = []

for scaler_name, scaler_factory in scaler_factories.items():
    for alpha in ridge_step_broad_alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_tr_step_best, y_tr)
        
        train_pred = model.predict(X_tr_step_best)
        val_pred = model.predict(X_val_step_best)
        
        ridge_step_rows.append({
            "model_class": "Ridge + Step Functions",
            "stage": "broad",
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

ridge_step_broad_results = pd.DataFrame(ridge_step_rows)

best_broad_by_scaler = (
    ridge_step_broad_results
    .loc[ridge_step_broad_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("\nBest broad Ridge-step result by scaler:")
print(best_broad_by_scaler.to_string(index=False))

# ------------------------------------------------------------
# Fine search around each broad winner
# ------------------------------------------------------------

ridge_step_fine_rows = []

for _, row in best_broad_by_scaler.iterrows():
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]
    best_alpha = row["alpha"]
    
    log_alpha = np.log10(best_alpha)
    fine_low = max(np.log10(ridge_step_broad_alphas.min()), log_alpha - 0.75)
    fine_high = min(np.log10(ridge_step_broad_alphas.max()), log_alpha + 0.75)
    
    ridge_step_fine_alphas = np.logspace(fine_low, fine_high, 41)
    
    for alpha in ridge_step_fine_alphas:
        model = make_pipeline(
            scaler_factory(),
            Ridge(alpha=alpha)
        )
        
        model.fit(X_tr_step_best, y_tr)
        
        train_pred = model.predict(X_tr_step_best)
        val_pred = model.predict(X_val_step_best)
        
        ridge_step_fine_rows.append({
            "model_class": "Ridge + Step Functions",
            "stage": "fine",
            "scaler": scaler_name,
            "alpha": alpha,
            "train_mse": mean_squared_error(y_tr, train_pred),
            "val_mse": mean_squared_error(y_val, val_pred),
            "aic": np.nan,
            "bic": np.nan
        })

ridge_step_fine_results = pd.DataFrame(ridge_step_fine_rows)

ridge_step_results = pd.concat(
    [ridge_step_broad_results, ridge_step_fine_results],
    ignore_index=True
)

best_ridge_step_by_scaler = (
    ridge_step_results
    .loc[ridge_step_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_ridge_step_overall = ridge_step_results.loc[
    ridge_step_results["val_mse"].idxmin()
]

elapsed = time.perf_counter() - start_time

print("\nBest Ridge-step result by scaler:")
print(best_ridge_step_by_scaler.to_string(index=False))

print("\nOverall best Ridge-step result:")
print(best_ridge_step_overall)

print(f"\nElapsed time: {elapsed:.2f} seconds")

if best_ridge_step_overall["alpha"] == ridge_step_broad_alphas.min():
    print("\nWARNING: Best alpha is at the lower broad-grid boundary. Ridge may prefer nearly no shrinkage.")
elif best_ridge_step_overall["alpha"] == ridge_step_broad_alphas.max():
    print("\nWARNING: Best alpha is at the upper broad-grid boundary. Consider expanding alpha upward.")

Best step-function design:
Base space: base_plus_poly_interactions
Candidate feature count: 30
Bins: 30
Step features added: 526
Total features: 708

OLS step-function sanity check:
Train MSE: 281.63428911174634
Validation MSE: 290.1093738555459

Best broad Ridge-step result by scaler:
           model_class stage   scaler    alpha  train_mse    val_mse  aic  bic
Ridge + Step Functions broad   robust 0.562341 281.672006 290.079990  NaN  NaN
Ridge + Step Functions broad standard 1.778279 281.656987 290.084438  NaN  NaN

Best Ridge-step result by scaler:
           model_class stage   scaler    alpha  train_mse    val_mse  aic  bic
Ridge + Step Functions  fine   robust 0.473151 281.664053 290.079273  NaN  NaN
Ridge + Step Functions  fine standard 2.304093 281.665268 290.083841  NaN  NaN

Overall best Ridge-step result:
model_class    Ridge + Step Functions
stage                            fine
scaler                         robust
alpha                        0.473151
train_mse          

### Interpretation: Step Functions + Ridge Regression

We constructed a high-dimensional feature space using step functions applied to a base set of predictors (including polynomial and interaction terms). This resulted in a total of 708 features, significantly expanding the model’s flexibility through piecewise constant approximations.

The unregularized OLS model achieved:
- Train MSE: 281.63  
- Validation MSE: 290.11  

Applying Ridge regularization led to a very small improvement:
- Best Validation MSE: 290.08 (robust scaler, α ≈ 0.47)

#### Key Takeaways

- **Minimal gain from Ridge:** The negligible improvement in validation MSE suggests that the model is not suffering from substantial overfitting, despite the large feature space.
- **Train vs Validation gap is small:** This indicates low variance and good generalization stability.
- **Bias likely dominates:** Since increasing model complexity (via step functions) and adding regularization does not materially improve performance, the model may still be underfitting the underlying structure.

#### Implication

Step functions increase dimensionality but impose rigid, piecewise constant structure. While they capture some nonlinearity, they may not be flexible enough to model smooth or complex relationships in the data. Further improvements may require models that learn structure more adaptively rather than relying solely on engineered basis expansions.

In [35]:
# ============================================================
# 12E. Lasso on best step-function feature space
# ============================================================

from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

start_time = time.perf_counter()

# ------------------------------------------------------------
# Check that the previous step-function design exists
# ------------------------------------------------------------

print("Lasso on best step-function feature space")
print("Training shape:", X_tr_step_best.shape)
print("Validation shape:", X_val_step_best.shape)
print("Reference OLS step validation MSE:", ols_step_val_mse)
print("Reference Ridge-step validation MSE:", best_ridge_step_overall["val_mse"])

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

LASSO_STEP_RESULTS_PATH = RESULTS_DIR / "lasso_step_results_holdout.csv"
LASSO_STEP_BEST_BY_SCALER_PATH = RESULTS_DIR / "lasso_step_best_by_scaler_holdout.csv"
LASSO_STEP_BEST_OVERALL_PATH = RESULTS_DIR / "lasso_step_best_overall_holdout.csv"

OVERWRITE_LASSO_STEP_RESULTS = True

if OVERWRITE_LASSO_STEP_RESULTS:
    for path in [
        LASSO_STEP_RESULTS_PATH,
        LASSO_STEP_BEST_BY_SCALER_PATH,
        LASSO_STEP_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Lasso helper functions
# ------------------------------------------------------------

def compute_lasso_alpha_max(X_scaled, y):
    """
    Computes alpha_max for sklearn's Lasso objective.

    At alpha >= alpha_max, all coefficients are zero.
    """
    y_arr = np.asarray(y, dtype=float)
    y_centered = y_arr - y_arr.mean()
    n = X_scaled.shape[0]
    return float(np.max(np.abs(X_scaled.T @ y_centered)) / n)

def fit_lasso_path_for_scaler(
    scaler_name,
    scaler_factory,
    alpha_ratios,
    stage,
    max_iter=50000,
    tol=1e-4
):
    """
    Fits Lasso over a sequence of alpha ratios for one scaler.
    Uses warm starts from larger alpha to smaller alpha.
    """
    print(f"\n--- {stage.upper()} search | scaler = {scaler_name} ---")
    
    scaler = scaler_factory()
    X_train_scaled = scaler.fit_transform(X_tr_step_best)
    X_val_scaled = scaler.transform(X_val_step_best)

    alpha_max = compute_lasso_alpha_max(X_train_scaled, y_tr)
    alphas = alpha_max * np.asarray(alpha_ratios)

    # Warm start works best when moving from stronger penalty to weaker penalty.
    order = np.argsort(alphas)[::-1]
    alphas = alphas[order]
    alpha_ratios_ordered = np.asarray(alpha_ratios)[order]

    rows = []

    model = Lasso(
        alpha=alphas[0],
        fit_intercept=True,
        max_iter=max_iter,
        tol=tol,
        warm_start=True,
        selection="cyclic",
        random_state=RANDOM_STATE
    )

    for alpha_ratio, alpha in zip(alpha_ratios_ordered, alphas):
        model.alpha = float(alpha)

        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_train_scaled, y_tr)

            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )

        train_pred = model.predict(X_train_scaled)
        val_pred = model.predict(X_val_scaled)

        train_mse = mean_squared_error(y_tr, train_pred)
        val_mse = mean_squared_error(y_val, val_pred)
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))

        row = {
            "model_class": "Lasso + Step Functions",
            "stage": stage,
            "scaler": scaler_name,
            "alpha_max": alpha_max,
            "alpha_ratio": alpha_ratio,
            "alpha": alpha,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "nonzero_coef": nonzero_coef,
            "n_iter": model.n_iter_,
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        }

        rows.append(row)
        append_checkpoint(pd.DataFrame([row]), LASSO_STEP_RESULTS_PATH)

        print(
            f"{stage:5s} | {scaler_name:8s} | "
            f"ratio={alpha_ratio:.2e} | alpha={alpha:.8f} | "
            f"train_mse={train_mse:.6f} | val_mse={val_mse:.6f} | "
            f"nonzero={nonzero_coef:4d} | n_iter={model.n_iter_:5d} | "
            f"warning={convergence_warning}"
        )

    results = pd.DataFrame(rows)

    del X_train_scaled, X_val_scaled, model
    gc.collect()

    return results

# ------------------------------------------------------------
# Stage 1: broad search
# ------------------------------------------------------------

scaler_factories = {
    "standard": StandardScaler,
    "robust": RobustScaler
}

# Ratios are relative to alpha_max.
# This searches from fairly strong Lasso shrinkage down to very weak shrinkage.
lasso_step_broad_ratios = np.logspace(-1, -6, 26)

lasso_step_broad_results_list = []

for scaler_name, scaler_factory in scaler_factories.items():
    results = fit_lasso_path_for_scaler(
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        alpha_ratios=lasso_step_broad_ratios,
        stage="broad",
        max_iter=50000,
        tol=1e-4
    )
    lasso_step_broad_results_list.append(results)

lasso_step_broad_results = pd.concat(lasso_step_broad_results_list, ignore_index=True)

best_lasso_step_broad_by_scaler = (
    lasso_step_broad_results
    .loc[lasso_step_broad_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

print("\nBest broad Lasso-step result by scaler:")
print(best_lasso_step_broad_by_scaler.to_string(index=False))

# ------------------------------------------------------------
# Stage 2: fine search around each scaler's broad winner
# ------------------------------------------------------------

lasso_step_fine_results_list = []

for _, row in best_lasso_step_broad_by_scaler.iterrows():
    scaler_name = row["scaler"]
    scaler_factory = scaler_factories[scaler_name]

    best_ratio = row["alpha_ratio"]

    fine_low = max(1e-7, best_ratio / 3.0)
    fine_high = min(1e-1, best_ratio * 3.0)

    lasso_step_fine_ratios = np.logspace(
        np.log10(fine_high),
        np.log10(fine_low),
        25
    )

    results = fit_lasso_path_for_scaler(
        scaler_name=scaler_name,
        scaler_factory=scaler_factory,
        alpha_ratios=lasso_step_fine_ratios,
        stage="fine",
        max_iter=70000,
        tol=1e-4
    )
    lasso_step_fine_results_list.append(results)

lasso_step_fine_results = pd.concat(lasso_step_fine_results_list, ignore_index=True)

# ------------------------------------------------------------
# Combine and summarize
# ------------------------------------------------------------

lasso_step_results = pd.concat(
    [lasso_step_broad_results, lasso_step_fine_results],
    ignore_index=True
)

best_lasso_step_by_scaler = (
    lasso_step_results
    .loc[lasso_step_results.groupby("scaler")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_lasso_step_overall = lasso_step_results.loc[
    lasso_step_results["val_mse"].idxmin()
]

best_lasso_step_by_scaler.to_csv(LASSO_STEP_BEST_BY_SCALER_PATH, index=False)
pd.DataFrame([best_lasso_step_overall]).to_csv(LASSO_STEP_BEST_OVERALL_PATH, index=False)

print("\nBest Lasso-step result by scaler:")
print(best_lasso_step_by_scaler.to_string(index=False))

print("\nOverall best Lasso-step result:")
print(best_lasso_step_overall)

# ------------------------------------------------------------
# Refit best Lasso-step model on the development training split
# ------------------------------------------------------------

best_lasso_step_model = make_pipeline(
    scaler_factories[best_lasso_step_overall["scaler"]](),
    Lasso(
        alpha=float(best_lasso_step_overall["alpha"]),
        fit_intercept=True,
        max_iter=100000,
        tol=1e-4,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
)

best_lasso_step_model.fit(X_tr_step_best, y_tr)

best_lasso_step_train_pred = best_lasso_step_model.predict(X_tr_step_best)
best_lasso_step_val_pred = best_lasso_step_model.predict(X_val_step_best)

best_lasso_step_train_mse = mean_squared_error(y_tr, best_lasso_step_train_pred)
best_lasso_step_val_mse = mean_squared_error(y_val, best_lasso_step_val_pred)

best_lasso_step_nonzero = int(
    np.sum(np.abs(best_lasso_step_model.named_steps["lasso"].coef_) > 1e-8)
)

print("\nRefit check for best Lasso-step model:")
print("Train MSE:", best_lasso_step_train_mse)
print("Validation MSE:", best_lasso_step_val_mse)
print("Nonzero coefficients:", best_lasso_step_nonzero)

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Lasso + Step Functions"] = {
    "model": best_lasso_step_model,
    "feature_space": "base_plus_poly_interactions + step_functions",
    "X_train_name": "X_tr_step_best",
    "X_val_name": "X_val_step_best",
    "scaler": best_lasso_step_overall["scaler"],
    "alpha": float(best_lasso_step_overall["alpha"]),
    "alpha_ratio": float(best_lasso_step_overall["alpha_ratio"]),
    "train_mse": float(best_lasso_step_train_mse),
    "val_mse": float(best_lasso_step_val_mse),
    "nonzero_coef": best_lasso_step_nonzero,
    "notes": "Two-stage holdout search on the 708-feature step-function design."
}

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")


Lasso on best step-function feature space
Training shape: (115936, 708)
Validation shape: (28985, 708)
Reference OLS step validation MSE: 290.1093738555459
Reference Ridge-step validation MSE: 290.0792732215163

--- BROAD search | scaler = standard ---
broad | standard | ratio=1.00e-01 | alpha=1.12558852 | train_mse=357.524507 | val_mse=363.873625 | nonzero=  45 | n_iter=  130 | warning=False
broad | standard | ratio=6.31e-02 | alpha=0.71019834 | train_mse=331.857870 | val_mse=338.933256 | nonzero=  50 | n_iter=   25 | warning=False
broad | standard | ratio=3.98e-02 | alpha=0.44810486 | train_mse=317.691771 | val_mse=325.009682 | nonzero=  69 | n_iter=   45 | warning=False
broad | standard | ratio=2.51e-02 | alpha=0.28273505 | train_mse=307.916339 | val_mse=315.566760 | nonzero= 122 | n_iter=   44 | warning=False
broad | standard | ratio=1.58e-02 | alpha=0.17839376 | train_mse=299.891079 | val_mse=307.729047 | nonzero= 208 | n_iter=  101 | warning=False
broad | standard | ratio=1.00e-0

/Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.951e+04, tolerance: 8.086e+03
  model = cd_fast.enet_coordinate_descent(



Refit check for best Lasso-step model:
Train MSE: 281.68849705581994
Validation MSE: 290.05901785647603
Nonzero coefficients: 658

Elapsed time: 96131.03 seconds


### Interpretation: Lasso on Step-Function Feature Space

We fit Lasso regression on the same 708-feature step-function design used for the OLS and Ridge step-function models. This model used a two-stage alpha search across both standard and robust scaling.

The best Lasso-step model used standard scaling with a very small penalty:

- Best validation MSE: 290.059
- Train MSE: 281.688
- Nonzero coefficients: 658 out of 708
- Best alpha: approximately 0.000310

This slightly improves on the previous step-function models:

- OLS step-function validation MSE: 290.109
- Ridge step-function validation MSE: 290.079
- Lasso step-function validation MSE: 290.059

#### Key Takeaways

The Lasso model gives the best validation MSE so far within the step-function feature family, but the improvement is very small. Compared with Ridge, the improvement is only about 0.02 MSE.

Lasso also retained nearly all of the step-function features, keeping 658 out of 708 coefficients nonzero. Therefore, in this feature space, Lasso is not acting as a strong feature-selection method. Instead, it behaves more like a lightly regularized linear model on the full expanded design.

The very long runtime and convergence difficulty suggest that further Lasso tuning on this same step-function feature space is unlikely to be worth the computational cost. The model is useful to record as the best step-function shrinkage result, but it does not materially change the broader conclusion: engineered linear basis expansions are beginning to plateau.

#### Implication

The best current model in the step-function family is Lasso + Step Functions, but the gains over Ridge and OLS are marginal. Future improvement likely requires either a different kind of nonlinear structure, such as splines, GAMs, trees, random forests, or boosting, or a different feature representation rather than more tuning of this same step-function design.

In [ ]:
# un-run model space DO NOT RUN! yet

# ============================================================
# 12F. Elastic Net sanity check on best step-function feature space
# Deferred model: run later if time permits
# ============================================================

from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

start_time = time.perf_counter()

print("Elastic Net on best step-function feature space")
print("Training shape:", X_tr_step_best.shape)
print("Validation shape:", X_val_step_best.shape)

print("\nReference models:")
print("OLS step-function validation MSE:", ols_step_val_mse)
print("Ridge-step validation MSE:", best_ridge_step_overall["val_mse"])
print("Lasso-step validation MSE:", best_lasso_step_val_mse)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ELASTIC_STEP_RESULTS_PATH = RESULTS_DIR / "elastic_step_results_holdout.csv"
ELASTIC_STEP_BEST_OVERALL_PATH = RESULTS_DIR / "elastic_step_best_overall_holdout.csv"

OVERWRITE_ELASTIC_STEP_RESULTS = True

if OVERWRITE_ELASTIC_STEP_RESULTS:
    for path in [ELASTIC_STEP_RESULTS_PATH, ELASTIC_STEP_BEST_OVERALL_PATH]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Scale once using training split only
# ------------------------------------------------------------

scaler = StandardScaler()
X_tr_step_scaled = scaler.fit_transform(X_tr_step_best)
X_val_step_scaled = scaler.transform(X_val_step_best)

# ------------------------------------------------------------
# Elastic Net search grid
# ------------------------------------------------------------
# l1_ratio = 1.0 is Lasso
# l1_ratio close to 0.0 is Ridge-like, but sklearn ElasticNet should not use exactly 0.
# We focus around mixed penalties and high-L1 penalties.

l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9]

# Use the best Lasso-step alpha as the center.
# Search one order of magnitude around it.
center_alpha = float(best_lasso_step_overall["alpha"])

alpha_multipliers = np.array([
    0.25,
    0.5,
    0.75,
    1.0,
    1.5,
    2.0,
    3.0,
    5.0,
    8.0,
    12.0
])

elastic_alphas = center_alpha * alpha_multipliers

print("\nCenter alpha from best Lasso-step:", center_alpha)
print("Elastic Net alpha grid:")
print(elastic_alphas)

# ------------------------------------------------------------
# Fit Elastic Net models
# ------------------------------------------------------------

elastic_step_rows = []

for l1_ratio in l1_ratios:
    print(f"\n--- Elastic Net search | l1_ratio = {l1_ratio} ---")
    
    # Start from stronger penalty and move downward with warm starts
    ordered_alphas = np.sort(elastic_alphas)[::-1]
    
    model = ElasticNet(
        alpha=ordered_alphas[0],
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=70000,
        tol=1e-4,
        warm_start=True,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
    
    for alpha in ordered_alphas:
        model.alpha = float(alpha)
        model.l1_ratio = float(l1_ratio)
        
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(X_tr_step_scaled, y_tr)
            
            convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )
        
        train_pred = model.predict(X_tr_step_scaled)
        val_pred = model.predict(X_val_step_scaled)
        
        train_mse = mean_squared_error(y_tr, train_pred)
        val_mse = mean_squared_error(y_val, val_pred)
        nonzero_coef = int(np.sum(np.abs(model.coef_) > 1e-8))
        
        row = {
            "model_class": "Elastic Net + Step Functions",
            "scaler": "standard",
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "nonzero_coef": nonzero_coef,
            "n_iter": model.n_iter_,
            "convergence_warning": convergence_warning,
            "aic": np.nan,
            "bic": np.nan
        }
        
        elastic_step_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), ELASTIC_STEP_RESULTS_PATH)
        
        print(
            f"l1_ratio={l1_ratio:4.2f} | "
            f"alpha={alpha:.8f} | "
            f"train_mse={train_mse:.6f} | "
            f"val_mse={val_mse:.6f} | "
            f"nonzero={nonzero_coef:4d} | "
            f"n_iter={model.n_iter_:5d} | "
            f"warning={convergence_warning}"
        )

elastic_step_results = pd.DataFrame(elastic_step_rows).sort_values("val_mse")

best_elastic_step_overall = elastic_step_results.iloc[0]
pd.DataFrame([best_elastic_step_overall]).to_csv(
    ELASTIC_STEP_BEST_OVERALL_PATH,
    index=False
)

print("\nElastic Net-step results sorted by validation MSE:")
print(elastic_step_results.to_string(index=False))

print("\nOverall best Elastic Net-step result:")
print(best_elastic_step_overall)

# ------------------------------------------------------------
# Refit best Elastic Net model
# ------------------------------------------------------------

best_elastic_step_model = make_pipeline(
    StandardScaler(),
    ElasticNet(
        alpha=float(best_elastic_step_overall["alpha"]),
        l1_ratio=float(best_elastic_step_overall["l1_ratio"]),
        fit_intercept=True,
        max_iter=100000,
        tol=1e-4,
        selection="cyclic",
        random_state=RANDOM_STATE
    )
)

best_elastic_step_model.fit(X_tr_step_best, y_tr)

best_elastic_step_train_pred = best_elastic_step_model.predict(X_tr_step_best)
best_elastic_step_val_pred = best_elastic_step_model.predict(X_val_step_best)

best_elastic_step_train_mse = mean_squared_error(y_tr, best_elastic_step_train_pred)
best_elastic_step_val_mse = mean_squared_error(y_val, best_elastic_step_val_pred)

best_elastic_step_nonzero = int(
    np.sum(np.abs(best_elastic_step_model.named_steps["elasticnet"].coef_) > 1e-8)
)

print("\nRefit check for best Elastic Net-step model:")
print("Train MSE:", best_elastic_step_train_mse)
print("Validation MSE:", best_elastic_step_val_mse)
print("Nonzero coefficients:", best_elastic_step_nonzero)

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Elastic Net + Step Functions"] = {
    "model": best_elastic_step_model,
    "feature_space": "base_plus_poly_interactions + step_functions",
    "X_train_name": "X_tr_step_best",
    "X_val_name": "X_val_step_best",
    "scaler": "standard",
    "alpha": float(best_elastic_step_overall["alpha"]),
    "l1_ratio": float(best_elastic_step_overall["l1_ratio"]),
    "train_mse": float(best_elastic_step_train_mse),
    "val_mse": float(best_elastic_step_val_mse),
    "nonzero_coef": best_elastic_step_nonzero,
    "notes": "Deferred sanity-check Elastic Net search around the best Lasso-step alpha."
}

del X_tr_step_scaled, X_val_step_scaled
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

### Deferred / Un-run Model Queue

The following model classes are currently deferred rather than discarded. They may be revisited later if time allows or if tree/boosting models plateau.

| Status | Model family | Feature space | Reason deferred |
|---|---|---|---|
| Un-run | Elastic Net + Step Functions | 708-feature step-function space | Ridge and Lasso were very close; full Elastic Net may be expensive and likely marginal |
| Un-run | PCR | Base / expanded linear feature spaces | Useful dimension-reduction baseline after linear models |
| Un-run | PLS | Base / expanded linear feature spaces | Supervised dimension reduction; may improve over PCR |
| Un-run | Regression Splines | Selected continuous predictors | More flexible/smoother alternative to step functions |
| Un-run | GAMs | Selected continuous predictors | Interpretable nonlinear additive model |
| Un-run | KNN Regression | Scaled reduced feature space | Could be expensive in high dimensions; likely needs dimensionality reduction |
| Un-run | SVR | Scaled reduced feature space | Potentially expensive for this sample size |
| Un-run | Neural Network | Scaled numeric/categorical feature space | Later-stage model; needs careful tuning |
| Un-run | BART | Reduced feature space | Useful tree-based Bayesian method, but may be package/runtime dependent |
| Un-run | Elastic Net | base_plus_poly_interactions or reduced step-function space | More sensible than running Elastic Net on all 708 step-function features; avoids repeating the expensive Lasso-step search with little expected gain |

Current priority after the step-function linear family is to move to tree-based models: regression tree, bagging, random forest, and gradient boosting.

In [29]:
# ============================================================
# 13A. Regression Tree baseline (quick holdout screen)
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

# ------------------------------------------------------------
# Design choice:
# Trees already learn threshold/step-like structure internally.
# So for the first tree baseline, we compare the controlled
# pre-step feature spaces rather than the 708-feature step space.
# ------------------------------------------------------------

tree_feature_spaces = {
    "base": feature_spaces["base"],
    "base_plus_poly": feature_spaces["base_plus_poly"],
    "base_plus_interactions": feature_spaces["base_plus_interactions"],
    "base_plus_poly_interactions": feature_spaces["base_plus_poly_interactions"]
}

print("Regression tree baseline")
print("Feature spaces:")
for name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print(f"  {name}: train {X_train_space.shape}, validation {X_val_space.shape}")

# ------------------------------------------------------------
# Reference models
# ------------------------------------------------------------

print("\nReference validation MSEs:")
if "best_ridge_overall" in globals():
    print("Best Ridge:", best_ridge_overall["val_mse"])

if "best_lasso_overall" in globals():
    print("Best Lasso:", best_lasso_overall["val_mse"])

if "best_elasticnet_overall" in globals():
    print("Best Elastic Net:", best_elasticnet_overall["val_mse"])

if "ols_step_val_mse" in globals():
    print("OLS step-functions:", ols_step_val_mse)

if "best_ridge_step_overall" in globals():
    print("Ridge + step-functions:", best_ridge_step_overall["val_mse"])

if "best_lasso_step_val_mse" in globals():
    print("Lasso + step-functions:", best_lasso_step_val_mse)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TREE_RESULTS_PATH = RESULTS_DIR / "regression_tree_results_holdout.csv"
TREE_BEST_BY_SPACE_PATH = RESULTS_DIR / "regression_tree_best_by_space_holdout.csv"
TREE_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_best_overall_holdout.csv"

OVERWRITE_TREE_RESULTS = True

if OVERWRITE_TREE_RESULTS:
    for path in [
        TREE_RESULTS_PATH,
        TREE_BEST_BY_SPACE_PATH,
        TREE_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# Tree hyperparameter grid
# ------------------------------------------------------------
# max_depth controls tree complexity.
# min_samples_leaf controls how small terminal leaves are allowed to be.
# Larger leaves usually reduce overfitting.

max_depth_grid = [3, 5, 7, 9, 12, 15, None]
min_samples_leaf_grid = [25, 50, 100, 250, 500, 1000, 2000]

tree_rows = []

for feature_space_name, (X_train_space, X_val_space) in tree_feature_spaces.items():
    print(f"\n--- Feature space: {feature_space_name} ---")
    
    for max_depth in max_depth_grid:
        for min_samples_leaf in min_samples_leaf_grid:
            
            model = DecisionTreeRegressor(
                criterion="squared_error",
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                min_samples_split=max(2, 2 * min_samples_leaf),
                random_state=RANDOM_STATE
            )
            
            model.fit(X_train_space, y_tr)
            
            train_pred = model.predict(X_train_space)
            val_pred = model.predict(X_val_space)
            
            train_mse = mean_squared_error(y_tr, train_pred)
            val_mse = mean_squared_error(y_val, val_pred)
            
            row = {
                "model_class": "Regression Tree",
                "feature_space": feature_space_name,
                "max_depth": "None" if max_depth is None else max_depth,
                "min_samples_leaf": min_samples_leaf,
                "min_samples_split": max(2, 2 * min_samples_leaf),
                "train_mse": train_mse,
                "val_mse": val_mse,
                "n_leaves": model.get_n_leaves(),
                "tree_depth": model.get_depth(),
                "aic": np.nan,
                "bic": np.nan
            }
            
            tree_rows.append(row)
            append_checkpoint(pd.DataFrame([row]), TREE_RESULTS_PATH)
            
            print(
                f"depth={str(max_depth):>4s} | "
                f"leaf={min_samples_leaf:4d} | "
                f"train_mse={train_mse:.6f} | "
                f"val_mse={val_mse:.6f} | "
                f"leaves={model.get_n_leaves():5d} | "
                f"actual_depth={model.get_depth():3d}"
            )

tree_results = pd.DataFrame(tree_rows).sort_values("val_mse")

best_tree_by_space = (
    tree_results
    .loc[tree_results.groupby("feature_space")["val_mse"].idxmin()]
    .sort_values("val_mse")
)

best_tree_overall = tree_results.iloc[0]

best_tree_by_space.to_csv(TREE_BEST_BY_SPACE_PATH, index=False)
pd.DataFrame([best_tree_overall]).to_csv(TREE_BEST_OVERALL_PATH, index=False)

print("\nBest regression tree result by feature space:")
print(best_tree_by_space.to_string(index=False))

print("\nTop 15 regression tree results overall:")
print(tree_results.head(15).to_string(index=False))

print("\nOverall best regression tree result:")
print(best_tree_overall)

# ------------------------------------------------------------
# Refit best tree on the development training split
# ------------------------------------------------------------

best_tree_feature_space_name = best_tree_overall["feature_space"]
X_train_best_tree, X_val_best_tree = tree_feature_spaces[best_tree_feature_space_name]

best_tree_max_depth = best_tree_overall["max_depth"]
if best_tree_max_depth == "None":
    best_tree_max_depth = None
else:
    best_tree_max_depth = int(best_tree_max_depth)

best_tree_min_samples_leaf = int(best_tree_overall["min_samples_leaf"])

best_tree_model = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=best_tree_max_depth,
    min_samples_leaf=best_tree_min_samples_leaf,
    min_samples_split=max(2, 2 * best_tree_min_samples_leaf),
    random_state=RANDOM_STATE
)

best_tree_model.fit(X_train_best_tree, y_tr)

best_tree_train_pred = best_tree_model.predict(X_train_best_tree)
best_tree_val_pred = best_tree_model.predict(X_val_best_tree)

best_tree_train_mse = mean_squared_error(y_tr, best_tree_train_pred)
best_tree_val_mse = mean_squared_error(y_val, best_tree_val_pred)

print("\nRefit check for best regression tree:")
print("Feature space:", best_tree_feature_space_name)
print("Train MSE:", best_tree_train_mse)
print("Validation MSE:", best_tree_val_mse)
print("Leaves:", best_tree_model.get_n_leaves())
print("Depth:", best_tree_model.get_depth())

# ------------------------------------------------------------
# Feature importances for the best tree
# If duplicate column names exist, group them safely.
# ------------------------------------------------------------

tree_importances = (
    pd.Series(best_tree_model.feature_importances_, index=X_train_best_tree.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 regression tree feature importances:")
print(tree_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Regression Tree"] = {
    "model": best_tree_model,
    "feature_space": best_tree_feature_space_name,
    "X_train_name": f"feature_spaces['{best_tree_feature_space_name}'][0]",
    "X_val_name": f"feature_spaces['{best_tree_feature_space_name}'][1]",
    "max_depth": best_tree_max_depth,
    "min_samples_leaf": best_tree_min_samples_leaf,
    "min_samples_split": max(2, 2 * best_tree_min_samples_leaf),
    "train_mse": float(best_tree_train_mse),
    "val_mse": float(best_tree_val_mse),
    "n_leaves": int(best_tree_model.get_n_leaves()),
    "depth": int(best_tree_model.get_depth()),
    "notes": "Single regression tree baseline across controlled pre-step feature spaces."
}

gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Regression tree baseline
Feature spaces:
  base: train (115936, 162), validation (28985, 162)
  base_plus_poly: train (115936, 172), validation (28985, 172)
  base_plus_interactions: train (115936, 172), validation (28985, 172)
  base_plus_poly_interactions: train (115936, 182), validation (28985, 182)

Reference validation MSEs:
Best Ridge: 308.35660869330025
OLS step-functions: 290.1093738555459
Ridge + step-functions: 290.0792732215163

--- Feature space: base ---
depth=   3 | leaf=  25 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=  50 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 100 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 250 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 500 | train_mse=520.811279 | val_mse=527.943002 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=1000 

### Interpretation: Regression Tree Holdout Baseline

The regression tree baseline produced a major improvement over the previous linear and step-function model families on the holdout validation set.

#### Best Holdout Regression Tree

- Validation MSE: approximately 240.57
- Train MSE: approximately 190.54
- Tree depth: 45
- Number of leaves: 3522
- Minimum samples per leaf: 25
- Best feature space: `base_plus_poly_interactions`

#### Comparison with Previous Best Models

| Model | Validation MSE |
|---|---:|
| OLS + Step Functions | ~290.11 |
| Ridge + Step Functions | ~290.08 |
| Lasso + Step Functions | ~290.06 |
| Regression Tree (holdout) | ~240.57 |

This represents a very large improvement relative to the previous linear-basis-expansion models.

#### Key Takeaways

The strong improvement suggests that the underlying data-generating structure contains important nonlinear interactions and threshold effects that are difficult for linear models and manually engineered step functions to capture efficiently.

Unlike the previous models, the regression tree learns interaction structure and nonlinear thresholds automatically. This appears particularly valuable for this dataset, where demographic, subgroup, grade-level, and assessment-related variables likely interact in complex ways.

The feature importances indicate that the tree heavily relies on variables related to:
- economic disadvantage,
- grade structure,
- assessment categories,
- subgroup indicators,
- attendance,
- school frequency,
- and demographic composition.

#### Important Caution

This result was obtained using the repeatedly-used holdout validation split rather than cross-validation. Because many modeling decisions have already been made using this holdout set, the result should currently be treated as an exploratory screening result rather than the final official regression-tree estimate.

Therefore, the next step is to run K-fold cross-validation for regression trees in order to determine whether this large improvement generalizes across folds or whether part of the gain reflects holdout-specific tuning.

In [43]:
# Preserve the quick holdout tree result before running CV tree selection

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    best_models_by_class["Regression Tree (holdout screen)"] = best_models_by_class["Regression Tree"]
    print("Saved holdout-screen tree as: Regression Tree (holdout screen)")
else:
    print("No holdout-screen Regression Tree found in best_models_by_class.")

Saved holdout-screen tree as: Regression Tree (holdout screen)


### Cross-Validation Status Tracker

| Model family | Current status | Notes |
|---|---|---|
| Simple Linear Regression | Holdout only | Exploratory baseline |
| Multiple Linear Regression | Holdout only | Exploratory baseline |
| Polynomial Regression | Holdout only | Used for feature-space development |
| Interaction Models | Holdout only | Used for feature-space development |
| Step Functions | Holdout only | Exploratory nonlinear basis expansion |
| Ridge Regression | Holdout only | Pre-tree regularization screen |
| Lasso | Holdout only | Pre-tree regularization screen |
| Elastic Net | Deferred / not fully run | Planned later on reduced feature space |
| Ridge + Step Functions | Holdout only | Exploratory shrinkage on expanded basis |
| Lasso + Step Functions | Holdout only | Computationally expensive exploratory result |
| Regression Tree (holdout screen) | Holdout only | Large improvement observed (~240 validation MSE) |
Regression Tree (5-fold CV) below | Complete | Official tree baseline: CV MSE ≈ 250.68; holdout MSE ≈ 241.35

### Planned Future CV Priorities

1. Regression Tree CV
2. Random Forest CV
3. Gradient Boosting / XGBoost CV
4. Compare top tree-based models against holdout-screen linear models
5. Optional: CV revisit for strongest linear-basis models if needed

### Important Validation Note

The original holdout validation split was useful for rapid exploratory screening and feature-space development. However, because many modeling decisions were made using that same split, future serious model-family comparisons should rely primarily on cross-validation, with the holdout set serving as a final secondary sanity check rather than the primary selection mechanism.

In [44]:
# ============================================================
# 13B. Regression Tree baseline with K-fold CV
# ============================================================

from pathlib import Path
from itertools import combinations
import time
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Regression tree baseline with K-fold CV")
print("Base training split shape:", X_tr.shape)
print("Holdout validation shape:", X_val.shape)

# ------------------------------------------------------------
# Important validation design:
#
# We do NOT use the existing feature_spaces dictionary here,
# because those engineered spaces were built after selecting
# top features using the holdout validation split.
#
# For CV, we rebuild top polynomial/interaction candidates
# inside each CV fold using only that fold's training data.
# This avoids leaking validation-fold information into the
# engineered feature space.
# ------------------------------------------------------------

def select_top_by_abs_corr(X_train_df, y_train_array, top_n):
    """
    Select top predictors by absolute correlation with y.
    This is computed using only the training portion of a CV fold.
    """
    y_arr = np.asarray(y_train_array, dtype=float)
    y_centered = y_arr - y_arr.mean()
    y_norm = np.sqrt(np.sum(y_centered ** 2))
    
    scores = []
    
    for col in X_train_df.columns:
        x_arr = X_train_df[col].to_numpy(dtype=float)
        x_centered = x_arr - x_arr.mean()
        x_norm = np.sqrt(np.sum(x_centered ** 2))
        
        denom = x_norm * y_norm
        
        if denom == 0 or not np.isfinite(denom):
            score = 0.0
        else:
            score = abs(float(np.sum(x_centered * y_centered) / denom))
        
        scores.append((col, score))
    
    scores = sorted(scores, key=lambda z: z[1], reverse=True)
    return [col for col, score in scores[:top_n]]


def build_controlled_tree_feature_spaces(X_train_raw, X_eval_raw, y_train_array):
    """
    Build the same four controlled feature spaces, but with top
    polynomial and interaction candidates selected from X_train_raw only.
    """
    top10_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=10)
    top5_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=5)
    
    # Base space
    X_train_base = X_train_raw.copy()
    X_eval_base = X_eval_raw.copy()
    
    # Base + squared terms
    X_train_poly = X_train_raw.copy()
    X_eval_poly = X_eval_raw.copy()
    
    for col in top10_features:
        new_col = f"{col}__squared"
        X_train_poly[new_col] = X_train_raw[col] ** 2
        X_eval_poly[new_col] = X_eval_raw[col] ** 2
    
    # Base + pairwise interactions
    X_train_inter = X_train_raw.copy()
    X_eval_inter = X_eval_raw.copy()
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_inter[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_inter[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    # Base + squared terms + interactions
    X_train_combined = X_train_poly.copy()
    X_eval_combined = X_eval_poly.copy()
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_combined[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_combined[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    feature_spaces_fold = {
        "base": (X_train_base, X_eval_base),
        "base_plus_poly": (X_train_poly, X_eval_poly),
        "base_plus_interactions": (X_train_inter, X_eval_inter),
        "base_plus_poly_interactions": (X_train_combined, X_eval_combined)
    }
    
    return feature_spaces_fold, top10_features, top5_features


# ------------------------------------------------------------
# Reference models already completed
# ------------------------------------------------------------

print("\nReference holdout validation MSEs:")
if "ols_step_val_mse" in globals():
    print("OLS step-functions:", ols_step_val_mse)

if "best_ridge_step_overall" in globals():
    print("Ridge + step-functions:", best_ridge_step_overall["val_mse"])

if "best_lasso_step_val_mse" in globals():
    print("Lasso + step-functions:", best_lasso_step_val_mse)
elif "best_lasso_step_overall" in globals():
    print("Lasso + step-functions:", best_lasso_step_overall["val_mse"])

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TREE_CV_FOLD_RESULTS_PATH = RESULTS_DIR / "regression_tree_cv_fold_results.csv"
TREE_CV_SUMMARY_PATH = RESULTS_DIR / "regression_tree_cv_summary.csv"
TREE_CV_BEST_OVERALL_PATH = RESULTS_DIR / "regression_tree_cv_best_overall.csv"

OVERWRITE_TREE_CV_RESULTS = True

if OVERWRITE_TREE_CV_RESULTS:
    for path in [
        TREE_CV_FOLD_RESULTS_PATH,
        TREE_CV_SUMMARY_PATH,
        TREE_CV_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# CV setup and tree grid
# ------------------------------------------------------------

N_SPLITS_TREE_CV = 5

kf = KFold(
    n_splits=N_SPLITS_TREE_CV,
    shuffle=True,
    random_state=RANDOM_STATE
)

max_depth_grid = [3, 5, 7, 9, 12, 15, None]
min_samples_leaf_grid = [25, 50, 100, 250, 500, 1000, 2000]

y_tr_array = np.asarray(y_tr, dtype=float)

tree_cv_rows = []

# ------------------------------------------------------------
# Run CV
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ Fold {fold_id} / {N_SPLITS_TREE_CV} ================")
    
    X_cv_train_raw = X_tr.iloc[cv_train_idx].copy()
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx].copy()
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    fold_feature_spaces, fold_top10, fold_top5 = build_controlled_tree_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )
    
    print("Top 10 fold features for squared terms:")
    print(fold_top10)
    print("Top 5 fold features for interactions:")
    print(fold_top5)
    
    for feature_space_name, (X_cv_train_space, X_cv_valid_space) in fold_feature_spaces.items():
        print(f"\n--- Fold {fold_id} | feature space: {feature_space_name} | shape: {X_cv_train_space.shape} ---")
        
        for max_depth in max_depth_grid:
            for min_samples_leaf in min_samples_leaf_grid:
                
                model = DecisionTreeRegressor(
                    criterion="squared_error",
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    min_samples_split=max(2, 2 * min_samples_leaf),
                    random_state=RANDOM_STATE
                )
                
                model.fit(X_cv_train_space, y_cv_train)
                
                train_pred = model.predict(X_cv_train_space)
                valid_pred = model.predict(X_cv_valid_space)
                
                train_mse = mean_squared_error(y_cv_train, train_pred)
                valid_mse = mean_squared_error(y_cv_valid, valid_pred)
                
                row = {
                    "model_class": "Regression Tree",
                    "fold": fold_id,
                    "feature_space": feature_space_name,
                    "max_depth": "None" if max_depth is None else max_depth,
                    "min_samples_leaf": min_samples_leaf,
                    "min_samples_split": max(2, 2 * min_samples_leaf),
                    "fold_train_mse": train_mse,
                    "fold_valid_mse": valid_mse,
                    "n_leaves": model.get_n_leaves(),
                    "tree_depth": model.get_depth()
                }
                
                tree_cv_rows.append(row)
                append_checkpoint(pd.DataFrame([row]), TREE_CV_FOLD_RESULTS_PATH)
                
                print(
                    f"depth={str(max_depth):>4s} | "
                    f"leaf={min_samples_leaf:4d} | "
                    f"fold_train_mse={train_mse:.6f} | "
                    f"fold_valid_mse={valid_mse:.6f} | "
                    f"leaves={model.get_n_leaves():5d} | "
                    f"actual_depth={model.get_depth():3d}"
                )
    
    del X_cv_train_raw, X_cv_valid_raw, fold_feature_spaces
    gc.collect()

# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

tree_cv_fold_results = pd.DataFrame(tree_cv_rows)

group_cols = [
    "model_class",
    "feature_space",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split"
]

tree_cv_summary = (
    tree_cv_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std"),
        mean_leaves=("n_leaves", "mean"),
        mean_depth=("tree_depth", "mean")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

tree_cv_summary.to_csv(TREE_CV_SUMMARY_PATH, index=False)

best_tree_cv_by_space = (
    tree_cv_summary
    .loc[tree_cv_summary.groupby("feature_space")["cv_valid_mse_mean"].idxmin()]
    .sort_values("cv_valid_mse_mean")
)

best_tree_cv_overall = tree_cv_summary.iloc[0]
pd.DataFrame([best_tree_cv_overall]).to_csv(TREE_CV_BEST_OVERALL_PATH, index=False)

print("\nBest CV regression tree result by feature space:")
print(best_tree_cv_by_space.to_string(index=False))

print("\nTop 15 CV regression tree results overall:")
print(tree_cv_summary.head(15).to_string(index=False))

print("\nOverall best CV regression tree result:")
print(best_tree_cv_overall)

# ------------------------------------------------------------
# Refit the CV-selected tree on the full development training split
# and evaluate once on the holdout validation split
# ------------------------------------------------------------

best_tree_feature_space_name = best_tree_cv_overall["feature_space"]

final_feature_spaces, final_top10, final_top5 = build_controlled_tree_feature_spaces(
    X_tr,
    X_val,
    y_tr_array
)

X_train_best_tree, X_val_best_tree = final_feature_spaces[best_tree_feature_space_name]

best_tree_max_depth = best_tree_cv_overall["max_depth"]
if best_tree_max_depth == "None":
    best_tree_max_depth = None
else:
    best_tree_max_depth = int(best_tree_max_depth)

best_tree_min_samples_leaf = int(best_tree_cv_overall["min_samples_leaf"])

best_tree_model = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=best_tree_max_depth,
    min_samples_leaf=best_tree_min_samples_leaf,
    min_samples_split=max(2, 2 * best_tree_min_samples_leaf),
    random_state=RANDOM_STATE
)

best_tree_model.fit(X_train_best_tree, y_tr)

best_tree_train_pred = best_tree_model.predict(X_train_best_tree)
best_tree_val_pred = best_tree_model.predict(X_val_best_tree)

best_tree_train_mse = mean_squared_error(y_tr, best_tree_train_pred)
best_tree_val_mse = mean_squared_error(y_val, best_tree_val_pred)

print("\nRefit check for CV-selected regression tree:")
print("Feature space:", best_tree_feature_space_name)
print("CV mean validation MSE:", best_tree_cv_overall["cv_valid_mse_mean"])
print("CV validation MSE std:", best_tree_cv_overall["cv_valid_mse_std"])
print("Holdout train MSE:", best_tree_train_mse)
print("Holdout validation MSE:", best_tree_val_mse)
print("Leaves:", best_tree_model.get_n_leaves())
print("Depth:", best_tree_model.get_depth())

print("\nFinal top 10 features used for squared terms:")
print(final_top10)

print("\nFinal top 5 features used for interactions:")
print(final_top5)

# ------------------------------------------------------------
# Feature importances for the final CV-selected tree
# ------------------------------------------------------------

tree_importances = (
    pd.Series(best_tree_model.feature_importances_, index=X_train_best_tree.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 regression tree feature importances:")
print(tree_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Regression Tree"] = {
    "model": best_tree_model,
    "feature_space": best_tree_feature_space_name,
    "X_train_name": "CV-built feature space from X_tr",
    "X_val_name": "CV-built feature space from X_val",
    "max_depth": best_tree_max_depth,
    "min_samples_leaf": best_tree_min_samples_leaf,
    "min_samples_split": max(2, 2 * best_tree_min_samples_leaf),
    "cv_valid_mse_mean": float(best_tree_cv_overall["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_tree_cv_overall["cv_valid_mse_std"]),
    "holdout_train_mse": float(best_tree_train_mse),
    "holdout_val_mse": float(best_tree_val_mse),
    "n_leaves": int(best_tree_model.get_n_leaves()),
    "depth": int(best_tree_model.get_depth()),
    "notes": (
        "Regression tree selected by 5-fold CV on X_tr. "
        "Holdout validation used only after CV selection."
    )
}

del final_feature_spaces
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Regression tree baseline with K-fold CV
Base training split shape: (115936, 162)
Holdout validation shape: (28985, 162)

Reference holdout validation MSEs:
OLS step-functions: 290.1093738555459
Ridge + step-functions: 290.0792732215163

================ Fold 1 / 5 ================


Top 10 fold features for squared terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'ATTENDANCE_RATE', 'PERCENT_DROPOUT']
Top 5 fold features for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']

--- Fold 1 | feature space: base | shape: (92748, 162) ---
depth=   3 | leaf=  25 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf=  50 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 100 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 250 | fold_train_mse=520.426897 | fold_valid_mse=522.403878 | leaves=    8 | actual_depth=  3
depth=   3 | leaf= 500 | fold_train_ms

### Regression Tree Cross-Validation Summary

The 5-fold cross-validation results confirm that regression trees substantially improve over the earlier linear and step-function baselines. The best CV-selected tree used the `base_plus_poly_interactions` feature space with `max_depth=None`, `min_samples_leaf=25`, and `min_samples_split=50`. This model achieved a mean CV validation MSE of approximately **250.68** with a standard deviation of approximately **4.65**.

When refit on the full training portion and evaluated on the original holdout validation set, the selected tree achieved a holdout validation MSE of approximately **241.35**. This supports the earlier holdout-screen result and suggests that the tree model captures important nonlinear and interaction structure that the linear-basis models missed.

The expanded polynomial and interaction tree feature spaces only modestly improved over the simpler base feature space. This is not surprising because regression trees already create nonlinear and interaction effects through recursive splitting. Therefore, the tree result should be treated as the first serious non-linear baseline, while random forests and gradient boosting are natural next steps.

### Interpretation: Regression Tree with 5-Fold Cross-Validation

We evaluated regression trees using 5-fold cross-validation on the development training split. Unlike the earlier holdout-only tree screen, this CV run gives a more reliable estimate of tree performance and reduces dependence on a single validation split.

The best CV-selected regression tree used:

- Feature space: `base_plus_poly_interactions`
- Maximum depth: unrestricted (`None`)
- Minimum samples per leaf: 25
- Mean CV validation MSE: 250.68
- CV validation MSE standard deviation: 4.65
- Mean number of leaves: about 2828
- Mean tree depth: about 40

After refitting this CV-selected tree on the full development training split, the holdout validation MSE was 241.35.

#### Comparison with Previous Best Models

| Model | Validation estimate |
|---|---:|
| Lasso + Step Functions | Holdout MSE ≈ 290.06 |
| Regression Tree | CV MSE ≈ 250.68 |
| Regression Tree | Holdout MSE ≈ 241.35 |

The regression tree represents a large improvement over the previous linear, regularized linear, and step-function models. The CV result confirms that the improvement is not merely an artifact of a single holdout split.

#### Key Takeaways

The tree model captures nonlinear threshold effects and interactions much more effectively than manually engineered linear basis expansions. This is consistent with the strong importance of variables related to economic disadvantage, grade level, assessment type, subgroup membership, attendance, region, school frequency, and demographic composition.

The selected tree is very large, with thousands of leaves and an unrestricted depth. This gives the model high flexibility, but it also creates a substantial gap between training error and CV validation error. Therefore, the single tree is useful as a strong baseline, but it is likely high-variance.

#### Implication

Tree-based methods are now the most promising model family. The next logical step is to use ensemble tree methods, especially bagging and random forests, to reduce variance while preserving the nonlinear and interaction-capturing strengths of regression trees.

In [30]:
# ============================================================
# 14A. Random Forest: controlled 3-fold CV screen
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest controlled CV screen")
print("Using base feature space only for first RF screen.")
print("X_tr shape:", X_tr.shape)
print("X_val shape:", X_val.shape)

# ------------------------------------------------------------
# Reference: current best regression tree
# ------------------------------------------------------------

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    tree_ref = best_models_by_class["Regression Tree"]
    print("\nReference Regression Tree:")
    print("CV validation MSE mean:", tree_ref.get("cv_valid_mse_mean"))
    print("Holdout validation MSE:", tree_ref.get("holdout_val_mse"))

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

RF_CV_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_screen.csv"
RF_CV_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_screen.csv"
RF_CV_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_screen.csv"

OVERWRITE_RF_CV_RESULTS = True

if OVERWRITE_RF_CV_RESULTS:
    for path in [
        RF_CV_FOLD_RESULTS_PATH,
        RF_CV_SUMMARY_PATH,
        RF_CV_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)

# ------------------------------------------------------------
# CV setup
# ------------------------------------------------------------

N_SPLITS_RF_CV = 3

kf = KFold(
    n_splits=N_SPLITS_RF_CV,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Keep this modest for the first RF pass.
# If RF is promising, we will refine later.
RF_N_ESTIMATORS = 120
RF_MAX_SAMPLES = 0.80

rf_param_grid = list(ParameterGrid({
    "max_features": ["sqrt", 0.50],
    "min_samples_leaf": [1, 5, 25],
}))

print("\nRF screen settings:")
print("CV folds:", N_SPLITS_RF_CV)
print("n_estimators:", RF_N_ESTIMATORS)
print("max_samples:", RF_MAX_SAMPLES)
print("parameter combinations:", len(rf_param_grid))

y_tr_array = np.asarray(y_tr, dtype=float)

rf_cv_rows = []

# ------------------------------------------------------------
# Run CV on base feature space
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ RF Fold {fold_id} / {N_SPLITS_RF_CV} ================")
    
    X_cv_train = X_tr.iloc[cv_train_idx]
    X_cv_valid = X_tr.iloc[cv_valid_idx]
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    for combo_id, params in enumerate(rf_param_grid, start=1):
        max_features = params["max_features"]
        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)
        
        model = RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            bootstrap=True,
            max_samples=RF_MAX_SAMPLES,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )
        
        model.fit(X_cv_train, y_cv_train)
        
        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)
        
        train_mse = mean_squared_error(y_cv_train, train_pred)
        valid_mse = mean_squared_error(y_cv_valid, valid_pred)
        
        # OOB predictions are an internal training-set generalization check.
        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)
        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan
        
        row = {
            "model_class": "Random Forest",
            "fold": fold_id,
            "feature_space": "base",
            "n_estimators": RF_N_ESTIMATORS,
            "max_samples": RF_MAX_SAMPLES,
            "max_features": str(max_features),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": train_mse,
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": valid_mse
        }
        
        rf_cv_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_CV_FOLD_RESULTS_PATH)
        
        print(
            f"combo={combo_id:2d} | "
            f"max_features={str(max_features):>4s} | "
            f"leaf={min_samples_leaf:3d} | "
            f"train_mse={train_mse:.6f} | "
            f"oob_mse={oob_mse:.6f} | "
            f"valid_mse={valid_mse:.6f}"
        )
    
    gc.collect()

# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

rf_cv_fold_results = pd.DataFrame(rf_cv_rows)

group_cols = [
    "model_class",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split"
]

rf_cv_summary = (
    rf_cv_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

rf_cv_summary.to_csv(RF_CV_SUMMARY_PATH, index=False)

best_rf_cv_overall = rf_cv_summary.iloc[0]
pd.DataFrame([best_rf_cv_overall]).to_csv(RF_CV_BEST_OVERALL_PATH, index=False)

print("\nRandom Forest CV summary:")
print(rf_cv_summary.to_string(index=False))

print("\nBest Random Forest CV result:")
print(best_rf_cv_overall)

# ------------------------------------------------------------
# Refit selected RF on full development training split
# and evaluate once on holdout validation split
# ------------------------------------------------------------

best_rf_max_features = best_rf_cv_overall["max_features"]
if best_rf_max_features != "sqrt":
    best_rf_max_features = float(best_rf_max_features)

best_rf_min_samples_leaf = int(best_rf_cv_overall["min_samples_leaf"])
best_rf_min_samples_split = max(2, 2 * best_rf_min_samples_leaf)

best_rf_model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=best_rf_min_samples_leaf,
    min_samples_split=best_rf_min_samples_split,
    max_features=best_rf_max_features,
    bootstrap=True,
    max_samples=RF_MAX_SAMPLES,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

best_rf_model.fit(X_tr, y_tr)

best_rf_train_pred = best_rf_model.predict(X_tr)
best_rf_val_pred = best_rf_model.predict(X_val)

best_rf_train_mse = mean_squared_error(y_tr, best_rf_train_pred)
best_rf_val_mse = mean_squared_error(y_val, best_rf_val_pred)

best_rf_oob_pred = best_rf_model.oob_prediction_
best_rf_oob_mask = np.isfinite(best_rf_oob_pred)

if best_rf_oob_mask.sum() > 0:
    best_rf_oob_mse = mean_squared_error(
        np.asarray(y_tr, dtype=float)[best_rf_oob_mask],
        best_rf_oob_pred[best_rf_oob_mask]
    )
else:
    best_rf_oob_mse = np.nan

print("\nRefit check for CV-selected Random Forest:")
print("Feature space: base")
print("CV mean validation MSE:", best_rf_cv_overall["cv_valid_mse_mean"])
print("CV validation MSE std:", best_rf_cv_overall["cv_valid_mse_std"])
print("Holdout train MSE:", best_rf_train_mse)
print("Holdout OOB MSE:", best_rf_oob_mse)
print("Holdout validation MSE:", best_rf_val_mse)
print("max_features:", best_rf_max_features)
print("min_samples_leaf:", best_rf_min_samples_leaf)
print("min_samples_split:", best_rf_min_samples_split)

# ------------------------------------------------------------
# Feature importances
# ------------------------------------------------------------

rf_importances = (
    pd.Series(best_rf_model.feature_importances_, index=X_tr.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 Random Forest feature importances:")
print(rf_importances.head(30).to_string())

# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Random Forest"] = {
    "model": best_rf_model,
    "feature_space": "base",
    "X_train_name": "X_tr",
    "X_val_name": "X_val",
    "n_estimators": RF_N_ESTIMATORS,
    "max_samples": RF_MAX_SAMPLES,
    "max_features": best_rf_max_features,
    "min_samples_leaf": best_rf_min_samples_leaf,
    "min_samples_split": best_rf_min_samples_split,
    "cv_valid_mse_mean": float(best_rf_cv_overall["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_rf_cv_overall["cv_valid_mse_std"]),
    "holdout_train_mse": float(best_rf_train_mse),
    "holdout_oob_mse": float(best_rf_oob_mse),
    "holdout_val_mse": float(best_rf_val_mse),
    "notes": (
        "First controlled Random Forest CV screen on base feature space only. "
        "Uses 3-fold CV, bootstrap sampling, and max_samples=0.80 for tractability."
    )
}

gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest controlled CV screen
Using base feature space only for first RF screen.
X_tr shape: (115936, 162)
X_val shape: (28985, 162)

Reference Regression Tree:
CV validation MSE mean: None
Holdout validation MSE: None

RF screen settings:
CV folds: 3
n_estimators: 120
max_samples: 0.8
parameter combinations: 6

================ RF Fold 1 / 3 ================
combo= 1 | max_features=sqrt | leaf=  1 | train_mse=38.007839 | oob_mse=186.637481 | valid_mse=183.452533
combo= 2 | max_features=sqrt | leaf=  5 | train_mse=187.298683 | oob_mse=239.712041 | valid_mse=238.688685
combo= 3 | max_features=sqrt | leaf= 25 | train_mse=271.703122 | oob_mse=294.923165 | valid_mse=295.280890
combo= 4 | max_features= 0.5 | leaf=  1 | train_mse=31.132973 | oob_mse=152.413389 | valid_mse=149.439472
combo= 5 | max_features= 0.5 | leaf=  5 | train_mse=109.617475 | oob_mse=182.720757 | valid_mse=180.841065
combo= 6 | max_features= 0.5 | leaf= 25 | train_mse=199.974906 | oob_mse=230.424653 | valid_mse=229.

### Random Forest Preliminary Screen

A first controlled random forest screen was run using the base feature space, 3-fold cross-validation, 120 trees, and `max_samples=0.80`.

The best preliminary random forest used:

- `max_features = 0.5`
- `min_samples_leaf = 1`
- `min_samples_split = 2`

This model achieved a mean 3-fold CV validation MSE of approximately **149.55**, with a CV standard deviation of approximately **0.28**. When refit on the full development training split, it achieved a holdout validation MSE of approximately **127.48**.

This is a major improvement over the single regression tree baseline, whose CV validation MSE was approximately **250.68** and holdout validation MSE was approximately **241.35**. The result strongly suggests that variance reduction through averaging many trees is highly valuable for this problem.

Because the preliminary screen was intentionally small and used only 3-fold CV, the next step is a stronger 5-fold random forest refinement. The refinement will focus on the promising region around low leaf sizes and moderate-to-large `max_features`, while also testing whether the fold-safe polynomial/interactions feature space adds value.

In [43]:
# ============================================================
# 14B. Random Forest: stronger 5-fold CV refinement
# ============================================================

from pathlib import Path
from itertools import combinations
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest 5-fold CV refinement")
print("Base training split shape:", X_tr.shape)
print("Holdout validation shape:", X_val.shape)

# ------------------------------------------------------------
# Helper functions
# These are redefined here so this cell does not depend on
# whether the earlier tree-CV helper functions are still in memory.
# ------------------------------------------------------------

def select_top_by_abs_corr(X_train_df, y_train_array, top_n):
    """
    Select top predictors by absolute correlation with y.
    This is computed using only the training portion of a CV fold.
    """
    y_arr = np.asarray(y_train_array, dtype=float)
    y_centered = y_arr - y_arr.mean()
    y_norm = np.sqrt(np.sum(y_centered ** 2))
    
    scores = []
    
    for col in X_train_df.columns:
        x_arr = X_train_df[col].to_numpy(dtype=float)
        x_centered = x_arr - x_arr.mean()
        x_norm = np.sqrt(np.sum(x_centered ** 2))
        
        denom = x_norm * y_norm
        
        if denom == 0 or not np.isfinite(denom):
            score = 0.0
        else:
            score = abs(float(np.sum(x_centered * y_centered) / denom))
        
        scores.append((col, score))
    
    scores = sorted(scores, key=lambda z: z[1], reverse=True)
    return [col for col, score in scores[:top_n]]


def build_rf_feature_spaces(X_train_raw, X_eval_raw, y_train_array):
    """
    Build fold-safe RF feature spaces.

    Important: top squared/interacted features are selected using only
    the training portion of the current fold.
    """
    top10_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=10)
    top5_features = select_top_by_abs_corr(X_train_raw, y_train_array, top_n=5)
    
    # Base
    X_train_base = X_train_raw.copy()
    X_eval_base = X_eval_raw.copy()
    
    # Base + squared terms + pairwise interactions
    X_train_combined = X_train_raw.copy()
    X_eval_combined = X_eval_raw.copy()
    
    for col in top10_features:
        new_col = f"{col}__squared"
        X_train_combined[new_col] = X_train_raw[col] ** 2
        X_eval_combined[new_col] = X_eval_raw[col] ** 2
    
    for f1, f2 in combinations(top5_features, 2):
        new_col = f"{f1}__x__{f2}"
        X_train_combined[new_col] = X_train_raw[f1] * X_train_raw[f2]
        X_eval_combined[new_col] = X_eval_raw[f1] * X_eval_raw[f2]
    
    feature_spaces_fold = {
        "base": (X_train_base, X_eval_base),
        "base_plus_poly_interactions": (X_train_combined, X_eval_combined)
    }
    
    return feature_spaces_fold, top10_features, top5_features


def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def parse_float_or_none(value):
    if str(value) == "None":
        return None
    return float(value)


# ------------------------------------------------------------
# Reference models
# ------------------------------------------------------------

print("\nReference models:")

if "best_models_by_class" in globals() and "Regression Tree" in best_models_by_class:
    tree_ref = best_models_by_class["Regression Tree"]
    print("Regression Tree CV MSE:", tree_ref.get("cv_valid_mse_mean"))
    print("Regression Tree holdout MSE:", tree_ref.get("holdout_val_mse"))

if "best_models_by_class" in globals() and "Random Forest" in best_models_by_class:
    rf_prelim_ref = best_models_by_class["Random Forest"]
    print("Preliminary RF CV MSE:", rf_prelim_ref.get("cv_valid_mse_mean"))
    print("Preliminary RF holdout MSE:", rf_prelim_ref.get("holdout_val_mse"))


# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

RF_REFINED_FOLD_RESULTS_PATH = RESULTS_DIR / "random_forest_cv_fold_results_refined.csv"
RF_REFINED_SUMMARY_PATH = RESULTS_DIR / "random_forest_cv_summary_refined.csv"
RF_REFINED_BEST_OVERALL_PATH = RESULTS_DIR / "random_forest_cv_best_overall_refined.csv"

OVERWRITE_RF_REFINED_RESULTS = True

if OVERWRITE_RF_REFINED_RESULTS:
    for path in [
        RF_REFINED_FOLD_RESULTS_PATH,
        RF_REFINED_SUMMARY_PATH,
        RF_REFINED_BEST_OVERALL_PATH,
    ]:
        if path.exists():
            path.unlink()


# ------------------------------------------------------------
# CV setup and refined RF grid
# ------------------------------------------------------------

N_SPLITS_RF_REFINED = 5
RF_N_ESTIMATORS_REFINED = 300

kf = KFold(
    n_splits=N_SPLITS_RF_REFINED,
    shuffle=True,
    random_state=RANDOM_STATE
)

rf_refined_param_grid = list(ParameterGrid({
    "feature_space": ["base", "base_plus_poly_interactions"],
    "max_features": [0.40, 0.50, 0.70],
    "min_samples_leaf": [1, 2, 5],
    "max_samples": [0.80, None]
}))

print("\nRF refined settings:")
print("CV folds:", N_SPLITS_RF_REFINED)
print("n_estimators:", RF_N_ESTIMATORS_REFINED)
print("parameter combinations:", len(rf_refined_param_grid))
print("total fits:", N_SPLITS_RF_REFINED * len(rf_refined_param_grid))

y_tr_array = np.asarray(y_tr, dtype=float)

rf_refined_rows = []

# ------------------------------------------------------------
# Run 5-fold CV
# ------------------------------------------------------------

for fold_id, (cv_train_idx, cv_valid_idx) in enumerate(kf.split(X_tr), start=1):
    print(f"\n================ RF Refined Fold {fold_id} / {N_SPLITS_RF_REFINED} ================")
    
    X_cv_train_raw = X_tr.iloc[cv_train_idx]
    X_cv_valid_raw = X_tr.iloc[cv_valid_idx]
    
    y_cv_train = y_tr_array[cv_train_idx]
    y_cv_valid = y_tr_array[cv_valid_idx]
    
    fold_feature_spaces, fold_top10, fold_top5 = build_rf_feature_spaces(
        X_cv_train_raw,
        X_cv_valid_raw,
        y_cv_train
    )
    
    print("Top 10 fold features for squared terms:")
    print(fold_top10)
    print("Top 5 fold features for interactions:")
    print(fold_top5)
    
    for combo_id, params in enumerate(rf_refined_param_grid, start=1):
        feature_space_name = params["feature_space"]
        max_features = params["max_features"]
        min_samples_leaf = int(params["min_samples_leaf"])
        min_samples_split = max(2, 2 * min_samples_leaf)
        max_samples = params["max_samples"]
        
        X_cv_train, X_cv_valid = fold_feature_spaces[feature_space_name]
        
        model = RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS_REFINED,
            criterion="squared_error",
            max_depth=None,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            bootstrap=True,
            max_samples=max_samples,
            oob_score=True,
            n_jobs=-1,
            random_state=RANDOM_STATE + 1000 * fold_id + combo_id
        )
        
        model.fit(X_cv_train, y_cv_train)
        
        train_pred = model.predict(X_cv_train)
        valid_pred = model.predict(X_cv_valid)
        
        train_mse = mean_squared_error(y_cv_train, train_pred)
        valid_mse = mean_squared_error(y_cv_valid, valid_pred)
        
        oob_pred = model.oob_prediction_
        oob_mask = np.isfinite(oob_pred)
        
        if oob_mask.sum() > 0:
            oob_mse = mean_squared_error(y_cv_train[oob_mask], oob_pred[oob_mask])
        else:
            oob_mse = np.nan
        
        row = {
            "model_class": "Random Forest",
            "screen": "refined_5fold",
            "fold": fold_id,
            "feature_space": feature_space_name,
            "n_estimators": RF_N_ESTIMATORS_REFINED,
            "max_samples": "None" if max_samples is None else str(max_samples),
            "max_features": str(max_features),
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "fold_train_mse": train_mse,
            "fold_oob_mse": oob_mse,
            "fold_valid_mse": valid_mse
        }
        
        rf_refined_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), RF_REFINED_FOLD_RESULTS_PATH)
        
        print(
            f"combo={combo_id:2d} | "
            f"space={feature_space_name:28s} | "
            f"max_samples={str(max_samples):>4s} | "
            f"max_features={str(max_features):>4s} | "
            f"leaf={min_samples_leaf:2d} | "
            f"train_mse={train_mse:.6f} | "
            f"oob_mse={oob_mse:.6f} | "
            f"valid_mse={valid_mse:.6f}"
        )
    
    del fold_feature_spaces
    gc.collect()


# ------------------------------------------------------------
# Summarize CV results
# ------------------------------------------------------------

rf_refined_fold_results = pd.DataFrame(rf_refined_rows)

group_cols = [
    "model_class",
    "screen",
    "feature_space",
    "n_estimators",
    "max_samples",
    "max_features",
    "min_samples_leaf",
    "min_samples_split"
]

rf_refined_summary = (
    rf_refined_fold_results
    .groupby(group_cols)
    .agg(
        cv_train_mse_mean=("fold_train_mse", "mean"),
        cv_train_mse_std=("fold_train_mse", "std"),
        cv_oob_mse_mean=("fold_oob_mse", "mean"),
        cv_oob_mse_std=("fold_oob_mse", "std"),
        cv_valid_mse_mean=("fold_valid_mse", "mean"),
        cv_valid_mse_std=("fold_valid_mse", "std")
    )
    .reset_index()
    .sort_values("cv_valid_mse_mean")
)

rf_refined_summary.to_csv(RF_REFINED_SUMMARY_PATH, index=False)

best_rf_refined = rf_refined_summary.iloc[0]
pd.DataFrame([best_rf_refined]).to_csv(RF_REFINED_BEST_OVERALL_PATH, index=False)

print("\nRandom Forest refined CV summary:")
print(rf_refined_summary.to_string(index=False))

print("\nBest refined Random Forest CV result:")
print(best_rf_refined)


# ------------------------------------------------------------
# Refit selected RF on full development training split
# and evaluate once on holdout validation split
# ------------------------------------------------------------

best_feature_space = best_rf_refined["feature_space"]
best_max_samples = parse_float_or_none(best_rf_refined["max_samples"])
best_max_features = float(best_rf_refined["max_features"])
best_min_samples_leaf = int(best_rf_refined["min_samples_leaf"])
best_min_samples_split = max(2, 2 * best_min_samples_leaf)

final_feature_spaces, final_top10, final_top5 = build_rf_feature_spaces(
    X_tr,
    X_val,
    y_tr
)

X_rf_train_final, X_rf_val_final = final_feature_spaces[best_feature_space]

best_rf_refined_model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS_REFINED,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=best_min_samples_leaf,
    min_samples_split=best_min_samples_split,
    max_features=best_max_features,
    bootstrap=True,
    max_samples=best_max_samples,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

best_rf_refined_model.fit(X_rf_train_final, y_tr)

rf_train_pred = best_rf_refined_model.predict(X_rf_train_final)
rf_val_pred = best_rf_refined_model.predict(X_rf_val_final)

rf_train_mse = mean_squared_error(y_tr, rf_train_pred)
rf_val_mse = mean_squared_error(y_val, rf_val_pred)

rf_oob_pred = best_rf_refined_model.oob_prediction_
rf_oob_mask = np.isfinite(rf_oob_pred)

if rf_oob_mask.sum() > 0:
    rf_oob_mse = mean_squared_error(
        np.asarray(y_tr, dtype=float)[rf_oob_mask],
        rf_oob_pred[rf_oob_mask]
    )
else:
    rf_oob_mse = np.nan

print("\nRefit check for CV-selected refined Random Forest:")
print("Feature space:", best_feature_space)
print("CV mean validation MSE:", best_rf_refined["cv_valid_mse_mean"])
print("CV validation MSE std:", best_rf_refined["cv_valid_mse_std"])
print("Holdout train MSE:", rf_train_mse)
print("Holdout OOB MSE:", rf_oob_mse)
print("Holdout validation MSE:", rf_val_mse)
print("n_estimators:", RF_N_ESTIMATORS_REFINED)
print("max_samples:", best_max_samples)
print("max_features:", best_max_features)
print("min_samples_leaf:", best_min_samples_leaf)
print("min_samples_split:", best_min_samples_split)

print("\nFinal top 10 features used for squared terms:")
print(final_top10)

print("\nFinal top 5 features used for interactions:")
print(final_top5)

rf_refined_importances = (
    pd.Series(best_rf_refined_model.feature_importances_, index=X_rf_train_final.columns)
    .groupby(level=0)
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 30 refined Random Forest feature importances:")
print(rf_refined_importances.head(30).to_string())


# ------------------------------------------------------------
# Store best model by model class
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

if "Random Forest" in best_models_by_class:
    best_models_by_class.setdefault(
        "Random Forest preliminary 3-fold",
        best_models_by_class["Random Forest"]
    )

best_models_by_class["Random Forest"] = {
    "model": best_rf_refined_model,
    "feature_space": best_feature_space,
    "X_train_final": X_rf_train_final,
    "X_val_final": X_rf_val_final,
    "n_estimators": RF_N_ESTIMATORS_REFINED,
    "max_samples": best_max_samples,
    "max_features": best_max_features,
    "min_samples_leaf": best_min_samples_leaf,
    "min_samples_split": best_min_samples_split,
    "cv_valid_mse_mean": float(best_rf_refined["cv_valid_mse_mean"]),
    "cv_valid_mse_std": float(best_rf_refined["cv_valid_mse_std"]),
    "holdout_train_mse": float(rf_train_mse),
    "holdout_oob_mse": float(rf_oob_mse),
    "holdout_val_mse": float(rf_val_mse),
    "notes": (
        "Refined 5-fold Random Forest screen over base and fold-safe "
        "base_plus_poly_interactions feature spaces."
    )
}

del final_feature_spaces
gc.collect()

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest 5-fold CV refinement
Base training split shape: (115936, 162)
Holdout validation shape: (28985, 162)

Reference models:
Regression Tree CV MSE: 250.67682419719927
Regression Tree holdout MSE: 241.35313171586995
Preliminary RF CV MSE: 149.54957077027083
Preliminary RF holdout MSE: 127.47963454073466

RF refined settings:
CV folds: 5
n_estimators: 300
parameter combinations: 36
total fits: 180

================ RF Refined Fold 1 / 5 ================
Top 10 fold features for squared terms:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS', 'PERCENT_BLACK', 'PERCENT_ENGLISH_LANGUAGE_LEANERS', 'PERCENT_WITH_DISABILITIES', 'ATTENDANCE_RATE', 'PERCENT_DROPOUT']
Top 5 fold features for interactions:
['PERCENT_ECONOMICALLY_DISADVANTAGED', 'PERCENT_FREE_LUNCH', 'PERCENT_DIPLOMA', 'PERCENT_STILL_ENROLLED', 'PERCENT_HOMELESS']
combo= 1 | space=base                         | max_samples= 0.8 | max_features= 0.4

### First Random Forest Submission Candidate

The refined 5-fold random forest screen selected the base feature space with 300 trees, `max_features = 0.5`, `max_samples = None`, `min_samples_leaf = 1`, and `min_samples_split = 2`.

This model achieved a mean 5-fold CV validation MSE of approximately **134.71** and a holdout validation MSE of approximately **123.21**. The holdout OOB MSE was approximately **123.64**, which is close to the holdout validation MSE and provides some reassurance that the model is not only exploiting the fixed holdout split.

Because this is the strongest model so far by a large margin, it is reasonable to train the same model specification on the full labeled training data and generate a first Kaggle submission.

In [46]:
# ============================================================
# 15A. First Kaggle submission: refined Random Forest on full training data
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Building first Random Forest submission")
print("Training feature matrix:", X_train_proc_model.shape)
print("Test feature matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature column count mismatch."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test feature columns are not aligned."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match test feature matrix."
assert test_ids.isna().sum() == 0, "Missing ASSESSMENT_ID values in test_ids."

# ------------------------------------------------------------
# Best refined RF specification from 5-fold CV
# ------------------------------------------------------------

final_rf_submission_model = RandomForestRegressor(
    n_estimators=300,
    criterion="squared_error",
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.5,
    bootstrap=True,
    max_samples=None,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

final_rf_submission_model.fit(X_train_proc_model, y_train)

# ------------------------------------------------------------
# Predict test set
# ------------------------------------------------------------

test_pred_raw = final_rf_submission_model.predict(X_test_proc_model)

print("\nRaw prediction summary:")
print(pd.Series(test_pred_raw).describe().to_string())

# Target is a percentage, so clip to the valid range.
test_pred_clipped = np.clip(test_pred_raw, 0, 100)

print("\nClipped prediction summary:")
print(pd.Series(test_pred_clipped).describe().to_string())

# ------------------------------------------------------------
# Build submission file
# ------------------------------------------------------------

submission_rf = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": test_pred_clipped
})

assert submission_rf.shape[0] == X_test_proc_model.shape[0], "Submission row count mismatch."
assert list(submission_rf.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"], "Submission columns are wrong."
assert submission_rf["ASSESSMENT_ID"].isna().sum() == 0, "Missing ASSESSMENT_ID in submission."
assert submission_rf["PERCENT_PROFICIENT"].isna().sum() == 0, "Missing predictions in submission."

submission_path = Path("submission_rf_refined_300_base.csv")
submission_rf.to_csv(submission_path, index=False)

print("\nSubmission file written to:", submission_path.resolve())
print("Submission shape:", submission_rf.shape)
print("\nFirst 5 rows:")
print(submission_rf.head().to_string(index=False))

Building first Random Forest submission
Training feature matrix: (144921, 162)
Test feature matrix: (48307, 162)

Raw prediction summary:
count    48307.000000
mean        54.055481
std         23.207649
min          0.093333
25%         35.896667
50%         52.250000
75%         72.523333
max        100.000000

Clipped prediction summary:
count    48307.000000
mean        54.055481
std         23.207649
min          0.093333
25%         35.896667
50%         52.250000
75%         72.523333
max        100.000000

Submission file written to: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_rf_refined_300_base.csv
Submission shape: (48307, 2)

First 5 rows:
ASSESSMENT_ID  PERCENT_PROFICIENT
 8af5e0382a81           59.140000
 e1591bf8db41           51.220000
 547ec44dcea6           33.786667
 0e200399fc40           67.336667
 c2c40438dac7           85.563333


### First Kaggle Submission File

A first submission file was generated using the refined random forest specification trained on the full labeled training data.

The submission file contains **48,307 rows** and the required two columns: `ASSESSMENT_ID` and `PERCENT_PROFICIENT`. Predicted values were already within the valid 0–100 range, so clipping did not alter the predictions. The prediction distribution appears plausible relative to the training target distribution, with a mean of approximately **54.06**, median of approximately **52.25**, and standard deviation of approximately **23.21**.

This file is ready for Kaggle submission as the first serious random forest benchmark.

### Random Forest Model-Class Continuation

The first full-data random forest submission achieved a public leaderboard MSE of **111.096**. This is worse than the external teammate LightGBM benchmark of **91.092**, but it is still a valid clean benchmark from the current notebook.

We will not switch directly to LightGBM yet. The project workflow is intentionally moving through model families in increasing complexity. Before leaving the random forest / bagging family, we want to extract two useful artifacts:

1. A clean out-of-fold random forest prediction vector for all labeled training rows.
2. A fold-averaged random forest test prediction file.

The out-of-fold predictions will be useful later for stacking, even if the random forest is not the final single best model.




# continue from here

In [47]:
# ============================================================
# 16A. Random Forest OOF predictions + fold-averaged test predictions
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Random Forest OOF + fold-averaged test predictions")
print("Training feature matrix:", X_train_proc_model.shape)
print("Test feature matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature column count mismatch."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test columns are not aligned."
assert len(y_train) == len(X_train_proc_model), "y_train length does not match training features."
assert len(train_ids) == len(X_train_proc_model), "train_ids length does not match training features."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match test features."

# ------------------------------------------------------------
# RF settings from refined CV
# ------------------------------------------------------------

RF_OOF_N_SPLITS = 5
RF_OOF_N_ESTIMATORS = 500

rf_oof_params = {
    "n_estimators": RF_OOF_N_ESTIMATORS,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "max_features": 0.5,
    "bootstrap": True,
    "max_samples": None,
    "oob_score": True,
    "n_jobs": -1
}

print("\nRF OOF settings:")
print(rf_oof_params)

kf = KFold(
    n_splits=RF_OOF_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

X_all = X_train_proc_model
X_test_all = X_test_proc_model
y_all = np.asarray(y_train, dtype=float)

oof_pred = np.zeros(len(X_all), dtype=float)
test_pred_folds = np.zeros((len(X_test_all), RF_OOF_N_SPLITS), dtype=float)

fold_rows = []
fold_models = []

# ------------------------------------------------------------
# Train fold models
# ------------------------------------------------------------

for fold_id, (tr_idx, val_idx) in enumerate(kf.split(X_all), start=1):
    print(f"\n================ RF OOF Fold {fold_id} / {RF_OOF_N_SPLITS} ================")
    
    X_fold_train = X_all.iloc[tr_idx]
    X_fold_valid = X_all.iloc[val_idx]
    y_fold_train = y_all[tr_idx]
    y_fold_valid = y_all[val_idx]
    
    model = RandomForestRegressor(
        **rf_oof_params,
        random_state=RANDOM_STATE + fold_id
    )
    
    model.fit(X_fold_train, y_fold_train)
    
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    test_pred = model.predict(X_test_all)
    
    oof_pred[val_idx] = valid_pred
    test_pred_folds[:, fold_id - 1] = test_pred
    
    train_mse = mean_squared_error(y_fold_train, train_pred)
    valid_mse = mean_squared_error(y_fold_valid, valid_pred)
    
    oob_pred = model.oob_prediction_
    oob_mask = np.isfinite(oob_pred)
    if oob_mask.sum() > 0:
        oob_mse = mean_squared_error(y_fold_train[oob_mask], oob_pred[oob_mask])
    else:
        oob_mse = np.nan
    
    row = {
        "model_class": "Random Forest",
        "artifact": "oof_fold_ensemble",
        "fold": fold_id,
        "n_estimators": RF_OOF_N_ESTIMATORS,
        "max_features": 0.5,
        "max_samples": "None",
        "min_samples_leaf": 1,
        "min_samples_split": 2,
        "fold_train_mse": train_mse,
        "fold_oob_mse": oob_mse,
        "fold_valid_mse": valid_mse
    }
    
    fold_rows.append(row)
    fold_models.append(model)
    
    print(f"fold_train_mse={train_mse:.6f}")
    print(f"fold_oob_mse={oob_mse:.6f}")
    print(f"fold_valid_mse={valid_mse:.6f}")
    
    gc.collect()

# ------------------------------------------------------------
# Summarize OOF performance
# ------------------------------------------------------------

rf_oof_fold_results = pd.DataFrame(fold_rows)
rf_oof_mse = mean_squared_error(y_all, oof_pred)

test_pred_mean_raw = test_pred_folds.mean(axis=1)
test_pred_mean = np.clip(test_pred_mean_raw, 0, 100)

print("\nRF OOF fold results:")
print(rf_oof_fold_results.to_string(index=False))

print("\nOverall RF OOF MSE:", rf_oof_mse)

print("\nRaw fold-averaged test prediction summary:")
print(pd.Series(test_pred_mean_raw).describe().to_string())

print("\nClipped fold-averaged test prediction summary:")
print(pd.Series(test_pred_mean).describe().to_string())

# ------------------------------------------------------------
# Save OOF predictions and fold-averaged submission
# ------------------------------------------------------------

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

rf_oof_path = RESULTS_DIR / "oof_rf_500_base.csv"
rf_fold_test_path = RESULTS_DIR / "testpred_rf_500_base_folds.csv"
rf_fold_submission_path = Path("submission_rf_500_base_oof_foldavg.csv")

rf_oof_df = pd.DataFrame({
    "ASSESSMENT_ID": train_ids.astype(str),
    "y_true": y_all,
    "rf_500_base_oof_pred": oof_pred
})

rf_test_fold_df = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str)
})

for fold_id in range(RF_OOF_N_SPLITS):
    rf_test_fold_df[f"rf_500_base_fold{fold_id + 1}_pred"] = test_pred_folds[:, fold_id]

rf_test_fold_df["rf_500_base_foldavg_pred"] = test_pred_mean

rf_submission_oof = pd.DataFrame({
    "ASSESSMENT_ID": test_ids.astype(str),
    "PERCENT_PROFICIENT": test_pred_mean
})

rf_oof_df.to_csv(rf_oof_path, index=False)
rf_test_fold_df.to_csv(rf_fold_test_path, index=False)
rf_submission_oof.to_csv(rf_fold_submission_path, index=False)

print("\nSaved RF OOF predictions to:", rf_oof_path.resolve())
print("Saved RF fold test predictions to:", rf_fold_test_path.resolve())
print("Saved RF fold-averaged submission to:", rf_fold_submission_path.resolve())

print("\nSubmission shape:", rf_submission_oof.shape)
print("\nFirst 5 submission rows:")
print(rf_submission_oof.head().to_string(index=False))

# ------------------------------------------------------------
# Store artifact in dictionary for later stacking
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["Random Forest OOF 500 base"] = {
    "fold_models": fold_models,
    "oof_predictions": oof_pred,
    "test_fold_predictions": test_pred_folds,
    "test_fold_average": test_pred_mean,
    "oof_mse": float(rf_oof_mse),
    "fold_results": rf_oof_fold_results,
    "submission_path": str(rf_fold_submission_path),
    "notes": "OOF/fold-averaged RF artifact for later stacking."
}

elapsed = time.perf_counter() - start_time
print(f"\nElapsed time: {elapsed:.2f} seconds")

Random Forest OOF + fold-averaged test predictions
Training feature matrix: (144921, 162)
Test feature matrix: (48307, 162)

RF OOF settings:
{'n_estimators': 500, 'criterion': 'squared_error', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_features': 0.5, 'bootstrap': True, 'max_samples': None, 'oob_score': True, 'n_jobs': -1}

================ RF OOF Fold 1 / 5 ================


fold_train_mse=16.708704
fold_oob_mse=122.960428
fold_valid_mse=122.881281

================ RF OOF Fold 2 / 5 ================
fold_train_mse=16.725163
fold_oob_mse=123.335287
fold_valid_mse=120.142245

================ RF OOF Fold 3 / 5 ================
fold_train_mse=16.707048
fold_oob_mse=122.797429
fold_valid_mse=123.546977

================ RF OOF Fold 4 / 5 ================
fold_train_mse=16.806578
fold_oob_mse=123.720044
fold_valid_mse=119.761836

================ RF OOF Fold 5 / 5 ================
fold_train_mse=16.672325
fold_oob_mse=122.655667
fold_valid_mse=125.307764

RF OOF fold results:
  model_class          artifact  fold  n_estimators  max_features max_samples  min_samples_leaf  min_samples_split  fold_train_mse  fold_oob_mse  fold_valid_mse
Random Forest oof_fold_ensemble     1           500           0.5        None                 1                  2       16.708704    122.960428      122.881281
Random Forest oof_fold_ensemble     2           500           0.5    

### Random Forest OOF and Fold-Averaged Submission

A 5-fold random forest OOF artifact was generated using the best refined RF settings with 500 trees, `max_features = 0.5`, `max_samples = None`, `min_samples_leaf = 1`, and `min_samples_split = 2`.

The overall random forest OOF MSE was approximately **122.33**. The fold validation MSEs were stable, ranging from approximately **119.76** to **125.31**. This is close to the earlier refined RF holdout MSE of approximately **123.21**, suggesting that the random forest generalization estimate is reasonably stable.

A fold-averaged test prediction file was also saved as `submission_rf_500_base_oof_foldavg.csv`. This file can be submitted as a second RF-family benchmark, but its larger value is that the saved OOF predictions can later be used as a stacking feature.

### Next Stage Plan After Random Forest OOF

After the random forest OOF artifact finishes, the next goal is to continue extracting value from the tree-ensemble / bagging family before moving to boosting.

The immediate next candidate is ExtraTrees. ExtraTrees is related to random forests but adds more randomization in split selection. This can sometimes reduce variance or produce errors that differ from random forests, which may be useful later in stacking even if it is not the best single model.

We will continue saving:
1. model-family validation metrics,
2. out-of-fold predictions,
3. fold-averaged test predictions,
4. Kaggle submission files,
5. a central model scoreboard.

The external teammate LightGBM score remains the current public benchmark, but we will not merge or copy that notebook into this workflow. When we reach the boosting stage, it can be inspected for ideas only.

In [31]:
# ============================================================
# 17A. Model scoreboard and submission tracker utilities
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# Submission tracker
# Manually update public scores after Kaggle submissions.
# ------------------------------------------------------------

submission_tracker = pd.DataFrame([
    {
        "submission_file": "submission_oof_lgbm.csv",
        "source": "external teammate reference",
        "model_family": "LightGBM",
        "public_mse": 91.092,
        "notes": "Current external team benchmark; do not commingle code."
    },
    {
        "submission_file": "submission_rf_refined_300_base.csv",
        "source": "current notebook",
        "model_family": "Random Forest",
        "public_mse": 111.096,
        "notes": "Full-data refit from refined RF CV-selected settings."
    },
    {
        "submission_file": "submission_rf_500_base_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "Random Forest",
        "public_mse": np.nan,
        "notes": "Pending RF OOF/fold-average submission candidate."
    }
])

submission_tracker_path = RESULTS_DIR / "submission_tracker.csv"
submission_tracker.to_csv(submission_tracker_path, index=False)

print("Submission tracker:")
print(submission_tracker.to_string(index=False))
print("\nSaved to:", submission_tracker_path.resolve())


# ------------------------------------------------------------
# Best-model scoreboard from best_models_by_class
# ------------------------------------------------------------

def build_model_scoreboard(best_models_by_class):
    rows = []
    
    for model_key, obj in best_models_by_class.items():
        if not isinstance(obj, dict):
            continue
        
        row = {
            "model_key": model_key,
            "feature_space": obj.get("feature_space", None),
            "cv_valid_mse_mean": obj.get("cv_valid_mse_mean", np.nan),
            "cv_valid_mse_std": obj.get("cv_valid_mse_std", np.nan),
            "holdout_val_mse": obj.get("holdout_val_mse", np.nan),
            "oof_mse": obj.get("oof_mse", np.nan),
            "holdout_oob_mse": obj.get("holdout_oob_mse", np.nan),
            "n_estimators": obj.get("n_estimators", np.nan),
            "max_features": obj.get("max_features", np.nan),
            "max_samples": obj.get("max_samples", np.nan),
            "min_samples_leaf": obj.get("min_samples_leaf", np.nan),
            "min_samples_split": obj.get("min_samples_split", np.nan),
            "submission_path": obj.get("submission_path", None),
            "notes": obj.get("notes", None)
        }
        
        rows.append(row)
    
    scoreboard = pd.DataFrame(rows)
    
    if len(scoreboard) > 0:
        sort_cols = []
        for col in ["oof_mse", "cv_valid_mse_mean", "holdout_val_mse"]:
            if col in scoreboard.columns:
                sort_cols.append(col)
        
        if sort_cols:
            scoreboard = scoreboard.sort_values(sort_cols, na_position="last")
    
    return scoreboard


if "best_models_by_class" in globals():
    model_scoreboard = build_model_scoreboard(best_models_by_class)
    model_scoreboard_path = RESULTS_DIR / "model_scoreboard.csv"
    model_scoreboard.to_csv(model_scoreboard_path, index=False)
    
    print("\nModel scoreboard:")
    print(model_scoreboard.to_string(index=False))
    print("\nSaved to:", model_scoreboard_path.resolve())
else:
    print("\nbest_models_by_class does not exist yet. Run this cell again after model cells finish.")

Submission tracker:
                       submission_file                      source  model_family  public_mse                                                   notes
               submission_oof_lgbm.csv external teammate reference      LightGBM      91.092 Current external team benchmark; do not commingle code.
    submission_rf_refined_300_base.csv            current notebook Random Forest     111.096   Full-data refit from refined RF CV-selected settings.
submission_rf_500_base_oof_foldavg.csv            current notebook Random Forest         NaN       Pending RF OOF/fold-average submission candidate.

Saved to: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/submission_tracker.csv

Model scoreboard:
      model_key  feature_space  cv_valid_mse_mean  cv_valid_mse_std  holdout_val_mse  oof_mse  holdout_oob_mse  n_estimators  max_features  max_samples  min_samples_leaf  min_samples_split submission_path        

In [32]:
# Patch metadata for RF OOF artifact in best_models_by_class

best_models_by_class["Random Forest OOF 500 base"].update({
    "feature_space": "base",
    "n_estimators": 500,
    "max_features": 0.5,
    "max_samples": None,
    "min_samples_leaf": 1,
    "min_samples_split": 2
})

KeyError: 'Random Forest OOF 500 base'

### Kernel Crash Recovery Note

The ExtraTrees OOF screen (deleted) crashed the Jupyter kernel, likely because the cell was too memory- and worker-intensive for the current environment. The crashed ExtraTrees output should not be used.

The notebook was recovered by rerunning only the preprocessing cells needed to rebuild `X_train_proc_model`, `X_test_proc_model`, `y_train`, `train_ids`, and `test_ids`. Expensive model-search cells were not rerun. Saved random forest artifacts were reloaded from disk and will continue to be used for tracking and future stacking.

# rerun these at least since kernel crash

0.. Setup and raw data loading


1.. Merge datasets


4.. Build X and y + preserve IDs


5.. Column typing


6A. Frequency encoding (safe, no leakage)


6B. Missing indicators + median imputation (more robust than mean)


6C. Drop raw high-cardinality categorical columns


6D. One-hot encode low-cardinality categorical variables


Create modeling copy WITHOUT ID (non-destructive)

7A. Simple Linear Regression (one feature at a time)


Extract best simple linear regression feature properly


8A. Build controlled candidate feature spaces


12A. Step Function Models


12B. Expanded Step Function Search

In [51]:
print(feature_spaces.keys())
print(X_tr.shape, X_val.shape)
print(make_step_features)
print(get_top_continuous_features)

dict_keys(['base', 'base_plus_poly', 'base_plus_interactions', 'base_plus_poly_interactions'])
(115936, 162) (28985, 162)
<function make_step_features at 0x334eff690>
<function get_top_continuous_features at 0x334eff3d0>


### 18B. Safe ExtraTrees screen and selected OOF artifact

The previous ExtraTrees OOF screen crashed the kernel, so this cell uses a safer two-stage design.

First, it runs a small ExtraTrees holdout screen on the base feature space only. Then it selects the best holdout configuration and, only if the validation MSE is below the safety cutoff, runs one 5-fold OOF/fold-averaged test-prediction artifact.

Design choices:
- Use only the base feature space because RF already preferred base over engineered polynomial/interaction features.
- Use fewer trees than the failed ExtraTrees run.
- Use `n_jobs=1` to reduce worker/memory pressure.
- Do not store all candidate test predictions.
- Save checkpoints after each screen configuration and after each OOF fold.
- Delete fitted model objects after each fit.

In [52]:
# ============================================================
# 18B. Safe ExtraTrees holdout screen + selected OOF artifact
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

start_time = time.perf_counter()

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 9890

print("Safe ExtraTrees screen + selected OOF artifact")
print("Development split:", X_tr.shape, X_val.shape)
print("Full training matrix:", X_train_proc_model.shape)
print("Test matrix:", X_test_proc_model.shape)

# ------------------------------------------------------------
# Runtime controls
# ------------------------------------------------------------
# This is intentionally conservative after the previous kernel crash.
# For a shorter run, set RUN_ET_OOF_AFTER_SCREEN = False.
# To force OOF even after a weak screen, increase ET_OOF_MSE_CUTOFF.

ET_N_JOBS = 1
ET_SCREEN_N_ESTIMATORS = 180
ET_OOF_N_ESTIMATORS = 250
ET_OOF_N_SPLITS = 5
ET_OOF_MSE_CUTOFF = 145.0
RUN_ET_OOF_AFTER_SCREEN = True

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ET_SCREEN_RESULTS_PATH = RESULTS_DIR / "extratrees_safe_holdout_screen.csv"
ET_SCREEN_BEST_PATH = RESULTS_DIR / "extratrees_safe_holdout_best.csv"
ET_OOF_FOLD_RESULTS_PATH = RESULTS_DIR / "extratrees_safe_oof_fold_results.csv"
ET_OOF_SUMMARY_PATH = RESULTS_DIR / "extratrees_safe_oof_summary.csv"
ET_OOF_PRED_PATH = RESULTS_DIR / "oof_extratrees_safe_base.csv"
ET_TESTPRED_PATH = RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv"
ET_SUBMISSION_PATH = Path("submission_extratrees_safe_base_oof_foldavg.csv")

OVERWRITE_ET_RESULTS = True

if OVERWRITE_ET_RESULTS:
    for path in [
        ET_SCREEN_RESULTS_PATH,
        ET_SCREEN_BEST_PATH,
        ET_OOF_FOLD_RESULTS_PATH,
        ET_OOF_SUMMARY_PATH,
        ET_OOF_PRED_PATH,
        ET_TESTPRED_PATH,
        RESULTS_DIR / "extratrees_safe_oof_pred_partial.npy",
        RESULTS_DIR / "extratrees_safe_test_pred_sum_partial.npy",
    ]:
        if path.exists():
            path.unlink()
    if ET_SUBMISSION_PATH.exists():
        ET_SUBMISSION_PATH.unlink()

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert "base" in feature_spaces, "feature_spaces['base'] is missing."
assert list(feature_spaces["base"][0].columns) == list(X_tr.columns), "Base feature-space columns do not match X_tr."
assert list(feature_spaces["base"][1].columns) == list(X_val.columns), "Base feature-space columns do not match X_val."
assert X_train_proc_model.shape[1] == X_test_proc_model.shape[1], "Train/test feature counts do not match."
assert list(X_train_proc_model.columns) == list(X_test_proc_model.columns), "Train/test columns are not aligned."
assert len(y_train) == len(X_train_proc_model), "y_train length does not match X_train_proc_model."
assert len(test_ids) == len(X_test_proc_model), "test_ids length does not match X_test_proc_model."

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------

def append_checkpoint(df, path):
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def make_extratrees_model(config, random_state):
    params = {k: v for k, v in config.items() if k != "config_name"}
    return ExtraTreesRegressor(
        criterion="squared_error",
        max_depth=None,
        min_samples_split=2,
        n_jobs=ET_N_JOBS,
        random_state=random_state,
        **params
    )


def one_dim_id_array(ids):
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]
    return pd.Series(ids).astype(str).to_numpy()

# ------------------------------------------------------------
# Holdout screen on the already-restored development split
# ------------------------------------------------------------
# Keep this deliberately small. We only use the base feature space because
# random forest already preferred base, and ExtraTrees should also learn
# threshold structure without hand-built polynomial/interactions.

et_screen_configs = [
    {
        "config_name": "et_base_180_mf0.5_leaf1_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.7_leaf1_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.70,
        "min_samples_leaf": 1,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf2_no_bootstrap",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 2,
        "bootstrap": False,
    },
    {
        "config_name": "et_base_180_mf0.5_leaf1_bootstrap0.8",
        "n_estimators": ET_SCREEN_N_ESTIMATORS,
        "max_features": 0.50,
        "min_samples_leaf": 1,
        "bootstrap": True,
        "max_samples": 0.80,
    },
]

print("\nExtraTrees holdout screen settings:")
print("n_jobs:", ET_N_JOBS)
print("screen n_estimators:", ET_SCREEN_N_ESTIMATORS)
print("configs:", len(et_screen_configs))

X_screen_tr = X_tr
X_screen_val = X_val
y_screen_tr = np.asarray(y_tr, dtype=float).ravel()
y_screen_val = np.asarray(y_val, dtype=float).ravel()

et_screen_rows = []

for config_id, config in enumerate(et_screen_configs, start=1):
    config_start = time.perf_counter()
    print("\n" + "=" * 80)
    print(f"ExtraTrees holdout config {config_id} / {len(et_screen_configs)}")
    print(config)
    print("=" * 80)

    model = make_extratrees_model(
        config,
        random_state=RANDOM_STATE + 3000 + config_id
    )
    model.fit(X_screen_tr, y_screen_tr)

    train_pred = model.predict(X_screen_tr)
    val_pred = model.predict(X_screen_val)

    train_mse = mean_squared_error(y_screen_tr, train_pred)
    val_mse = mean_squared_error(y_screen_val, val_pred)
    elapsed = time.perf_counter() - config_start

    row = {
        "model_class": "ExtraTrees",
        "stage": "holdout_screen",
        "feature_space": "base",
        "config_name": config["config_name"],
        "n_estimators": config["n_estimators"],
        "max_features": config["max_features"],
        "min_samples_leaf": config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": config["bootstrap"],
        "max_samples": config.get("max_samples", np.nan),
        "train_mse": train_mse,
        "holdout_val_mse": val_mse,
        "elapsed_sec": elapsed,
    }

    et_screen_rows.append(row)
    append_checkpoint(pd.DataFrame([row]), ET_SCREEN_RESULTS_PATH)

    print(f"train_mse={train_mse:.6f}")
    print(f"holdout_val_mse={val_mse:.6f}")
    print(f"elapsed_sec={elapsed:.2f}")

    del model, train_pred, val_pred
    gc.collect()

et_screen_results = pd.DataFrame(et_screen_rows).sort_values("holdout_val_mse")
et_screen_results.to_csv(ET_SCREEN_RESULTS_PATH, index=False)

best_et_screen = et_screen_results.iloc[0].to_dict()
pd.DataFrame([best_et_screen]).to_csv(ET_SCREEN_BEST_PATH, index=False)

print("\n" + "=" * 80)
print("ExtraTrees holdout screen complete")
print("=" * 80)
print(et_screen_results.to_string(index=False))
print("\nBest holdout config:")
print(pd.DataFrame([best_et_screen]).to_string(index=False))
print("\nSaved screen results to:", ET_SCREEN_RESULTS_PATH.resolve())
print("Saved best screen result to:", ET_SCREEN_BEST_PATH.resolve())

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["ExtraTrees holdout screen safe base"] = {
    "model_class": "ExtraTrees",
    "feature_space": "base",
    "holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
    "train_mse": float(best_et_screen["train_mse"]),
    "n_estimators": int(best_et_screen["n_estimators"]),
    "max_features": best_et_screen["max_features"],
    "max_samples": best_et_screen["max_samples"],
    "min_samples_leaf": int(best_et_screen["min_samples_leaf"]),
    "min_samples_split": 2,
    "notes": "Safe ExtraTrees holdout screen after previous crash."
}

# ------------------------------------------------------------
# OOF artifact for the selected screen config only
# ------------------------------------------------------------

should_run_oof = (
    RUN_ET_OOF_AFTER_SCREEN
    and np.isfinite(best_et_screen["holdout_val_mse"])
    and best_et_screen["holdout_val_mse"] <= ET_OOF_MSE_CUTOFF
)

print("\nOOF gate:")
print("RUN_ET_OOF_AFTER_SCREEN:", RUN_ET_OOF_AFTER_SCREEN)
print("ET_OOF_MSE_CUTOFF:", ET_OOF_MSE_CUTOFF)
print("Best holdout MSE:", best_et_screen["holdout_val_mse"])
print("Will run OOF:", should_run_oof)

if should_run_oof:
    best_config_name = best_et_screen["config_name"]
    selected_config = [c for c in et_screen_configs if c["config_name"] == best_config_name][0].copy()
    selected_config["config_name"] = "et_safe_oof_from_" + best_config_name
    selected_config["n_estimators"] = ET_OOF_N_ESTIMATORS

    print("\nSelected OOF config:")
    print(selected_config)
    print("OOF folds:", ET_OOF_N_SPLITS)
    print("OOF n_estimators:", ET_OOF_N_ESTIMATORS)

    X_all = X_train_proc_model
    X_test_all = X_test_proc_model
    y_all = np.asarray(y_train, dtype=float).ravel()

    oof_pred = np.full(len(X_all), np.nan, dtype=float)
    test_pred_sum = np.zeros(len(X_test_all), dtype=float)

    fold_rows = []

    kf = KFold(
        n_splits=ET_OOF_N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for fold_id, (tr_idx, val_idx) in enumerate(kf.split(X_all), start=1):
        fold_start = time.perf_counter()
        print("\n" + "=" * 80)
        print(f"ExtraTrees OOF fold {fold_id} / {ET_OOF_N_SPLITS}")
        print("=" * 80)

        model = make_extratrees_model(
            selected_config,
            random_state=RANDOM_STATE + 4000 + fold_id
        )

        model.fit(X_all.iloc[tr_idx], y_all[tr_idx])

        valid_pred = model.predict(X_all.iloc[val_idx])
        test_pred = model.predict(X_test_all)

        oof_pred[val_idx] = valid_pred
        test_pred_sum += test_pred / ET_OOF_N_SPLITS

        valid_mse = mean_squared_error(y_all[val_idx], valid_pred)
        fold_elapsed = time.perf_counter() - fold_start

        row = {
            "model_class": "ExtraTrees",
            "stage": "oof_selected_config",
            "feature_space": "base",
            "fold": fold_id,
            "config_name": selected_config["config_name"],
            "n_estimators": selected_config["n_estimators"],
            "max_features": selected_config["max_features"],
            "min_samples_leaf": selected_config["min_samples_leaf"],
            "min_samples_split": 2,
            "bootstrap": selected_config["bootstrap"],
            "max_samples": selected_config.get("max_samples", np.nan),
            "fold_valid_mse": valid_mse,
            "elapsed_sec": fold_elapsed,
        }

        fold_rows.append(row)
        append_checkpoint(pd.DataFrame([row]), ET_OOF_FOLD_RESULTS_PATH)

        # Light checkpointing after each fold. These are small arrays, not model objects.
        np.save(RESULTS_DIR / "extratrees_safe_oof_pred_partial.npy", oof_pred)
        np.save(RESULTS_DIR / "extratrees_safe_test_pred_sum_partial.npy", test_pred_sum)

        print(f"fold_valid_mse={valid_mse:.6f}")
        print(f"elapsed_sec={fold_elapsed:.2f}")
        print("Partial OOF predictions filled:", np.isfinite(oof_pred).sum(), "/", len(oof_pred))

        del model, valid_pred, test_pred
        gc.collect()

    assert np.isfinite(oof_pred).all(), "OOF predictions contain missing values."

    et_oof_fold_results = pd.DataFrame(fold_rows)
    et_oof_fold_results.to_csv(ET_OOF_FOLD_RESULTS_PATH, index=False)

    et_oof_mse = mean_squared_error(y_all, oof_pred)
    test_pred_avg = np.clip(test_pred_sum, 0, 100)
    oof_pred_clipped = np.clip(oof_pred, 0, 100)

    train_id_values = one_dim_id_array(train_ids)
    test_id_values = one_dim_id_array(test_ids)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_values,
        "y_true": y_all,
        "oof_pred": oof_pred,
        "oof_pred_clipped": oof_pred_clipped,
    })
    oof_df.to_csv(ET_OOF_PRED_PATH, index=False)

    test_pred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_values,
        "PERCENT_PROFICIENT": test_pred_avg,
    })
    test_pred_df.to_csv(ET_TESTPRED_PATH, index=False)

    submission_et = test_pred_df.copy()
    assert submission_et.shape[0] == X_test_proc_model.shape[0], "Submission row count mismatch."
    assert list(submission_et.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"], "Submission columns are wrong."
    assert submission_et["ASSESSMENT_ID"].isna().sum() == 0, "Missing ASSESSMENT_ID in submission."
    assert submission_et["PERCENT_PROFICIENT"].isna().sum() == 0, "Missing predictions in submission."
    submission_et.to_csv(ET_SUBMISSION_PATH, index=False)

    total_elapsed = time.perf_counter() - start_time

    et_oof_summary = pd.DataFrame([{
        "model_key": "ExtraTrees OOF safe base",
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "screen_holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
        "oof_mse": float(et_oof_mse),
        "fold_valid_mse_mean": float(et_oof_fold_results["fold_valid_mse"].mean()),
        "fold_valid_mse_std": float(et_oof_fold_results["fold_valid_mse"].std()),
        "n_splits": ET_OOF_N_SPLITS,
        "n_estimators": ET_OOF_N_ESTIMATORS,
        "max_features": selected_config["max_features"],
        "min_samples_leaf": selected_config["min_samples_leaf"],
        "min_samples_split": 2,
        "bootstrap": selected_config["bootstrap"],
        "max_samples": selected_config.get("max_samples", np.nan),
        "n_jobs": ET_N_JOBS,
        "submission_path": str(ET_SUBMISSION_PATH),
        "elapsed_sec": total_elapsed,
    }])
    et_oof_summary.to_csv(ET_OOF_SUMMARY_PATH, index=False)

    best_models_by_class["ExtraTrees OOF safe base"] = {
        "model_class": "ExtraTrees",
        "feature_space": "base",
        "holdout_val_mse": float(best_et_screen["holdout_val_mse"]),
        "oof_mse": float(et_oof_mse),
        "cv_valid_mse_mean": float(et_oof_fold_results["fold_valid_mse"].mean()),
        "cv_valid_mse_std": float(et_oof_fold_results["fold_valid_mse"].std()),
        "n_estimators": ET_OOF_N_ESTIMATORS,
        "max_features": selected_config["max_features"],
        "max_samples": selected_config.get("max_samples", np.nan),
        "min_samples_leaf": int(selected_config["min_samples_leaf"]),
        "min_samples_split": 2,
        "submission_path": str(ET_SUBMISSION_PATH),
        "notes": "Selected safe ExtraTrees OOF artifact; test predictions are fold-averaged."
    }

    new_submission_row = {
        "submission_file": ET_SUBMISSION_PATH.name,
        "source": "current notebook",
        "model_family": "ExtraTrees",
        "public_mse": np.nan,
        "notes": "Safe ExtraTrees OOF/fold-average submission candidate."
    }

    if "submission_tracker" in globals():
        submission_tracker = pd.concat(
            [submission_tracker, pd.DataFrame([new_submission_row])],
            ignore_index=True
        )
        submission_tracker = submission_tracker.drop_duplicates(
            subset=["submission_file"],
            keep="last"
        )
    else:
        submission_tracker = pd.DataFrame([new_submission_row])

    submission_tracker.to_csv(RESULTS_DIR / "submission_tracker.csv", index=False)

    print("\n" + "=" * 80)
    print("ExtraTrees OOF artifact complete")
    print("=" * 80)
    print("Fold results:")
    print(et_oof_fold_results.to_string(index=False))
    print("\nOOF MSE:", et_oof_mse)
    print("\nTest prediction summary:")
    print(pd.Series(test_pred_avg).describe().to_string())
    print("\nSaved files:")
    print("OOF:", ET_OOF_PRED_PATH.resolve())
    print("Fold-avg test predictions:", ET_TESTPRED_PATH.resolve())
    print("Submission:", ET_SUBMISSION_PATH.resolve())
    print("OOF summary:", ET_OOF_SUMMARY_PATH.resolve())
    print("\nSubmission shape:", submission_et.shape)
    print("First 5 submission rows:")
    print(submission_et.head().to_string(index=False))
    print("\nTotal elapsed seconds:", total_elapsed)

else:
    total_elapsed = time.perf_counter() - start_time
    print("\nOOF was skipped by the safety gate.")
    print("The holdout screen is still saved and can be reviewed before deciding whether to run OOF.")
    print("Total elapsed seconds:", total_elapsed)

Safe ExtraTrees screen + selected OOF artifact
Development split: (115936, 162) (28985, 162)
Full training matrix: (144921, 162)
Test matrix: (48307, 162)

ExtraTrees holdout screen settings:
n_jobs: 1
screen n_estimators: 180
configs: 4

ExtraTrees holdout config 1 / 4
{'config_name': 'et_base_180_mf0.5_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse=0.025122
holdout_val_mse=111.627310
elapsed_sec=131.23

ExtraTrees holdout config 2 / 4
{'config_name': 'et_base_180_mf0.7_leaf1_no_bootstrap', 'n_estimators': 180, 'max_features': 0.7, 'min_samples_leaf': 1, 'bootstrap': False}
train_mse=0.025122
holdout_val_mse=111.874048
elapsed_sec=163.68

ExtraTrees holdout config 3 / 4
{'config_name': 'et_base_180_mf0.5_leaf2_no_bootstrap', 'n_estimators': 180, 'max_features': 0.5, 'min_samples_leaf': 2, 'bootstrap': False}
train_mse=19.159447
holdout_val_mse=119.026918
elapsed_sec=107.84

ExtraTrees holdout config 4 / 4
{'config_na

### ExtraTrees Kaggle submission result

The safe ExtraTrees model produced the best clean non-boosting result so far.

| Submission file | Model | Feature space | OOF MSE | Public leaderboard MSE |
|---|---|---|---:|---:|
| `submission_extratrees_safe_base_oof_foldavg.csv` | ExtraTreesRegressor | `base` | 110.7493 | 102.56 |

The selected ExtraTrees configuration used:
- `n_estimators = 250` for the OOF/fold-averaged artifact
- `max_features = 0.5`
- `min_samples_leaf = 1`
- `bootstrap = False`
- feature space: `base`

This improves substantially over the earlier clean Random Forest public benchmark of 111.096. The OOF MSE and public leaderboard MSE are not directly comparable because they are evaluated on different data, but both indicate that ExtraTrees is now the strongest clean bagging-family model in the notebook.

Next, we will test whether a lightweight OOF-weighted blend of Random Forest and ExtraTrees improves over pure ExtraTrees.

### 19A. OOF blend of Random Forest and ExtraTrees

ExtraTrees is now the strongest clean non-boosting model by OOF MSE, but the earlier Random Forest may still contain complementary signal. This cell below performs a lightweight OOF blend between the saved RF 500-base artifact and the new safe ExtraTrees artifact.

No new tree models are trained here. The cell only:
- loads saved OOF predictions,
- searches blend weights using OOF MSE,
- blends the matching test predictions,
- saves a new blend submission if the OOF blend is useful.

In [ ]:
# ============================================================
# 19A. OOF blend: RF 500 base + safe ExtraTrees base
# Full corrected replacement cell
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RESULTS_DIR = Path("model_results")
assert RESULTS_DIR.exists(), "model_results folder was not found."

# ------------------------------------------------------------
# Locate saved artifacts
# ------------------------------------------------------------

def first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n" +
        "\n".join(str(p) for p in candidates)
    )

rf_oof_path = first_existing_path(
    [
        RESULTS_DIR / "oof_rf_500_base.csv",
    ],
    "RF OOF file"
)

rf_test_path = first_existing_path(
    [
        RESULTS_DIR / "testpred_rf_500_base_folds.csv",
        RESULTS_DIR / "testpred_rf_500_base_foldavg.csv",
        RESULTS_DIR / "testpred_rf_500_base.csv",
    ],
    "RF test-prediction file"
)

et_oof_path = first_existing_path(
    [
        RESULTS_DIR / "oof_extratrees_safe_base.csv",
    ],
    "ExtraTrees OOF file"
)

et_test_path = first_existing_path(
    [
        RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    ],
    "ExtraTrees test-prediction file"
)

print("Using files:")
print("RF OOF:       ", rf_oof_path)
print("RF testpred:  ", rf_test_path)
print("ET OOF:       ", et_oof_path)
print("ET testpred:  ", et_test_path)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})


def one_dim_id_array(ids):
    if isinstance(ids, pd.DataFrame):
        ids = ids.iloc[:, 0]
    return pd.Series(ids).astype(str).to_numpy()


def pick_prediction_column(df, preferred_cols, label):
    """
    Pick a prediction column robustly.
    Handles custom names such as rf_500_base_oof_pred.
    """
    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = {"ASSESSMENT_ID", "y_true"}

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "fold" in c.lower()
        )
    ]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    priority_terms = [
        "percent_proficient",
        "foldavg",
        "fold_avg",
        "average",
        "avg",
        "test_pred",
        "oof_pred",
        "oof",
        "pred",
    ]

    for term in priority_terms:
        matches = [c for c in pred_like_cols if term in c.lower()]
        if len(matches) == 1:
            return matches[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns available: {list(df.columns)}\n"
        f"Numeric columns found: {numeric_cols}\n"
        f"Prediction-like columns found: {pred_like_cols}"
    )


def standardize_oof(df, model_name):
    assert "ASSESSMENT_ID" in df.columns, f"{model_name} OOF missing ASSESSMENT_ID."
    assert "y_true" in df.columns, f"{model_name} OOF missing y_true."

    pred_col = pick_prediction_column(
        df,
        preferred_cols=[
            "oof_pred",
            "rf_500_base_oof_pred",
            "et_oof_pred",
            "oof_pred_clipped",
            "pred",
            "prediction",
        ],
        label=f"{model_name} OOF"
    )

    print(f"{model_name.upper()} OOF prediction column used:", pred_col)

    out = df[["ASSESSMENT_ID", "y_true", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={pred_col: f"{model_name}_oof_pred"})
    return out


def standardize_testpred(df, model_name):
    assert "ASSESSMENT_ID" in df.columns, f"{model_name} test predictions missing ASSESSMENT_ID."

    # First try to pick a clear aggregate prediction column.
    try:
        pred_col = pick_prediction_column(
            df,
            preferred_cols=[
                "PERCENT_PROFICIENT",
                "test_pred",
                "test_pred_avg",
                "foldavg_pred",
                "fold_avg_pred",
                "rf_500_base_test_pred",
                "rf_500_base_test_pred_foldavg",
                "rf_500_base_foldavg_pred",
                "et_test_pred",
                "pred",
                "prediction",
            ],
            label=f"{model_name} test predictions"
        )

        print(f"{model_name.upper()} test prediction column used:", pred_col)

        out = df[["ASSESSMENT_ID", pred_col]].copy()
        out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
        out = out.rename(columns={pred_col: f"{model_name}_test_pred"})
        return out

    except ValueError:
        # If the RF test file contains only fold columns and no aggregate,
        # average fold prediction columns automatically.
        excluded_cols = {"ASSESSMENT_ID", "y_true"}
        numeric_cols = [
            c for c in df.columns
            if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
        ]

        fold_cols = [
            c for c in numeric_cols
            if "fold" in c.lower()
        ]

        if len(fold_cols) >= 2:
            print(f"{model_name.upper()} test prediction columns averaged:", fold_cols)

            out = df[["ASSESSMENT_ID"]].copy()
            out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
            out[f"{model_name}_test_pred"] = df[fold_cols].mean(axis=1)
            return out

        raise

# ------------------------------------------------------------
# Load and align OOF predictions
# ------------------------------------------------------------

rf_oof_raw = read_pred_csv(rf_oof_path)
et_oof_raw = read_pred_csv(et_oof_path)

print("\nRF OOF columns:")
print(list(rf_oof_raw.columns))

print("\nExtraTrees OOF columns:")
print(list(et_oof_raw.columns))

rf_oof = standardize_oof(rf_oof_raw, "rf")
et_oof = standardize_oof(et_oof_raw, "et")

assert rf_oof["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in RF OOF."
assert et_oof["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in ExtraTrees OOF."

oof_blend_df = rf_oof.merge(
    et_oof,
    on="ASSESSMENT_ID",
    how="inner",
    suffixes=("_rf", "_et")
)

assert len(oof_blend_df) == len(rf_oof) == len(et_oof), "OOF merge lost rows."

assert np.allclose(
    oof_blend_df["y_true_rf"].to_numpy(dtype=float),
    oof_blend_df["y_true_et"].to_numpy(dtype=float)
), "RF and ExtraTrees y_true values do not match after merge."

oof_blend_df["y_true"] = oof_blend_df["y_true_rf"].astype(float)
oof_blend_df = oof_blend_df.drop(columns=["y_true_rf", "y_true_et"])

y_oof = oof_blend_df["y_true"].to_numpy(dtype=float)
rf_oof_pred = oof_blend_df["rf_oof_pred"].to_numpy(dtype=float)
et_oof_pred = oof_blend_df["et_oof_pred"].to_numpy(dtype=float)

rf_oof_mse = mean_squared_error(y_oof, np.clip(rf_oof_pred, 0, 100))
et_oof_mse = mean_squared_error(y_oof, np.clip(et_oof_pred, 0, 100))

print("\nComponent OOF MSE:")
print(f"RF 500 base:        {rf_oof_mse:.6f}")
print(f"ExtraTrees safe:    {et_oof_mse:.6f}")

# ------------------------------------------------------------
# OOF weight search
# ------------------------------------------------------------
# w_et = 1 means pure ExtraTrees.
# w_et = 0 means pure Random Forest.

weight_rows = []

for w_et in np.linspace(0, 1, 1001):
    w_rf = 1.0 - w_et

    blend_pred_raw = w_rf * rf_oof_pred + w_et * et_oof_pred
    blend_pred_clipped = np.clip(blend_pred_raw, 0, 100)

    weight_rows.append({
        "w_rf": w_rf,
        "w_et": w_et,
        "oof_mse_raw": mean_squared_error(y_oof, blend_pred_raw),
        "oof_mse_clipped": mean_squared_error(y_oof, blend_pred_clipped),
    })

blend_weight_results = pd.DataFrame(weight_rows)
blend_weight_results = blend_weight_results.sort_values("oof_mse_clipped").reset_index(drop=True)

best_blend = blend_weight_results.iloc[0].to_dict()
best_w_rf = float(best_blend["w_rf"])
best_w_et = float(best_blend["w_et"])

print("\nBest OOF blend:")
print(pd.DataFrame([best_blend]).to_string(index=False))

print("\nTop 10 blend weights:")
print(blend_weight_results.head(10).to_string(index=False))

# Save OOF blend details
oof_blend_df["blend_pred_raw"] = best_w_rf * rf_oof_pred + best_w_et * et_oof_pred
oof_blend_df["blend_pred_clipped"] = np.clip(oof_blend_df["blend_pred_raw"], 0, 100)

weight_results_path = RESULTS_DIR / "blend_rf500_extratrees_safe_weight_screen.csv"
oof_blend_path = RESULTS_DIR / "oof_blend_rf500_extratrees_safe.csv"

blend_weight_results.to_csv(weight_results_path, index=False)
oof_blend_df.to_csv(oof_blend_path, index=False)

# ------------------------------------------------------------
# Load and align test predictions
# ------------------------------------------------------------

rf_test_raw = read_pred_csv(rf_test_path)
et_test_raw = read_pred_csv(et_test_path)

print("\nRF test prediction columns:")
print(list(rf_test_raw.columns))

print("\nExtraTrees test prediction columns:")
print(list(et_test_raw.columns))

rf_test = standardize_testpred(rf_test_raw, "rf")
et_test = standardize_testpred(et_test_raw, "et")

assert rf_test["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in RF test predictions."
assert et_test["ASSESSMENT_ID"].duplicated().sum() == 0, "Duplicate ASSESSMENT_ID in ExtraTrees test predictions."

test_blend_df = rf_test.merge(et_test, on="ASSESSMENT_ID", how="inner")
assert len(test_blend_df) == len(rf_test) == len(et_test), "Test prediction merge lost rows."

# Put submission back in original test_ids order when available.
if "test_ids" in globals():
    test_order = pd.DataFrame({
        "ASSESSMENT_ID": one_dim_id_array(test_ids),
        "_test_order": np.arange(len(test_ids))
    })

    test_blend_df = test_order.merge(test_blend_df, on="ASSESSMENT_ID", how="left")

    assert test_blend_df["rf_test_pred"].isna().sum() == 0, "Missing RF predictions after test_ids alignment."
    assert test_blend_df["et_test_pred"].isna().sum() == 0, "Missing ET predictions after test_ids alignment."

    test_blend_df = test_blend_df.sort_values("_test_order").drop(columns=["_test_order"])

test_blend_df["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf * test_blend_df["rf_test_pred"].to_numpy(dtype=float)
    + best_w_et * test_blend_df["et_test_pred"].to_numpy(dtype=float),
    0,
    100
)

test_blend_path = RESULTS_DIR / "testpred_blend_rf500_extratrees_safe.csv"
submission_blend_path = Path("submission_blend_rf500_extratrees_safe_oof_weighted.csv")

test_blend_df.to_csv(test_blend_path, index=False)

submission_blend = test_blend_df[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()

expected_test_n = len(test_ids) if "test_ids" in globals() else 48307

assert submission_blend.shape[0] == expected_test_n, f"Submission row count is not {expected_test_n}."
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

# ------------------------------------------------------------
# Update trackers if they exist
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["RF500 + ExtraTrees safe OOF blend"] = {
    "model_class": "OOF weighted blend",
    "component_models": "RF 500 base + ExtraTrees safe base",
    "feature_space": "base",
    "oof_mse": float(best_blend["oof_mse_clipped"]),
    "oof_mse_raw": float(best_blend["oof_mse_raw"]),
    "w_rf": best_w_rf,
    "w_et": best_w_et,
    "submission_path": str(submission_blend_path),
    "notes": "Lightweight OOF-tuned blend of saved RF and ExtraTrees artifacts."
}

et_submission_row = {
    "submission_file": "submission_extratrees_safe_base_oof_foldavg.csv",
    "source": "current notebook",
    "model_family": "ExtraTrees",
    "public_mse": 102.56,
    "notes": "Safe ExtraTrees OOF/fold-average submission. Kaggle public leaderboard MSE = 102.56."
}

blend_submission_row = {
    "submission_file": submission_blend_path.name,
    "source": "current notebook",
    "model_family": "RF + ExtraTrees blend",
    "public_mse": np.nan,
    "notes": f"OOF-weighted blend; w_rf={best_w_rf:.3f}, w_et={best_w_et:.3f}, OOF MSE={best_blend['oof_mse_clipped']:.6f}."
}

tracker_new_rows = pd.DataFrame([et_submission_row, blend_submission_row])

if "submission_tracker" in globals():
    submission_tracker = pd.concat(
        [submission_tracker, tracker_new_rows],
        ignore_index=True
    )
    submission_tracker = submission_tracker.drop_duplicates(
        subset=["submission_file"],
        keep="last"
    )
else:
    submission_tracker = tracker_new_rows.copy()

submission_tracker.to_csv(RESULTS_DIR / "submission_tracker.csv", index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nSaved files:")
print("Weight screen:", weight_results_path.resolve())
print("OOF blend:", oof_blend_path.resolve())
print("Test blend predictions:", test_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())
print("Updated submission tracker:", (RESULTS_DIR / "submission_tracker.csv").resolve())

print("\nBlend test prediction summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nSubmission shape:", submission_blend.shape)

print("\nFirst 5 submission rows:")
print(submission_blend.head().to_string(index=False))

print("\nDecision note:")
if best_w_et >= 0.999:
    print("Best OOF blend is effectively pure ExtraTrees. Prioritize the pure ExtraTrees submission.")
elif best_blend["oof_mse_clipped"] < et_oof_mse:
    print("Blend improves OOF over ExtraTrees alone. This blend submission is worth trying after the pure ExtraTrees submission.")
else:
    print("Blend did not improve OOF over ExtraTrees alone. Prefer the pure ExtraTrees submission.")

Using files:
RF OOF:        model_results/oof_rf_500_base.csv
RF testpred:   model_results/testpred_rf_500_base_folds.csv
ET OOF:        model_results/oof_extratrees_safe_base.csv
ET testpred:   model_results/testpred_extratrees_safe_base_foldavg.csv

RF OOF columns:
['ASSESSMENT_ID', 'y_true', 'rf_500_base_oof_pred']

ExtraTrees OOF columns:
['ASSESSMENT_ID', 'y_true', 'oof_pred', 'oof_pred_clipped']
RF OOF prediction column used: rf_500_base_oof_pred
ET OOF prediction column used: oof_pred

Component OOF MSE:
RF 500 base:        122.328024
ExtraTrees safe:    110.749294

Best OOF blend:
 w_rf  w_et  oof_mse_raw  oof_mse_clipped
0.152 0.848   110.363092       110.363092

Top 10 blend weights:
 w_rf  w_et  oof_mse_raw  oof_mse_clipped
0.152 0.848   110.363092       110.363092
0.153 0.847   110.363099       110.363099
0.151 0.849   110.363118       110.363118
0.154 0.846   110.363139       110.363139
0.150 0.850   110.363178       110.363178
0.155 0.845   110.363212       110.363212
0.1

### RF + ExtraTrees OOF-weighted blend

After the safe ExtraTrees model became the strongest clean bagging-family model, we tested a lightweight OOF-weighted blend with the earlier RF 500-base OOF artifact.

No new models were trained in this step. The cell loaded saved OOF and test-prediction files, searched weights from 0 to 1, and selected the weight with the lowest OOF MSE.

| Model / blend | OOF MSE |
|---|---:|
| RF 500 base | 122.3280 |
| ExtraTrees safe base | 110.7493 |
| RF + ExtraTrees OOF blend | 110.3631 |

Best blend weights:

| Component | Weight |
|---|---:|
| RF 500 base | 0.152 |
| ExtraTrees safe base | 0.848 |

The blend improves OOF MSE by about 0.386 relative to pure ExtraTrees. This is a modest improvement, but it suggests that the RF predictions contain some complementary signal even though RF is weaker as a standalone model.

The generated blend submission file is:

`submission_blend_rf500_extratrees_safe_oof_weighted.csv`

This submission should be tested on Kaggle, but the public leaderboard result may differ from the OOF ranking because the public test split is a different evaluation set.

### Blend submission decision

The RF + ExtraTrees OOF-weighted blend improved OOF MSE only modestly:

| Candidate | OOF MSE |
|---|---:|
| ExtraTrees safe base | 110.7493 |
| RF + ExtraTrees blend | 110.3631 |

The OOF improvement is approximately 0.386 MSE points. Because Kaggle submissions are limited to 2 per day and there are 12 days remaining, this improvement is too small to justify spending a leaderboard submission slot immediately.

Decision:
- Do not submit `submission_blend_rf500_extratrees_safe_oof_weighted.csv` for now.
- Keep it saved as a fallback candidate.
- Use future Kaggle slots only for models or blends with materially stronger offline evidence, preferably from a new model family or a larger OOF improvement.

Current clean public leaderboard benchmark:
- `submission_extratrees_safe_base_oof_foldavg.csv`
- Public MSE: 102.56

In [33]:
# ============================================================
# Post-crash recovery diagnostic: no model fitting, no LightGBM
# ============================================================

from pathlib import Path
import sys
import platform
import gc

import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("scikit-learn:", sklearn.__version__)
print("Current working directory:", Path.cwd())

print("\nCore object checks:")
for name in [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "train_ids",
    "test_ids",
    "X_tr",
    "X_val",
    "y_tr",
    "y_val",
    "feature_spaces",
    "make_step_features",
    "get_top_continuous_features",
]:
    exists = name in globals()
    obj = globals().get(name, None)
    shape = getattr(obj, "shape", None)
    print(f"{name:28s} exists={str(exists):5s} shape={shape}")

print("\nFeature spaces:")
if "feature_spaces" in globals():
    print(feature_spaces.keys())

print("\nRecovery helper functions:")
print("make_step_features:", globals().get("make_step_features", None))
print("get_top_continuous_features:", globals().get("get_top_continuous_features", None))

print("\nSaved model_results artifacts:")
results_dir = Path("model_results")
print("model_results exists:", results_dir.exists())
if results_dir.exists():
    for path in sorted(results_dir.glob("*")):
        print(" -", path.name)

gc.collect()

Python: 3.14.2
Platform: macOS-15.6-arm64-arm-64bit-Mach-O
scikit-learn: 1.8.0
Current working directory: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart

Core object checks:
X_train_proc_model           exists=True  shape=(144921, 162)
X_test_proc_model            exists=True  shape=(48307, 162)
y_train                      exists=True  shape=(144921,)
train_ids                    exists=True  shape=(144921,)
test_ids                     exists=True  shape=(48307,)
X_tr                         exists=True  shape=(115936, 162)
X_val                        exists=True  shape=(28985, 162)
y_tr                         exists=True  shape=(115936,)
y_val                        exists=True  shape=(28985,)
feature_spaces               exists=True  shape=None
make_step_features           exists=True  shape=None
get_top_continuous_features  exists=True  shape=None

Feature spaces:
dict_keys(['base', 'base_plus_poly', 'base_plus_interactio

1795

In [60]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


### Post-crash status checkpoint

The kernel recovery check passed. The core preprocessing objects, feature spaces, train/validation split, and saved model artifacts are available. LightGBM imports successfully, but fitting stability has not yet been verified after the crash. Before rerunning any LightGBM or other expensive model, we inspect saved result files and continue with small, checkpointed cells only.

In [34]:
# ============================================================
# Inspect saved result artifacts after LightGBM crash/recovery
# No model fitting in this cell.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path("model_results")

files_to_check = [
    "model_scoreboard.csv",
    "submission_tracker.csv",
    "lgbm_clean_base_holdout_screen.csv",
    "extratrees_safe_oof_summary.csv",
    "extratrees_safe_oof_fold_results.csv",
    "oof_blend_rf500_extratrees_safe.csv",
    "blend_rf500_extratrees_safe_weight_screen.csv",
]

for fname in files_to_check:
    path = results_dir / fname
    print("\n" + "=" * 80)
    print(fname)
    print("exists:", path.exists())

    if not path.exists():
        continue

    df = pd.read_csv(path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))

    # Try to show the most informative ordering without assuming exact column names.
    sort_candidates = [
        "public_mse",
        "oof_mse",
        "valid_mse",
        "cv_valid_mse_mean",
        "holdout_mse",
        "mse",
    ]

    sort_col = None
    for col in sort_candidates:
        if col in df.columns:
            sort_col = col
            break

    if sort_col is not None:
        print(f"sorted by: {sort_col}")
        display(df.sort_values(sort_col).head(10))
    else:
        display(df.head(10))


model_scoreboard.csv
exists: True
shape: (2, 14)
columns: ['model_key', 'feature_space', 'cv_valid_mse_mean', 'cv_valid_mse_std', 'holdout_val_mse', 'oof_mse', 'holdout_oob_mse', 'n_estimators', 'max_features', 'max_samples', 'min_samples_leaf', 'min_samples_split', 'submission_path', 'notes']
sorted by: oof_mse


,model_key,feature_space,cv_valid_mse_mean,cv_valid_mse_std,holdout_val_mse,oof_mse,holdout_oob_mse,n_estimators,max_features,max_samples,min_samples_leaf,min_samples_split,submission_path,notes
0,Random Forest,base,149.549571,0.282329,127.479635,NaN,129.17549,120.0,0.5,0.8,1,2,NaN,First controlled Random Forest CV screen on ba...
1,Regression Tree,base_plus_poly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25,50,NaN,Single regression tree baseline across control...



submission_tracker.csv
exists: True
shape: (3, 5)
columns: ['submission_file', 'source', 'model_family', 'public_mse', 'notes']
sorted by: public_mse


,submission_file,source,model_family,public_mse,notes
0,submission_oof_lgbm.csv,external teammate reference,LightGBM,91.092,Current external team benchmark; do not commin...
1,submission_rf_refined_300_base.csv,current notebook,Random Forest,111.096,Full-data refit from refined RF CV-selected se...
2,submission_rf_500_base_oof_foldavg.csv,current notebook,Random Forest,NaN,Pending RF OOF/fold-average submission candidate.



lgbm_clean_base_holdout_screen.csv
exists: True
shape: (2, 19)
columns: ['model_class', 'backend', 'stage', 'feature_space', 'config_name', 'train_mse_clipped', 'holdout_val_mse_clipped', 'n_iter_used', 'elapsed_sec', 'n_estimators', 'learning_rate', 'num_leaves', 'min_child_samples', 'subsample', 'subsample_freq', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'max_depth']


...

,model_class,stage,feature_space,fold,config_name,n_estimators,max_features,min_samples_leaf,min_samples_split,bootstrap,max_samples,fold_valid_mse,elapsed_sec
0,ExtraTrees,oof_selected_config,base,1,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,111.639330,169.659820
1,ExtraTrees,oof_selected_config,base,2,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,109.155334,166.818236
2,ExtraTrees,oof_selected_config,base,3,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,111.870290,165.977457
3,ExtraTrees,oof_selected_config,base,4,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,108.405409,166.278418
4,ExtraTrees,oof_selected_config,base,5,et_safe_oof_from_et_base_180_mf0.5_leaf1_no_bo...,250,0.5,1,2,False,NaN,112.676076,166.069512



oof_blend_rf500_extratrees_safe.csv
exists: True
shape: (144921, 6)
columns: ['ASSESSMENT_ID', 'rf_oof_pred', 'et_oof_pred', 'y_true', 'blend_pred_raw', 'blend_pred_clipped']


,ASSESSMENT_ID,rf_oof_pred,et_oof_pred,y_true,blend_pred_raw,blend_pred_clipped
0,3b6deef53665,36.480,37.668,38.0,37.487424,37.487424
1,962a3bfbfe84,60.716,61.660,65.0,61.516512,61.516512
2,ffe086287b6e,73.440,74.560,65.0,74.389760,74.389760
3,e6f80847409d,51.054,51.868,52.0,51.744272,51.744272
4,676cc6d81961,49.046,49.600,46.0,49.515792,49.515792
5,c9292c7cecae,63.482,67.844,74.0,67.180976,67.180976
6,121afff188a5,10.578,11.476,13.0,11.339504,11.339504
7,2611f38e41a1,77.030,77.668,80.0,77.571024,77.571024
8,fc3217bcfd05,86.298,87.172,91.0,87.039152,87.039152
9,0394ccf86692,60.324,53.588,56.0,54.611872,54.611872



blend_rf500_extratrees_safe_weight_screen.csv
exists: True
shape: (1001, 4)
columns: ['w_rf', 'w_et', 'oof_mse_raw', 'oof_mse_clipped']


,w_rf,w_et,oof_mse_raw,oof_mse_clipped
0,0.152,0.848,110.363092,110.363092
1,0.153,0.847,110.363099,110.363099
2,0.151,0.849,110.363118,110.363118
3,0.154,0.846,110.363139,110.363139
4,0.150,0.850,110.363178,110.363178
5,0.155,0.845,110.363212,110.363212
6,0.149,0.851,110.363272,110.363272
7,0.156,0.844,110.363318,110.363318
8,0.148,0.852,110.363398,110.363398
9,0.157,0.843,110.363458,110.363458


### LightGBM post-crash interpretation

A saved LightGBM holdout-screen artifact exists and contains two completed base-feature-space fits. The better saved configuration reached holdout validation MSE around 114.08, which is useful but not stronger than the current ExtraTrees / RF+ExtraTrees artifacts. Because the previous kernel crash occurred around the LightGBM installation/fitting stage, we will not rerun a full LightGBM screen yet. First we run a small, single-threaded smoke fit with early stopping to confirm that LightGBM fitting is stable in the current kernel.

### 20A. Clean boosting screen and selected OOF artifact

We now move from bagging-style tree ensembles to boosting.

This cell is a clean implementation from the current notebook pipeline. It does not use or copy any teammate/reference notebook code. If `lightgbm` is installed, it will run a small LightGBM screen. If `lightgbm` is not installed, it will fall back to scikit-learn's `HistGradientBoostingRegressor`.

Design:
- Use the `base` feature space because Random Forest and ExtraTrees both preferred the base feature matrix.
- Run a small holdout screen first.
- Select the best boosting configuration by holdout validation MSE.
- Run a 5-fold OOF/fold-averaged test artifact only if the screen is competitive enough.
- Save OOF predictions and a submission candidate for later use.
- Do not spend a Kaggle submission slot yet unless offline evidence is materially strong.

In [35]:
# ============================================================
# LightGBM post-crash smoke test
# Small sample, single-threaded, early stopping, no submission.
# ============================================================

from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

RNG_SEED = globals().get("RANDOM_STATE", 9890)
rng = np.random.default_rng(RNG_SEED)

def take_rows(X, idx):
    """Works for both pandas objects and numpy arrays."""
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

# Keep this intentionally small so we test stability without stressing the kernel.
n_train_smoke = min(20_000, X_tr.shape[0])
n_val_smoke = min(8_000, X_val.shape[0])

tr_idx = rng.choice(X_tr.shape[0], size=n_train_smoke, replace=False)
val_idx = rng.choice(X_val.shape[0], size=n_val_smoke, replace=False)

X_smoke = np.asarray(take_rows(X_tr, tr_idx), dtype=np.float32)
y_smoke = np.asarray(take_rows(y_tr, tr_idx), dtype=np.float32).ravel()

X_smoke_val = np.asarray(take_rows(X_val, val_idx), dtype=np.float32)
y_smoke_val = np.asarray(take_rows(y_val, val_idx), dtype=np.float32).ravel()

print("LightGBM version:", lgb.__version__)
print("Smoke train shape:", X_smoke.shape)
print("Smoke validation shape:", X_smoke_val.shape)

smoke_params = dict(
    objective="regression",
    metric="l2",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
    random_state=RNG_SEED,
    n_jobs=1,              # important: avoid worker/thread pressure after crash
    verbosity=-1,
    force_col_wise=True,
)

start = time.time()

smoke_model = lgb.LGBMRegressor(**smoke_params)

smoke_model.fit(
    X_smoke,
    y_smoke,
    eval_set=[(X_smoke_val, y_smoke_val)],
    eval_metric="l2",
    callbacks=[
        lgb.early_stopping(stopping_rounds=40, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)

elapsed = time.time() - start

best_iter = getattr(smoke_model, "best_iteration_", None)
pred_raw = smoke_model.predict(X_smoke_val, num_iteration=best_iter)
pred_clipped = np.clip(pred_raw, 0, 100)

smoke_summary = pd.DataFrame([{
    "model_class": "LightGBM",
    "stage": "post_crash_smoke_test",
    "feature_space": "base",
    "n_train_smoke": n_train_smoke,
    "n_val_smoke": n_val_smoke,
    "best_iteration": best_iter,
    "val_mse_raw": mean_squared_error(y_smoke_val, pred_raw),
    "val_mse_clipped": mean_squared_error(y_smoke_val, pred_clipped),
    "elapsed_sec": elapsed,
    **smoke_params,
}])

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

smoke_path = results_dir / "lgbm_post_crash_smoke_test.csv"
smoke_summary.to_csv(smoke_path, index=False)

print("\nSaved:", smoke_path)
display(smoke_summary)

# Clean up immediately.
del smoke_model, X_smoke, y_smoke, X_smoke_val, y_smoke_val, pred_raw, pred_clipped
gc.collect()

LightGBM version: 4.6.0
Smoke train shape: (20000, 162)
Smoke validation shape: (8000, 162)
Training until validation scores don't improve for 40 rounds
[50]	valid_0's l2: 302.248
[100]	valid_0's l2: 254.695
[150]	valid_0's l2: 238.767
[200]	valid_0's l2: 228.813
[250]	valid_0's l2: 222.504
[300]	valid_0's l2: 217.859
[350]	valid_0's l2: 213.881
[400]	valid_0's l2: 210.339
Did not meet early stopping. Best iteration is:
[400]	valid_0's l2: 210.339


/Users/saadmanchowdhury/Desktop/All Github projects/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Saved: model_results/lgbm_post_crash_smoke_test.csv


,model_class,stage,feature_space,n_train_smoke,n_val_smoke,best_iteration,val_mse_raw,val_mse_clipped,elapsed_sec,objective,metric,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,max_depth,random_state,n_jobs,verbosity,force_col_wise
0,LightGBM,post_crash_smoke_test,base,20000,8000,400,210.339065,209.938381,1.535752,regression,l2,400,0.05,31,80,0.8,1,0.8,0.0,5.0,-1,9890,1,-1,True


1728

In [36]:
# ============================================================
# Controlled LightGBM holdout screen
# Full X_tr / X_val split, base feature space only.
# Single-threaded, early stopping, checkpoint after each config.
# No submission and no OOF artifacts in this cell.
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RNG_SEED = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

screen_path = results_dir / "lgbm_controlled_base_holdout_screen.csv"

# Use NumPy arrays consistently for fit and predict.
# float32 reduces memory pressure and is fine for tree models.
X_train_lgb = np.ascontiguousarray(np.asarray(X_tr, dtype=np.float32))
X_val_lgb = np.ascontiguousarray(np.asarray(X_val, dtype=np.float32))
y_train_lgb = np.asarray(y_tr, dtype=np.float32).ravel()
y_val_lgb = np.asarray(y_val, dtype=np.float32).ravel()

print("LightGBM version:", lgb.__version__)
print("Train shape:", X_train_lgb.shape)
print("Validation shape:", X_val_lgb.shape)
print("Checkpoint file:", screen_path)

common_params = dict(
    objective="regression",
    metric="l2",
    random_state=RNG_SEED,
    n_jobs=1,                 # keep conservative after kernel crash
    verbosity=-1,
    force_col_wise=True,
)

lgbm_configs = [
    dict(
        config_name="lgbm_c01_lr03_l63_child80_l2_5_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=5.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c02_lr03_l63_child120_l2_10_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=120,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=10.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c03_lr02_l63_child80_l2_10_sub085_col085",
        n_estimators=3500,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=10.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_c04_lr03_l127_child120_l2_20_sub08_col08",
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=127,
        min_child_samples=120,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=20.0,
        max_depth=-1,
    ),
]

if screen_path.exists():
    screen_df = pd.read_csv(screen_path)
    if "status" in screen_df.columns:
        done_configs = set(
            screen_df.loc[
                screen_df["status"].eq("completed"),
                "config_name"
            ].astype(str)
        )
    elif "config_name" in screen_df.columns:
        done_configs = set(screen_df["config_name"].astype(str))
    else:
        done_configs = set()
else:
    screen_df = pd.DataFrame()
    done_configs = set()

print("Already completed configs:", sorted(done_configs))

for cfg in lgbm_configs:
    config_name = cfg["config_name"]

    if config_name in done_configs:
        print(f"\nSkipping already completed config: {config_name}")
        continue

    print("\n" + "=" * 80)
    print("Running:", config_name)

    params = {**common_params, **cfg}
    params_for_model = params.copy()
    params_for_model.pop("config_name")

    start = time.time()

    model = lgb.LGBMRegressor(**params_for_model)

    model.fit(
        X_train_lgb,
        y_train_lgb,
        eval_set=[(X_val_lgb, y_val_lgb)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=250),
        ],
    )

    elapsed = time.time() - start

    best_iter = getattr(model, "best_iteration_", None)

    pred_train_raw = model.predict(X_train_lgb, num_iteration=best_iter)
    pred_val_raw = model.predict(X_val_lgb, num_iteration=best_iter)

    pred_train_clip = np.clip(pred_train_raw, 0, 100)
    pred_val_clip = np.clip(pred_val_raw, 0, 100)

    row = {
        "model_class": "LightGBM",
        "backend": "lgbm",
        "stage": "controlled_holdout_screen",
        "feature_space": "base",
        "status": "completed",
        "config_name": config_name,
        "train_mse_raw": mean_squared_error(y_train_lgb, pred_train_raw),
        "train_mse_clipped": mean_squared_error(y_train_lgb, pred_train_clip),
        "holdout_val_mse_raw": mean_squared_error(y_val_lgb, pred_val_raw),
        "holdout_val_mse_clipped": mean_squared_error(y_val_lgb, pred_val_clip),
        "best_iteration": best_iter,
        "elapsed_sec": elapsed,
    }

    for key, value in cfg.items():
        if key != "config_name":
            row[key] = value

    screen_df = pd.concat(
        [screen_df, pd.DataFrame([row])],
        ignore_index=True
    )

    screen_df.to_csv(screen_path, index=False)

    print("\nCompleted:", config_name)
    print("Best iteration:", best_iter)
    print("Train MSE clipped:", row["train_mse_clipped"])
    print("Holdout MSE clipped:", row["holdout_val_mse_clipped"])
    print("Elapsed seconds:", round(elapsed, 2))
    print("Checkpoint saved:", screen_path)

    del model
    del pred_train_raw, pred_val_raw, pred_train_clip, pred_val_clip
    gc.collect()

print("\n" + "=" * 80)
print("Controlled LightGBM screen results:")

screen_df = pd.read_csv(screen_path)

display_cols = [
    "config_name",
    "train_mse_clipped",
    "holdout_val_mse_clipped",
    "best_iteration",
    "elapsed_sec",
    "n_estimators",
    "learning_rate",
    "num_leaves",
    "min_child_samples",
    "subsample",
    "colsample_bytree",
    "reg_lambda",
]

display(
    screen_df
    .sort_values("holdout_val_mse_clipped")
    [display_cols]
    .reset_index(drop=True)
)

# Clean up full LightGBM arrays.
del X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb
gc.collect()

LightGBM version: 4.6.0
Train shape: (115936, 162)
Validation shape: (28985, 162)
Checkpoint file: model_results/lgbm_controlled_base_holdout_screen.csv
Already completed configs: []

Running: lgbm_c01_lr03_l63_child80_l2_5_sub08_col08
Training until validation scores don't improve for 100 rounds
[250]	valid_0's l2: 199.162
[500]	valid_0's l2: 174.406
[750]	valid_0's l2: 163.192
[1000]	valid_0's l2: 156.177
[1250]	valid_0's l2: 151.088
[1500]	valid_0's l2: 147.206
[1750]	valid_0's l2: 144.006
[2000]	valid_0's l2: 141.114
[2250]	valid_0's l2: 138.747
[2500]	valid_0's l2: 136.724
Did not meet early stopping. Best iteration is:
[2500]	valid_0's l2: 136.724

Completed: lgbm_c01_lr03_l63_child80_l2_5_sub08_col08
Best iteration: 2500
Train MSE clipped: 92.78294057631074
Holdout MSE clipped: 136.40299287236442
Elapsed seconds: 36.1
Checkpoint saved: model_results/lgbm_controlled_base_holdout_screen.csv

Running: lgbm_c02_lr03_l63_child120_l2_10_sub08_col08
Training until validation scores don

,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_c04_lr03_l127_child120_l2_20_sub08_col08,70.014712,126.393035,2500,51.681790,2500,0.03,127,120,0.80,0.80,20.0
1,lgbm_c01_lr03_l63_child80_l2_5_sub08_col08,92.782941,136.402993,2500,36.097008,2500,0.03,63,80,0.80,0.80,5.0
2,lgbm_c03_lr02_l63_child80_l2_10_sub085_col085,95.873512,137.583865,3500,49.708509,3500,0.02,63,80,0.85,0.85,10.0
3,lgbm_c02_lr03_l63_child120_l2_10_sub08_col08,97.946833,139.495044,2500,37.048055,2500,0.03,63,120,0.80,0.80,10.0


0

In [37]:
# ============================================================
# 20B. LightGBM targeted holdout refinement
# Conservative, checkpointed, base feature space only.
# No OOF and no submission in this cell.
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RNG_SEED = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

refine_path = results_dir / "lgbm_targeted_base_holdout_refinement.csv"

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = ["X_tr", "X_val", "y_tr", "y_val"]
missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing objects: {missing_objects}"

print("LightGBM version:", lgb.__version__)
print("Train shape:", X_tr.shape)
print("Validation shape:", X_val.shape)
print("Checkpoint file:", refine_path)

# Use NumPy arrays consistently for fit and predict.
# float32 reduces memory pressure and avoids DataFrame/name warnings.
X_train_lgb = np.ascontiguousarray(np.asarray(X_tr, dtype=np.float32))
X_val_lgb = np.ascontiguousarray(np.asarray(X_val, dtype=np.float32))
y_train_lgb = np.asarray(y_tr, dtype=np.float32).ravel()
y_val_lgb = np.asarray(y_val, dtype=np.float32).ravel()

common_params = dict(
    objective="regression",
    metric="l2",
    random_state=RNG_SEED,
    n_jobs=1,              # keep conservative after kernel crashes
    verbosity=-1,
    force_col_wise=True,
)

# These configs refine around the strongest saved completed LightGBM region:
# leaves around 63, min_child_samples around 40, subsample/colsample around 0.9.
# We are not doing OOF yet; this is just a targeted holdout refinement.
targeted_configs = [
    dict(
        config_name="lgbm_t01_lr03_l63_child40_l2_1_sub09_col09",
        n_estimators=8000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_t02_lr02_l63_child40_l2_1_sub09_col09",
        n_estimators=10000,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
        max_depth=-1,
    ),
    dict(
        config_name="lgbm_t03_lr03_l95_child60_l2_5_sub085_col09",
        n_estimators=8000,
        learning_rate=0.03,
        num_leaves=95,
        min_child_samples=60,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=5.0,
        max_depth=-1,
    ),
]

if refine_path.exists():
    refine_df = pd.read_csv(refine_path)
    done_configs = set(refine_df["config_name"].astype(str)) if "config_name" in refine_df.columns else set()
else:
    refine_df = pd.DataFrame()
    done_configs = set()

print("Already completed targeted configs:", sorted(done_configs))

for cfg in targeted_configs:
    config_name = cfg["config_name"]

    if config_name in done_configs:
        print(f"\nSkipping already completed config: {config_name}")
        continue

    print("\n" + "=" * 80)
    print("Running targeted LightGBM config:", config_name)
    print("=" * 80)

    params = {**common_params, **cfg}
    params_for_model = params.copy()
    params_for_model.pop("config_name")

    start = time.time()

    model = lgb.LGBMRegressor(**params_for_model)

    model.fit(
        X_train_lgb,
        y_train_lgb,
        eval_set=[(X_val_lgb, y_val_lgb)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=300, verbose=True),
            lgb.log_evaluation(period=500),
        ],
    )

    elapsed = time.time() - start

    best_iter = getattr(model, "best_iteration_", None)
    if best_iter is None or best_iter <= 0:
        best_iter = cfg["n_estimators"]

    pred_train_raw = model.predict(X_train_lgb, num_iteration=best_iter)
    pred_val_raw = model.predict(X_val_lgb, num_iteration=best_iter)

    pred_train_clip = np.clip(pred_train_raw, 0, 100)
    pred_val_clip = np.clip(pred_val_raw, 0, 100)

    row = {
        "model_class": "LightGBM",
        "backend": "lgbm",
        "stage": "targeted_holdout_refinement",
        "feature_space": "base",
        "status": "completed",
        "config_name": config_name,
        "train_mse_raw": mean_squared_error(y_train_lgb, pred_train_raw),
        "train_mse_clipped": mean_squared_error(y_train_lgb, pred_train_clip),
        "holdout_val_mse_raw": mean_squared_error(y_val_lgb, pred_val_raw),
        "holdout_val_mse_clipped": mean_squared_error(y_val_lgb, pred_val_clip),
        "best_iteration": best_iter,
        "elapsed_sec": elapsed,
    }

    for key, value in cfg.items():
        if key != "config_name":
            row[key] = value

    refine_df = pd.concat(
        [refine_df, pd.DataFrame([row])],
        ignore_index=True
    )
    refine_df.to_csv(refine_path, index=False)

    print("\nCompleted:", config_name)
    print("Best iteration:", best_iter)
    print("Train MSE clipped:", row["train_mse_clipped"])
    print("Holdout MSE clipped:", row["holdout_val_mse_clipped"])
    print("Elapsed seconds:", round(elapsed, 2))
    print("Checkpoint saved:", refine_path)

    del model
    del pred_train_raw, pred_val_raw, pred_train_clip, pred_val_clip
    gc.collect()

# ------------------------------------------------------------
# Consolidate all current LightGBM holdout screens
# ------------------------------------------------------------

screen_files = [
    "lgbm_clean_base_holdout_screen.csv",
    "lgbm_controlled_base_holdout_screen.csv",
    "lgbm_targeted_base_holdout_refinement.csv",
]

frames = []

for fname in screen_files:
    path = results_dir / fname
    if not path.exists():
        continue

    df = pd.read_csv(path)
    df["source_file"] = fname

    if "best_iteration" not in df.columns and "n_iter_used" in df.columns:
        df["best_iteration"] = df["n_iter_used"]

    frames.append(df)

if len(frames) > 0:
    all_lgbm_screens = pd.concat(frames, ignore_index=True)

    display_cols = [
        "source_file",
        "config_name",
        "train_mse_clipped",
        "holdout_val_mse_clipped",
        "best_iteration",
        "elapsed_sec",
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "min_child_samples",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
    ]
    display_cols = [c for c in display_cols if c in all_lgbm_screens.columns]

    all_lgbm_screens = all_lgbm_screens.sort_values("holdout_val_mse_clipped")

    print("\n" + "=" * 80)
    print("Combined LightGBM holdout screen leaderboard:")
    display(all_lgbm_screens[display_cols].head(15).reset_index(drop=True))

    best_lgbm = all_lgbm_screens.iloc[0]

    print("\nBest LightGBM holdout result so far:")
    print("source_file:", best_lgbm["source_file"])
    print("config_name:", best_lgbm["config_name"])
    print("holdout_val_mse_clipped:", best_lgbm["holdout_val_mse_clipped"])
    print("best_iteration:", best_lgbm.get("best_iteration", None))

    print("\nDecision guide:")
    if best_lgbm["holdout_val_mse_clipped"] <= 111.0:
        print("LightGBM is now close enough to ExtraTrees/RF+ET offline results to consider a selected OOF artifact next.")
    else:
        print("Do not run LightGBM OOF yet. Holdout is still not clearly better than the ExtraTrees/RF+ET artifacts.")

else:
    print("No LightGBM screen files found.")

# Clean up large arrays from this cell.
del X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb
gc.collect()

LightGBM version: 4.6.0
Train shape: (115936, 162)
Validation shape: (28985, 162)
Checkpoint file: model_results/lgbm_targeted_base_holdout_refinement.csv
Already completed targeted configs: []

Running targeted LightGBM config: lgbm_t01_lr03_l63_child40_l2_1_sub09_col09
Training until validation scores don't improve for 300 rounds
[500]	valid_0's l2: 173.611
[1000]	valid_0's l2: 153.444
[1500]	valid_0's l2: 142.397
[2000]	valid_0's l2: 134.757
[2500]	valid_0's l2: 129.388
[3000]	valid_0's l2: 125.208
[3500]	valid_0's l2: 121.699
[4000]	valid_0's l2: 118.833
[4500]	valid_0's l2: 116.477
[5000]	valid_0's l2: 114.247
[5500]	valid_0's l2: 112.313
[6000]	valid_0's l2: 110.83
[6500]	valid_0's l2: 109.438
[7000]	valid_0's l2: 108.097
[7500]	valid_0's l2: 107.004
[8000]	valid_0's l2: 105.968
Did not meet early stopping. Best iteration is:
[8000]	valid_0's l2: 105.968

Completed: lgbm_t01_lr03_l63_child40_l2_1_sub09_col09
Best iteration: 8000
Train MSE clipped: 35.15224971100176
Holdout MSE cl

,source_file,config_name,train_mse_clipped,holdout_val_mse_clipped,best_iteration,elapsed_sec,n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda
0,lgbm_targeted_base_holdout_refinement.csv,lgbm_t03_lr03_l95_child60_l2_5_sub085_col09,27.643127,104.496354,8000,118.549880,8000,0.03,95,60,0.85,0.90,5.0
1,lgbm_targeted_base_holdout_refinement.csv,lgbm_t01_lr03_l63_child40_l2_1_sub09_col09,35.152250,105.586469,8000,103.653627,8000,0.03,63,40,0.90,0.90,1.0
2,lgbm_targeted_base_holdout_refinement.csv,lgbm_t02_lr02_l63_child40_l2_1_sub09_col09,41.509765,108.300128,10000,122.553700,10000,0.02,63,40,0.90,0.90,1.0
3,lgbm_clean_base_holdout_screen.csv,lgbm_lr03_leaves63_l2_1_sub09_col09,52.948577,114.079714,4999,62.768896,5000,0.03,63,40,0.90,0.90,1.0
4,lgbm_controlled_base_holdout_screen.csv,lgbm_c04_lr03_l127_child120_l2_20_sub08_col08,70.014712,126.393035,2500,51.681790,2500,0.03,127,120,0.80,0.80,20.0
5,lgbm_clean_base_holdout_screen.csv,lgbm_lr03_leaves31_l2_1_sub09_col09,85.639976,130.751393,5000,42.012063,5000,0.03,31,40,0.90,0.90,1.0
6,lgbm_controlled_base_holdout_screen.csv,lgbm_c01_lr03_l63_child80_l2_5_sub08_col08,92.782941,136.402993,2500,36.097008,2500,0.03,63,80,0.80,0.80,5.0
7,lgbm_controlled_base_holdout_screen.csv,lgbm_c03_lr02_l63_child80_l2_10_sub085_col085,95.873512,137.583865,3500,49.708509,3500,0.02,63,80,0.85,0.85,10.0
8,lgbm_controlled_base_holdout_screen.csv,lgbm_c02_lr03_l63_child120_l2_10_sub08_col08,97.946833,139.495044,2500,37.048055,2500,0.03,63,120,0.80,0.80,10.0



Best LightGBM holdout result so far:
source_file: lgbm_targeted_base_holdout_refinement.csv
config_name: lgbm_t03_lr03_l95_child60_l2_5_sub085_col09
holdout_val_mse_clipped: 104.49635410873547
best_iteration: 8000

Decision guide:
LightGBM is now close enough to ExtraTrees/RF+ET offline results to consider a selected OOF artifact next.


0

In [38]:
# ============================================================
# 21A. Selected LightGBM OOF artifact + fold-averaged submission
# Uses the best targeted holdout config:
# lgbm_t03_lr03_l95_child60_l2_5_sub085_col09
#
# Conservative design:
# - one config only
# - base feature space only
# - n_jobs=1
# - checkpoint after every fold
# - resume-safe
# - no old crashed ExtraTrees code involved
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

results_dir = Path("model_results")
results_dir.mkdir(exist_ok=True)

artifact_name = "lgbm_t03_base_5fold_oof"

metrics_path = results_dir / f"{artifact_name}_fold_metrics.csv"
oof_npy_path = results_dir / f"{artifact_name}_oof.npy"
test_sum_npy_path = results_dir / f"{artifact_name}_test_pred_sum.npy"

oof_csv_path = results_dir / f"oof_{artifact_name}.csv"
testpred_csv_path = results_dir / f"testpred_{artifact_name}_foldavg.csv"
submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

print("LightGBM version:", lgb.__version__)
print("Artifact:", artifact_name)
print("Metrics checkpoint:", metrics_path)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing required objects: {missing_objects}"

assert X_train_proc_model.shape[0] == len(y_train), (
    X_train_proc_model.shape,
    len(y_train),
)
assert X_test_proc_model.shape[0] == len(test_ids), (
    X_test_proc_model.shape,
    len(test_ids),
)

print("Full train shape:", X_train_proc_model.shape)
print("Full test shape:", X_test_proc_model.shape)
print("Target length:", len(y_train))
print("Test ID length:", len(test_ids))

# Convert once to memory-conscious contiguous arrays.
# This avoids repeated DataFrame slicing overhead and keeps LightGBM stable.
X_full_lgb = np.ascontiguousarray(np.asarray(X_train_proc_model, dtype=np.float32))
X_test_lgb = np.ascontiguousarray(np.asarray(X_test_proc_model, dtype=np.float32))
y_full_lgb = np.asarray(y_train, dtype=np.float32).ravel()

n_train = X_full_lgb.shape[0]
n_test = X_test_lgb.shape[0]

# ------------------------------------------------------------
# Selected LightGBM config
# ------------------------------------------------------------

selected_lgbm_params = dict(
    objective="regression",
    metric="l2",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=-1,
    force_col_wise=True,

    # Selected from targeted holdout refinement
    n_estimators=12000,
    learning_rate=0.03,
    num_leaves=95,
    min_child_samples=60,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
)

print("\nSelected LightGBM parameters:")
for k, v in selected_lgbm_params.items():
    print(f"  {k}: {v}")

# ------------------------------------------------------------
# Resume-safe checkpoint setup
# ------------------------------------------------------------

if metrics_path.exists():
    fold_metrics = pd.read_csv(metrics_path)
    completed_folds = set(fold_metrics["fold"].astype(int).tolist())
else:
    fold_metrics = pd.DataFrame()
    completed_folds = set()

if oof_npy_path.exists():
    oof_pred = np.load(oof_npy_path)
    assert len(oof_pred) == n_train
else:
    oof_pred = np.full(n_train, np.nan, dtype=np.float32)

if test_sum_npy_path.exists():
    test_pred_sum = np.load(test_sum_npy_path)
    assert len(test_pred_sum) == n_test
else:
    test_pred_sum = np.zeros(n_test, dtype=np.float32)

print("\nAlready completed folds:", sorted(completed_folds))

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

start_all = time.time()

# ------------------------------------------------------------
# 5-fold OOF loop
# ------------------------------------------------------------

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full_lgb), start=1):

    if fold in completed_folds:
        print(f"\nSkipping already completed fold {fold}")
        continue

    print("\n" + "=" * 80)
    print(f"Starting LightGBM fold {fold}/5")
    print("=" * 80)

    X_fold_tr = X_full_lgb[tr_idx]
    y_fold_tr = y_full_lgb[tr_idx]
    X_fold_val = X_full_lgb[val_idx]
    y_fold_val = y_full_lgb[val_idx]

    fold_params = selected_lgbm_params.copy()
    fold_params["random_state"] = RANDOM_STATE + fold

    model = lgb.LGBMRegressor(**fold_params)

    fold_start = time.time()

    model.fit(
        X_fold_tr,
        y_fold_tr,
        eval_set=[(X_fold_val, y_fold_val)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=True),
            lgb.log_evaluation(period=1000),
        ],
    )

    fold_elapsed = time.time() - fold_start

    best_iter = getattr(model, "best_iteration_", None)
    if best_iter is None or best_iter <= 0:
        best_iter = selected_lgbm_params["n_estimators"]

    val_pred_raw = model.predict(X_fold_val, num_iteration=best_iter)
    val_pred_clip = np.clip(val_pred_raw, 0, 100).astype(np.float32)

    test_pred_raw = model.predict(X_test_lgb, num_iteration=best_iter)
    test_pred_clip = np.clip(test_pred_raw, 0, 100).astype(np.float32)

    oof_pred[val_idx] = val_pred_clip
    test_pred_sum += test_pred_clip / 5.0

    row = {
        "artifact_name": artifact_name,
        "model_class": "LightGBM",
        "feature_space": "base",
        "fold": fold,
        "fold_train_n": len(tr_idx),
        "fold_valid_n": len(val_idx),
        "best_iteration": best_iter,
        "fold_valid_mse_raw": mean_squared_error(y_fold_val, val_pred_raw),
        "fold_valid_mse_clipped": mean_squared_error(y_fold_val, val_pred_clip),
        "elapsed_sec": fold_elapsed,
    }

    for key, value in selected_lgbm_params.items():
        row[key] = value

    fold_metrics = pd.concat(
        [fold_metrics, pd.DataFrame([row])],
        ignore_index=True,
    )

    # Checkpoint immediately after the fold.
    fold_metrics.to_csv(metrics_path, index=False)
    np.save(oof_npy_path, oof_pred)
    np.save(test_sum_npy_path, test_pred_sum)

    print(f"\nCompleted fold {fold}")
    print("Best iteration:", best_iter)
    print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
    print("Elapsed seconds:", round(fold_elapsed, 2))
    print("Checkpointed metrics/oof/test sum.")

    del model
    del X_fold_tr, y_fold_tr, X_fold_val, y_fold_val
    del val_pred_raw, val_pred_clip, test_pred_raw, test_pred_clip
    gc.collect()

elapsed_all = time.time() - start_all

# ------------------------------------------------------------
# Final artifact construction
# ------------------------------------------------------------

n_missing_oof = int(np.isnan(oof_pred).sum())
print("\n" + "=" * 80)
print("OOF completion check")
print("=" * 80)
print("Missing OOF predictions:", n_missing_oof)

if n_missing_oof == 0:
    overall_oof_mse = mean_squared_error(y_full_lgb, oof_pred)
    print("Overall LightGBM OOF MSE clipped:", overall_oof_mse)

    # Training IDs are helpful for stacking if available.
    if "train_ids" in globals():
        train_id_series = pd.Series(train_ids).reset_index(drop=True)
    else:
        train_id_series = pd.Series(np.arange(n_train), name="TRAIN_ROW_ID")

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_series,
        "PERCENT_PROFICIENT_TRUE": y_full_lgb,
        f"OOF_{artifact_name}": oof_pred,
    })
    oof_df.to_csv(oof_csv_path, index=False)

    test_id_series = pd.Series(test_ids).reset_index(drop=True)
    final_test_pred = np.clip(test_pred_sum, 0, 100).astype(np.float32)

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        f"TESTPRED_{artifact_name}": final_test_pred,
    })
    testpred_df.to_csv(testpred_csv_path, index=False)

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        "PERCENT_PROFICIENT": final_test_pred,
    })
    submission_df.to_csv(submission_path, index=False)

    print("\nSaved OOF file:", oof_csv_path)
    print("Saved test prediction file:", testpred_csv_path)
    print("Saved Kaggle submission:", submission_path)
    print("\nSubmission shape:", submission_df.shape)
    print("\nSubmission prediction summary:")
    print(submission_df["PERCENT_PROFICIENT"].describe())

    print("\nFold metrics:")
    display(
        fold_metrics.sort_values("fold")[
            [
                "fold",
                "fold_valid_mse_clipped",
                "fold_valid_mse_raw",
                "best_iteration",
                "elapsed_sec",
            ]
        ].reset_index(drop=True)
    )

    print("\nTotal elapsed seconds in this run:", round(elapsed_all, 2))

else:
    print(
        "OOF artifact is not complete yet. Re-run this same cell to resume from the latest checkpoint."
    )

# Clean only local arrays from this cell.
del X_full_lgb, X_test_lgb, y_full_lgb
gc.collect()

LightGBM version: 4.6.0
Artifact: lgbm_t03_base_5fold_oof
Metrics checkpoint: model_results/lgbm_t03_base_5fold_oof_fold_metrics.csv
Full train shape: (144921, 162)
Full test shape: (48307, 162)
Target length: 144921
Test ID length: 48307

Selected LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  n_estimators: 12000
  learning_rate: 0.03
  num_leaves: 95
  min_child_samples: 60
  subsample: 0.85
  subsample_freq: 1
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 5.0
  max_depth: -1

Already completed folds: []

Starting LightGBM fold 1/5
Training until validation scores don't improve for 500 rounds
[1000]	valid_0's l2: 144.55
[2000]	valid_0's l2: 129.408
[3000]	valid_0's l2: 121.318
[4000]	valid_0's l2: 115.861
[5000]	valid_0's l2: 112.045
[6000]	valid_0's l2: 109.066
[7000]	valid_0's l2: 106.881
[8000]	valid_0's l2: 105.062
[9000]	valid_0's l2: 103.675
[10000]	valid_0's l2: 102.592
[11000]	valid_

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,100.265396,100.821338,11999,187.305308
1,2,96.495171,97.028716,11998,177.751144
2,3,98.903893,99.598858,12000,181.971037
3,4,96.720520,97.372209,11998,179.059395
4,5,101.512009,102.124080,12000,364.619435



Total elapsed seconds in this run: 3695.62


0

### LightGBM OOF Submission Result

The selected LightGBM model produced the strongest clean pipeline result so far. The 5-fold OOF artifact used the base feature space and the targeted LightGBM configuration:

- `num_leaves = 95`
- `min_child_samples = 60`
- `learning_rate = 0.03`
- `subsample = 0.85`
- `colsample_bytree = 0.90`
- `reg_lambda = 5.0`
- `n_estimators = 12000`

The completed 5-fold OOF run gave an overall clipped OOF MSE of approximately **98.78**. The resulting fold-averaged Kaggle submission file was:

`submission_lgbm_t03_base_5fold_oof_foldavg.csv`

This submission achieved a new public leaderboard MSE of **90.644**, which is the best result from the clean current notebook pipeline so far.

This confirms that gradient boosting is currently the strongest model family for this problem. The result also validates the decision to move from bagging/tree ensembles into LightGBM while preserving earlier Random Forest and ExtraTrees OOF artifacts for possible blending or stacking.

One important observation is that all five LightGBM folds reached the maximum estimator limit of `12000` without early stopping. This suggests the model may still benefit from a longer selected OOF run with a higher tree cap. However, before rerunning a longer LightGBM OOF model, the next low-risk step is to run the saved OOF blend screen using Random Forest, ExtraTrees, and LightGBM. This requires no model refitting and may improve the submission by combining complementary model errors.

In [39]:
# ============================================================
# 22A. Three-model OOF blend screen:
# Random Forest + ExtraTrees + LightGBM
#
# No model fitting in this cell.
# Reads saved OOF/test prediction artifacts and creates
# a convex OOF-weighted blend submission.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RESULTS_DIR = Path("model_results")
assert RESULTS_DIR.exists(), "model_results folder was not found."

# ------------------------------------------------------------
# Locate required saved artifacts
# ------------------------------------------------------------

artifact_paths = {
    "rf_oof": RESULTS_DIR / "oof_rf_500_base.csv",
    "rf_test": RESULTS_DIR / "testpred_rf_500_base_folds.csv",
    "et_oof": RESULTS_DIR / "oof_extratrees_safe_base.csv",
    "et_test": RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    "lgbm_oof": RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
    "lgbm_test": RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
}

for label, path in artifact_paths.items():
    assert path.exists(), f"Missing {label}: {path}"

print("Using saved artifacts:")
for label, path in artifact_paths.items():
    print(f"  {label}: {path}")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})


def one_dim_id_array(ids):
    arr = np.asarray(ids)
    if arr.ndim > 1:
        arr = arr.ravel()
    return pd.Series(arr).astype(str).to_numpy()


def first_existing_column(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find {label}. "
        f"Tried {candidates}. Available columns: {list(df.columns)}"
    )


def pick_prediction_column(df, preferred_cols, label, target_cols=None):
    if target_cols is None:
        target_cols = []

    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = set(["ASSESSMENT_ID"] + target_cols)

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "foldavg" in c.lower()
            or "fold_avg" in c.lower()
            or "average" in c.lower()
            or "avg" in c.lower()
        )
    ]

    priority_terms = [
        "foldavg",
        "fold_avg",
        "testpred",
        "test_pred",
        "percent_proficient",
        "oof",
        "pred",
        "avg",
    ]

    for term in priority_terms:
        matches = [c for c in pred_like_cols if term in c.lower()]
        if len(matches) == 1:
            return matches[0]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns available: {list(df.columns)}\n"
        f"Numeric columns found: {numeric_cols}\n"
        f"Prediction-like columns found: {pred_like_cols}"
    )


def standardize_oof(path, model_name, preferred_pred_cols):
    df = read_pred_csv(path)

    assert "ASSESSMENT_ID" in df.columns, f"{model_name} OOF missing ASSESSMENT_ID."

    y_col = first_existing_column(
        df,
        [
            "y_true",
            "PERCENT_PROFICIENT_TRUE",
            "target",
            "true",
            "actual",
        ],
        f"{model_name} OOF target column"
    )

    pred_col = pick_prediction_column(
        df,
        preferred_cols=preferred_pred_cols,
        label=f"{model_name} OOF prediction",
        target_cols=[y_col],
    )

    print(f"{model_name.upper()} OOF target column: {y_col}")
    print(f"{model_name.upper()} OOF prediction column: {pred_col}")

    out = df[["ASSESSMENT_ID", y_col, pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={
        y_col: f"{model_name}_y_true",
        pred_col: f"{model_name}_oof_pred",
    })

    return out


def standardize_testpred(path, model_name, preferred_pred_cols):
    df = read_pred_csv(path)

    assert "ASSESSMENT_ID" in df.columns, f"{model_name} test predictions missing ASSESSMENT_ID."

    pred_col = pick_prediction_column(
        df,
        preferred_cols=preferred_pred_cols,
        label=f"{model_name} test prediction",
        target_cols=[],
    )

    print(f"{model_name.upper()} test prediction column: {pred_col}")

    out = df[["ASSESSMENT_ID", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)
    out = out.rename(columns={pred_col: f"{model_name}_test_pred"})

    return out


# ------------------------------------------------------------
# Load and standardize OOF artifacts
# ------------------------------------------------------------

rf_oof = standardize_oof(
    artifact_paths["rf_oof"],
    "rf",
    preferred_pred_cols=[
        "rf_500_base_oof_pred",
        "oof_pred",
        "pred",
        "prediction",
    ],
)

et_oof = standardize_oof(
    artifact_paths["et_oof"],
    "et",
    preferred_pred_cols=[
        "oof_pred",          # raw ExtraTrees OOF, preferred for blending
        "oof_pred_clipped",
        "et_oof_pred",
        "pred",
        "prediction",
    ],
)

lgbm_oof = standardize_oof(
    artifact_paths["lgbm_oof"],
    "lgbm",
    preferred_pred_cols=[
        "OOF_lgbm_t03_base_5fold_oof",
        "oof_pred",
        "lgbm_oof_pred",
        "pred",
        "prediction",
    ],
)

for name, df in [("rf", rf_oof), ("et", et_oof), ("lgbm", lgbm_oof)]:
    assert df["ASSESSMENT_ID"].duplicated().sum() == 0, f"Duplicate ASSESSMENT_ID in {name} OOF."

oof_blend_df = (
    rf_oof
    .merge(et_oof, on="ASSESSMENT_ID", how="inner")
    .merge(lgbm_oof, on="ASSESSMENT_ID", how="inner")
)

assert len(oof_blend_df) == len(rf_oof) == len(et_oof) == len(lgbm_oof), "OOF merge lost rows."

# Check target alignment
assert np.allclose(
    oof_blend_df["rf_y_true"].to_numpy(dtype=float),
    oof_blend_df["et_y_true"].to_numpy(dtype=float),
), "RF and ExtraTrees targets do not match."

assert np.allclose(
    oof_blend_df["rf_y_true"].to_numpy(dtype=float),
    oof_blend_df["lgbm_y_true"].to_numpy(dtype=float),
), "RF and LightGBM targets do not match."

oof_blend_df["y_true"] = oof_blend_df["rf_y_true"].astype(float)

y_oof = oof_blend_df["y_true"].to_numpy(dtype=float)

rf_pred = oof_blend_df["rf_oof_pred"].to_numpy(dtype=float)
et_pred = oof_blend_df["et_oof_pred"].to_numpy(dtype=float)
lgbm_pred = oof_blend_df["lgbm_oof_pred"].to_numpy(dtype=float)

component_rows = []
for model_name, pred in [
    ("rf", rf_pred),
    ("et", et_pred),
    ("lgbm", lgbm_pred),
]:
    component_rows.append({
        "model": model_name,
        "oof_mse_raw": mean_squared_error(y_oof, pred),
        "oof_mse_clipped": mean_squared_error(y_oof, np.clip(pred, 0, 100)),
        "pred_mean": np.mean(np.clip(pred, 0, 100)),
        "pred_std": np.std(np.clip(pred, 0, 100)),
    })

component_results = pd.DataFrame(component_rows).sort_values("oof_mse_clipped")

print("\nComponent OOF results:")
display(component_results.reset_index(drop=True))

print("\nResidual correlation matrix, using clipped predictions:")
resid_df = pd.DataFrame({
    "rf_resid": y_oof - np.clip(rf_pred, 0, 100),
    "et_resid": y_oof - np.clip(et_pred, 0, 100),
    "lgbm_resid": y_oof - np.clip(lgbm_pred, 0, 100),
})
display(resid_df.corr())

lgbm_oof_mse = float(
    component_results.loc[
        component_results["model"] == "lgbm",
        "oof_mse_clipped"
    ].iloc[0]
)

# ------------------------------------------------------------
# Build candidate convex weights
# ------------------------------------------------------------

candidate_rows = []
seen_weights = set()

def add_weight(w_rf, w_et, w_lgbm, source):
    weights = np.array([w_rf, w_et, w_lgbm], dtype=float)

    if np.any(weights < -1e-9):
        return

    weights[np.abs(weights) < 1e-12] = 0.0

    total = weights.sum()
    if total <= 0:
        return

    weights = weights / total

    key = tuple(np.round(weights, 6))
    if key in seen_weights:
        return

    seen_weights.add(key)

    candidate_rows.append({
        "w_rf": weights[0],
        "w_et": weights[1],
        "w_lgbm": weights[2],
        "source": source,
    })


# Pure models
add_weight(1.0, 0.0, 0.0, "pure_rf")
add_weight(0.0, 1.0, 0.0, "pure_et")
add_weight(0.0, 0.0, 1.0, "pure_lgbm")

# Fine pairwise grids
pair_grid = np.linspace(0, 1, 1001)

for w_lgbm in pair_grid:
    add_weight(0.0, 1.0 - w_lgbm, w_lgbm, "pair_et_lgbm")

for w_lgbm in pair_grid:
    add_weight(1.0 - w_lgbm, 0.0, w_lgbm, "pair_rf_lgbm")

for w_et in pair_grid:
    add_weight(1.0 - w_et, w_et, 0.0, "pair_rf_et")

# Coarse three-way grid focused around LightGBM dominance.
# Since LightGBM is much stronger OOF than RF/ET, we do not waste
# many candidates with tiny LightGBM weights.
step = 0.01

for w_lgbm in np.arange(0.60, 1.0 + step / 2, step):
    remainder = 1.0 - w_lgbm
    n_steps = int(round(remainder / step))

    for k in range(n_steps + 1):
        w_rf = k * step
        w_et = remainder - w_rf
        add_weight(w_rf, w_et, w_lgbm, "three_way_lgbm_focused_coarse")

candidate_df = pd.DataFrame(candidate_rows)

print("\nInitial candidate weight count:", len(candidate_df))

# ------------------------------------------------------------
# Evaluate candidate weights
# ------------------------------------------------------------

pred_matrix = np.vstack([
    rf_pred,
    et_pred,
    lgbm_pred,
]).astype(np.float32)

def evaluate_weight_df(weight_df):
    rows = []

    for row in weight_df.itertuples(index=False):
        weights = np.array([row.w_rf, row.w_et, row.w_lgbm], dtype=np.float32)

        blend_raw = (
            weights[0] * pred_matrix[0]
            + weights[1] * pred_matrix[1]
            + weights[2] * pred_matrix[2]
        )

        blend_clipped = np.clip(blend_raw, 0, 100)

        rows.append({
            "w_rf": float(weights[0]),
            "w_et": float(weights[1]),
            "w_lgbm": float(weights[2]),
            "source": row.source,
            "oof_mse_raw": mean_squared_error(y_oof, blend_raw),
            "oof_mse_clipped": mean_squared_error(y_oof, blend_clipped),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("oof_mse_clipped")
        .reset_index(drop=True)
    )

blend_weight_results_initial = evaluate_weight_df(candidate_df)
best_initial = blend_weight_results_initial.iloc[0]

print("\nBest initial blend:")
display(pd.DataFrame([best_initial]))

# Local fine search around the best initial weights.
local_step = 0.002
window = 0.04

center_rf = float(best_initial["w_rf"])
center_et = float(best_initial["w_et"])
center_lgbm = float(best_initial["w_lgbm"])

for w_rf in np.arange(
    max(0.0, center_rf - window),
    min(1.0, center_rf + window) + local_step / 2,
    local_step,
):
    for w_et in np.arange(
        max(0.0, center_et - window),
        min(1.0, center_et + window) + local_step / 2,
        local_step,
    ):
        w_lgbm = 1.0 - w_rf - w_et
        add_weight(w_rf, w_et, w_lgbm, "local_fine_around_best")

candidate_df = pd.DataFrame(candidate_rows)

print("Final candidate weight count after local fine search:", len(candidate_df))

blend_weight_results = evaluate_weight_df(candidate_df)
best_blend = blend_weight_results.iloc[0].to_dict()

best_w_rf = float(best_blend["w_rf"])
best_w_et = float(best_blend["w_et"])
best_w_lgbm = float(best_blend["w_lgbm"])
best_oof_mse = float(best_blend["oof_mse_clipped"])

print("\nFinal best OOF blend:")
display(pd.DataFrame([best_blend]))

print("\nTop 15 blend weights:")
display(blend_weight_results.head(15))

blend_gain_vs_lgbm = lgbm_oof_mse - best_oof_mse

print("\nBlend gain versus pure LightGBM OOF:")
print("Pure LightGBM OOF MSE:", lgbm_oof_mse)
print("Best blend OOF MSE:   ", best_oof_mse)
print("OOF MSE gain:         ", blend_gain_vs_lgbm)

# ------------------------------------------------------------
# Save OOF blend artifacts
# ------------------------------------------------------------

oof_blend_df["blend_pred_raw"] = (
    best_w_rf * rf_pred
    + best_w_et * et_pred
    + best_w_lgbm * lgbm_pred
)

oof_blend_df["blend_pred_clipped"] = np.clip(
    oof_blend_df["blend_pred_raw"],
    0,
    100,
)

weight_results_path = RESULTS_DIR / "blend_rf_et_lgbm_weight_screen.csv"
oof_blend_path = RESULTS_DIR / "oof_blend_rf_et_lgbm_weighted.csv"

blend_weight_results.to_csv(weight_results_path, index=False)
oof_blend_df.to_csv(oof_blend_path, index=False)

# ------------------------------------------------------------
# Load, standardize, and align test predictions
# ------------------------------------------------------------

rf_test = standardize_testpred(
    artifact_paths["rf_test"],
    "rf",
    preferred_pred_cols=[
        "rf_500_base_foldavg_pred",
        "rf_500_base_test_pred",
        "PERCENT_PROFICIENT",
        "test_pred",
        "pred",
        "prediction",
    ],
)

et_test = standardize_testpred(
    artifact_paths["et_test"],
    "et",
    preferred_pred_cols=[
        "PERCENT_PROFICIENT",
        "et_test_pred",
        "test_pred_avg",
        "test_pred",
        "pred",
        "prediction",
    ],
)

lgbm_test = standardize_testpred(
    artifact_paths["lgbm_test"],
    "lgbm",
    preferred_pred_cols=[
        "TESTPRED_lgbm_t03_base_5fold_oof",
        "PERCENT_PROFICIENT",
        "lgbm_test_pred",
        "test_pred",
        "pred",
        "prediction",
    ],
)

for name, df in [("rf", rf_test), ("et", et_test), ("lgbm", lgbm_test)]:
    assert df["ASSESSMENT_ID"].duplicated().sum() == 0, f"Duplicate ASSESSMENT_ID in {name} test predictions."

test_blend_df = (
    rf_test
    .merge(et_test, on="ASSESSMENT_ID", how="inner")
    .merge(lgbm_test, on="ASSESSMENT_ID", how="inner")
)

assert len(test_blend_df) == len(rf_test) == len(et_test) == len(lgbm_test), "Test prediction merge lost rows."

# Restore original test_ids order when available.
if "test_ids" in globals():
    test_order = pd.DataFrame({
        "ASSESSMENT_ID": one_dim_id_array(test_ids),
        "_test_order": np.arange(len(test_ids)),
    })

    test_blend_df = test_order.merge(test_blend_df, on="ASSESSMENT_ID", how="left")

    assert test_blend_df["rf_test_pred"].isna().sum() == 0, "Missing RF predictions after test_ids alignment."
    assert test_blend_df["et_test_pred"].isna().sum() == 0, "Missing ExtraTrees predictions after test_ids alignment."
    assert test_blend_df["lgbm_test_pred"].isna().sum() == 0, "Missing LightGBM predictions after test_ids alignment."

    test_blend_df = (
        test_blend_df
        .sort_values("_test_order")
        .drop(columns=["_test_order"])
        .reset_index(drop=True)
    )

test_blend_df["PERCENT_PROFICIENT"] = np.clip(
    best_w_rf * test_blend_df["rf_test_pred"].to_numpy(dtype=float)
    + best_w_et * test_blend_df["et_test_pred"].to_numpy(dtype=float)
    + best_w_lgbm * test_blend_df["lgbm_test_pred"].to_numpy(dtype=float),
    0,
    100,
)

test_blend_path = RESULTS_DIR / "testpred_blend_rf_et_lgbm_weighted.csv"
submission_blend_path = Path("submission_blend_rf_et_lgbm_oof_weighted.csv")

test_blend_df.to_csv(test_blend_path, index=False)

submission_blend = test_blend_df[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()

expected_test_n = len(test_ids) if "test_ids" in globals() else 48307

assert submission_blend.shape[0] == expected_test_n, f"Submission row count is not {expected_test_n}."
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

# ------------------------------------------------------------
# Update notebook trackers
# ------------------------------------------------------------

if "best_models_by_class" not in globals():
    best_models_by_class = {}

best_models_by_class["LightGBM T03 base OOF"] = {
    "model_class": "LightGBM",
    "feature_space": "base",
    "oof_mse": lgbm_oof_mse,
    "submission_path": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
    "notes": "Selected LightGBM OOF artifact from targeted holdout refinement."
}

best_models_by_class["RF + ExtraTrees + LightGBM OOF blend"] = {
    "model_class": "OOF weighted blend",
    "component_models": "RF 500 base + ExtraTrees safe base + LightGBM T03 base",
    "feature_space": "base",
    "oof_mse": best_oof_mse,
    "oof_mse_raw": float(best_blend["oof_mse_raw"]),
    "w_rf": best_w_rf,
    "w_et": best_w_et,
    "w_lgbm": best_w_lgbm,
    "submission_path": str(submission_blend_path),
    "notes": "Convex OOF-weighted blend of saved RF, ExtraTrees, and LightGBM artifacts."
}

tracker_path = RESULTS_DIR / "submission_tracker.csv"

if "submission_tracker" in globals():
    tracker_base = submission_tracker.copy()
elif tracker_path.exists():
    tracker_base = pd.read_csv(tracker_path)
else:
    tracker_base = pd.DataFrame()

tracker_new_rows = pd.DataFrame([
    {
        "submission_file": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "LightGBM",
        "public_mse": np.nan,
        "notes": f"Selected LightGBM OOF artifact; OOF MSE={lgbm_oof_mse:.6f}.",
    },
    {
        "submission_file": submission_blend_path.name,
        "source": "current notebook",
        "model_family": "RF + ExtraTrees + LightGBM blend",
        "public_mse": np.nan,
        "notes": (
            f"OOF-weighted blend; "
            f"w_rf={best_w_rf:.3f}, "
            f"w_et={best_w_et:.3f}, "
            f"w_lgbm={best_w_lgbm:.3f}, "
            f"OOF MSE={best_oof_mse:.6f}."
        ),
    },
])

if len(tracker_base) > 0:
    submission_tracker = pd.concat(
        [tracker_base, tracker_new_rows],
        ignore_index=True,
    )
else:
    submission_tracker = tracker_new_rows.copy()

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last",
)

submission_tracker.to_csv(tracker_path, index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nSaved files:")
print("Weight screen:", weight_results_path.resolve())
print("OOF blend:", oof_blend_path.resolve())
print("Test blend predictions:", test_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())
print("Updated submission tracker:", tracker_path.resolve())

print("\nBlend submission prediction summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nSubmission shape:", submission_blend.shape)

print("\nFirst 5 submission rows:")
print(submission_blend.head().to_string(index=False))

print("\nDecision note:")
if best_w_lgbm >= 0.995:
    print("Best blend is effectively pure LightGBM. Submit/keep the pure LightGBM file first.")
elif blend_gain_vs_lgbm >= 0.25:
    print("Blend improves OOF over pure LightGBM by at least 0.25 MSE. This blend is worth a Kaggle submission after pure LightGBM.")
elif blend_gain_vs_lgbm > 0:
    print("Blend improves OOF only modestly. Save it, but prioritize pure LightGBM unless you have extra submission slots.")
else:
    print("Blend does not improve OOF over pure LightGBM. Prefer the pure LightGBM submission.")

Using saved artifacts:
  rf_oof: model_results/oof_rf_500_base.csv
  rf_test: model_results/testpred_rf_500_base_folds.csv
  et_oof: model_results/oof_extratrees_safe_base.csv
  et_test: model_results/testpred_extratrees_safe_base_foldavg.csv
  lgbm_oof: model_results/oof_lgbm_t03_base_5fold_oof.csv
  lgbm_test: model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv
RF OOF target column: y_true
RF OOF prediction column: rf_500_base_oof_pred
ET OOF target column: y_true
ET OOF prediction column: oof_pred
LGBM OOF target column: PERCENT_PROFICIENT_TRUE
LGBM OOF prediction column: OOF_lgbm_t03_base_5fold_oof

Component OOF results:


,model,oof_mse_raw,oof_mse_clipped,pred_mean,pred_std
0,lgbm,98.779406,98.779406,54.152808,24.283770
1,et,110.749294,110.749294,54.127367,23.857745
2,rf,122.328024,122.328024,54.134143,22.839348



Residual correlation matrix, using clipped predictions:


,rf_resid,et_resid,lgbm_resid
rf_resid,1.000000,0.929709,0.842448
et_resid,0.929709,1.000000,0.802689
lgbm_resid,0.842448,0.802689,1.000000



Initial candidate weight count: 3780

Best initial blend:


...

Final candidate weight count after local fine search: 4574

Final best OOF blend:


,w_rf,w_et,w_lgbm,source,oof_mse_raw,oof_mse_clipped
0,0.0,0.356,0.644,pair_et_lgbm,93.499458,93.499458



Top 15 blend weights:


,w_rf,w_et,w_lgbm,source,oof_mse_raw,oof_mse_clipped
0,0.0,0.356,0.644,pair_et_lgbm,93.499458,93.499458
1,0.0,0.357,0.643,pair_et_lgbm,93.499484,93.499484
2,0.0,0.355,0.645,pair_et_lgbm,93.499516,93.499516
3,0.0,0.358,0.642,pair_et_lgbm,93.499592,93.499592
4,0.0,0.354,0.646,pair_et_lgbm,93.499655,93.499655
5,0.0,0.359,0.641,pair_et_lgbm,93.499786,93.499786
6,0.0,0.353,0.647,pair_et_lgbm,93.499879,93.499879
7,0.0,0.360,0.640,pair_et_lgbm,93.500061,93.500061
8,0.0,0.352,0.648,pair_et_lgbm,93.500187,93.500187
9,0.0,0.361,0.639,pair_et_lgbm,93.500420,93.500420



Blend gain versus pure LightGBM OOF:
Pure LightGBM OOF MSE: 98.77940606294962
Best blend OOF MSE:    93.49945771419596
OOF MSE gain:          5.279948348753663
RF test prediction column: rf_500_base_foldavg_pred
ET test prediction column: PERCENT_PROFICIENT
LGBM test prediction column: TESTPRED_lgbm_t03_base_5fold_oof

Saved files:
Weight screen: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/blend_rf_et_lgbm_weight_screen.csv
OOF blend: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/oof_blend_rf_et_lgbm_weighted.csv
Test blend predictions: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/testpred_blend_rf_et_lgbm_weighted.csv
Blend submission: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_blend_rf_et_lgbm

### RF + ExtraTrees + LightGBM OOF Blend Result

The saved OOF blend screen compared the current Random Forest, ExtraTrees, and LightGBM artifacts without refitting any models.

Component OOF results:

- Random Forest OOF MSE: **122.33**
- ExtraTrees OOF MSE: **110.75**
- LightGBM OOF MSE: **98.78**

The best convex blend excluded Random Forest and used:

- `w_rf = 0.000`
- `w_et = 0.356`
- `w_lgbm = 0.644`

This produced a blended OOF MSE of approximately **93.50**, improving over pure LightGBM by about **5.28 MSE**.

The resulting blend submission file is:

`submission_blend_rf_et_lgbm_oof_weighted.csv`

Interpretation: although ExtraTrees is weaker than LightGBM as a standalone model, its residuals are not identical to LightGBM's residuals, so it contributes useful complementary signal. Random Forest does not add value in this convex blend once ExtraTrees and LightGBM are included.

The pure LightGBM OOF submission already achieved a public leaderboard MSE of **90.644**, the best clean current-notebook result so far. Because the blend improves OOF substantially, the blend submission is worth a Kaggle submission before starting the next longer LightGBM run.

Next modeling step: run a longer selected LightGBM OOF artifact using the same successful configuration but a higher tree cap, since all five folds reached the previous `12000` estimator limit without early stopping.

### RF + ExtraTrees + LightGBM Blend Submission Result

The saved OOF blend screen combined the existing Random Forest, ExtraTrees, and LightGBM OOF artifacts without refitting any models.

Standalone OOF results:

- Random Forest OOF MSE: **122.33**
- ExtraTrees OOF MSE: **110.75**
- LightGBM OOF MSE: **98.78**

The best convex blend assigned zero weight to Random Forest and used:

- `w_rf = 0.000`
- `w_et = 0.356`
- `w_lgbm = 0.644`

This produced a blended OOF MSE of approximately **93.50**, improving over pure LightGBM by about **5.28 OOF MSE**.

The blend submission file was:

`submission_blend_rf_et_lgbm_oof_weighted.csv`

Kaggle public leaderboard result:

- Pure selected LightGBM OOF submission: **90.644**
- RF + ExtraTrees + LightGBM OOF blend submission: **87.054**

This is the strongest clean current-notebook result so far. 

Interpretation: ExtraTrees is weaker than LightGBM as a standalone model, but it adds complementary signal when blended with LightGBM. Random Forest does not add marginal value in this blend after ExtraTrees and LightGBM are included.

Next modeling decision: because the 12k LightGBM OOF run reached the estimator cap on every fold and still produced the best single-model public score so far, the next step is a longer selected LightGBM OOF run. After that finishes, we should rebuild the blend screen using ExtraTrees, the original 12k LightGBM, and the new longer LightGBM artifacts.

In [40]:
# ============================================================
# 23A. Overnight selected LightGBM OOF suite + automatic re-blend
#
# Why this replaces the earlier 25k cell:
# - Pure 12k LGBM scored 90.644 public MSE.
# - ET + LGBM blend scored 87.054 public MSE.
# - All 12k LGBM folds reached the estimator cap.
# - So overnight time is better used by running longer selected
#   LightGBM artifacts and then re-blending automatically.
#
# Runs:
#   1. T03 long80k, lr=0.03
#   2. T03 long100k, lr=0.02
#
# Conservative safeguards:
# - n_jobs=1
# - one fold checkpoint at a time
# - per-fold test prediction files
# - resume-safe
# - no parameter grid explosion
# ============================================================

from pathlib import Path
import time
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("LightGBM version:", lgb.__version__)
print("Results folder:", RESULTS_DIR.resolve())

# ------------------------------------------------------------
# Update tracker with known public scores
# ------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"

if tracker_path.exists():
    submission_tracker = pd.read_csv(tracker_path)
elif "submission_tracker" in globals():
    submission_tracker = submission_tracker.copy()
else:
    submission_tracker = pd.DataFrame(
        columns=[
            "submission_file",
            "source",
            "model_family",
            "public_mse",
            "notes",
        ]
    )

tracker_updates = pd.DataFrame([
    {
        "submission_file": "submission_lgbm_t03_base_5fold_oof_foldavg.csv",
        "source": "current notebook",
        "model_family": "LightGBM",
        "public_mse": 90.644,
        "notes": "Selected 12k LightGBM OOF/fold-average submission.",
    },
    {
        "submission_file": "submission_blend_rf_et_lgbm_oof_weighted.csv",
        "source": "current notebook",
        "model_family": "ExtraTrees + LightGBM blend",
        "public_mse": 87.054,
        "notes": "OOF blend: w_rf=0.000, w_et=0.356, w_lgbm=0.644; OOF MSE=93.499458.",
    },
])

submission_tracker = pd.concat(
    [submission_tracker, tracker_updates],
    ignore_index=True
)

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last"
)

submission_tracker.to_csv(tracker_path, index=False)

print("\nUpdated submission tracker with latest public scores.")
print("Tracker path:", tracker_path)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]
assert len(missing_objects) == 0, f"Missing required objects: {missing_objects}"

assert X_train_proc_model.shape[0] == len(y_train), (
    X_train_proc_model.shape,
    len(y_train),
)

assert X_test_proc_model.shape[0] == len(test_ids), (
    X_test_proc_model.shape,
    len(test_ids),
)

print("\nFull train shape:", X_train_proc_model.shape)
print("Full test shape:", X_test_proc_model.shape)
print("Target length:", len(y_train))
print("Test ID length:", len(test_ids))

def as_1d_str_series(x):
    return pd.Series(np.asarray(x).ravel()).astype(str)

# Convert once to memory-conscious contiguous arrays.
X_full_lgb = np.ascontiguousarray(np.asarray(X_train_proc_model, dtype=np.float32))
X_test_lgb = np.ascontiguousarray(np.asarray(X_test_proc_model, dtype=np.float32))
y_full_lgb = np.asarray(y_train, dtype=np.float32).ravel()

n_train = X_full_lgb.shape[0]
n_test = X_test_lgb.shape[0]

# ------------------------------------------------------------
# Overnight LightGBM jobs
# ------------------------------------------------------------

base_lgbm_params = dict(
    objective="regression",
    metric="l2",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=-1,
    force_col_wise=True,

    # Selected T03 structure
    num_leaves=95,
    min_child_samples=60,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=5.0,
    max_depth=-1,
)

overnight_jobs = [
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long80k_lr03",
        "learning_rate": 0.03,
        "n_estimators": 80000,
        "stopping_rounds": 2500,
        "log_period": 2000,
    },
    {
        "artifact_name": "lgbm_t03_base_5fold_oof_long100k_lr02",
        "learning_rate": 0.02,
        "n_estimators": 100000,
        "stopping_rounds": 3000,
        "log_period": 2500,
    },
]

print("\nPlanned overnight jobs:")
for job in overnight_jobs:
    print(
        f"  {job['artifact_name']}: "
        f"lr={job['learning_rate']}, "
        f"n_estimators={job['n_estimators']}, "
        f"early_stopping={job['stopping_rounds']}"
    )

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

completed_lgbm_artifacts = []

# ------------------------------------------------------------
# Function: run one selected LightGBM OOF artifact
# ------------------------------------------------------------

def run_lgbm_oof_artifact(job):
    artifact_name = job["artifact_name"]

    metrics_path = RESULTS_DIR / f"{artifact_name}_fold_metrics.csv"
    oof_npy_path = RESULTS_DIR / f"{artifact_name}_oof.npy"

    oof_csv_path = RESULTS_DIR / f"oof_{artifact_name}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact_name}_foldavg.csv"
    submission_path = Path(f"submission_{artifact_name}_foldavg.csv")

    print("\n" + "#" * 90)
    print("Starting artifact:", artifact_name)
    print("#" * 90)

    params = base_lgbm_params.copy()
    params.update(
        dict(
            n_estimators=job["n_estimators"],
            learning_rate=job["learning_rate"],
        )
    )

    print("\nLightGBM parameters:")
    for k, v in params.items():
        print(f"  {k}: {v}")

    if metrics_path.exists():
        fold_metrics = pd.read_csv(metrics_path)
    else:
        fold_metrics = pd.DataFrame()

    if oof_npy_path.exists():
        oof_pred = np.load(oof_npy_path)
        assert len(oof_pred) == n_train
    else:
        oof_pred = np.full(n_train, np.nan, dtype=np.float32)

    artifact_start = time.time()

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_full_lgb), start=1):

        fold_test_pred_path = RESULTS_DIR / f"{artifact_name}_fold{fold}_test_pred.npy"

        completed_metric_folds = (
            set(fold_metrics["fold"].astype(int).tolist())
            if len(fold_metrics) > 0 and "fold" in fold_metrics.columns
            else set()
        )

        fold_done_in_metrics = fold in completed_metric_folds
        fold_oof_done = np.isfinite(oof_pred[val_idx]).all()
        fold_test_done = fold_test_pred_path.exists()

        if fold_done_in_metrics and fold_oof_done and fold_test_done:
            print(f"\nSkipping already completed fold {fold} for {artifact_name}")
            continue

        if fold_done_in_metrics or fold_oof_done or fold_test_done:
            print(f"\nFold {fold} has incomplete checkpoint state. Rerunning fold cleanly.")

            if len(fold_metrics) > 0 and "fold" in fold_metrics.columns:
                fold_metrics = fold_metrics[fold_metrics["fold"].astype(int) != fold].copy()
                fold_metrics.to_csv(metrics_path, index=False)

            if fold_test_pred_path.exists():
                fold_test_pred_path.unlink()

            oof_pred[val_idx] = np.nan
            np.save(oof_npy_path, oof_pred)

        print("\n" + "=" * 80)
        print(f"Starting {artifact_name}, fold {fold}/5")
        print("=" * 80)

        X_fold_tr = X_full_lgb[tr_idx]
        y_fold_tr = y_full_lgb[tr_idx]
        X_fold_val = X_full_lgb[val_idx]
        y_fold_val = y_full_lgb[val_idx]

        fold_params = params.copy()
        fold_params["random_state"] = RANDOM_STATE + fold

        model = lgb.LGBMRegressor(**fold_params)

        fold_start = time.time()

        model.fit(
            X_fold_tr,
            y_fold_tr,
            eval_set=[(X_fold_val, y_fold_val)],
            eval_metric="l2",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=job["stopping_rounds"],
                    verbose=True,
                ),
                lgb.log_evaluation(period=job["log_period"]),
            ],
        )

        fold_elapsed = time.time() - fold_start

        best_iter = getattr(model, "best_iteration_", None)
        if best_iter is None or best_iter <= 0:
            best_iter = params["n_estimators"]

        val_pred_raw = model.predict(X_fold_val, num_iteration=best_iter)
        val_pred_clip = np.clip(val_pred_raw, 0, 100).astype(np.float32)

        test_pred_raw = model.predict(X_test_lgb, num_iteration=best_iter)
        test_pred_clip = np.clip(test_pred_raw, 0, 100).astype(np.float32)

        oof_pred[val_idx] = val_pred_clip
        np.save(oof_npy_path, oof_pred)
        np.save(fold_test_pred_path, test_pred_clip)

        row = {
            "artifact_name": artifact_name,
            "model_class": "LightGBM",
            "feature_space": "base",
            "fold": fold,
            "fold_train_n": len(tr_idx),
            "fold_valid_n": len(val_idx),
            "best_iteration": best_iter,
            "fold_valid_mse_raw": mean_squared_error(y_fold_val, val_pred_raw),
            "fold_valid_mse_clipped": mean_squared_error(y_fold_val, val_pred_clip),
            "elapsed_sec": fold_elapsed,
        }

        for key, value in params.items():
            row[key] = value

        fold_metrics = pd.concat(
            [fold_metrics, pd.DataFrame([row])],
            ignore_index=True,
        )

        fold_metrics = fold_metrics.sort_values("fold").reset_index(drop=True)
        fold_metrics.to_csv(metrics_path, index=False)

        print(f"\nCompleted {artifact_name}, fold {fold}")
        print("Best iteration:", best_iter)
        print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
        print("Elapsed seconds:", round(fold_elapsed, 2))
        print("Saved metrics checkpoint:", metrics_path)
        print("Saved OOF checkpoint:", oof_npy_path)
        print("Saved fold test prediction:", fold_test_pred_path)

        del model
        del X_fold_tr, y_fold_tr, X_fold_val, y_fold_val
        del val_pred_raw, val_pred_clip, test_pred_raw, test_pred_clip
        gc.collect()

    # --------------------------------------------------------
    # Finalize artifact
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("Completion check for:", artifact_name)
    print("=" * 80)

    n_missing_oof = int(np.isnan(oof_pred).sum())
    print("Missing OOF predictions:", n_missing_oof)

    fold_test_preds = []
    missing_fold_test_files = []

    for fold in range(1, 6):
        fold_test_pred_path = RESULTS_DIR / f"{artifact_name}_fold{fold}_test_pred.npy"

        if not fold_test_pred_path.exists():
            missing_fold_test_files.append(str(fold_test_pred_path))
            continue

        arr = np.load(fold_test_pred_path)
        assert len(arr) == n_test, f"Bad test prediction length for fold {fold}: {len(arr)}"
        fold_test_preds.append(arr.astype(np.float32))

    print("Missing fold test prediction files:", len(missing_fold_test_files))

    if n_missing_oof != 0 or len(missing_fold_test_files) != 0:
        print("\nArtifact is incomplete. Re-run this same cell to resume.")
        for p in missing_fold_test_files:
            print("Missing:", p)

        return {
            "artifact_name": artifact_name,
            "completed": False,
            "oof_mse": np.nan,
            "oof_csv_path": oof_csv_path,
            "testpred_csv_path": testpred_csv_path,
            "submission_path": submission_path,
            "metrics_path": metrics_path,
        }

    overall_oof_mse = mean_squared_error(y_full_lgb, oof_pred)

    final_test_pred = np.clip(
        np.mean(np.vstack(fold_test_preds), axis=0),
        0,
        100,
    ).astype(np.float32)

    if "train_ids" in globals():
        train_id_series = as_1d_str_series(train_ids)
    else:
        train_id_series = pd.Series(np.arange(n_train)).astype(str)

    test_id_series = as_1d_str_series(test_ids)

    oof_df = pd.DataFrame({
        "ASSESSMENT_ID": train_id_series,
        "PERCENT_PROFICIENT_TRUE": y_full_lgb,
        f"OOF_{artifact_name}": oof_pred,
    })
    oof_df.to_csv(oof_csv_path, index=False)

    testpred_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        f"TESTPRED_{artifact_name}": final_test_pred,
    })
    testpred_df.to_csv(testpred_csv_path, index=False)

    submission_df = pd.DataFrame({
        "ASSESSMENT_ID": test_id_series,
        "PERCENT_PROFICIENT": final_test_pred,
    })

    assert submission_df.shape[0] == n_test
    assert list(submission_df.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
    assert submission_df["ASSESSMENT_ID"].isna().sum() == 0
    assert submission_df["PERCENT_PROFICIENT"].isna().sum() == 0

    submission_df.to_csv(submission_path, index=False)

    artifact_elapsed = time.time() - artifact_start

    print("\nOverall OOF MSE clipped:", overall_oof_mse)
    print("Saved OOF file:", oof_csv_path)
    print("Saved test prediction file:", testpred_csv_path)
    print("Saved Kaggle submission:", submission_path)

    print("\nSubmission shape:", submission_df.shape)

    print("\nSubmission prediction summary:")
    print(submission_df["PERCENT_PROFICIENT"].describe().to_string())

    print("\nFold metrics:")
    display(
        fold_metrics.sort_values("fold")[
            [
                "fold",
                "fold_valid_mse_clipped",
                "fold_valid_mse_raw",
                "best_iteration",
                "elapsed_sec",
            ]
        ].reset_index(drop=True)
    )

    print("\nArtifact elapsed seconds:", round(artifact_elapsed, 2))

    # Update trackers
    if "best_models_by_class" not in globals():
        globals()["best_models_by_class"] = {}

    best_models_by_class[artifact_name] = {
        "model_class": "LightGBM",
        "feature_space": "base",
        "oof_mse": float(overall_oof_mse),
        "submission_path": str(submission_path),
        "notes": "Overnight selected LightGBM OOF artifact.",
    }

    tracker_row = pd.DataFrame([
        {
            "submission_file": submission_path.name,
            "source": "current notebook",
            "model_family": "LightGBM",
            "public_mse": np.nan,
            "notes": f"Overnight LightGBM artifact {artifact_name}; OOF MSE={overall_oof_mse:.6f}.",
        }
    ])

    global submission_tracker
    submission_tracker = pd.concat(
        [submission_tracker, tracker_row],
        ignore_index=True,
    )

    submission_tracker = submission_tracker.drop_duplicates(
        subset=["submission_file"],
        keep="last",
    )

    submission_tracker.to_csv(tracker_path, index=False)

    return {
        "artifact_name": artifact_name,
        "completed": True,
        "oof_mse": float(overall_oof_mse),
        "oof_csv_path": oof_csv_path,
        "testpred_csv_path": testpred_csv_path,
        "submission_path": submission_path,
        "metrics_path": metrics_path,
    }


# ------------------------------------------------------------
# Run overnight LightGBM jobs
# ------------------------------------------------------------

suite_start = time.time()

for job in overnight_jobs:
    result = run_lgbm_oof_artifact(job)
    completed_lgbm_artifacts.append(result)
    gc.collect()

suite_elapsed = time.time() - suite_start

print("\n" + "#" * 90)
print("Overnight LightGBM suite summary")
print("#" * 90)

suite_summary = pd.DataFrame(completed_lgbm_artifacts)
display(suite_summary)

print("Suite elapsed seconds:", round(suite_elapsed, 2))

# ------------------------------------------------------------
# Automatic re-blend screen
# Includes:
# - RF 500
# - ExtraTrees safe
# - original 12k LightGBM
# - completed overnight LightGBM artifacts
# ------------------------------------------------------------

print("\n" + "#" * 90)
print("Automatic re-blend screen")
print("#" * 90)

blend_components = [
    {
        "name": "rf500",
        "oof_path": RESULTS_DIR / "oof_rf_500_base.csv",
        "test_path": RESULTS_DIR / "testpred_rf_500_base_folds.csv",
        "oof_pred_preferred": ["rf_500_base_oof_pred", "oof_pred", "pred"],
        "test_pred_preferred": ["rf_500_base_foldavg_pred", "PERCENT_PROFICIENT", "test_pred", "pred"],
    },
    {
        "name": "et_safe",
        "oof_path": RESULTS_DIR / "oof_extratrees_safe_base.csv",
        "test_path": RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
        "oof_pred_preferred": ["oof_pred", "oof_pred_clipped", "pred"],
        "test_pred_preferred": ["PERCENT_PROFICIENT", "test_pred_avg", "test_pred", "pred"],
    },
    {
        "name": "lgbm12k",
        "oof_path": RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
        "test_path": RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
        "oof_pred_preferred": ["OOF_lgbm_t03_base_5fold_oof", "oof_pred", "pred"],
        "test_pred_preferred": ["TESTPRED_lgbm_t03_base_5fold_oof", "PERCENT_PROFICIENT", "test_pred", "pred"],
    },
]

for result in completed_lgbm_artifacts:
    if result["completed"]:
        artifact_name = result["artifact_name"]
        blend_components.append({
            "name": artifact_name.replace("lgbm_t03_base_5fold_oof_", "lgbm_"),
            "oof_path": Path(result["oof_csv_path"]),
            "test_path": Path(result["testpred_csv_path"]),
            "oof_pred_preferred": [f"OOF_{artifact_name}", "oof_pred", "pred"],
            "test_pred_preferred": [f"TESTPRED_{artifact_name}", "PERCENT_PROFICIENT", "test_pred", "pred"],
        })

available_components = []

for comp in blend_components:
    if comp["oof_path"].exists() and comp["test_path"].exists():
        available_components.append(comp)
    else:
        print("Skipping missing component:", comp["name"])

print("\nAvailable blend components:")
for comp in available_components:
    print(" ", comp["name"])

assert len(available_components) >= 2, "Need at least two components for blending."

def read_pred_csv(path):
    return pd.read_csv(path, dtype={"ASSESSMENT_ID": str})

def first_existing_column(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find {label}. Tried {candidates}. Available columns: {list(df.columns)}"
    )

def pick_prediction_column(df, preferred_cols, label, target_cols=None):
    if target_cols is None:
        target_cols = []

    for col in preferred_cols:
        if col in df.columns:
            return col

    excluded_cols = set(["ASSESSMENT_ID"] + target_cols)

    numeric_cols = [
        c for c in df.columns
        if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    pred_like_cols = [
        c for c in numeric_cols
        if (
            "pred" in c.lower()
            or "proficient" in c.lower()
            or "oof" in c.lower()
            or "foldavg" in c.lower()
            or "fold_avg" in c.lower()
            or "avg" in c.lower()
        )
    ]

    if len(pred_like_cols) == 1:
        return pred_like_cols[0]

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise ValueError(
        f"Could not identify prediction column for {label}.\n"
        f"Columns: {list(df.columns)}\n"
        f"Numeric columns: {numeric_cols}\n"
        f"Prediction-like columns: {pred_like_cols}"
    )

def load_oof_component(comp):
    df = read_pred_csv(comp["oof_path"])

    y_col = first_existing_column(
        df,
        ["y_true", "PERCENT_PROFICIENT_TRUE", "target", "true", "actual"],
        f"{comp['name']} target column",
    )

    pred_col = pick_prediction_column(
        df,
        comp["oof_pred_preferred"],
        f"{comp['name']} OOF prediction",
        target_cols=[y_col],
    )

    out = df[["ASSESSMENT_ID", y_col, pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)

    out = out.rename(columns={
        y_col: "y_true",
        pred_col: comp["name"],
    })

    print(f"{comp['name']} OOF prediction column:", pred_col)

    return out

def load_test_component(comp):
    df = read_pred_csv(comp["test_path"])

    pred_col = pick_prediction_column(
        df,
        comp["test_pred_preferred"],
        f"{comp['name']} test prediction",
        target_cols=[],
    )

    out = df[["ASSESSMENT_ID", pred_col]].copy()
    out["ASSESSMENT_ID"] = out["ASSESSMENT_ID"].astype(str)

    out = out.rename(columns={pred_col: comp["name"]})

    print(f"{comp['name']} test prediction column:", pred_col)

    return out

# Load and merge OOF components
merged_oof = None
component_names = []

for comp in available_components:
    this_oof = load_oof_component(comp)
    component_names.append(comp["name"])

    if merged_oof is None:
        merged_oof = this_oof.copy()
    else:
        merged_oof = merged_oof.merge(
            this_oof,
            on="ASSESSMENT_ID",
            how="inner",
            suffixes=("", "_new"),
        )

        assert np.allclose(
            merged_oof["y_true"].to_numpy(dtype=float),
            merged_oof["y_true_new"].to_numpy(dtype=float),
        ), f"Target mismatch after merging {comp['name']}"

        merged_oof = merged_oof.drop(columns=["y_true_new"])

assert len(merged_oof) == n_train, f"OOF merge row count {len(merged_oof)} != {n_train}"

y_blend = merged_oof["y_true"].to_numpy(dtype=float)

component_oof_rows = []

for name in component_names:
    pred = merged_oof[name].to_numpy(dtype=float)

    component_oof_rows.append({
        "component": name,
        "oof_mse_clipped": mean_squared_error(y_blend, np.clip(pred, 0, 100)),
        "pred_mean": np.mean(np.clip(pred, 0, 100)),
        "pred_std": np.std(np.clip(pred, 0, 100)),
    })

component_oof_results = (
    pd.DataFrame(component_oof_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

print("\nComponent OOF results:")
display(component_oof_results)

pred_matrix = np.vstack([
    merged_oof[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

# ------------------------------------------------------------
# Weight screen
# For 3+ components, use strong candidates:
# - pure models
# - pairwise fine grids
# - lgbm-focused random Dirichlet candidates
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

weights = []
sources = []

def add_weights(w, source):
    w = np.asarray(w, dtype=float)

    if np.any(w < -1e-12):
        return

    total = w.sum()
    if total <= 0:
        return

    w = w / total
    w[np.abs(w) < 1e-12] = 0.0

    key = tuple(np.round(w, 6))

    if key in seen:
        return

    seen.add(key)
    weights.append(w)
    sources.append(source)

seen = set()
n_comp = len(component_names)

# Pure models
for j in range(n_comp):
    w = np.zeros(n_comp)
    w[j] = 1.0
    add_weights(w, "pure")

# Pairwise fine grids
grid = np.linspace(0, 1, 1001)

for i in range(n_comp):
    for j in range(i + 1, n_comp):
        for a in grid:
            w = np.zeros(n_comp)
            w[i] = a
            w[j] = 1.0 - a
            add_weights(w, "pairwise_grid")

# Random lgbm-focused convex search.
# This lets the search consider 3+ way blends without a huge grid.
# Stronger components receive larger Dirichlet concentration.
oof_mse_by_name = {
    row["component"]: row["oof_mse_clipped"]
    for _, row in component_oof_results.iterrows()
}

best_single_mse = min(oof_mse_by_name.values())

alpha = []

for name in component_names:
    mse = oof_mse_by_name[name]

    if mse <= best_single_mse + 2:
        alpha.append(8.0)
    elif "lgbm" in name.lower():
        alpha.append(5.0)
    elif "et" in name.lower():
        alpha.append(3.0)
    else:
        alpha.append(1.0)

alpha = np.asarray(alpha, dtype=float)

for _ in range(30000):
    add_weights(rng.dirichlet(alpha), "dirichlet_focused")

weight_matrix = np.vstack(weights).astype(np.float32)

print("\nNumber of blend candidates:", len(weight_matrix))

blend_rows = []

for idx, w in enumerate(weight_matrix):
    pred = np.dot(w, pred_matrix)
    pred_clip = np.clip(pred, 0, 100)

    row = {
        "source": sources[idx],
        "oof_mse_clipped": mean_squared_error(y_blend, pred_clip),
        "oof_mse_raw": mean_squared_error(y_blend, pred),
    }

    for name, val in zip(component_names, w):
        row[f"w_{name}"] = float(val)

    blend_rows.append(row)

blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values("oof_mse_clipped")
    .reset_index(drop=True)
)

best_blend = blend_results.iloc[0].to_dict()

print("\nBest automatic re-blend:")
display(pd.DataFrame([best_blend]))

print("\nTop 20 automatic re-blend candidates:")
display(blend_results.head(20))

# ------------------------------------------------------------
# Create automatic re-blend submission
# ------------------------------------------------------------

merged_test = None

for comp in available_components:
    this_test = load_test_component(comp)

    if merged_test is None:
        merged_test = this_test.copy()
    else:
        merged_test = merged_test.merge(this_test, on="ASSESSMENT_ID", how="inner")

assert len(merged_test) == n_test, f"Test merge row count {len(merged_test)} != {n_test}"

# Restore original test_ids order
test_order = pd.DataFrame({
    "ASSESSMENT_ID": as_1d_str_series(test_ids),
    "_test_order": np.arange(n_test),
})

merged_test = test_order.merge(merged_test, on="ASSESSMENT_ID", how="left")

for name in component_names:
    assert merged_test[name].isna().sum() == 0, f"Missing test predictions for {name}"

merged_test = (
    merged_test
    .sort_values("_test_order")
    .drop(columns=["_test_order"])
    .reset_index(drop=True)
)

best_weights = np.array(
    [best_blend[f"w_{name}"] for name in component_names],
    dtype=np.float32,
)

test_matrix = np.vstack([
    merged_test[name].to_numpy(dtype=float)
    for name in component_names
]).astype(np.float32)

final_blend_pred = np.clip(
    np.dot(best_weights, test_matrix),
    0,
    100,
).astype(np.float32)

auto_blend_name = "blend_auto_et_lgbm_long_oof_weighted"

weight_screen_path = RESULTS_DIR / f"{auto_blend_name}_weight_screen.csv"
testpred_blend_path = RESULTS_DIR / f"testpred_{auto_blend_name}.csv"
submission_blend_path = Path(f"submission_{auto_blend_name}.csv")

blend_results.to_csv(weight_screen_path, index=False)

testpred_blend_df = merged_test[["ASSESSMENT_ID"] + component_names].copy()
testpred_blend_df["PERCENT_PROFICIENT"] = final_blend_pred
testpred_blend_df.to_csv(testpred_blend_path, index=False)

submission_blend = pd.DataFrame({
    "ASSESSMENT_ID": as_1d_str_series(test_ids),
    "PERCENT_PROFICIENT": final_blend_pred,
})

assert submission_blend.shape == (n_test, 2)
assert list(submission_blend.columns) == ["ASSESSMENT_ID", "PERCENT_PROFICIENT"]
assert submission_blend["ASSESSMENT_ID"].isna().sum() == 0
assert submission_blend["PERCENT_PROFICIENT"].isna().sum() == 0

submission_blend.to_csv(submission_blend_path, index=False)

print("\nSaved automatic re-blend files:")
print("Weight screen:", weight_screen_path.resolve())
print("Test blend predictions:", testpred_blend_path.resolve())
print("Blend submission:", submission_blend_path.resolve())

print("\nAutomatic re-blend submission summary:")
print(submission_blend["PERCENT_PROFICIENT"].describe().to_string())

print("\nAutomatic re-blend submission shape:", submission_blend.shape)

print("\nFirst 5 rows:")
print(submission_blend.head().to_string(index=False))

# Update tracker for automatic blend
best_auto_blend_oof = float(best_blend["oof_mse_clipped"])

weight_note_parts = []
for name in component_names:
    weight_note_parts.append(f"{name}={best_blend[f'w_{name}']:.4f}")

tracker_auto_blend_row = pd.DataFrame([
    {
        "submission_file": submission_blend_path.name,
        "source": "current notebook",
        "model_family": "automatic OOF blend",
        "public_mse": np.nan,
        "notes": (
            f"Automatic post-long-run OOF blend; "
            f"OOF MSE={best_auto_blend_oof:.6f}; "
            + ", ".join(weight_note_parts)
        ),
    }
])

submission_tracker = pd.concat(
    [submission_tracker, tracker_auto_blend_row],
    ignore_index=True,
)

submission_tracker = submission_tracker.drop_duplicates(
    subset=["submission_file"],
    keep="last",
)

submission_tracker.to_csv(tracker_path, index=False)

print("\nUpdated submission tracker:", tracker_path.resolve())

print("\nDecision guide:")
print("Known public scores:")
print("  Pure 12k LightGBM: 90.644")
print("  ET + 12k LightGBM blend: 87.054")
print("\nCompare the new long-run OOF results and automatic blend OOF against:")
print("  Old pure LightGBM OOF: 98.7794")
print("  Old ET + LightGBM blend OOF: 93.4995")
print("\nIf the automatic long blend improves OOF versus 93.4995, it is the next natural Kaggle submission.")

# Clean local arrays
del X_full_lgb, X_test_lgb, y_full_lgb
gc.collect()

LightGBM version: 4.6.0
Results folder: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results

Updated submission tracker with latest public scores.
Tracker path: model_results/submission_tracker.csv

Full train shape: (144921, 162)
Full test shape: (48307, 162)
Target length: 144921
Test ID length: 48307

Planned overnight jobs:
  lgbm_t03_base_5fold_oof_long80k_lr03: lr=0.03, n_estimators=80000, early_stopping=2500
  lgbm_t03_base_5fold_oof_long100k_lr02: lr=0.02, n_estimators=100000, early_stopping=3000

##########################################################################################
Starting artifact: lgbm_t03_base_5fold_oof_long80k_lr03
##########################################################################################

LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  num_leaves: 95
  min_child_samples: 60
  subsample:

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.764534,96.739767,39024,2553.609186
1,2,91.844963,92.593493,39385,1485.937457
2,3,94.334686,95.260975,34065,591.087250
3,4,92.318062,93.309677,36884,628.429420
4,5,96.555138,97.543099,40463,688.799824



Artifact elapsed seconds: 13004.18

##########################################################################################
Starting artifact: lgbm_t03_base_5fold_oof_long100k_lr02
##########################################################################################

LightGBM parameters:
  objective: regression
  metric: l2
  random_state: 9890
  n_jobs: 1
  verbosity: -1
  force_col_wise: True
  num_leaves: 95
  min_child_samples: 60
  subsample: 0.85
  subsample_freq: 1
  colsample_bytree: 0.9
  reg_alpha: 0.0
  reg_lambda: 5.0
  max_depth: -1
  n_estimators: 100000
  learning_rate: 0.02

Starting lgbm_t03_base_5fold_oof_long100k_lr02, fold 1/5
Training until validation scores don't improve for 3000 rounds
[2500]	valid_0's l2: 132.896
[5000]	valid_0's l2: 119.103
[7500]	valid_0's l2: 111.575
[10000]	valid_0's l2: 107.136
[12500]	valid_0's l2: 104.191
[15000]	valid_0's l2: 102.111
[17500]	valid_0's l2: 100.632
[20000]	valid_0's l2: 99.4575
[22500]	valid_0's l2: 98.6736
[25000

,fold,fold_valid_mse_clipped,fold_valid_mse_raw,best_iteration,elapsed_sec
0,1,95.241920,96.194398,54700,935.653679
1,2,91.291603,92.035591,52940,884.257007
2,3,93.777443,94.710169,59506,1025.118408
3,4,92.410759,93.340667,53392,894.905689
4,5,95.909294,96.860579,64138,1080.716788



Artifact elapsed seconds: 11441.8

##########################################################################################
Overnight LightGBM suite summary
##########################################################################################


...

,source,oof_mse_clipped,oof_mse_raw,w_rf500,w_et_safe,w_lgbm12k,w_lgbm_long80k_lr03,w_lgbm_long100k_lr02
0,pairwise_grid,88.966368,88.966368,0.0,0.319,0.0,0.0,0.681



Top 20 automatic re-blend candidates:


,source,oof_mse_clipped,oof_mse_raw,w_rf500,w_et_safe,w_lgbm12k,w_lgbm_long80k_lr03,w_lgbm_long100k_lr02
0,pairwise_grid,88.966368,88.966368,0.0,0.319,0.0,0.0,0.681
1,pairwise_grid,88.966373,88.966373,0.0,0.318,0.0,0.0,0.682
2,pairwise_grid,88.966458,88.966458,0.0,0.320,0.0,0.0,0.680
3,pairwise_grid,88.966470,88.966470,0.0,0.317,0.0,0.0,0.683
4,pairwise_grid,88.966640,88.966640,0.0,0.321,0.0,0.0,0.679
5,pairwise_grid,88.966662,88.966662,0.0,0.316,0.0,0.0,0.684
6,pairwise_grid,88.966919,88.966919,0.0,0.322,0.0,0.0,0.678
7,pairwise_grid,88.966949,88.966949,0.0,0.315,0.0,0.0,0.685
8,pairwise_grid,88.967288,88.967288,0.0,0.323,0.0,0.0,0.677
9,pairwise_grid,88.967329,88.967329,0.0,0.314,0.0,0.0,0.686


rf500 test prediction column: rf_500_base_foldavg_pred
et_safe test prediction column: PERCENT_PROFICIENT
lgbm12k test prediction column: TESTPRED_lgbm_t03_base_5fold_oof
lgbm_long80k_lr03 test prediction column: TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03
lgbm_long100k_lr02 test prediction column: TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02

Saved automatic re-blend files:
Weight screen: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/blend_auto_et_lgbm_long_oof_weighted_weight_screen.csv
Test blend predictions: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/model_results/testpred_blend_auto_et_lgbm_long_oof_weighted.csv
Blend submission: /Users/saadmanchowdhury/Desktop/All Github projects/02._ml_prediction_competition/Working on GPT restart/submission_blend_auto_et_lgbm_long_oof_weighted.csv

Automatic re-blend submission summary:
count    48307.000000

0

### Overnight selected LightGBM long-run suite and automatic re-blend

The earlier 12k selected LightGBM artifact reached the estimator cap on all folds, so two longer versions of the same T03 configuration were trained with fold-level checkpointing:

- `long80k_lr03`: learning rate 0.03, 80,000 maximum estimators, early stopping 2,500
- `long100k_lr02`: learning rate 0.02, 100,000 maximum estimators, early stopping 3,000

Both completed successfully. The 12k LightGBM OOF MSE was 98.7794. The long 80k run improved OOF MSE to 94.1635, and the long 100k run improved OOF MSE to 93.7262. This confirms that the original 12k LightGBM was under-trained.

An automatic convex re-blend was then run using the saved OOF artifacts from Random Forest, ExtraTrees, the original 12k LightGBM, and the completed long LightGBM runs. The best blend used:

- ExtraTrees safe base: 0.319
- LightGBM long100k_lr02: 0.681

All other components received zero weight. The new blend OOF MSE was 88.9664, improving substantially over the previous ET + 12k LightGBM blend OOF MSE of 93.4995. The corresponding submission file is:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`

This is the next primary Kaggle submission candidate.

### Kaggle result: automatic long LightGBM + ExtraTrees blend

The automatic long-run blend was submitted to Kaggle as:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`

This submission achieved a public leaderboard MSE of **80.662**, making it the best clean current-notebook result so far.

This is a large improvement over the previous best clean blend:

- Previous best clean blend: `submission_blend_rf_et_lgbm_oof_weighted.csv`
  - Public MSE: **87.054**
- New best clean blend: `submission_blend_auto_et_lgbm_long_oof_weighted.csv`
  - Public MSE: **80.662**
- Public leaderboard improvement: **6.392 MSE points**

The local OOF evidence was directionally consistent with the public leaderboard result. The previous ExtraTrees + 12k LightGBM blend had OOF MSE **93.4995**, while the new automatic blend had OOF MSE **88.9664**. The best automatic blend used:

- ExtraTrees safe base: **0.319**
- LightGBM `long100k_lr02`: **0.681**

Random Forest, the original 12k LightGBM, and the `long80k_lr03` LightGBM variant received zero weight in the best automatic blend.

This result confirms that the original 12k LightGBM was under-trained and that the longer `long100k_lr02` LightGBM artifact added substantial predictive strength. It also confirms that ExtraTrees, although weaker as a standalone model, provides useful complementary signal when blended with LightGBM.

Current best clean current-notebook submission:

`submission_blend_auto_et_lgbm_long_oof_weighted.csv`  
Public MSE: **80.662**

In [41]:
# 22A. Post-80.662: update tracker + refine/stack saved OOF artifacts
# This cell is mostly fast. It:
# 1. records the new 80.662 public score,
# 2. reloads saved RF / ExtraTrees / LightGBM OOF artifacts,
# 3. refines the ET + long100k blend continuously,
# 4. tries conservative calibration and simple OOF meta-stacking,
# 5. saves candidate submissions for review.

import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize, minimize_scalar
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV, LinearRegression

warnings.filterwarnings("ignore")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

TARGET_NAME = "PERCENT_PROFICIENT"
ID_NAME = "ASSESSMENT_ID"

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clipped(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = clip100(pred)
    return float(np.mean((y_true - pred) ** 2))

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of these files exist:\n" + "\n".join(map(str, paths)))

def ids_from_global(primary_name, fallback_name, expected_len):
    if primary_name in globals():
        obj = globals()[primary_name]
        if isinstance(obj, pd.DataFrame):
            if ID_NAME in obj.columns:
                s = obj[ID_NAME]
            else:
                s = obj.iloc[:, 0]
        else:
            s = pd.Series(obj)
    elif fallback_name in globals():
        obj = globals()[fallback_name]
        if isinstance(obj, pd.DataFrame) and ID_NAME in obj.columns:
            s = obj[ID_NAME]
        else:
            return None
    else:
        return None

    s = pd.Series(s).astype(str).reset_index(drop=True)
    if len(s) != expected_len:
        raise ValueError(f"{primary_name}/{fallback_name} length mismatch: {len(s)} vs {expected_len}")
    return pd.Series(s.values, name=ID_NAME)

def pick_pred_col(df, kind="oof", preferred=None):
    if preferred is not None and preferred in df.columns:
        return preferred

    exclude = {ID_NAME, "fold", "FOLD", "index", "Unnamed: 0"}
    if kind == "oof":
        exclude |= {TARGET_NAME, "target", "TARGET", "y", "Y", "actual", "truth"}

    numeric_like = []
    for c in df.columns:
        if c in exclude:
            continue
        converted = pd.to_numeric(df[c], errors="coerce")
        if converted.notna().mean() > 0.95:
            numeric_like.append(c)

    if not numeric_like:
        raise ValueError(f"No numeric-like prediction columns found. Columns were: {list(df.columns)}")

    priority_keywords = ["oof", "testpred", "foldavg", "pred", "percent_proficient"]
    lower_map = {c: c.lower() for c in numeric_like}

    for kw in priority_keywords:
        for c in numeric_like:
            if kw in lower_map[c]:
                return c

    if len(numeric_like) == 1:
        return numeric_like[0]

    print("Multiple numeric-like columns found; using the last one:")
    print(numeric_like)
    return numeric_like[-1]

def load_prediction(path, expected_len, expected_ids=None, preferred_col=None, kind="oof"):
    path = Path(path)
    df = pd.read_csv(path)
    pred_col = pick_pred_col(df, kind=kind, preferred=preferred_col)

    if ID_NAME in df.columns and expected_ids is not None:
        left = pd.DataFrame({ID_NAME: expected_ids.astype(str).values})
        right = df[[ID_NAME, pred_col]].copy()
        right[ID_NAME] = right[ID_NAME].astype(str)

        merged = left.merge(right, on=ID_NAME, how="left")
        missing = merged[pred_col].isna().sum()
        if missing > 0:
            raise ValueError(f"{path} has {missing} missing predictions after ID merge.")

        return pd.to_numeric(merged[pred_col], errors="raise").to_numpy(dtype=float), pred_col

    if len(df) != expected_len:
        raise ValueError(f"{path} length mismatch: {len(df)} vs expected {expected_len}")

    return pd.to_numeric(df[pred_col], errors="raise").to_numpy(dtype=float), pred_col

def save_candidate(candidate_name, test_pred, oof_pred=None, extra_info=None):
    test_pred = clip100(test_pred)

    testpred_path = RESULTS_DIR / f"testpred_{candidate_name}.csv"
    sub_path = Path(f"submission_{candidate_name}.csv")

    pd.DataFrame({
        ID_NAME: test_id_series.astype(str).values,
        f"TESTPRED_{candidate_name}": test_pred
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_NAME: test_id_series.astype(str).values,
        TARGET_NAME: test_pred
    }).to_csv(sub_path, index=False)

    if oof_pred is not None:
        oof_pred = clip100(oof_pred)
        if train_id_series is not None:
            oof_df = pd.DataFrame({
                ID_NAME: train_id_series.astype(str).values,
                TARGET_NAME: y,
                f"OOF_{candidate_name}": oof_pred
            })
        else:
            oof_df = pd.DataFrame({
                TARGET_NAME: y,
                f"OOF_{candidate_name}": oof_pred
            })
        oof_df.to_csv(RESULTS_DIR / f"oof_{candidate_name}.csv", index=False)

    saved_submissions.append({
        "candidate": candidate_name,
        "submission_path": str(sub_path),
        "testpred_path": str(testpred_path),
        **(extra_info or {})
    })

    return sub_path

# --------------------------------------------------------------------------------------
# Update submission tracker with the 80.662 public score
# --------------------------------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"

new_public_row = {
    "file": "submission_blend_auto_et_lgbm_long_oof_weighted.csv",
    "source": "current notebook",
    "model_family": "OOF blend",
    "public_mse": 80.662,
    "notes": "Automatic long-run blend: 0.319 ExtraTrees safe + 0.681 LightGBM long100k_lr02"
}

if tracker_path.exists():
    tracker = pd.read_csv(tracker_path)
else:
    tracker = pd.DataFrame()

for col in new_public_row:
    if col not in tracker.columns:
        tracker[col] = np.nan

row_to_add = {col: new_public_row.get(col, np.nan) for col in tracker.columns}
tracker = pd.concat([tracker, pd.DataFrame([row_to_add])], ignore_index=True)

file_col = "file" if "file" in tracker.columns else tracker.columns[0]
tracker = tracker.drop_duplicates(subset=[file_col], keep="last")
tracker.to_csv(tracker_path, index=False)

print(f"Updated tracker: {tracker_path}")

# --------------------------------------------------------------------------------------
# Load target and IDs
# --------------------------------------------------------------------------------------

y = np.asarray(y_train, dtype=float).ravel()
n_train = len(y)

train_id_series = ids_from_global("train_ids", "scores_training", n_train)
test_id_series = ids_from_global("test_ids", "scores_test", len(X_test_proc_model))

if test_id_series is None:
    raise ValueError("Could not find test IDs. Expected test_ids or scores_test with ASSESSMENT_ID.")

# --------------------------------------------------------------------------------------
# Load saved OOF/test prediction components
# --------------------------------------------------------------------------------------

component_specs = {
    "rf500": {
        "oof_paths": ["model_results/oof_rf_500_base.csv"],
        "test_paths": ["model_results/testpred_rf_500_base_folds.csv"],
        "oof_col": "rf_500_base_oof_pred",
        "test_col": "rf_500_base_foldavg_pred",
    },
    "et_safe": {
        "oof_paths": ["model_results/oof_extratrees_safe_base.csv"],
        "test_paths": [
            "model_results/testpred_extratrees_safe_base_foldavg.csv",
            "model_results/testpred_extratrees_safe_base.csv",
        ],
        "oof_col": "oof_pred",
        "test_col": TARGET_NAME,
    },
    "lgbm12k": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof",
    },
    "lgbm_long80k_lr03": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof_long80k_lr03",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03",
    },
    "lgbm_long100k_lr02": {
        "oof_paths": ["model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv"],
        "test_paths": ["model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv"],
        "oof_col": "OOF_lgbm_t03_base_5fold_oof_long100k_lr02",
        "test_col": "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02",
    },
}

oof_components = {}
test_components = {}
loaded_rows = []

for name, spec in component_specs.items():
    try:
        oof_path = first_existing(spec["oof_paths"])
        test_path = first_existing(spec["test_paths"])

        oof_pred, oof_col = load_prediction(
            oof_path,
            expected_len=n_train,
            expected_ids=train_id_series,
            preferred_col=spec.get("oof_col"),
            kind="oof"
        )

        test_pred, test_col = load_prediction(
            test_path,
            expected_len=len(test_id_series),
            expected_ids=test_id_series,
            preferred_col=spec.get("test_col"),
            kind="test"
        )

        oof_components[name] = clip100(oof_pred)
        test_components[name] = clip100(test_pred)

        loaded_rows.append({
            "component": name,
            "oof_path": str(oof_path),
            "oof_col": oof_col,
            "test_path": str(test_path),
            "test_col": test_col,
            "oof_mse_clipped": mse_clipped(y, oof_pred),
            "oof_pred_mean": float(np.mean(clip100(oof_pred))),
            "oof_pred_std": float(np.std(clip100(oof_pred))),
            "test_pred_mean": float(np.mean(clip100(test_pred))),
            "test_pred_std": float(np.std(clip100(test_pred))),
        })

    except Exception as e:
        print(f"Skipped {name}: {e}")

loaded_df = pd.DataFrame(loaded_rows).sort_values("oof_mse_clipped")
print("\nLoaded components:")
try:
    display(loaded_df)
except NameError:
    print(loaded_df.to_string(index=False))

required = {"et_safe", "lgbm_long100k_lr02"}
missing_required = required - set(oof_components)
if missing_required:
    raise ValueError(f"Missing required components for the main refinement: {missing_required}")

# --------------------------------------------------------------------------------------
# Candidate screen
# --------------------------------------------------------------------------------------

summary_rows = []
saved_submissions = []

def add_summary(name, oof_pred, test_pred=None, source="candidate", details=None):
    row = {
        "name": name,
        "source": source,
        "oof_mse_clipped": mse_clipped(y, oof_pred),
        "pred_mean_oof": float(np.mean(clip100(oof_pred))),
        "pred_std_oof": float(np.std(clip100(oof_pred))),
    }

    if test_pred is not None:
        row.update({
            "pred_mean_test": float(np.mean(clip100(test_pred))),
            "pred_std_test": float(np.std(clip100(test_pred))),
            "pred_min_test": float(np.min(clip100(test_pred))),
            "pred_max_test": float(np.max(clip100(test_pred))),
        })

    if details:
        row.update(details)

    summary_rows.append(row)

# Standalone components
for name in oof_components:
    add_summary(
        name=name,
        oof_pred=oof_components[name],
        test_pred=test_components[name],
        source="component"
    )

# Recreate submitted 80.662 blend
submitted_oof = (
    0.319 * oof_components["et_safe"]
    + 0.681 * oof_components["lgbm_long100k_lr02"]
)
submitted_test = (
    0.319 * test_components["et_safe"]
    + 0.681 * test_components["lgbm_long100k_lr02"]
)
add_summary(
    name="submitted_public80p662_recreated",
    oof_pred=submitted_oof,
    test_pred=submitted_test,
    source="known_submission",
    details={"w_et_safe": 0.319, "w_lgbm_long100k_lr02": 0.681}
)

# 1D continuous refinement of ET + long100k weight
def scalar_blend_loss(w_et):
    p = (
        w_et * oof_components["et_safe"]
        + (1.0 - w_et) * oof_components["lgbm_long100k_lr02"]
    )
    return mse_clipped(y, p)

res_scalar = minimize_scalar(
    scalar_blend_loss,
    bounds=(0.0, 1.0),
    method="bounded",
    options={"xatol": 1e-10}
)

w_et_refined = float(res_scalar.x)
w_lgbm_refined = 1.0 - w_et_refined

scalar_oof = (
    w_et_refined * oof_components["et_safe"]
    + w_lgbm_refined * oof_components["lgbm_long100k_lr02"]
)
scalar_test = (
    w_et_refined * test_components["et_safe"]
    + w_lgbm_refined * test_components["lgbm_long100k_lr02"]
)

add_summary(
    name="blend_et_lgbm_long100_scalar_refined",
    oof_pred=scalar_oof,
    test_pred=scalar_test,
    source="scalar_minimize",
    details={"w_et_safe": w_et_refined, "w_lgbm_long100k_lr02": w_lgbm_refined}
)

save_candidate(
    "blend_et_lgbm_long100_scalar_refined",
    scalar_test,
    scalar_oof,
    extra_info={"w_et_safe": w_et_refined, "w_lgbm_long100k_lr02": w_lgbm_refined}
)

# Convex all-component optimization
component_order = [
    name for name in [
        "rf500",
        "et_safe",
        "lgbm12k",
        "lgbm_long80k_lr03",
        "lgbm_long100k_lr02",
    ]
    if name in oof_components
]

X_meta = np.column_stack([oof_components[name] for name in component_order])
X_test_meta = np.column_stack([test_components[name] for name in component_order])

def convex_loss(w):
    return mse_clipped(y, X_meta @ np.asarray(w))

x0 = np.ones(len(component_order)) / len(component_order)
if "et_safe" in component_order and "lgbm_long100k_lr02" in component_order:
    x0 = np.zeros(len(component_order))
    x0[component_order.index("et_safe")] = w_et_refined
    x0[component_order.index("lgbm_long100k_lr02")] = w_lgbm_refined

res_convex = minimize(
    convex_loss,
    x0=x0,
    method="SLSQP",
    bounds=[(0.0, 1.0)] * len(component_order),
    constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    options={"maxiter": 1000, "ftol": 1e-12}
)

w_convex = np.asarray(res_convex.x, dtype=float)
convex_oof = X_meta @ w_convex
convex_test = X_test_meta @ w_convex

details_convex = {
    "convex_success": bool(res_convex.success),
    "convex_message": str(res_convex.message),
}
for name, w in zip(component_order, w_convex):
    details_convex[f"w_{name}"] = float(w)

add_summary(
    name="blend_all_components_convex_refined",
    oof_pred=convex_oof,
    test_pred=convex_test,
    source="slsqp_convex",
    details=details_convex
)

save_candidate(
    "blend_all_components_convex_refined",
    convex_test,
    convex_oof,
    extra_info=details_convex
)

# Conservative affine calibration around the refined ET + long100 blend
# Bounds are intentionally conservative to avoid making a wild leaderboard-overfit correction.
def calibration_loss(ab):
    a, b = ab
    return mse_clipped(y, a * scalar_oof + b)

res_cal = minimize(
    calibration_loss,
    x0=np.array([1.0, 0.0]),
    method="L-BFGS-B",
    bounds=[(0.90, 1.10), (-3.0, 3.0)],
    options={"maxiter": 1000, "ftol": 1e-12}
)

a_cal, b_cal = map(float, res_cal.x)
cal_oof = a_cal * scalar_oof + b_cal
cal_test = a_cal * scalar_test + b_cal

add_summary(
    name="blend_et_lgbm_long100_scalar_refined_calibrated",
    oof_pred=cal_oof,
    test_pred=cal_test,
    source="conservative_affine_calibration",
    details={
        "a": a_cal,
        "b": b_cal,
        "calibration_success": bool(res_cal.success),
        "base_w_et_safe": w_et_refined,
        "base_w_lgbm_long100k_lr02": w_lgbm_refined,
    }
)

save_candidate(
    "blend_et_lgbm_long100_scalar_refined_calibrated",
    cal_test,
    cal_oof,
    extra_info={
        "a": a_cal,
        "b": b_cal,
        "base_w_et_safe": w_et_refined,
        "base_w_lgbm_long100k_lr02": w_lgbm_refined,
    }
)

# Ridge meta-stack with 5-fold CV over OOF-prediction features
def run_meta_cv(model_factory, model_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta_oof = np.zeros(n_train, dtype=float)

    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_meta), start=1):
        model = model_factory()
        model.fit(X_meta[tr_idx], y[tr_idx])
        meta_oof[va_idx] = model.predict(X_meta[va_idx])

        fold_rows.append({
            "fold": fold,
            "fold_mse_clipped": mse_clipped(y[va_idx], meta_oof[va_idx])
        })

    final_model = model_factory()
    final_model.fit(X_meta, y)
    meta_test = final_model.predict(X_test_meta)

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(RESULTS_DIR / f"{model_name}_meta_cv_fold_metrics.csv", index=False)

    details = {
        "meta_cv_mse_clipped": mse_clipped(y, meta_oof),
        "meta_fold_mse_mean": float(fold_df["fold_mse_clipped"].mean()),
        "meta_fold_mse_std": float(fold_df["fold_mse_clipped"].std(ddof=1)),
    }

    if hasattr(final_model, "intercept_"):
        details["intercept"] = float(np.ravel(final_model.intercept_)[0])

    if hasattr(final_model, "coef_"):
        coefs = np.ravel(final_model.coef_)
        for name, coef in zip(component_order, coefs):
            details[f"coef_{name}"] = float(coef)

    if hasattr(final_model, "alpha_"):
        details["alpha"] = float(final_model.alpha_)

    add_summary(
        name=model_name,
        oof_pred=meta_oof,
        test_pred=meta_test,
        source="meta_model_cv",
        details=details
    )

    save_candidate(model_name, meta_test, meta_oof, extra_info=details)

    return final_model, fold_df

alphas = np.logspace(-6, 3, 60)

ridge_model, ridge_folds = run_meta_cv(
    model_factory=lambda: RidgeCV(alphas=alphas, fit_intercept=True),
    model_name="stack_ridge_oof_components"
)

positive_lr_model, positive_lr_folds = run_meta_cv(
    model_factory=lambda: LinearRegression(positive=True),
    model_name="stack_positive_linear_oof_components"
)

# --------------------------------------------------------------------------------------
# Save and display screen
# --------------------------------------------------------------------------------------

screen_df = pd.DataFrame(summary_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
screen_path = RESULTS_DIR / "post80p662_blend_refinement_screen.csv"
screen_df.to_csv(screen_path, index=False)

saved_df = pd.DataFrame(saved_submissions)
saved_path = RESULTS_DIR / "post80p662_saved_candidate_submissions.csv"
saved_df.to_csv(saved_path, index=False)

print("\nPost-80.662 blend / stack refinement screen:")
try:
    display(screen_df)
except NameError:
    print(screen_df.to_string(index=False))

print("\nSaved candidate submissions:")
try:
    display(saved_df)
except NameError:
    print(saved_df.to_string(index=False))

print("\nKey files:")
print("Screen:", screen_path)
print("Saved submissions list:", saved_path)
print("Tracker:", tracker_path)

print("\nDecision guide:")
print("- Do not submit these automatically before review.")
print("- If scalar_refined or convex_refined only improves by ~0.001 OOF, it is probably not worth a Kaggle slot.")
print("- If ridge/positive stack improves OOF materially and has sane coefficients/test summary, it may be a candidate.")
print("- Paste this output when you return.")

Updated tracker: model_results/submission_tracker.csv

Loaded components:


,component,oof_path,oof_col,test_path,test_col,oof_mse_clipped,oof_pred_mean,oof_pred_std,test_pred_mean,test_pred_std
4,lgbm_long100k_lr02,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long100k_lr02,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02,93.726214,54.153144,24.734667,54.074208,24.605725
3,lgbm_long80k_lr03,model_results/oof_lgbm_t03_base_5fold_oof_long...,OOF_lgbm_t03_base_5fold_oof_long80k_lr03,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03,94.163485,54.147019,24.743266,54.067601,24.607933
2,lgbm12k,model_results/oof_lgbm_t03_base_5fold_oof.csv,OOF_lgbm_t03_base_5fold_oof,model_results/testpred_lgbm_t03_base_5fold_oof...,TESTPRED_lgbm_t03_base_5fold_oof,98.779406,54.152808,24.283770,54.086671,24.207260
1,et_safe,model_results/oof_extratrees_safe_base.csv,oof_pred,model_results/testpred_extratrees_safe_base_fo...,PERCENT_PROFICIENT,110.749294,54.127367,23.857745,54.040173,23.782901
0,rf500,model_results/oof_rf_500_base.csv,rf_500_base_oof_pred,model_results/testpred_rf_500_base_folds.csv,rf_500_base_foldavg_pred,122.328024,54.134143,22.839348,54.056767,22.817463



Post-80.662 blend / stack refinement screen:


,name,source,oof_mse_clipped,pred_mean_oof,pred_std_oof,pred_mean_test,pred_std_test,pred_min_test,pred_max_test,w_et_safe,w_lgbm_long100k_lr02,convex_success,convex_message,w_rf500,w_lgbm12k,w_lgbm_long80k_lr03,a,b,calibration_success,base_w_et_safe,base_w_lgbm_long100k_lr02,meta_cv_mse_clipped,meta_fold_mse_mean,meta_fold_mse_std,intercept,coef_rf500,coef_et_safe,coef_lgbm12k,coef_lgbm_long80k_lr03,coef_lgbm_long100k_lr02,alpha
0,stack_ridge_oof_components,meta_model_cv,87.976479,54.179374,24.697050,54.095697,24.611060,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.976479,87.976470,1.775049,-0.333024,-0.171527,0.480278,-0.159616,0.402591,0.455197,1000.0
1,stack_positive_linear_oof_components,meta_model_cv,88.658727,54.175720,24.678509,54.094807,24.614145,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.658727,88.658718,1.776098,-0.983962,0.000000,0.333300,0.000000,0.248013,0.437605,NaN
2,blend_et_lgbm_long100_scalar_refined_calibrated,conservative_affine_calibration,88.755227,54.188640,24.709775,54.107978,24.643046,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.019759,-1.017553,True,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,blend_all_components_convex_refined,slsqp_convex,88.865535,54.143501,24.251630,54.061842,24.181663,0.076160,100.0000,0.315230,0.437056,True,Optimization terminated successfully,1.406089e-14,2.778263e-15,0.247714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,blend_et_lgbm_long100_scalar_refined,scalar_minimize,88.966359,54.144933,24.249691,54.063366,24.178358,0.076961,100.0000,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,submitted_public80p662_recreated,known_submission,88.966368,54.144921,24.249135,54.063351,24.177857,0.077070,100.0000,0.319000,0.681000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lgbm_long100k_lr02,component,93.726214,54.153144,24.734667,54.074208,24.605725,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,lgbm_long80k_lr03,component,94.163485,54.147019,24.743266,54.067601,24.607933,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lgbm12k,component,98.779406,54.152808,24.283770,54.086671,24.207260,0.000000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,et_safe,component,110.749294,54.127367,23.857745,54.040173,23.782901,0.100800,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved candidate submissions:


,candidate,submission_path,testpred_path,w_et_safe,w_lgbm_long100k_lr02,convex_success,convex_message,w_rf500,w_lgbm12k,w_lgbm_long80k_lr03,a,b,base_w_et_safe,base_w_lgbm_long100k_lr02,meta_cv_mse_clipped,meta_fold_mse_mean,meta_fold_mse_std,intercept,coef_rf500,coef_et_safe,coef_lgbm12k,coef_lgbm_long80k_lr03,coef_lgbm_long100k_lr02,alpha
0,blend_et_lgbm_long100_scalar_refined,submission_blend_et_lgbm_long100_scalar_refine...,model_results/testpred_blend_et_lgbm_long100_s...,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,blend_all_components_convex_refined,submission_blend_all_components_convex_refined...,model_results/testpred_blend_all_components_co...,0.315230,0.437056,True,Optimization terminated successfully,1.406089e-14,2.778263e-15,0.247714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,blend_et_lgbm_long100_scalar_refined_calibrated,submission_blend_et_lgbm_long100_scalar_refine...,model_results/testpred_blend_et_lgbm_long100_s...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.019759,-1.017553,0.318548,0.681452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,stack_ridge_oof_components,submission_stack_ridge_oof_components.csv,model_results/testpred_stack_ridge_oof_compone...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.976479,87.976470,1.775049,-0.333024,-0.171527,0.480278,-0.159616,0.402591,0.455197,1000.0
4,stack_positive_linear_oof_components,submission_stack_positive_linear_oof_component...,model_results/testpred_stack_positive_linear_o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.658727,88.658718,1.776098,-0.983962,0.000000,0.333300,0.000000,0.248013,0.437605,NaN



Key files:
Screen: model_results/post80p662_blend_refinement_screen.csv
Saved submissions list: model_results/post80p662_saved_candidate_submissions.csv
Tracker: model_results/submission_tracker.csv

Decision guide:
- Do not submit these automatically before review.
- If scalar_refined or convex_refined only improves by ~0.001 OOF, it is probably not worth a Kaggle slot.
- If ridge/positive stack improves OOF materially and has sane coefficients/test summary, it may be a candidate.
- Paste this output when you return.


In [43]:
# 22B fixed: LightGBM diversity holdout screen with safe feature names
# Fix: convert X_tr / X_val to NumPy arrays and pass simple feature names to LightGBM.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from scipy import sparse

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse(y_true, pred):
    pred = clip100(pred)
    return float(np.mean((np.asarray(y_true, dtype=float) - pred) ** 2))

def to_lgbm_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32)
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

# Use existing development split if available; otherwise recreate it.
if all(name in globals() for name in ["X_tr", "X_val", "y_tr", "y_val"]):
    X_tr_use, X_val_use = X_tr, X_val
    y_tr_use = np.asarray(y_tr, dtype=float).ravel()
    y_val_use = np.asarray(y_val, dtype=float).ravel()
    print("Using existing X_tr / X_val split.")
else:
    X_tr_use, X_val_use, y_tr_use, y_val_use = train_test_split(
        X_train_proc_model,
        np.asarray(y_train, dtype=float).ravel(),
        test_size=0.20,
        random_state=RANDOM_STATE
    )
    print("Recreated 80/20 split.")

X_tr_lgbm = to_lgbm_matrix(X_tr_use)
X_val_lgbm = to_lgbm_matrix(X_val_use)

safe_feature_names = [f"f{i}" for i in range(X_tr_lgbm.shape[1])]

print("Train shape:", X_tr_lgbm.shape)
print("Validation shape:", X_val_lgbm.shape)

base_params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
}

jobs = [
    {
        "name": "lgbm_div01_l63_child80_l2_10_lr02",
        "params": {
            **base_params,
            "learning_rate": 0.02,
            "n_estimators": 70000,
            "num_leaves": 63,
            "min_child_samples": 80,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.05,
            "reg_lambda": 10.0,
            "max_depth": -1,
        },
        "stopping_rounds": 2500,
        "log_period": 2500,
    },
    {
        "name": "lgbm_div02_l127_child50_l2_15_lr02",
        "params": {
            **base_params,
            "learning_rate": 0.02,
            "n_estimators": 80000,
            "num_leaves": 127,
            "min_child_samples": 50,
            "subsample": 0.80,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.0,
            "reg_lambda": 15.0,
            "max_depth": -1,
        },
        "stopping_rounds": 3000,
        "log_period": 2500,
    },
    {
        "name": "lgbm_div03_extra_trees_l95_child60_l2_8_lr025",
        "params": {
            **base_params,
            "learning_rate": 0.025,
            "n_estimators": 70000,
            "num_leaves": 95,
            "min_child_samples": 60,
            "subsample": 0.90,
            "subsample_freq": 1,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.0,
            "reg_lambda": 8.0,
            "max_depth": -1,
            "extra_trees": True,
        },
        "stopping_rounds": 2500,
        "log_period": 2500,
    },
]

screen_path = RESULTS_DIR / "lgbm_diversity_holdout_screen_post80p662.csv"

if screen_path.exists():
    screen_df = pd.read_csv(screen_path)
    rows = screen_df.to_dict("records")
    completed = set(screen_df["name"].astype(str)) if "name" in screen_df.columns else set()
    print(f"Loaded existing screen with {len(completed)} completed jobs.")
else:
    rows = []
    completed = set()

reference_t03_holdout_mse = 104.4964
overall_start = time.time()

for job in jobs:
    name = job["name"]

    if name in completed:
        print(f"\nSkipping completed job: {name}")
        continue

    print("\n" + "#" * 90)
    print("Starting:", name)
    print("#" * 90)

    start = time.time()

    model = lgb.LGBMRegressor(**job["params"])
    model.fit(
        X_tr_lgbm,
        y_tr_use,
        eval_set=[(X_val_lgbm, y_val_use)],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(job["stopping_rounds"], verbose=True),
            lgb.log_evaluation(period=job["log_period"]),
        ],
    )

    best_iter = model.best_iteration_ or job["params"]["n_estimators"]

    train_pred = model.predict(X_tr_lgbm, num_iteration=best_iter)
    val_pred = model.predict(X_val_lgbm, num_iteration=best_iter)

    row = {
        "name": name,
        "best_iteration": int(best_iter),
        "train_mse_clipped": mse(y_tr_use, train_pred),
        "valid_mse_clipped": mse(y_val_use, val_pred),
        "valid_pred_mean": float(np.mean(clip100(val_pred))),
        "valid_pred_std": float(np.std(clip100(val_pred))),
        "elapsed_sec": float(time.time() - start),
        "delta_vs_old_t03_holdout": mse(y_val_use, val_pred) - reference_t03_holdout_mse,
    }

    for k, v in job["params"].items():
        row[f"param_{k}"] = v

    rows.append(row)

    screen_df = pd.DataFrame(rows).sort_values("valid_mse_clipped").reset_index(drop=True)
    screen_df.to_csv(screen_path, index=False)

    print("\nCompleted:", name)
    print("Best iteration:", row["best_iteration"])
    print("Validation MSE clipped:", row["valid_mse_clipped"])
    print("Delta vs old T03 holdout:", row["delta_vs_old_t03_holdout"])
    print("Saved checkpoint:", screen_path)

    del model, train_pred, val_pred
    gc.collect()

print("\nFinal LightGBM diversity holdout screen:")
screen_df = pd.DataFrame(rows).sort_values("valid_mse_clipped").reset_index(drop=True)
screen_df.to_csv(screen_path, index=False)

try:
    display(screen_df)
except NameError:
    print(screen_df.to_string(index=False))

print("\nScreen path:", screen_path)
print("Total elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Using existing X_tr / X_val split.
Train shape: (115936, 162)
Validation shape: (28985, 162)

##########################################################################################
Starting: lgbm_div01_l63_child80_l2_10_lr02
##########################################################################################
Training until validation scores don't improve for 2500 rounds
[2500]	valid_0's l2: 145.07
[5000]	valid_0's l2: 130.559
[7500]	valid_0's l2: 122.539
[10000]	valid_0's l2: 116.925
[12500]	valid_0's l2: 113.127
[15000]	valid_0's l2: 110.18
[17500]	valid_0's l2: 107.911
[20000]	valid_0's l2: 106.053
[22500]	valid_0's l2: 104.635
[25000]	valid_0's l2: 103.55
[27500]	valid_0's l2: 102.57
[30000]	valid_0's l2: 101.815
[32500]	valid_0's l2: 101.095
[35000]	valid_0's l2: 100.581
[37500]	valid_0's l2: 100.137
[40000]	valid_0's l2: 99.7268
[42500]	valid_0's l2: 99.404
[45000]	valid_0's l2: 99.0987
[47500]	valid_0's l2: 98.8982
[50000]	valid_0's l2: 98.7031
[52500]	valid_0's l2: 98.

,name,best_iteration,train_mse_clipped,valid_mse_clipped,valid_pred_mean,valid_pred_std,elapsed_sec,delta_vs_old_t03_holdout,param_objective,param_metric,param_random_state,param_n_jobs,param_verbosity,param_force_col_wise,param_learning_rate,param_n_estimators,param_num_leaves,param_min_child_samples,param_subsample,param_subsample_freq,param_colsample_bytree,param_reg_alpha,param_reg_lambda,param_max_depth,param_extra_trees
0,lgbm_div02_l127_child50_l2_15_lr02,47124,3.485951,95.127949,54.119015,24.777036,3551.669870,-9.368451,regression,l2,9890,1,-1,True,0.020,80000,127,50,0.80,1,0.85,0.00,15.0,-1,NaN
1,lgbm_div01_l63_child80_l2_10_lr02,69974,7.093457,96.932557,54.121766,24.721065,3513.916109,-7.563843,regression,l2,9890,1,-1,True,0.020,70000,63,80,0.85,1,0.85,0.05,10.0,-1,NaN
2,lgbm_div03_extra_trees_l95_child60_l2_8_lr025,69799,10.281223,97.325755,54.107728,24.648221,4286.634597,-7.170645,regression,l2,9890,1,-1,True,0.025,70000,95,60,0.90,1,0.90,0.00,8.0,-1,True



Screen path: model_results/lgbm_diversity_holdout_screen_post80p662.csv
Total elapsed minutes: 189.22


In [44]:
# 23A. Promote best diversity LightGBM to 3-fold OOF artifact + automatic blend
# Purpose:
# - Train the best holdout diversity candidate with 3-fold OOF.
# - Save fold checkpoints so the cell can resume.
# - If all 3 folds finish, create a fold-averaged submission.
# - Then automatically re-blend RF / ET / old LGBMs / new div02.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from scipy.optimize import minimize
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def to_lgbm_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32)
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def get_ids(name, fallback_df_name, expected_len):
    if name in globals():
        s = pd.Series(globals()[name]).astype(str).reset_index(drop=True)
    elif fallback_df_name in globals() and ID_COL in globals()[fallback_df_name].columns:
        s = globals()[fallback_df_name][ID_COL].astype(str).reset_index(drop=True)
    else:
        return None
    if len(s) != expected_len:
        raise ValueError(f"{name} length mismatch: {len(s)} vs {expected_len}")
    return s

X_all = to_lgbm_matrix(X_train_proc_model)
X_test_all = to_lgbm_matrix(X_test_proc_model)
y_all = np.asarray(y_train, dtype=float).ravel()

n_train = X_all.shape[0]
n_test = X_test_all.shape[0]

train_id_series = get_ids("train_ids", "scores_training", n_train)
test_id_series = get_ids("test_ids", "scores_test", n_test)

if test_id_series is None:
    raise ValueError("Could not find test IDs from test_ids or scores_test.")

safe_feature_names = [f"f{i}" for i in range(X_all.shape[1])]

params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.02,
    "n_estimators": 80000,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 15.0,
    "max_depth": -1,
}

metrics_path = RESULTS_DIR / f"{artifact}_fold_metrics.csv"
oof_npy_path = RESULTS_DIR / f"{artifact}_oof.npy"

if oof_npy_path.exists():
    oof_pred = np.load(oof_npy_path)
else:
    oof_pred = np.full(n_train, np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

print("Artifact:", artifact)
print("Train matrix:", X_all.shape)
print("Test matrix:", X_test_all.shape)
print("Existing completed folds:", [] if metrics_df.empty else list(metrics_df["fold"]))

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all), start=1):
    fold_test_path = RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(oof_pred[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/3")
        continue

    print("\n" + "=" * 80)
    print(f"Starting {artifact}, fold {fold}/3")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_all[tr_idx],
        y_all[tr_idx],
        eval_set=[(X_all[va_idx], y_all[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=3000, verbose=True),
            lgb.log_evaluation(period=2500),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_pred = model.predict(X_all[va_idx], num_iteration=best_iter)
    test_pred = model.predict(X_test_all, num_iteration=best_iter)

    oof_pred[va_idx] = va_pred.astype(np.float32)
    np.save(oof_npy_path, oof_pred)
    np.save(fold_test_path, test_pred.astype(np.float32))

    row = {
        "fold": fold,
        "fold_valid_mse_clipped": mse_clip(y_all[va_idx], va_pred),
        "fold_valid_mse_raw": float(np.mean((y_all[va_idx] - va_pred) ** 2)),
        "best_iteration": int(best_iter),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("\nCompleted fold:", fold)
    print("Fold valid MSE clipped:", row["fold_valid_mse_clipped"])
    print("Best iteration:", row["best_iteration"])
    print("Saved checkpoint:", metrics_path)

    del model, va_pred, test_pred
    gc.collect()

completed = (not np.isnan(oof_pred).any()) and all(
    (RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy").exists()
    for fold in range(1, 4)
)

print("\n" + "#" * 90)
print("3-fold artifact completion check")
print("#" * 90)
print("Missing OOF predictions:", int(np.isnan(oof_pred).sum()))
print("Completed:", completed)

if not completed:
    print("\nNot all folds are complete yet. Rerun this same cell later to resume.")
else:
    oof_mse = mse_clip(y_all, oof_pred)
    print("Overall 3-fold OOF MSE clipped:", oof_mse)

    fold_test_preds = [
        np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_pred.npy")
        for fold in range(1, 4)
    ]
    test_foldavg = np.mean(fold_test_preds, axis=0)

    oof_csv_path = RESULTS_DIR / f"oof_{artifact}.csv"
    testpred_csv_path = RESULTS_DIR / f"testpred_{artifact}_foldavg.csv"
    submission_path = Path(f"submission_{artifact}_foldavg.csv")

    oof_df = pd.DataFrame({
        TARGET_COL: y_all,
        f"OOF_{artifact}": oof_pred
    })
    if train_id_series is not None:
        oof_df.insert(0, ID_COL, train_id_series.values)

    oof_df.to_csv(oof_csv_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{artifact}": clip100(test_foldavg)
    }).to_csv(testpred_csv_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: clip100(test_foldavg)
    }).to_csv(submission_path, index=False)

    print("Saved OOF:", oof_csv_path)
    print("Saved test predictions:", testpred_csv_path)
    print("Saved standalone submission:", submission_path)

    print("\nStandalone div02 3-fold submission summary:")
    print(pd.Series(clip100(test_foldavg)).describe())

    # ------------------------------------------------------------------
    # Automatic convex blend including new div02 artifact
    # ------------------------------------------------------------------

    def read_pred(path, preferred_col):
        df = pd.read_csv(path)
        if preferred_col in df.columns:
            return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

        numeric_cols = []
        for c in df.columns:
            if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
                continue
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().mean() > 0.95:
                numeric_cols.append(c)

        if len(numeric_cols) == 0:
            raise ValueError(f"No usable prediction column found in {path}")

        print(f"Using fallback column {numeric_cols[-1]} from {path}")
        return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

    components = {
        "rf500": {
            "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
            "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
        },
        "et_safe": {
            "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
            "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
        },
        "lgbm12k": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
        },
        "lgbm_long80k_lr03": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
        },
        "lgbm_long100k_lr02": {
            "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
            "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
        },
        "lgbm_div02_3fold": {
            "oof": (str(oof_csv_path), f"OOF_{artifact}"),
            "test": (str(testpred_csv_path), f"TESTPRED_{artifact}"),
        },
    }

    names = []
    oof_list = []
    test_list = []
    comp_rows = []

    for name, spec in components.items():
        try:
            oof_p = clip100(read_pred(*spec["oof"]))
            test_p = clip100(read_pred(*spec["test"]))

            if len(oof_p) != n_train or len(test_p) != n_test:
                print(f"Skipping {name}: length mismatch")
                continue

            names.append(name)
            oof_list.append(oof_p)
            test_list.append(test_p)

            comp_rows.append({
                "component": name,
                "oof_mse_clipped": mse_clip(y_all, oof_p),
                "oof_mean": float(np.mean(oof_p)),
                "oof_std": float(np.std(oof_p)),
                "test_mean": float(np.mean(test_p)),
                "test_std": float(np.std(test_p)),
            })

        except Exception as e:
            print(f"Skipping {name}: {e}")

    Xo = np.column_stack(oof_list)
    Xt = np.column_stack(test_list)

    def blend_loss(w):
        return mse_clip(y_all, Xo @ w)

    starts = []

    starts.append(np.ones(len(names)) / len(names))

    known = np.zeros(len(names))
    if "et_safe" in names and "lgbm_long100k_lr02" in names:
        known[names.index("et_safe")] = 0.319
        known[names.index("lgbm_long100k_lr02")] = 0.681
        starts.append(known)

    for i in range(len(names)):
        pure = np.zeros(len(names))
        pure[i] = 1.0
        starts.append(pure)

    best_res = None
    for x0 in starts:
        res = minimize(
            blend_loss,
            x0=x0,
            method="SLSQP",
            bounds=[(0.0, 1.0)] * len(names),
            constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
            options={"maxiter": 1000, "ftol": 1e-12},
        )
        if best_res is None or res.fun < best_res.fun:
            best_res = res

    weights = np.asarray(best_res.x, dtype=float)
    blend_oof = Xo @ weights
    blend_test = Xt @ weights

    blend_name = "blend_auto_with_lgbm_div02_3fold_oof_weighted"

    weight_df = pd.DataFrame({
        "component": names,
        "weight": weights,
    }).sort_values("weight", ascending=False)

    comp_df = pd.DataFrame(comp_rows).sort_values("oof_mse_clipped")

    weight_path = RESULTS_DIR / f"{blend_name}_weights.csv"
    testblend_path = RESULTS_DIR / f"testpred_{blend_name}.csv"
    blend_submission_path = Path(f"submission_{blend_name}.csv")

    weight_df.to_csv(weight_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{blend_name}": clip100(blend_test)
    }).to_csv(testblend_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: clip100(blend_test)
    }).to_csv(blend_submission_path, index=False)

    print("\nBlend component OOF results:")
    try:
        display(comp_df)
    except NameError:
        print(comp_df.to_string(index=False))

    print("\nBest blend weights:")
    try:
        display(weight_df)
    except NameError:
        print(weight_df.to_string(index=False))

    print("\nBest blend OOF MSE clipped:", mse_clip(y_all, blend_oof))
    print("Previous submitted blend OOF MSE was about: 88.9664")
    print("Saved weights:", weight_path)
    print("Saved blend test predictions:", testblend_path)
    print("Saved blend submission:", blend_submission_path)

    print("\nBlend submission summary:")
    print(pd.Series(clip100(blend_test)).describe())

print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Artifact: lgbm_div02_l127_child50_l2_15_lr02_3fold_oof
Train matrix: (144921, 162)
Test matrix: (48307, 162)
Existing completed folds: []

Starting lgbm_div02_l127_child50_l2_15_lr02_3fold_oof, fold 1/3
Training until validation scores don't improve for 3000 rounds
[2500]	valid_0's l2: 131.203
[5000]	valid_0's l2: 119.548
[7500]	valid_0's l2: 113.556
[10000]	valid_0's l2: 110.141
[12500]	valid_0's l2: 108.057
[15000]	valid_0's l2: 106.683
[17500]	valid_0's l2: 105.74
[20000]	valid_0's l2: 105.083
[22500]	valid_0's l2: 104.662
[25000]	valid_0's l2: 104.348
[27500]	valid_0's l2: 104.128
[30000]	valid_0's l2: 103.975
[32500]	valid_0's l2: 103.875
[35000]	valid_0's l2: 103.791
[37500]	valid_0's l2: 103.718
[40000]	valid_0's l2: 103.715
[42500]	valid_0's l2: 103.706
Early stopping, best iteration is:
[41059]	valid_0's l2: 103.693

Completed fold: 1
Fold valid MSE clipped: 102.8387506655704
Best iteration: 41059
Saved checkpoint: model_results/lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_fol

,component,oof_mse_clipped,oof_mean,oof_std,test_mean,test_std
4,lgbm_long100k_lr02,93.726214,54.153144,24.734667,54.074208,24.605725
3,lgbm_long80k_lr03,94.163485,54.147019,24.743266,54.067601,24.607933
2,lgbm12k,98.779406,54.152808,24.283770,54.086671,24.207260
5,lgbm_div02_3fold,104.083036,54.131471,24.596497,54.084634,24.384543
1,et_safe,110.749294,54.127367,23.857745,54.040173,23.782901
0,rf500,122.328024,54.134143,22.839348,54.056767,22.817463



Best blend weights:


,component,weight
4,lgbm_long100k_lr02,0.391029
1,et_safe,0.308273
3,lgbm_long80k_lr03,0.220189
5,lgbm_div02_3fold,0.080509
0,rf500,0.000000
2,lgbm12k,0.000000



Best blend OOF MSE clipped: 88.76684691394617
Previous submitted blend OOF MSE was about: 88.9664
Saved weights: model_results/blend_auto_with_lgbm_div02_3fold_oof_weighted_weights.csv
Saved blend test predictions: model_results/testpred_blend_auto_with_lgbm_div02_3fold_oof_weighted.csv
Saved blend submission: submission_blend_auto_with_lgbm_div02_3fold_oof_weighted.csv

Blend submission summary:
count    48307.000000
mean        54.063100
std         24.170248
min          0.074479
25%         35.017689
50%         52.562427
75%         73.413524
max        100.000000
dtype: float64

Total elapsed minutes: 117.43


In [45]:
# 24A. Conservative residual stack on top of the current best 80.662 blend
# Goal:
# - Use the current best submitted blend as the base prediction.
# - Train a small LightGBM residual model with proper OOF validation.
# - Choose a conservative shrinkage factor for the residual correction.
# - Save candidate submissions, but do NOT submit automatically.

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "residstack_lgbm_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

def to_float_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def add_meta_features(X, M):
    M = np.asarray(M, dtype=np.float32)
    if sparse.issparse(X):
        return sparse.hstack([X, sparse.csr_matrix(M)], format="csr")
    return np.hstack([X, M])

# ---------------------------------------------------------------------
# Load base matrices and saved OOF/test prediction artifacts
# ---------------------------------------------------------------------

X_base = to_float_matrix(X_train_proc_model)
X_test_base = to_float_matrix(X_test_proc_model)
y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

components = {
    "rf500": {
        "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
        "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
    },
    "et_safe": {
        "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
        "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
    },
    "lgbm12k": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
    },
    "lgbm_long80k_lr03": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
    },
    "lgbm_long100k_lr02": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
    },
    "lgbm_div02_3fold": {
        "oof": ("model_results/oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv", "OOF_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
        "test": ("model_results/testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv", "TESTPRED_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
    },
}

oof_preds = {}
test_preds = {}

for name, spec in components.items():
    try:
        oof_p = clip100(read_pred(*spec["oof"]))
        test_p = clip100(read_pred(*spec["test"]))

        if len(oof_p) != len(y) or len(test_p) != len(test_id_series):
            print(f"Skipping {name}: length mismatch")
            continue

        oof_preds[name] = oof_p
        test_preds[name] = test_p
        print(f"Loaded {name}: OOF MSE = {mse_clip(y, oof_p):.6f}")

    except Exception as e:
        print(f"Skipping {name}: {e}")

if "et_safe" not in oof_preds or "lgbm_long100k_lr02" not in oof_preds:
    raise ValueError("Need et_safe and lgbm_long100k_lr02 for the current best blend.")

# Current best submitted 80.662 blend
base_oof = 0.319 * oof_preds["et_safe"] + 0.681 * oof_preds["lgbm_long100k_lr02"]
base_test = 0.319 * test_preds["et_safe"] + 0.681 * test_preds["lgbm_long100k_lr02"]

base_mse = mse_clip(y, base_oof)
print("\nCurrent best submitted blend OOF MSE:", base_mse)

# ---------------------------------------------------------------------
# Build meta features
# ---------------------------------------------------------------------

component_order = list(oof_preds.keys())

M_train_main = np.column_stack([oof_preds[name] for name in component_order])
M_test_main = np.column_stack([test_preds[name] for name in component_order])

lgbm_names = [n for n in component_order if n.startswith("lgbm")]
lgbm_train = np.column_stack([oof_preds[n] for n in lgbm_names])
lgbm_test = np.column_stack([test_preds[n] for n in lgbm_names])

M_train_extra = np.column_stack([
    base_oof,
    M_train_main.mean(axis=1),
    M_train_main.std(axis=1),
    M_train_main.max(axis=1) - M_train_main.min(axis=1),
    lgbm_train.mean(axis=1),
    lgbm_train.std(axis=1),
])

M_test_extra = np.column_stack([
    base_test,
    M_test_main.mean(axis=1),
    M_test_main.std(axis=1),
    M_test_main.max(axis=1) - M_test_main.min(axis=1),
    lgbm_test.mean(axis=1),
    lgbm_test.std(axis=1),
])

M_train = np.hstack([M_train_main, M_train_extra]).astype(np.float32)
M_test = np.hstack([M_test_main, M_test_extra]).astype(np.float32)

Z_train = add_meta_features(X_base, M_train)
Z_test = add_meta_features(X_test_base, M_test)

residual_y = y - base_oof

print("Residual target summary:")
print(pd.Series(residual_y).describe())

print("\nMeta training matrix:", Z_train.shape)
print("Meta test matrix:", Z_test.shape)

# ---------------------------------------------------------------------
# 5-fold residual model with checkpointing
# ---------------------------------------------------------------------

params = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.03,
    "n_estimators": 20000,
    "num_leaves": 31,
    "max_depth": 6,
    "min_child_samples": 250,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 1.0,
    "reg_lambda": 50.0,
}

resid_oof_path = RESULTS_DIR / f"{artifact}_resid_oof.npy"
metrics_path = RESULTS_DIR / f"{artifact}_fold_metrics.csv"

if resid_oof_path.exists():
    resid_oof = np.load(resid_oof_path)
else:
    resid_oof = np.full(len(y), np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

safe_feature_names = [f"f{i}" for i in range(Z_train.shape[1])]
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf.split(Z_train), start=1):
    fold_test_path = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(resid_oof[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/5")
        continue

    print("\n" + "=" * 80)
    print(f"Starting residual stack fold {fold}/5")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        Z_train[tr_idx],
        residual_y[tr_idx],
        eval_set=[(Z_train[va_idx], residual_y[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=True),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_resid_pred = model.predict(Z_train[va_idx], num_iteration=best_iter)
    test_resid_pred = model.predict(Z_test, num_iteration=best_iter)

    resid_oof[va_idx] = va_resid_pred.astype(np.float32)
    np.save(resid_oof_path, resid_oof)
    np.save(fold_test_path, test_resid_pred.astype(np.float32))

    row = {
        "fold": fold,
        "best_iteration": int(best_iter),
        "residual_valid_mse": float(np.mean((residual_y[va_idx] - va_resid_pred) ** 2)),
        "corrected_mse_lambda_1": mse_clip(y[va_idx], base_oof[va_idx] + va_resid_pred),
        "base_mse_on_fold": mse_clip(y[va_idx], base_oof[va_idx]),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("Completed fold:", fold)
    print("Base fold MSE:", row["base_mse_on_fold"])
    print("Corrected fold MSE with lambda=1:", row["corrected_mse_lambda_1"])
    print("Best iteration:", row["best_iteration"])
    print("Saved checkpoint:", metrics_path)

    del model, va_resid_pred, test_resid_pred
    gc.collect()

completed = (not np.isnan(resid_oof).any()) and all(
    (RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy").exists()
    for fold in range(1, 6)
)

print("\nCompleted residual stack:", completed)
print("Missing residual OOF predictions:", int(np.isnan(resid_oof).sum()))

if completed:
    test_resids = [
        np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy")
        for fold in range(1, 6)
    ]
    resid_test = np.mean(test_resids, axis=0)

    lambda_grid = np.linspace(0.0, 0.60, 121)
    lambda_rows = []

    for lam in lambda_grid:
        pred = base_oof + lam * resid_oof
        lambda_rows.append({
            "lambda": float(lam),
            "oof_mse_clipped": mse_clip(y, pred),
            "gain_vs_base_oof": base_mse - mse_clip(y, pred),
            "pred_mean": float(np.mean(clip100(pred))),
            "pred_std": float(np.std(clip100(pred))),
        })

    lambda_df = pd.DataFrame(lambda_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    lambda_path = RESULTS_DIR / f"{artifact}_lambda_screen.csv"
    lambda_df.to_csv(lambda_path, index=False)

    print("\nResidual shrinkage screen:")
    try:
        display(lambda_df.head(15))
    except NameError:
        print(lambda_df.head(15).to_string(index=False))

    best_lambda = float(lambda_df.loc[0, "lambda"])
    conservative_lambda = round(best_lambda * 0.5, 4)

    save_lambdas = sorted(set([
        best_lambda,
        conservative_lambda,
        0.10,
        0.20,
        0.30,
    ]))

    saved_rows = []

    for lam in save_lambdas:
        final_oof = clip100(base_oof + lam * resid_oof)
        final_test = clip100(base_test + lam * resid_test)

        name = f"{artifact}_lambda_{str(lam).replace('.', 'p')}"
        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            f"TESTPRED_{name}": final_test
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            TARGET_COL: final_test
        }).to_csv(sub_path, index=False)

        saved_rows.append({
            "lambda": lam,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_base_oof": base_mse - mse_clip(y, final_oof),
            "submission_path": str(sub_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        })

    saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    saved_path = RESULTS_DIR / f"{artifact}_saved_candidates.csv"
    saved_df.to_csv(saved_path, index=False)

    print("\nSaved residual-stack candidate submissions:")
    try:
        display(saved_df)
    except NameError:
        print(saved_df.to_string(index=False))

    corr = np.corrcoef(residual_y, resid_oof)[0, 1]
    print("\nResidual OOF prediction correlation with true residual:", corr)
    print("Base OOF MSE:", base_mse)
    print("Best residual-stack OOF MSE:", float(saved_df.loc[0, "oof_mse_clipped"]))
    print("Best OOF gain:", float(saved_df.loc[0, "gain_vs_base_oof"]))
    print("Lambda screen:", lambda_path)
    print("Saved candidates:", saved_path)

print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Loaded rf500: OOF MSE = 122.328024
Loaded et_safe: OOF MSE = 110.749294
Loaded lgbm12k: OOF MSE = 98.779406
Loaded lgbm_long80k_lr03: OOF MSE = 94.163485
Loaded lgbm_long100k_lr02: OOF MSE = 93.726214
Loaded lgbm_div02_3fold: OOF MSE = 104.083036

Current best submitted blend OOF MSE: 88.96636822734054
Residual target summary:
count    144921.000000
mean          0.038600
std           9.432152
min         -74.711172
25%          -4.382987
50%           0.127390
75%           4.493513
max          84.650332
dtype: float64

Meta training matrix: (144921, 174)
Meta test matrix: (48307, 174)

Starting residual stack fold 1/5
Training until validation scores don't improve for 1000 rounds
[1000]	valid_0's l2: 89.8431
Early stopping, best iteration is:
[248]	valid_0's l2: 88.7583
Completed fold: 1
Base fold MSE: 90.31439437895932
Corrected fold MSE with lambda=1: 88.75596944176321
Best iteration: 248
Saved checkpoint: model_results/residstack_lgbm_on_best80p662_blend_5fold_fold_metrics.csv



,lambda,oof_mse_clipped,gain_vs_base_oof,pred_mean,pred_std
0,0.600,87.543031,1.423337,54.167175,24.526060
1,0.595,87.550667,1.415702,54.166992,24.523684
2,0.590,87.558373,1.407995,54.166809,24.521309
3,0.585,87.566151,1.400217,54.166626,24.518935
4,0.580,87.573999,1.392369,54.166443,24.516562
5,0.575,87.581919,1.384449,54.166260,24.514191
6,0.570,87.589909,1.376459,54.166077,24.511821
7,0.565,87.597971,1.368397,54.165894,24.509451
8,0.560,87.606104,1.360265,54.165711,24.507083
9,0.555,87.614307,1.352061,54.165527,24.504716



Saved residual-stack candidate submissions:


,lambda,oof_mse_clipped,gain_vs_base_oof,submission_path,test_mean,test_std,test_min,test_max
0,0.6,87.543031,1.423337,submission_residstack_lgbm_on_best80p662_blend...,54.081737,24.436239,0.148440,100.0
1,0.3,88.126863,0.839505,submission_residstack_lgbm_on_best80p662_blend...,54.072594,24.305647,0.125898,100.0
2,0.2,88.378293,0.588075,submission_residstack_lgbm_on_best80p662_blend...,54.069531,24.262751,0.118384,100.0
3,0.1,88.658121,0.308247,submission_residstack_lgbm_on_best80p662_blend...,54.066449,24.220152,0.110870,100.0



Residual OOF prediction correlation with true residual: 0.14331489148998913
Base OOF MSE: 88.96636822734054
Best residual-stack OOF MSE: 87.54303119204958
Best OOF gain: 1.423337035290956
Lambda screen: model_results/residstack_lgbm_on_best80p662_blend_5fold_lambda_screen.csv
Saved candidates: model_results/residstack_lgbm_on_best80p662_blend_5fold_saved_candidates.csv

Total elapsed minutes: 1.42


In [46]:
# 24B. Expanded residual-stack lambda screen
# No model training here. This reuses the saved residual OOF/test predictions from 24A.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

artifact = "residstack_lgbm_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y, p):
    return float(np.mean((np.asarray(y, dtype=float) - clip100(p)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

# Rebuild the current best submitted 80.662 blend.
et_oof = clip100(read_pred("model_results/oof_extratrees_safe_base.csv", "oof_pred"))
et_test = clip100(read_pred("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL))

lgb_oof = clip100(read_pred(
    "model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv",
    "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"
))
lgb_test = clip100(read_pred(
    "model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv",
    "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"
))

base_oof = 0.319 * et_oof + 0.681 * lgb_oof
base_test = 0.319 * et_test + 0.681 * lgb_test

base_mse = mse_clip(y, base_oof)

# Load residual OOF and averaged test residuals from 24A.
resid_oof = np.load(RESULTS_DIR / f"{artifact}_resid_oof.npy")

resid_test_parts = []
for fold in range(1, 6):
    p = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"
    if not p.exists():
        raise FileNotFoundError(f"Missing residual test prediction file: {p}")
    resid_test_parts.append(np.load(p))

resid_test = np.mean(resid_test_parts, axis=0)

print("Base OOF MSE:", base_mse)
print("Residual OOF shape:", resid_oof.shape)
print("Residual test shape:", resid_test.shape)

# Expanded lambda screen.
lambda_grid = np.round(np.arange(0.00, 1.505, 0.005), 3)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(kf.split(base_oof))

rows = []

for lam in lambda_grid:
    oof_pred = base_oof + lam * resid_oof
    overall_mse = mse_clip(y, oof_pred)

    fold_gains = []
    fold_mses = []

    for fold, (_, va_idx) in enumerate(fold_indices, start=1):
        fold_base_mse = mse_clip(y[va_idx], base_oof[va_idx])
        fold_new_mse = mse_clip(y[va_idx], oof_pred[va_idx])
        fold_gains.append(fold_base_mse - fold_new_mse)
        fold_mses.append(fold_new_mse)

    test_pred = clip100(base_test + lam * resid_test)

    rows.append({
        "lambda": float(lam),
        "oof_mse_clipped": overall_mse,
        "gain_vs_base_oof": base_mse - overall_mse,
        "min_fold_gain": float(np.min(fold_gains)),
        "mean_fold_gain": float(np.mean(fold_gains)),
        "max_fold_gain": float(np.max(fold_gains)),
        "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        "test_mean": float(np.mean(test_pred)),
        "test_std": float(np.std(test_pred)),
        "test_min": float(np.min(test_pred)),
        "test_max": float(np.max(test_pred)),
    })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)
screen_path = RESULTS_DIR / f"{artifact}_expanded_lambda_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop expanded lambda candidates:")
try:
    display(screen_df.head(25))
except NameError:
    print(screen_df.head(25).to_string(index=False))

best_lambda = float(screen_df.loc[0, "lambda"])
print("\nBest lambda:", best_lambda)
print("Best OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best OOF gain vs base:", float(screen_df.loc[0, "gain_vs_base_oof"]))
print("Minimum fold gain at best lambda:", float(screen_df.loc[0, "min_fold_gain"]))

# Save a few sensible candidate submissions.
candidate_lambdas = sorted(set([
    0.60,
    0.80,
    1.00,
    round(best_lambda, 3),
    round(0.75 * best_lambda, 3),
]))

saved_rows = []

for lam in candidate_lambdas:
    final_oof = clip100(base_oof + lam * resid_oof)
    final_test = clip100(base_test + lam * resid_test)

    name = f"{artifact}_expanded_lambda_{str(lam).replace('.', 'p')}"

    oof_path = RESULTS_DIR / f"oof_{name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
    sub_path = Path(f"submission_{name}.csv")

    pd.DataFrame({
        TARGET_COL: y,
        f"OOF_{name}": final_oof
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        f"TESTPRED_{name}": final_test
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: test_id_series.values,
        TARGET_COL: final_test
    }).to_csv(sub_path, index=False)

    saved_rows.append({
        "lambda": lam,
        "oof_mse_clipped": mse_clip(y, final_oof),
        "gain_vs_base_oof": base_mse - mse_clip(y, final_oof),
        "submission_path": str(sub_path),
        "test_mean": float(np.mean(final_test)),
        "test_std": float(np.std(final_test)),
        "test_min": float(np.min(final_test)),
        "test_max": float(np.max(final_test)),
    })

saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
saved_path = RESULTS_DIR / f"{artifact}_expanded_saved_candidates.csv"
saved_df.to_csv(saved_path, index=False)

print("\nSaved expanded residual-stack candidates:")
try:
    display(saved_df)
except NameError:
    print(saved_df.to_string(index=False))

print("\nScreen saved to:", screen_path)
print("Saved candidates list:", saved_path)

Base OOF MSE: 88.96636822734054
Residual OOF shape: (144921,)
Residual test shape: (48307,)

Top expanded lambda candidates:


,lambda,oof_mse_clipped,gain_vs_base_oof,min_fold_gain,mean_fold_gain,max_fold_gain,fold_mse_std,test_mean,test_std,test_min,test_max
0,1.135,87.136049,1.830319,1.552019,1.830321,2.114976,1.919453,54.097842,24.676033,0.188640,100.0
1,1.140,87.136071,1.830297,1.550795,1.830299,2.115073,1.919480,54.097991,24.678314,0.189016,100.0
2,1.130,87.136097,1.830271,1.553172,1.830273,2.114798,1.919418,54.097693,24.673752,0.188265,100.0
3,1.145,87.136164,1.830204,1.549500,1.830206,2.115089,1.919500,54.098139,24.680596,0.189392,100.0
4,1.125,87.136217,1.830152,1.554254,1.830153,2.114540,1.919376,54.097543,24.671472,0.187889,100.0
5,1.150,87.136329,1.830040,1.548135,1.830042,2.115024,1.919513,54.098288,24.682879,0.189768,100.0
6,1.120,87.136407,1.829961,1.555266,1.829963,2.114201,1.919326,54.097394,24.669192,0.187513,100.0
7,1.155,87.136564,1.829805,1.546699,1.829806,2.114879,1.919518,54.098437,24.685163,0.190143,100.0
8,1.115,87.136668,1.829700,1.556207,1.829702,2.113781,1.919269,54.097245,24.666913,0.187138,100.0
9,1.160,87.136870,1.829499,1.545192,1.829500,2.114653,1.919516,54.098586,24.687448,0.190519,100.0



Best lambda: 1.135
Best OOF MSE: 87.1360487514482
Best OOF gain vs base: 1.8303194758923382
Minimum fold gain at best lambda: 1.552018866620017

Saved expanded residual-stack candidates:


,lambda,oof_mse_clipped,gain_vs_base_oof,submission_path,test_mean,test_std,test_min,test_max
0,1.135,87.136049,1.830319,submission_residstack_lgbm_on_best80p662_blend...,54.097842,24.676033,0.188640,100.0
1,1.000,87.162181,1.804188,submission_residstack_lgbm_on_best80p662_blend...,54.093804,24.614705,0.178496,100.0
2,0.851,87.250959,1.715410,submission_residstack_lgbm_on_best80p662_blend...,54.089329,24.547659,0.167301,100.0
3,0.800,87.295825,1.670544,submission_residstack_lgbm_on_best80p662_blend...,54.087791,24.524863,0.163468,100.0
4,0.600,87.543031,1.423337,submission_residstack_lgbm_on_best80p662_blend...,54.081737,24.436239,0.148440,100.0



Screen saved to: model_results/residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_screen.csv
Saved candidates list: model_results/residstack_lgbm_on_best80p662_blend_5fold_expanded_saved_candidates.csv


### Expanded Residual Stack on Best 80.662 Blend

This experiment fit a residual model on top of the current best submitted blend, whose OOF MSE was 88.9664. The expanded residual-stack screen improved the best clipped OOF MSE to 87.1360 at lambda = 1.135, corresponding to an OOF gain of approximately 1.83 MSE points. Importantly, the improvement was positive on every fold, with the minimum fold-level gain around 1.55, suggesting that the residual correction is not driven by a single lucky validation fold.

The resulting test prediction distribution appears reasonable, with mean near 54.10, standard deviation near 24.68, and predictions clipped to the valid [0, 100] target range. However, because this model is a residual correction on top of the already-best ET/LGBM blend, it may still be exploiting similar structure rather than adding a clearly independent source of predictive signal. Therefore, this candidate is saved as a strong backlog submission candidate, but it is not submitted immediately.

The next priority is to seek genuinely different model diversity through the planned div02 / diversity-style OOF artifact and blend experiment, rather than continuing to tune small residual-stack or lambda refinements.

### Kaggle result: expanded residual stack on best 80.662 blend

The expanded residual-stack candidate was submitted as:

`submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv`

This submission improved the public leaderboard MSE from the prior best `80.662` to `77.277`.

This is a meaningful improvement, so the residual-stack model is now the new leaderboard anchor. The result also confirms that the previous ET + long LightGBM blend had systematic residual structure that could be learned from the existing training features and model-prediction meta-features.

The best internal lambda-screen candidate used `lambda = 1.135`, with OOF MSE approximately `87.1360`, OOF gain approximately `1.8303` versus the old base blend, and positive fold-level gains across all folds. Because the public leaderboard also improved substantially, future work should focus on adding diversity to the residual-correction layer rather than returning to small base-blend or lambda-only tuning.

In [48]:
# ============================================================
# 25A. Residual-stack diversity model after 77.277 public score
# ============================================================
#
# Current confirmed public anchor:
# submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
# Public MSE: 77.277
#
# Goal:
# - Do NOT tune only lambda again.
# - Do NOT build a stage-2 residual on top of the 77.277 model yet.
# - Instead, train a different first-level residual correction on the original
#   80.662 base blend residuals.
# - Then blend the original successful residual correction with this new diverse
#   residual correction using OOF weights.
#
# Submit only if this cell finds a real OOF gain versus the 1.135 residual-stack
# anchor, preferably with positive fold-gain diagnostics.

import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy import sparse
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

PUBLIC_ANCHOR_FILE = "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"
PUBLIC_ANCHOR_SCORE = 77.277

old_resid_artifact = "residstack_lgbm_on_best80p662_blend_5fold"
new_artifact = "residstack_huber_extra_on_best80p662_blend_5fold"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def read_pred(path, preferred_col):
    df = pd.read_csv(path)

    if preferred_col in df.columns:
        return pd.to_numeric(df[preferred_col], errors="raise").to_numpy(dtype=float)

    numeric_cols = []
    for c in df.columns:
        if c in [ID_COL, TARGET_COL, "fold", "Unnamed: 0"]:
            continue
        vals = pd.to_numeric(df[c], errors="coerce")
        if vals.notna().mean() > 0.95:
            numeric_cols.append(c)

    if not numeric_cols:
        raise ValueError(f"No prediction column found in {path}")

    print(f"Using fallback prediction column {numeric_cols[-1]} from {path}")
    return pd.to_numeric(df[numeric_cols[-1]], errors="raise").to_numpy(dtype=float)

def get_test_ids():
    if "test_ids" in globals():
        obj = globals()["test_ids"]
        if isinstance(obj, pd.DataFrame):
            if ID_COL in obj.columns:
                return obj[ID_COL].astype(str).reset_index(drop=True)
            return obj.iloc[:, 0].astype(str).reset_index(drop=True)
        return pd.Series(obj).astype(str).reset_index(drop=True)

    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)

    raise ValueError("Could not find test IDs.")

def to_float_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def add_meta_features(X, M):
    M = np.asarray(M, dtype=np.float32)
    if sparse.issparse(X):
        return sparse.hstack([X, sparse.csr_matrix(M)], format="csr")
    return np.hstack([X, M])

def mean_test_residual_parts(artifact, n_folds=5):
    parts = []
    for fold in range(1, n_folds + 1):
        p = RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy"
        if not p.exists():
            raise FileNotFoundError(f"Missing residual test prediction file: {p}")
        parts.append(np.load(p))
    return np.mean(parts, axis=0)

# ------------------------------------------------------------
# Log public checkpoint in tracker
# ------------------------------------------------------------

tracker_path = RESULTS_DIR / "submission_tracker.csv"
checkpoint_row = {
    "file": PUBLIC_ANCHOR_FILE,
    "public_mse": PUBLIC_ANCHOR_SCORE,
    "private_mse": np.nan,
    "notes": (
        "New public anchor. Expanded residual stack on prior 80.662 ET+longLGBM blend; "
        "lambda=1.135; public MSE improved to 77.277."
    ),
}

if tracker_path.exists():
    tracker_df = pd.read_csv(tracker_path)
else:
    tracker_df = pd.DataFrame()

if len(tracker_df) == 0 or PUBLIC_ANCHOR_FILE not in set(tracker_df.get("file", pd.Series(dtype=str)).astype(str)):
    tracker_df = pd.concat([tracker_df, pd.DataFrame([checkpoint_row])], ignore_index=True)
    tracker_df.to_csv(tracker_path, index=False)
    print("Added public checkpoint to tracker:", tracker_path)
else:
    print("Public checkpoint already present in tracker:", tracker_path)

# ------------------------------------------------------------
# Load base matrices and model prediction artifacts
# ------------------------------------------------------------

X_base = to_float_matrix(X_train_proc_model)
X_test_base = to_float_matrix(X_test_proc_model)
y = np.asarray(y_train, dtype=float).ravel()
test_id_series = get_test_ids()

components = {
    "rf500": {
        "oof": ("model_results/oof_rf_500_base.csv", "rf_500_base_oof_pred"),
        "test": ("model_results/testpred_rf_500_base_folds.csv", "rf_500_base_foldavg_pred"),
    },
    "et_safe": {
        "oof": ("model_results/oof_extratrees_safe_base.csv", "oof_pred"),
        "test": ("model_results/testpred_extratrees_safe_base_foldavg.csv", TARGET_COL),
    },
    "lgbm12k": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof.csv", "OOF_lgbm_t03_base_5fold_oof"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof"),
    },
    "lgbm_long80k_lr03": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv", "OOF_lgbm_t03_base_5fold_oof_long80k_lr03"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03"),
    },
    "lgbm_long100k_lr02": {
        "oof": ("model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv", "OOF_lgbm_t03_base_5fold_oof_long100k_lr02"),
        "test": ("model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv", "TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02"),
    },
    "lgbm_div02_3fold": {
        "oof": ("model_results/oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv", "OOF_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
        "test": ("model_results/testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv", "TESTPRED_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof"),
    },
}

oof_preds = {}
test_preds = {}

for name, spec in components.items():
    try:
        oof_p = clip100(read_pred(*spec["oof"]))
        test_p = clip100(read_pred(*spec["test"]))

        if len(oof_p) != len(y) or len(test_p) != len(test_id_series):
            print(f"Skipping {name}: length mismatch")
            continue

        oof_preds[name] = oof_p
        test_preds[name] = test_p
        print(f"Loaded {name}: OOF MSE = {mse_clip(y, oof_p):.6f}")

    except Exception as e:
        print(f"Skipping {name}: {e}")

if "et_safe" not in oof_preds or "lgbm_long100k_lr02" not in oof_preds:
    raise ValueError("Need et_safe and lgbm_long100k_lr02 to rebuild the old 80.662 base blend.")

# Old submitted 80.662 base blend.
base_oof = 0.319 * oof_preds["et_safe"] + 0.681 * oof_preds["lgbm_long100k_lr02"]
base_test = 0.319 * test_preds["et_safe"] + 0.681 * test_preds["lgbm_long100k_lr02"]
base_mse = mse_clip(y, base_oof)

# Existing successful residual correction from 24A/24B.
old_resid_oof = np.load(RESULTS_DIR / f"{old_resid_artifact}_resid_oof.npy")
old_resid_test = mean_test_residual_parts(old_resid_artifact, n_folds=5)

old_lambda = 1.135
current_anchor_oof = clip100(base_oof + old_lambda * old_resid_oof)
current_anchor_test = clip100(base_test + old_lambda * old_resid_test)
current_anchor_mse = mse_clip(y, current_anchor_oof)

print("\nOld 80.662 base OOF MSE:", base_mse)
print("Current residual-stack anchor lambda:", old_lambda)
print("Current residual-stack anchor OOF MSE:", current_anchor_mse)
print("Current public anchor MSE:", PUBLIC_ANCHOR_SCORE)

# ------------------------------------------------------------
# Build feature matrix for a diverse first-level residual model
# ------------------------------------------------------------

component_order = list(oof_preds.keys())

M_train_main = np.column_stack([oof_preds[name] for name in component_order])
M_test_main = np.column_stack([test_preds[name] for name in component_order])

lgbm_names = [n for n in component_order if n.startswith("lgbm")]
lgbm_train = np.column_stack([oof_preds[n] for n in lgbm_names])
lgbm_test = np.column_stack([test_preds[n] for n in lgbm_names])

extra_train_cols = [
    base_oof,
    M_train_main.mean(axis=1),
    M_train_main.std(axis=1),
    M_train_main.min(axis=1),
    M_train_main.max(axis=1),
    M_train_main.max(axis=1) - M_train_main.min(axis=1),
    lgbm_train.mean(axis=1),
    lgbm_train.std(axis=1),
]

extra_test_cols = [
    base_test,
    M_test_main.mean(axis=1),
    M_test_main.std(axis=1),
    M_test_main.min(axis=1),
    M_test_main.max(axis=1),
    M_test_main.max(axis=1) - M_test_main.min(axis=1),
    lgbm_test.mean(axis=1),
    lgbm_test.std(axis=1),
]

# Explicit disagreement features. The model could learn these from the raw
# component predictions, but adding them helps a constrained residual model.
if "et_safe" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["et_safe"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["et_safe"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

if "lgbm_long80k_lr03" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["lgbm_long80k_lr03"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["lgbm_long80k_lr03"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

if "lgbm_div02_3fold" in oof_preds and "lgbm_long100k_lr02" in oof_preds:
    d_tr = oof_preds["lgbm_div02_3fold"] - oof_preds["lgbm_long100k_lr02"]
    d_te = test_preds["lgbm_div02_3fold"] - test_preds["lgbm_long100k_lr02"]
    extra_train_cols.extend([d_tr, np.abs(d_tr)])
    extra_test_cols.extend([d_te, np.abs(d_te)])

M_train_extra = np.column_stack(extra_train_cols).astype(np.float32)
M_test_extra = np.column_stack(extra_test_cols).astype(np.float32)

M_train = np.hstack([M_train_main.astype(np.float32), M_train_extra])
M_test = np.hstack([M_test_main.astype(np.float32), M_test_extra])

Z_train = add_meta_features(X_base, M_train)
Z_test = add_meta_features(X_test_base, M_test)

# Important: target the original base residual, not the 77.277 anchor residual.
# This keeps this as another first-level residual correction.
residual_y = y - base_oof

print("\nResidual target summary:")
print(pd.Series(residual_y).describe())

print("\nNew residual model train matrix:", Z_train.shape)
print("New residual model test matrix:", Z_test.shape)

# ------------------------------------------------------------
# Diverse residual model: Huber + Extra-Trees-style LightGBM
# ------------------------------------------------------------

params = {
    "objective": "huber",
    "metric": "l2",
    "alpha": 0.85,
    "random_state": RANDOM_STATE + 2026,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,
    "learning_rate": 0.025,
    "n_estimators": 30000,
    "num_leaves": 63,
    "max_depth": 8,
    "min_child_samples": 180,
    "subsample": 0.75,
    "subsample_freq": 1,
    "colsample_bytree": 0.75,
    "reg_alpha": 2.0,
    "reg_lambda": 80.0,
    "extra_trees": True,
}

resid_oof_path = RESULTS_DIR / f"{new_artifact}_resid_oof.npy"
metrics_path = RESULTS_DIR / f"{new_artifact}_fold_metrics.csv"

if resid_oof_path.exists():
    new_resid_oof = np.load(resid_oof_path)
else:
    new_resid_oof = np.full(len(y), np.nan, dtype=np.float32)

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame()

safe_feature_names = [f"f{i}" for i in range(Z_train.shape[1])]

# Different seed/split for model diversity.
kf_model = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2026)

overall_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(kf_model.split(Z_train), start=1):
    fold_test_path = RESULTS_DIR / f"{new_artifact}_fold{fold}_test_resid.npy"

    already_done = (
        (not metrics_df.empty)
        and (fold in set(metrics_df["fold"]))
        and fold_test_path.exists()
        and (not np.isnan(new_resid_oof[va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed fold {fold}/5")
        continue

    print("\n" + "=" * 80)
    print(f"Starting diverse residual model fold {fold}/5")
    print("=" * 80)

    start = time.time()

    model = lgb.LGBMRegressor(**params)
    model.fit(
        Z_train[tr_idx],
        residual_y[tr_idx],
        eval_set=[(Z_train[va_idx], residual_y[va_idx])],
        eval_metric="l2",
        feature_name=safe_feature_names,
        callbacks=[
            lgb.early_stopping(stopping_rounds=1500, verbose=True),
            lgb.log_evaluation(period=1500),
        ],
    )

    best_iter = model.best_iteration_ or params["n_estimators"]

    va_resid_pred = model.predict(Z_train[va_idx], num_iteration=best_iter)
    test_resid_pred = model.predict(Z_test, num_iteration=best_iter)

    new_resid_oof[va_idx] = va_resid_pred.astype(np.float32)
    np.save(resid_oof_path, new_resid_oof)
    np.save(fold_test_path, test_resid_pred.astype(np.float32))

    row = {
        "fold": fold,
        "best_iteration": int(best_iter),
        "residual_valid_mse": float(np.mean((residual_y[va_idx] - va_resid_pred) ** 2)),
        "corrected_mse_lambda_1": mse_clip(y[va_idx], base_oof[va_idx] + va_resid_pred),
        "base_mse_on_fold": mse_clip(y[va_idx], base_oof[va_idx]),
        "elapsed_sec": float(time.time() - start),
    }

    metrics_df = metrics_df[metrics_df["fold"] != fold] if not metrics_df.empty else metrics_df
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
    metrics_df = metrics_df.sort_values("fold").reset_index(drop=True)
    metrics_df.to_csv(metrics_path, index=False)

    print("\nCompleted fold:", fold)
    print("Fold corrected MSE at lambda=1:", row["corrected_mse_lambda_1"])
    print("Fold base MSE:", row["base_mse_on_fold"])
    print("Best iteration:", row["best_iteration"])

    del model, va_resid_pred, test_resid_pred
    gc.collect()

completed = (not np.isnan(new_resid_oof).any()) and all(
    (RESULTS_DIR / f"{new_artifact}_fold{fold}_test_resid.npy").exists()
    for fold in range(1, 6)
)

print("\n" + "#" * 90)
print("Diverse residual artifact completion check")
print("#" * 90)
print("Missing OOF residual predictions:", int(np.isnan(new_resid_oof).sum()))
print("Completed:", completed)

if not completed:
    print("\nNot all folds are complete yet. Re-execute this same cell to resume from checkpoints.")

else:
    new_resid_test = mean_test_residual_parts(new_artifact, n_folds=5)

    standalone_new_mse = mse_clip(y, base_oof + new_resid_oof)
    corr_old_new = float(np.corrcoef(old_resid_oof, new_resid_oof)[0, 1])
    corr_new_true = float(np.corrcoef(residual_y, new_resid_oof)[0, 1])

    print("\nStandalone diverse residual correction OOF MSE at lambda=1:", standalone_new_mse)
    print("Correlation old residual correction vs new residual correction:", corr_old_new)
    print("Correlation new residual correction vs true base residual:", corr_new_true)

    # --------------------------------------------------------
    # Blend old successful residual with new diverse residual
    # --------------------------------------------------------

    a_grid = sorted(set(np.round(np.arange(0.85, 1.306, 0.025), 3).tolist() + [old_lambda]))
    b_grid = sorted(set(np.round(np.arange(0.00, 0.801, 0.025), 3).tolist()))

    # Fold diagnostics against the current 1.135 residual-stack anchor.
    # These folds are diagnostic only; the OOF predictions themselves are already cross-fit.
    kf_diag = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    diag_folds = list(kf_diag.split(y))

    rows = []

    for a in a_grid:
        for b in b_grid:
            pred_oof = clip100(base_oof + a * old_resid_oof + b * new_resid_oof)
            overall_mse = mse_clip(y, pred_oof)

            fold_gains = []
            fold_mses = []

            for fold, (_, va_idx) in enumerate(diag_folds, start=1):
                anchor_fold_mse = mse_clip(y[va_idx], current_anchor_oof[va_idx])
                new_fold_mse = mse_clip(y[va_idx], pred_oof[va_idx])
                fold_gains.append(anchor_fold_mse - new_fold_mse)
                fold_mses.append(new_fold_mse)

            pred_test = clip100(base_test + a * old_resid_test + b * new_resid_test)

            rows.append({
                "old_resid_weight": float(a),
                "new_resid_weight": float(b),
                "oof_mse_clipped": overall_mse,
                "gain_vs_current_anchor_oof": current_anchor_mse - overall_mse,
                "min_fold_gain_vs_current": float(np.min(fold_gains)),
                "mean_fold_gain_vs_current": float(np.mean(fold_gains)),
                "max_fold_gain_vs_current": float(np.max(fold_gains)),
                "fold_mse_std": float(np.std(fold_mses, ddof=1)),
                "test_mean": float(np.mean(pred_test)),
                "test_std": float(np.std(pred_test)),
                "test_min": float(np.min(pred_test)),
                "test_max": float(np.max(pred_test)),
            })

    screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

    screen_path = RESULTS_DIR / f"{new_artifact}_blend_with_old_resid_screen.csv"
    screen_df.to_csv(screen_path, index=False)

    print("\nTop residual-diversity blend candidates:")
    try:
        display(screen_df.head(25))
    except NameError:
        print(screen_df.head(25).to_string(index=False))

    print("\nCurrent anchor OOF MSE:", current_anchor_mse)
    print("Best new blend OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
    print("Best new blend OOF gain vs current anchor:", float(screen_df.loc[0, "gain_vs_current_anchor_oof"]))
    print("Best new blend min fold gain vs current:", float(screen_df.loc[0, "min_fold_gain_vs_current"]))
    print("Screen saved to:", screen_path)

    # --------------------------------------------------------
    # Save only candidates that beat the current OOF anchor
    # --------------------------------------------------------

    improved = screen_df[
        (screen_df["gain_vs_current_anchor_oof"] > 0.0)
    ].copy()

    saved_rows = []

    if improved.empty:
        print("\nNo OOF-improving residual-diversity blend was found. Do not submit from this cell.")
    else:
        # Prefer candidates with positive fold-gain evidence, but keep the best OOF candidate too.
        positive_fold = improved[improved["min_fold_gain_vs_current"] > 0].copy()

        candidate_pool = positive_fold if len(positive_fold) > 0 else improved
        candidate_pool = candidate_pool.sort_values("oof_mse_clipped").head(5).reset_index(drop=True)

        for _, row in candidate_pool.iterrows():
            a = float(row["old_resid_weight"])
            b = float(row["new_resid_weight"])

            final_oof = clip100(base_oof + a * old_resid_oof + b * new_resid_oof)
            final_test = clip100(base_test + a * old_resid_test + b * new_resid_test)

            name = (
                f"{new_artifact}_oldw_{str(round(a, 3)).replace('.', 'p')}"
                f"_neww_{str(round(b, 3)).replace('.', 'p')}"
            )

            oof_path = RESULTS_DIR / f"oof_{name}.csv"
            testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
            sub_path = Path(f"submission_{name}.csv")

            pd.DataFrame({
                TARGET_COL: y,
                f"OOF_{name}": final_oof
            }).to_csv(oof_path, index=False)

            pd.DataFrame({
                ID_COL: test_id_series.values,
                f"TESTPRED_{name}": final_test
            }).to_csv(testpred_path, index=False)

            pd.DataFrame({
                ID_COL: test_id_series.values,
                TARGET_COL: final_test
            }).to_csv(sub_path, index=False)

            saved_rows.append({
                "candidate": name,
                "old_resid_weight": a,
                "new_resid_weight": b,
                "oof_mse_clipped": mse_clip(y, final_oof),
                "gain_vs_current_anchor_oof": current_anchor_mse - mse_clip(y, final_oof),
                "submission_path": str(sub_path),
                "testpred_path": str(testpred_path),
                "test_mean": float(np.mean(final_test)),
                "test_std": float(np.std(final_test)),
                "test_min": float(np.min(final_test)),
                "test_max": float(np.max(final_test)),
            })

        saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
        saved_path = RESULTS_DIR / f"{new_artifact}_saved_candidates.csv"
        saved_df.to_csv(saved_path, index=False)

        print("\nSaved OOF-improving residual-diversity candidates:")
        try:
            display(saved_df)
        except NameError:
            print(saved_df.to_string(index=False))

        print("Saved candidates list:", saved_path)

        best_gain = float(saved_df.loc[0, "gain_vs_current_anchor_oof"])
        best_sub = saved_df.loc[0, "submission_path"]

        if best_gain >= 0.15:
            print("\nSubmission candidate worth considering:", best_sub)
            print("Reason: OOF gain versus current anchor is at least 0.15.")
        else:
            print("\nGain is small. Hold this candidate unless there are no better modeling directions.")

    print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Public checkpoint already present in tracker: model_results/submission_tracker.csv
Loaded rf500: OOF MSE = 122.328024
Loaded et_safe: OOF MSE = 110.749294
Loaded lgbm12k: OOF MSE = 98.779406
Loaded lgbm_long80k_lr03: OOF MSE = 94.163485
Loaded lgbm_long100k_lr02: OOF MSE = 93.726214
Loaded lgbm_div02_3fold: OOF MSE = 104.083036

Old 80.662 base OOF MSE: 88.96636822734054
Current residual-stack anchor lambda: 1.135
Current residual-stack anchor OOF MSE: 87.13604874858268
Current public anchor MSE: 77.277

Residual target summary:
count    144921.000000
mean          0.038600
std           9.432152
min         -74.711172
25%          -4.382987
50%           0.127390
75%           4.493513
max          84.650332
dtype: float64

New residual model train matrix: (144921, 182)
New residual model test matrix: (48307, 182)

Skipping completed fold 1/5

Skipping completed fold 2/5

Skipping completed fold 3/5

Skipping completed fold 4/5

Skipping completed fold 5/5

###########################

,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,min_fold_gain_vs_current,mean_fold_gain_vs_current,max_fold_gain_vs_current,fold_mse_std,test_mean,test_std,test_min,test_max
0,0.850,0.525,86.101600,1.034449,0.832125,1.034449,1.260125,2.024438,54.135944,24.990570,0.000000,100.0
1,0.850,0.500,86.103925,1.032124,0.838290,1.032124,1.246989,2.019299,54.133858,24.969032,0.000000,100.0
2,0.850,0.550,86.104379,1.031670,0.820974,1.031670,1.268000,2.029641,54.137978,25.012076,0.000000,100.0
3,0.850,0.475,86.111472,1.024576,0.839445,1.024576,1.228558,2.014096,54.131729,24.947484,0.000000,100.0
4,0.850,0.575,86.112234,1.023815,0.804855,1.023815,1.270733,2.034937,54.139975,25.033576,0.000000,100.0
5,0.850,0.450,86.124204,1.011845,0.835587,1.011844,1.204820,2.008816,54.129571,24.925946,0.000000,100.0
6,0.850,0.600,86.125240,1.010808,0.783856,1.010808,1.268266,2.040195,54.141928,25.055058,0.000000,100.0
7,0.875,0.525,86.129413,1.006636,0.804662,1.006635,1.232918,2.023738,54.136588,25.002140,0.000000,100.0
8,0.875,0.500,86.129549,1.006500,0.812929,1.006500,1.222086,2.018712,54.134528,24.980623,0.000000,100.0
9,0.875,0.550,86.134341,1.001708,0.791425,1.001707,1.238502,2.028886,54.138599,25.023630,0.000000,100.0



Current anchor OOF MSE: 87.13604874858268
Best new blend OOF MSE: 86.10159951777065
Best new blend OOF gain vs current anchor: 1.0344492308120294
Best new blend min fold gain vs current: 0.8321252420166019
Screen saved to: model_results/residstack_huber_extra_on_best80p662_blend_5fold_blend_with_old_resid_screen.csv

Saved OOF-improving residual-diversity candidates:


,candidate,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.525,86.101600,1.034449,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.135944,24.990570,0.0,100.0
1,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.500,86.103925,1.032124,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.133858,24.969032,0.0,100.0
2,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.550,86.104379,1.031670,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.137978,25.012076,0.0,100.0
3,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.475,86.111472,1.024576,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.131729,24.947484,0.0,100.0
4,residstack_huber_extra_on_best80p662_blend_5fo...,0.85,0.575,86.112234,1.023815,submission_residstack_huber_extra_on_best80p66...,model_results/testpred_residstack_huber_extra_...,54.139975,25.033576,0.0,100.0


Saved candidates list: model_results/residstack_huber_extra_on_best80p662_blend_5fold_saved_candidates.csv

Submission candidate worth considering: submission_residstack_huber_extra_on_best80p662_blend_5fold_oldw_0p85_neww_0p525.csv
Reason: OOF gain versus current anchor is at least 0.15.

Total elapsed minutes: 0.03


### 25A Huber/extra-trees-style residual model checkpoint

The diverse residual model completed successfully and all five fold artifacts were available from checkpointed predictions.

This model was trained as a new first-level residual correction on the original best 80.662 base blend residuals, rather than as a second-stage correction on top of the 77.277 model. The standalone new residual correction achieved OOF MSE `85.6994`, compared with `87.1360` for the current 77.277 public anchor, giving an OOF gain of approximately `1.4367`.

The new residual correction is correlated with the previous residual correction at about `0.7401`, so it is related but not identical. Its correlation with the original base residual target is about `0.1931`, which is enough to produce a large OOF improvement.

The initial old+new residual blend screen found a best constrained blend at old residual weight `0.850` and new residual weight `0.525`, with OOF MSE `86.1016` and positive fold gains versus the current anchor. However, this constrained screen did not allow the old residual weight to approach zero, while the standalone new residual model already has better OOF MSE than the saved blend candidates.

Therefore, the next step is an expanded coefficient screen over both residual corrections, including the standalone new residual model and low/zero old-residual weights. No additional model training is needed for this step.

In [49]:
# ============================================================
# 25B. Expanded coefficient screen for old vs new residuals
# ============================================================
#
# 25A found:
# - current anchor OOF MSE: 87.1360
# - standalone new residual OOF MSE: 85.6994
# - constrained old+new blend best OOF MSE: 86.1016
#
# The constrained blend screen did not allow old_resid_weight near zero.
# This cell does no training. It only screens:
#
#     base + old_weight * old_residual + new_weight * new_residual
#
# over a wider grid, then saves the best OOF-backed candidates.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(x):
    return str(round(float(x), 4)).replace("-", "m").replace(".", "p")

def mean_test_residual_parts(artifact, n_folds=5):
    return np.mean(
        [np.load(RESULTS_DIR / f"{artifact}_fold{fold}_test_resid.npy") for fold in range(1, n_folds + 1)],
        axis=0,
    )

# These should already exist because 25A just ran.
# The fallbacks only reload residual artifacts if needed.
old_artifact = "residstack_lgbm_on_best80p662_blend_5fold"
new_artifact = "residstack_huber_extra_on_best80p662_blend_5fold"

if "old_resid_oof" not in globals():
    old_resid_oof = np.load(RESULTS_DIR / f"{old_artifact}_resid_oof.npy")

if "old_resid_test" not in globals():
    old_resid_test = mean_test_residual_parts(old_artifact, n_folds=5)

if "new_resid_oof" not in globals():
    new_resid_oof = np.load(RESULTS_DIR / f"{new_artifact}_resid_oof.npy")

if "new_resid_test" not in globals():
    new_resid_test = mean_test_residual_parts(new_artifact, n_folds=5)

if "current_anchor_oof" not in globals():
    current_anchor_oof = clip100(base_oof + 1.135 * old_resid_oof)

current_anchor_mse = mse_clip(y, current_anchor_oof)

standalone_new_oof = clip100(base_oof + 1.0 * new_resid_oof)
standalone_new_mse = mse_clip(y, standalone_new_oof)

print("Current anchor OOF MSE:", current_anchor_mse)
print("Standalone new residual OOF MSE:", standalone_new_mse)
print("Standalone new gain vs current anchor:", current_anchor_mse - standalone_new_mse)

# Wider coefficient screen.
# old weight is allowed to fall to zero or slightly negative because the new residual
# may already contain much of the old residual signal.
old_grid = np.round(np.arange(-0.300, 1.251, 0.025), 3)
new_grid = np.round(np.arange(0.000, 1.601, 0.025), 3)

# Add exact known points.
old_grid = np.array(sorted(set(old_grid.tolist() + [0.0, 0.85, 1.135])))
new_grid = np.array(sorted(set(new_grid.tolist() + [0.525, 1.0])))

diag_seed = globals().get("RANDOM_STATE", 9890)
diag_folds = list(KFold(n_splits=5, shuffle=True, random_state=diag_seed).split(y))

rows = []

for ow in old_grid:
    old_part_oof = base_oof + ow * old_resid_oof
    old_part_test = base_test + ow * old_resid_test

    for nw in new_grid:
        pred_oof = clip100(old_part_oof + nw * new_resid_oof)
        oof_mse = mse_clip(y, pred_oof)

        fold_gains = []
        fold_mses = []

        for fold, (_, va_idx) in enumerate(diag_folds, start=1):
            anchor_fold_mse = mse_clip(y[va_idx], current_anchor_oof[va_idx])
            cand_fold_mse = mse_clip(y[va_idx], pred_oof[va_idx])
            fold_gains.append(anchor_fold_mse - cand_fold_mse)
            fold_mses.append(cand_fold_mse)

        pred_test = clip100(old_part_test + nw * new_resid_test)

        rows.append({
            "old_resid_weight": float(ow),
            "new_resid_weight": float(nw),
            "oof_mse_clipped": float(oof_mse),
            "gain_vs_current_anchor_oof": float(current_anchor_mse - oof_mse),
            "gain_vs_standalone_new_oof": float(standalone_new_mse - oof_mse),
            "min_fold_gain_vs_current": float(np.min(fold_gains)),
            "mean_fold_gain_vs_current": float(np.mean(fold_gains)),
            "max_fold_gain_vs_current": float(np.max(fold_gains)),
            "fold_mse_std": float(np.std(fold_mses, ddof=1)),
            "test_mean": float(np.mean(pred_test)),
            "test_std": float(np.std(pred_test)),
            "test_min": float(np.min(pred_test)),
            "test_max": float(np.max(pred_test)),
        })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

screen_path = RESULTS_DIR / "residstack_oldnew_expanded_weight_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop expanded old/new residual coefficient candidates:")
try:
    display(screen_df.head(30))
except NameError:
    print(screen_df.head(30).to_string(index=False))

print("\nScreen saved to:", screen_path)
print("Best expanded OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best expanded gain vs current anchor:", float(screen_df.loc[0, "gain_vs_current_anchor_oof"]))
print("Best expanded gain vs standalone new:", float(screen_df.loc[0, "gain_vs_standalone_new_oof"]))
print("Best expanded min fold gain vs current:", float(screen_df.loc[0, "min_fold_gain_vs_current"]))

# Save top candidates that beat the current anchor and have positive fold-gain diagnostics.
eligible = screen_df[
    (screen_df["gain_vs_current_anchor_oof"] > 0)
    & (screen_df["min_fold_gain_vs_current"] > 0)
].copy()

if eligible.empty:
    print("\nNo eligible expanded candidate found. This would be unexpected given standalone new OOF.")
else:
    saved_rows = []

    # Save top 8, but avoid saving many nearly identical rows if desired later.
    for _, row in eligible.head(8).iterrows():
        ow = float(row["old_resid_weight"])
        nw = float(row["new_resid_weight"])

        final_oof = clip100(base_oof + ow * old_resid_oof + nw * new_resid_oof)
        final_test = clip100(base_test + ow * old_resid_test + nw * new_resid_test)

        name = f"residstack_oldnew_expanded_oldw_{safe_token(ow)}_neww_{safe_token(nw)}"

        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof,
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            f"TESTPRED_{name}": final_test,
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: test_id_series.values,
            TARGET_COL: final_test,
        }).to_csv(sub_path, index=False)

        saved_rows.append({
            "candidate": name,
            "old_resid_weight": ow,
            "new_resid_weight": nw,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_current_anchor_oof": current_anchor_mse - mse_clip(y, final_oof),
            "gain_vs_standalone_new_oof": standalone_new_mse - mse_clip(y, final_oof),
            "submission_path": str(sub_path),
            "testpred_path": str(testpred_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        })

    saved_df = pd.DataFrame(saved_rows).sort_values("oof_mse_clipped").reset_index(drop=True)
    saved_path = RESULTS_DIR / "residstack_oldnew_expanded_saved_candidates.csv"
    saved_df.to_csv(saved_path, index=False)

    print("\nSaved expanded old/new residual candidates:")
    try:
        display(saved_df)
    except NameError:
        print(saved_df.to_string(index=False))

    print("\nSaved candidates list:", saved_path)
    print("Best submission candidate:", saved_df.loc[0, "submission_path"])

Current anchor OOF MSE: 87.13604874858268
Standalone new residual OOF MSE: 85.69939570622593
Standalone new gain vs current anchor: 1.4366530423567525

Top expanded old/new residual coefficient candidates:


,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,gain_vs_standalone_new_oof,min_fold_gain_vs_current,mean_fold_gain_vs_current,max_fold_gain_vs_current,fold_mse_std,test_mean,test_std,test_min,test_max
0,0.025,0.875,85.647385,1.488664,0.052011,1.181558,1.488663,1.776519,2.075271,54.142717,24.913286,0.0,100.0
1,-0.025,0.900,85.647596,1.488453,0.051800,1.175174,1.488451,1.777796,2.076937,54.143403,24.912080,0.0,100.0
2,0.000,0.875,85.647722,1.488326,0.051673,1.182816,1.488325,1.770613,2.071575,54.141995,24.901855,0.0,100.0
3,0.000,0.900,85.647759,1.488290,0.051637,1.173405,1.488289,1.783309,2.080667,54.144118,24.923507,0.0,100.0
4,0.050,0.875,85.648805,1.487244,0.050591,1.178659,1.487243,1.780495,2.078763,54.143432,24.924726,0.0,100.0
5,-0.050,0.900,85.649191,1.486858,0.050205,1.175296,1.486856,1.770344,2.073001,54.142681,24.900660,0.0,100.0
6,0.050,0.850,85.649498,1.486551,0.049898,1.187350,1.486550,1.767150,2.069890,54.141305,24.903074,0.0,100.0
7,0.025,0.900,85.649672,1.486377,0.049724,1.169992,1.486376,1.786900,2.084213,54.144829,24.934944,0.0,100.0
8,-0.025,0.875,85.649823,1.486225,0.049572,1.182432,1.486224,1.762759,2.067673,54.141270,24.890437,0.0,100.0
9,-0.050,0.925,85.649944,1.486105,0.049451,1.165134,1.486103,1.782429,2.082340,54.144801,24.922316,0.0,100.0



Screen saved to: model_results/residstack_oldnew_expanded_weight_screen.csv
Best expanded OOF MSE: 85.64738488864343
Best expanded gain vs current anchor: 1.4886638599392512
Best expanded gain vs standalone new: 0.0520108175824987
Best expanded min fold gain vs current: 1.181558077853552

Saved expanded old/new residual candidates:


,candidate,old_resid_weight,new_resid_weight,oof_mse_clipped,gain_vs_current_anchor_oof,gain_vs_standalone_new_oof,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,residstack_oldnew_expanded_oldw_0p025_neww_0p875,0.025,0.875,85.647385,1.488664,0.052011,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.142717,24.913286,0.0,100.0
1,residstack_oldnew_expanded_oldw_m0p025_neww_0p9,-0.025,0.900,85.647596,1.488453,0.051800,submission_residstack_oldnew_expanded_oldw_m0p...,model_results/testpred_residstack_oldnew_expan...,54.143403,24.912080,0.0,100.0
2,residstack_oldnew_expanded_oldw_0p0_neww_0p875,0.000,0.875,85.647722,1.488326,0.051673,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.141995,24.901855,0.0,100.0
3,residstack_oldnew_expanded_oldw_0p0_neww_0p9,0.000,0.900,85.647759,1.488290,0.051637,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.144118,24.923507,0.0,100.0
4,residstack_oldnew_expanded_oldw_0p05_neww_0p875,0.050,0.875,85.648805,1.487244,0.050591,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.143432,24.924726,0.0,100.0
5,residstack_oldnew_expanded_oldw_m0p05_neww_0p9,-0.050,0.900,85.649191,1.486858,0.050205,submission_residstack_oldnew_expanded_oldw_m0p...,model_results/testpred_residstack_oldnew_expan...,54.142681,24.900660,0.0,100.0
6,residstack_oldnew_expanded_oldw_0p05_neww_0p85,0.050,0.850,85.649498,1.486551,0.049898,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.141305,24.903074,0.0,100.0
7,residstack_oldnew_expanded_oldw_0p025_neww_0p9,0.025,0.900,85.649672,1.486377,0.049724,submission_residstack_oldnew_expanded_oldw_0p0...,model_results/testpred_residstack_oldnew_expan...,54.144829,24.934944,0.0,100.0



Saved candidates list: model_results/residstack_oldnew_expanded_saved_candidates.csv
Best submission candidate: submission_residstack_oldnew_expanded_oldw_0p025_neww_0p875.csv


### 25B expanded old/new residual coefficient screen

The expanded old/new residual screen tested combinations of the original LightGBM residual correction and the newer Huber/extra-trees-style residual correction.

The best internal candidate was:

`residstack_oldnew_expanded_oldw_0p025_neww_0p875`

with OOF MSE `85.6474`, an OOF gain of approximately `1.4887` versus the current submitted 77.277 public anchor, and a positive minimum fold gain of approximately `1.1816`.

However, the gain versus the standalone new residual correction was only about `0.0520` OOF MSE. The top region of the screen was also very flat, with old residual weights near zero and new residual weights around `0.875–0.900`. This suggests that the newer Huber/extra residual model mostly supersedes the older residual correction.

Because this coefficient screen is mainly a small refinement of an already-discovered direction, it is not submitted immediately. Instead, this candidate is kept as the new internal OOF anchor for further modeling.

After the strong residual models, are there still systematic residual biases by
school, district, assessment, subgroup, region, or combinations of these?

In [50]:
# ============================================================
# 26A. Cross-fitted grouped residual calibration
# ============================================================
#
# Purpose:
# - Do NOT submit the 25B coefficient tweak yet.
# - Use the best 25B candidate as an internal OOF anchor.
# - Test whether remaining residuals contain systematic group-level structure:
#   school, district, county, assessment, subgroup, region, and combinations.
#
# This is a leakage-safe residual target encoding / empirical Bayes correction:
# - OOF corrections are built fold-by-fold.
# - A validation row's group residual mean is computed only from other folds.
# - Test corrections use full training residual statistics.
#
# This cell is much faster than training another full model.

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(s):
    return str(s).replace(" ", "_").replace("/", "_").replace("-", "m").replace(".", "p")

def get_test_ids():
    if "test_id_series" in globals():
        return pd.Series(test_id_series).astype(str).reset_index(drop=True)
    if "test_ids" in globals():
        return pd.Series(test_ids).astype(str).reset_index(drop=True)
    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)
    raise ValueError("Could not find test IDs.")

def make_key(df, cols):
    tmp = df.loc[:, list(cols)].copy()
    for c in cols:
        tmp[c] = tmp[c].astype("string").fillna("__NA__")
    return tmp.astype(str).agg("||".join, axis=1).to_numpy()

def smooth_group_map(keys, values, alpha):
    d = pd.DataFrame({"key": keys, "value": values})
    stats = d.groupby("key")["value"].agg(["count", "mean"])
    global_mean = float(np.mean(values))
    stats["smooth"] = (
        stats["count"] * stats["mean"] + alpha * global_mean
    ) / (stats["count"] + alpha)
    return stats["smooth"], global_mean

def map_with_default(keys, smooth_map, default):
    return pd.Series(keys).map(smooth_map).fillna(default).to_numpy(dtype=float)

# ------------------------------------------------------------
# Internal anchor from 25B
# ------------------------------------------------------------

# Best 25B candidate:
# old_resid_weight = 0.025
# new_resid_weight = 0.875

internal_old_w = 0.025
internal_new_w = 0.875

internal_anchor_name = (
    f"oldnew_internal_oldw_{str(internal_old_w).replace('.', 'p')}"
    f"_neww_{str(internal_new_w).replace('.', 'p')}"
)

internal_anchor_oof = clip100(
    base_oof + internal_old_w * old_resid_oof + internal_new_w * new_resid_oof
)

internal_anchor_test = clip100(
    base_test + internal_old_w * old_resid_test + internal_new_w * new_resid_test
)

internal_anchor_mse = mse_clip(y, internal_anchor_oof)

print("Internal 25B anchor:", internal_anchor_name)
print("Internal 25B anchor OOF MSE:", internal_anchor_mse)

# Residual left after current best internal candidate.
resid_left = np.asarray(y, dtype=float) - internal_anchor_oof

print("\nRemaining residual summary:")
print(pd.Series(resid_left).describe())

# ------------------------------------------------------------
# Metadata table for group corrections
# ------------------------------------------------------------

train_meta = train_full.copy()
test_meta = test_full.copy()

# Add N_STUDENTS bins, because assessment size may change residual behavior.
if "N_STUDENTS" in train_meta.columns and "N_STUDENTS" in test_meta.columns:
    train_bins, bin_edges = pd.qcut(
        train_meta["N_STUDENTS"],
        q=10,
        duplicates="drop",
        retbins=True,
    )
    train_meta["N_STUDENTS_BIN"] = train_bins.astype(str)
    test_meta["N_STUDENTS_BIN"] = pd.cut(
        test_meta["N_STUDENTS"],
        bins=bin_edges,
        include_lowest=True,
    ).astype(str)

# Candidate group structures.
raw_group_specs = [
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("SCHOOL",),
    ("DISTRICT",),
    ("COUNTY",),
    ("REGION",),
    ("DISTRICT_TYPE",),

    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "N_STUDENTS_BIN"),
    ("SUBGROUP_NAME", "N_STUDENTS_BIN"),

    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),

    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
]

group_specs = []
for spec in raw_group_specs:
    if all(c in train_meta.columns and c in test_meta.columns for c in spec):
        group_specs.append(spec)

print("\nGroup specs used:")
for spec in group_specs:
    print("  ", spec)

alpha_grid = [5, 20, 100, 300]

# ------------------------------------------------------------
# Build cross-fitted group residual correction features
# ------------------------------------------------------------

n = len(y)
nt = len(test_meta)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

group_feature_names = []
group_oof_cols = []
group_test_cols = []
individual_scores = []

for spec in group_specs:
    train_keys = make_key(train_meta, spec)
    test_keys = make_key(test_meta, spec)

    for alpha in alpha_grid:
        feature_name = "grp_" + "__".join(spec) + f"__a{alpha}"
        print("\nBuilding:", feature_name)

        corr_oof = np.zeros(n, dtype=float)

        for fold, (tr_idx, va_idx) in enumerate(kf.split(train_meta), start=1):
            smap, default = smooth_group_map(
                train_keys[tr_idx],
                resid_left[tr_idx],
                alpha=alpha,
            )
            corr_oof[va_idx] = map_with_default(
                train_keys[va_idx],
                smap,
                default,
            )

        # Full-train mapping for test.
        smap_full, default_full = smooth_group_map(
            train_keys,
            resid_left,
            alpha=alpha,
        )
        corr_test = map_with_default(
            test_keys,
            smap_full,
            default_full,
        )

        # Individual correction screen: best scalar lambda on this correction.
        best_lam = None
        best_mse = np.inf
        for lam in np.round(np.arange(-0.50, 1.501, 0.025), 3):
            m = mse_clip(y, internal_anchor_oof + lam * corr_oof)
            if m < best_mse:
                best_mse = m
                best_lam = float(lam)

        individual_scores.append({
            "feature": feature_name,
            "group_spec": "|".join(spec),
            "alpha": alpha,
            "best_lambda": best_lam,
            "oof_mse": best_mse,
            "gain_vs_internal_anchor": internal_anchor_mse - best_mse,
            "corr_oof_std": float(np.std(corr_oof)),
            "corr_test_std": float(np.std(corr_test)),
        })

        group_feature_names.append(feature_name)
        group_oof_cols.append(corr_oof)
        group_test_cols.append(corr_test)

group_oof = np.column_stack(group_oof_cols)
group_test = np.column_stack(group_test_cols)

individual_df = pd.DataFrame(individual_scores).sort_values(
    "oof_mse"
).reset_index(drop=True)

individual_path = RESULTS_DIR / "group_residual_individual_screen.csv"
individual_df.to_csv(individual_path, index=False)

print("\nTop individual group residual corrections:")
try:
    display(individual_df.head(25))
except NameError:
    print(individual_df.head(25).to_string(index=False))

print("Individual screen saved to:", individual_path)

# ------------------------------------------------------------
# Cross-fitted ridge meta-combination of group corrections
# ------------------------------------------------------------

meta_oof = np.zeros(n, dtype=float)
meta_test_folds = []

ridge_alphas = np.logspace(-3, 4, 20)
meta_folds = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2601).split(group_oof))

meta_fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(meta_folds, start=1):
    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )

    model.fit(group_oof[tr_idx], resid_left[tr_idx])

    pred_va = model.predict(group_oof[va_idx])
    pred_test = model.predict(group_test)

    meta_oof[va_idx] = pred_va
    meta_test_folds.append(pred_test)

    fold_mse_before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
    fold_mse_after_raw = mse_clip(y[va_idx], internal_anchor_oof[va_idx] + pred_va)

    meta_fold_rows.append({
        "fold": fold,
        "mse_before": fold_mse_before,
        "mse_after_raw_meta": fold_mse_after_raw,
        "gain_raw_meta": fold_mse_before - fold_mse_after_raw,
    })

meta_test = np.mean(meta_test_folds, axis=0)

meta_fold_df = pd.DataFrame(meta_fold_rows)
print("\nMeta fold diagnostics before lambda shrink:")
try:
    display(meta_fold_df)
except NameError:
    print(meta_fold_df.to_string(index=False))

# ------------------------------------------------------------
# Lambda screen for the meta correction
# ------------------------------------------------------------

rows = []

for lam in np.round(np.arange(-0.50, 1.501, 0.005), 3):
    pred_oof = clip100(internal_anchor_oof + lam * meta_oof)

    fold_gains = []
    fold_mses = []

    for fold, (_, va_idx) in enumerate(meta_folds, start=1):
        before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
        after = mse_clip(y[va_idx], pred_oof[va_idx])
        fold_gains.append(before - after)
        fold_mses.append(after)

    pred_test = clip100(internal_anchor_test + lam * meta_test)

    rows.append({
        "lambda": float(lam),
        "oof_mse_clipped": mse_clip(y, pred_oof),
        "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, pred_oof),
        "min_fold_gain": float(np.min(fold_gains)),
        "mean_fold_gain": float(np.mean(fold_gains)),
        "max_fold_gain": float(np.max(fold_gains)),
        "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        "test_mean": float(np.mean(pred_test)),
        "test_std": float(np.std(pred_test)),
        "test_min": float(np.min(pred_test)),
        "test_max": float(np.max(pred_test)),
    })

screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

screen_path = RESULTS_DIR / "group_residual_meta_lambda_screen.csv"
screen_df.to_csv(screen_path, index=False)

print("\nTop grouped residual meta-correction lambda candidates:")
try:
    display(screen_df.head(30))
except NameError:
    print(screen_df.head(30).to_string(index=False))

print("\nGrouped residual screen saved to:", screen_path)
print("Internal anchor OOF MSE:", internal_anchor_mse)
print("Best grouped residual OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
print("Best grouped residual gain:", float(screen_df.loc[0, "gain_vs_internal_anchor"]))
print("Best grouped residual min fold gain:", float(screen_df.loc[0, "min_fold_gain"]))

# ------------------------------------------------------------
# Save candidate only if it improves the internal anchor
# ------------------------------------------------------------

eligible = screen_df[
    (screen_df["gain_vs_internal_anchor"] > 0)
    & (screen_df["min_fold_gain"] > 0)
].copy()

if eligible.empty:
    print("\nNo positive-fold grouped residual candidate found. Do not submit from this cell.")
else:
    best = eligible.iloc[0]
    lam = float(best["lambda"])

    final_oof = clip100(internal_anchor_oof + lam * meta_oof)
    final_test = clip100(internal_anchor_test + lam * meta_test)

    name = f"group_resid_meta_on_{internal_anchor_name}_lambda_{safe_token(lam)}"

    oof_path = RESULTS_DIR / f"oof_{name}.csv"
    testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
    sub_path = Path(f"submission_{name}.csv")

    pd.DataFrame({
        TARGET_COL: y,
        f"OOF_{name}": final_oof,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        ID_COL: get_test_ids().values,
        f"TESTPRED_{name}": final_test,
    }).to_csv(testpred_path, index=False)

    pd.DataFrame({
        ID_COL: get_test_ids().values,
        TARGET_COL: final_test,
    }).to_csv(sub_path, index=False)

    saved = pd.DataFrame([{
        "candidate": name,
        "lambda": lam,
        "oof_mse_clipped": mse_clip(y, final_oof),
        "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, final_oof),
        "submission_path": str(sub_path),
        "testpred_path": str(testpred_path),
        "test_mean": float(np.mean(final_test)),
        "test_std": float(np.std(final_test)),
        "test_min": float(np.min(final_test)),
        "test_max": float(np.max(final_test)),
    }])

    saved_path = RESULTS_DIR / "group_residual_meta_saved_candidate.csv"
    saved.to_csv(saved_path, index=False)

    print("\nSaved grouped residual candidate:")
    try:
        display(saved)
    except NameError:
        print(saved.to_string(index=False))

    print("Saved candidate list:", saved_path)
    print("Candidate file:", sub_path)

    # Coefficient inspection from full model, for insight only.
    final_model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )
    final_model.fit(group_oof, resid_left)

    ridge = final_model.named_steps["ridgecv"]
    coef_df = pd.DataFrame({
        "feature": group_feature_names,
        "coef_scaled_space": ridge.coef_,
    }).assign(abs_coef=lambda d: d["coef_scaled_space"].abs())

    coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
    coef_path = RESULTS_DIR / "group_residual_meta_top_coefficients.csv"
    coef_df.to_csv(coef_path, index=False)

    print("\nTop grouped residual meta features by coefficient magnitude:")
    try:
        display(coef_df.head(25))
    except NameError:
        print(coef_df.head(25).to_string(index=False))

    print("Coefficient table saved to:", coef_path)

Internal 25B anchor: oldnew_internal_oldw_0p025_neww_0p875
Internal 25B anchor OOF MSE: 85.64738488718324

Remaining residual summary:
count    144921.000000
mean         -0.057888
std           9.254438
min         -79.243600
25%          -4.158437
50%           0.027997
75%           4.053417
max          84.574910
dtype: float64

Group specs used:
   ('ASSESSMENT_NAME',)
   ('SUBGROUP_NAME',)
   ('SCHOOL',)
   ('DISTRICT',)
   ('COUNTY',)
   ('REGION',)
   ('DISTRICT_TYPE',)
   ('ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('ASSESSMENT_NAME', 'N_STUDENTS_BIN')
   ('SUBGROUP_NAME', 'N_STUDENTS_BIN')
   ('SCHOOL', 'ASSESSMENT_NAME')
   ('SCHOOL', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME')
   ('DISTRICT', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_NAME')
   ('COUNTY', 'SUBGROUP_NAME')
   ('REGION', 'ASSESSMENT_NAME')
   ('DISTRICT_TYPE', 'ASSESSMENT_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_

,feature,group_spec,alpha,best_lambda,oof_mse,gain_vs_internal_anchor,corr_oof_std,corr_test_std
0,grp_SCHOOL__SUBGROUP_NAME__a5,SCHOOL|SUBGROUP_NAME,5,-0.500,81.334289,4.313096,1.719158,1.465856
1,grp_SCHOOL__ASSESSMENT_NAME__a5,SCHOOL|ASSESSMENT_NAME,5,-0.500,82.666780,2.980605,1.617578,1.946658
2,grp_SCHOOL__SUBGROUP_NAME__a20,SCHOOL|SUBGROUP_NAME,20,-0.500,83.776380,1.871004,0.722063,0.653960
3,grp_DISTRICT__ASSESSMENT_NAME__a5,DISTRICT|ASSESSMENT_NAME,5,-0.500,83.835018,1.812367,1.379330,1.437408
4,grp_SCHOOL__a5,SCHOOL,5,-0.500,84.060439,1.586946,1.341112,1.079777
5,grp_DISTRICT__SUBGROUP_NAME__a5,DISTRICT|SUBGROUP_NAME,5,-0.500,84.329519,1.317866,1.020284,0.779819
6,grp_SCHOOL__ASSESSMENT_NAME__a20,SCHOOL|ASSESSMENT_NAME,20,-0.500,84.529056,1.118329,0.534157,0.656322
7,grp_SCHOOL__a20,SCHOOL,20,-0.500,84.551904,1.095481,0.894762,0.764166
8,grp_DISTRICT__ASSESSMENT_NAME__a20,DISTRICT|ASSESSMENT_NAME,20,-0.500,84.773057,0.874328,0.696646,0.684216
9,grp_DISTRICT__SUBGROUP_NAME__a20,DISTRICT|SUBGROUP_NAME,20,-0.500,84.818798,0.828587,0.640020,0.520420


Individual screen saved to: model_results/group_residual_individual_screen.csv

Meta fold diagnostics before lambda shrink:


,fold,mse_before,mse_after_raw_meta,gain_raw_meta
0,1,86.161319,71.362821,14.798498
1,2,83.772147,68.903692,14.868455
2,3,85.657469,71.209126,14.448343
3,4,86.072973,71.273155,14.799818
4,5,86.572999,71.271056,15.301943



Top grouped residual meta-correction lambda candidates:


...


Grouped residual screen saved to: model_results/group_residual_meta_lambda_screen.csv
Internal anchor OOF MSE: 85.64738488718324
Best grouped residual OOF MSE: 70.8011606501738
Best grouped residual gain: 14.846224237009437
Best grouped residual min fold gain: 14.442496884824408

Saved grouped residual candidate:


,candidate,lambda,oof_mse_clipped,gain_vs_internal_anchor,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,group_resid_meta_on_oldnew_internal_oldw_0p025...,1.015,70.801161,14.846224,submission_group_resid_meta_on_oldnew_internal...,model_results/testpred_group_resid_meta_on_old...,54.115372,24.496373,0.0,100.0


Saved candidate list: model_results/group_residual_meta_saved_candidate.csv
Candidate file: submission_group_resid_meta_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p015.csv

Top grouped residual meta features by coefficient magnitude:


,feature,coef_scaled_space,abs_coef
0,grp_SCHOOL__a100,64.191121,64.191121
1,grp_SCHOOL__a300,-41.944601,41.944601
2,grp_SCHOOL__SUBGROUP_NAME__a100,30.898091,30.898091
3,grp_SCHOOL__a20,-28.024627,28.024627
4,grp_SUBGROUP_NAME__a300,-24.430791,24.430791
5,grp_SCHOOL__SUBGROUP_NAME__a300,-23.708511,23.708511
6,grp_SCHOOL__ASSESSMENT_NAME__a20,15.344819,15.344819
7,grp_SUBGROUP_NAME__a5,12.852220,12.852220
8,grp_SUBGROUP_NAME__a20,10.935674,10.935674
9,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a100,10.711082,10.711082


Coefficient table saved to: model_results/group_residual_meta_top_coefficients.csv


### 26A grouped residual calibration checkpoint

The grouped residual calibration experiment found a very large apparent OOF improvement. Starting from the internal 25B anchor with OOF MSE `85.6474`, the grouped residual meta-correction reached OOF MSE `70.8012` at lambda `1.015`, an apparent gain of approximately `14.8462`. Fold-level gains were also large and positive, with minimum fold gain around `14.4425`.

The strongest individual group residual signals were mostly school- and district-based combinations, especially `SCHOOL × SUBGROUP_NAME`, `SCHOOL × ASSESSMENT_NAME`, `DISTRICT × ASSESSMENT_NAME`, `SCHOOL`, and `DISTRICT × SUBGROUP_NAME`. The ridge meta-combination was also dominated by school-level smoothed residual features.

However, this result is too large to submit immediately. The current grouped residual feature construction used global cross-fitted encodings before a second meta-model cross-validation step. This can allow validation rows to indirectly influence each other through shared group residual statistics, especially for high-cardinality groups such as school and school-by-subgroup. The individual group screens also frequently selected negative lambdas at the lower search boundary, which suggests that the grouped residual structure requires further audit.

Therefore, the 26A result is treated as a major modeling insight rather than a valid submission candidate. The next step is a strictly nested, leakage-safe grouped residual calibration audit in which each outer validation fold is completely excluded from both the group residual maps and the meta-model fit.

In [51]:
# ============================================================
# 26B. Strict nested leakage-safe grouped residual audit
# ============================================================
#
# 26A found a huge grouped-residual gain, but the meta-CV may have allowed
# validation rows to influence each other through precomputed global group encodings.
#
# This cell performs a stricter nested audit:
#
# Outer fold:
#   - outer validation rows are fully held out.
#   - all group residual maps for outer validation are built only from outer-train rows.
#   - the ridge meta-model is fit only on outer-train rows.
#
# Inner fold inside each outer-train split:
#   - group features for outer-train rows are themselves built out-of-fold.
#   - this prevents the meta-model from learning from target-encoded features that
#     directly include the row's own residual.
#
# No submission should be made until this nested audit confirms a meaningful gain.

import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
TARGET_COL = "PERCENT_PROFICIENT"
RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

def clip100(x):
    return np.clip(np.asarray(x, dtype=float), 0, 100)

def mse_clip(y_true, pred):
    return float(np.mean((np.asarray(y_true, dtype=float) - clip100(pred)) ** 2))

def safe_token(x):
    return str(round(float(x), 4)).replace("-", "m").replace(".", "p")

def get_test_ids_safe():
    if "test_id_series" in globals():
        return pd.Series(test_id_series).astype(str).reset_index(drop=True)
    if "test_ids" in globals():
        return pd.Series(test_ids).astype(str).reset_index(drop=True)
    if "scores_test" in globals() and ID_COL in scores_test.columns:
        return scores_test[ID_COL].astype(str).reset_index(drop=True)
    raise ValueError("Could not find test IDs.")

def make_key(df, cols):
    tmp = df.loc[:, list(cols)].copy()
    for c in cols:
        tmp[c] = tmp[c].astype("string").fillna("__NA__")
    return tmp.astype(str).agg("||".join, axis=1).to_numpy()

def smooth_group_map(keys, values, alpha):
    d = pd.DataFrame({"key": keys, "value": values})
    stats = d.groupby("key")["value"].agg(["count", "mean"])
    global_mean = float(np.mean(values))
    stats["smooth"] = (
        stats["count"] * stats["mean"] + alpha * global_mean
    ) / (stats["count"] + alpha)
    return stats["smooth"], global_mean

def map_with_default(keys, smooth_map, default):
    return pd.Series(keys).map(smooth_map).fillna(default).to_numpy(dtype=float)

def build_nested_train_and_valid_features(
    train_meta,
    train_keys_by_feature,
    resid,
    outer_tr_idx,
    outer_va_idx,
    feature_specs,
    inner_splits=5,
    inner_seed=12345,
):
    """
    Returns:
      X_outer_train: inner-OOF group residual features for outer-train rows
      X_outer_valid: group residual features for outer-valid rows, mapped from all outer-train rows

    This is the leakage-safe part.
    """
    n_outer_tr = len(outer_tr_idx)
    n_outer_va = len(outer_va_idx)
    p = len(feature_specs)

    X_tr = np.zeros((n_outer_tr, p), dtype=np.float32)
    X_va = np.zeros((n_outer_va, p), dtype=np.float32)

    inner_kf = KFold(n_splits=inner_splits, shuffle=True, random_state=inner_seed)

    for j, (feature_name, spec, alpha) in enumerate(feature_specs):
        keys_all = train_keys_by_feature[feature_name]

        # Build inner-OOF features for outer-train rows.
        for inner_tr_rel, inner_va_rel in inner_kf.split(outer_tr_idx):
            inner_tr_abs = outer_tr_idx[inner_tr_rel]
            inner_va_abs = outer_tr_idx[inner_va_rel]

            smap, default = smooth_group_map(
                keys_all[inner_tr_abs],
                resid[inner_tr_abs],
                alpha=alpha,
            )

            X_tr[inner_va_rel, j] = map_with_default(
                keys_all[inner_va_abs],
                smap,
                default,
            )

        # Build outer-valid features from all outer-train rows only.
        smap_outer, default_outer = smooth_group_map(
            keys_all[outer_tr_idx],
            resid[outer_tr_idx],
            alpha=alpha,
        )

        X_va[:, j] = map_with_default(
            keys_all[outer_va_idx],
            smap_outer,
            default_outer,
        )

    return X_tr, X_va

def build_full_oof_and_test_features(
    train_meta,
    test_meta,
    train_keys_by_feature,
    test_keys_by_feature,
    resid,
    feature_specs,
    inner_splits=5,
    seed=12345,
):
    """
    Final training features are full-data OOF encodings.
    Final test features are full-train maps applied to test.
    """
    n = len(train_meta)
    nt = len(test_meta)
    p = len(feature_specs)

    X_oof = np.zeros((n, p), dtype=np.float32)
    X_test = np.zeros((nt, p), dtype=np.float32)

    kf = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    for j, (feature_name, spec, alpha) in enumerate(feature_specs):
        keys_train = train_keys_by_feature[feature_name]
        keys_test = test_keys_by_feature[feature_name]

        for tr_idx, va_idx in kf.split(train_meta):
            smap, default = smooth_group_map(
                keys_train[tr_idx],
                resid[tr_idx],
                alpha=alpha,
            )

            X_oof[va_idx, j] = map_with_default(
                keys_train[va_idx],
                smap,
                default,
            )

        smap_full, default_full = smooth_group_map(
            keys_train,
            resid,
            alpha=alpha,
        )

        X_test[:, j] = map_with_default(
            keys_test,
            smap_full,
            default_full,
        )

    return X_oof, X_test

# ------------------------------------------------------------
# Recreate current internal 25B anchor
# ------------------------------------------------------------

internal_old_w = 0.025
internal_new_w = 0.875

internal_anchor_name = (
    f"oldnew_internal_oldw_{str(internal_old_w).replace('.', 'p')}"
    f"_neww_{str(internal_new_w).replace('.', 'p')}"
)

internal_anchor_oof = clip100(
    base_oof + internal_old_w * old_resid_oof + internal_new_w * new_resid_oof
)

internal_anchor_test = clip100(
    base_test + internal_old_w * old_resid_test + internal_new_w * new_resid_test
)

internal_anchor_mse = mse_clip(y, internal_anchor_oof)
resid_left = np.asarray(y, dtype=float) - internal_anchor_oof

print("Internal 25B anchor:", internal_anchor_name)
print("Internal 25B anchor OOF MSE:", internal_anchor_mse)
print("Residual-left std:", float(np.std(resid_left)))

# ------------------------------------------------------------
# Metadata and feature specs
# ------------------------------------------------------------

train_meta = train_full.copy().reset_index(drop=True)
test_meta = test_full.copy().reset_index(drop=True)

if "N_STUDENTS" in train_meta.columns and "N_STUDENTS" in test_meta.columns:
    train_bins, bin_edges = pd.qcut(
        train_meta["N_STUDENTS"],
        q=10,
        duplicates="drop",
        retbins=True,
    )
    train_meta["N_STUDENTS_BIN"] = train_bins.astype(str)
    test_meta["N_STUDENTS_BIN"] = pd.cut(
        test_meta["N_STUDENTS"],
        bins=bin_edges,
        include_lowest=True,
    ).astype(str)

# Compact but targeted list based on the 26A individual screen and coefficient table.
# This keeps the nested audit feasible while focusing on the strongest discovered structures.
raw_specs = [
    ("SCHOOL",),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),

    ("DISTRICT",),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),

    ("COUNTY",),
    ("COUNTY", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),

    ("SUBGROUP_NAME",),
    ("ASSESSMENT_NAME",),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("SUBGROUP_NAME", "N_STUDENTS_BIN"),
    ("ASSESSMENT_NAME", "N_STUDENTS_BIN"),
]

group_specs = []
for spec in raw_specs:
    if all(c in train_meta.columns and c in test_meta.columns for c in spec):
        group_specs.append(spec)

alpha_grid = [5, 20, 100, 300]

feature_specs = []
for spec in group_specs:
    for alpha in alpha_grid:
        feature_name = "grp_" + "__".join(spec) + f"__a{alpha}"
        feature_specs.append((feature_name, spec, alpha))

print("\nNested grouped audit feature count:", len(feature_specs))
print("Group specs:")
for spec in group_specs:
    print("  ", spec)

# Precompute keys once.
train_keys_by_feature = {}
test_keys_by_feature = {}

for feature_name, spec, alpha in feature_specs:
    # Same key for different alphas under same spec, but this simple cache name is fine.
    train_keys_by_feature[feature_name] = make_key(train_meta, spec)
    test_keys_by_feature[feature_name] = make_key(test_meta, spec)

# ------------------------------------------------------------
# Strict nested outer-CV meta residual prediction
# ------------------------------------------------------------

nested_oof_path = RESULTS_DIR / "group_residual_nested_meta_oof.npy"
nested_fold_path = RESULTS_DIR / "group_residual_nested_meta_fold_metrics.csv"

if nested_oof_path.exists():
    nested_meta_oof = np.load(nested_oof_path)
else:
    nested_meta_oof = np.full(len(y), np.nan, dtype=np.float32)

if nested_fold_path.exists():
    nested_fold_df = pd.read_csv(nested_fold_path)
else:
    nested_fold_df = pd.DataFrame()

outer_kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 2602)
outer_folds = list(outer_kf.split(train_meta))

ridge_alphas = np.logspace(-3, 5, 25)

overall_start = time.time()

for outer_fold, (outer_tr_idx, outer_va_idx) in enumerate(outer_folds, start=1):
    already_done = (
        (not nested_fold_df.empty)
        and (outer_fold in set(nested_fold_df["fold"]))
        and (not np.isnan(nested_meta_oof[outer_va_idx]).any())
    )

    if already_done:
        print(f"\nSkipping completed nested outer fold {outer_fold}/5")
        continue

    print("\n" + "=" * 90)
    print(f"Nested grouped residual audit: outer fold {outer_fold}/5")
    print("=" * 90)

    start = time.time()

    X_outer_tr, X_outer_va = build_nested_train_and_valid_features(
        train_meta=train_meta,
        train_keys_by_feature=train_keys_by_feature,
        resid=resid_left,
        outer_tr_idx=outer_tr_idx,
        outer_va_idx=outer_va_idx,
        feature_specs=feature_specs,
        inner_splits=5,
        inner_seed=RANDOM_STATE + 2700 + outer_fold,
    )

    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alphas)
    )

    model.fit(X_outer_tr, resid_left[outer_tr_idx])
    pred_outer_va = model.predict(X_outer_va)

    nested_meta_oof[outer_va_idx] = pred_outer_va.astype(np.float32)
    np.save(nested_oof_path, nested_meta_oof)

    before_mse = mse_clip(y[outer_va_idx], internal_anchor_oof[outer_va_idx])
    after_raw_mse = mse_clip(y[outer_va_idx], internal_anchor_oof[outer_va_idx] + pred_outer_va)

    row = {
        "fold": outer_fold,
        "mse_before": before_mse,
        "mse_after_raw_meta": after_raw_mse,
        "gain_raw_meta": before_mse - after_raw_mse,
        "elapsed_sec": float(time.time() - start),
    }

    nested_fold_df = nested_fold_df[nested_fold_df["fold"] != outer_fold] if not nested_fold_df.empty else nested_fold_df
    nested_fold_df = pd.concat([nested_fold_df, pd.DataFrame([row])], ignore_index=True)
    nested_fold_df = nested_fold_df.sort_values("fold").reset_index(drop=True)
    nested_fold_df.to_csv(nested_fold_path, index=False)

    print("Fold before MSE:", before_mse)
    print("Fold after raw nested meta MSE:", after_raw_mse)
    print("Fold raw gain:", before_mse - after_raw_mse)
    print("Elapsed minutes:", round((time.time() - start) / 60, 2))

    del X_outer_tr, X_outer_va, model, pred_outer_va
    gc.collect()

missing = int(np.isnan(nested_meta_oof).sum())
print("\nNested OOF missing rows:", missing)

if missing > 0:
    print("Not all nested folds are complete. Re-run this same cell to resume.")
else:
    print("\nNested outer-fold diagnostics:")
    try:
        display(nested_fold_df)
    except NameError:
        print(nested_fold_df.to_string(index=False))

    # --------------------------------------------------------
    # Lambda screen using strictly nested OOF meta correction
    # --------------------------------------------------------

    rows = []

    for lam in np.round(np.arange(-1.000, 1.501, 0.005), 3):
        pred_oof = clip100(internal_anchor_oof + lam * nested_meta_oof)

        fold_gains = []
        fold_mses = []

        for fold, (_, va_idx) in enumerate(outer_folds, start=1):
            before = mse_clip(y[va_idx], internal_anchor_oof[va_idx])
            after = mse_clip(y[va_idx], pred_oof[va_idx])
            fold_gains.append(before - after)
            fold_mses.append(after)

        rows.append({
            "lambda": float(lam),
            "oof_mse_clipped": mse_clip(y, pred_oof),
            "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, pred_oof),
            "min_fold_gain": float(np.min(fold_gains)),
            "mean_fold_gain": float(np.mean(fold_gains)),
            "max_fold_gain": float(np.max(fold_gains)),
            "fold_mse_std": float(np.std(fold_mses, ddof=1)),
        })

    screen_df = pd.DataFrame(rows).sort_values("oof_mse_clipped").reset_index(drop=True)

    screen_path = RESULTS_DIR / "group_residual_nested_meta_lambda_screen.csv"
    screen_df.to_csv(screen_path, index=False)

    print("\nTop strict nested grouped residual lambda candidates:")
    try:
        display(screen_df.head(30))
    except NameError:
        print(screen_df.head(30).to_string(index=False))

    print("\nStrict nested grouped screen saved to:", screen_path)
    print("Internal anchor OOF MSE:", internal_anchor_mse)
    print("Best strict nested grouped OOF MSE:", float(screen_df.loc[0, "oof_mse_clipped"]))
    print("Best strict nested grouped gain:", float(screen_df.loc[0, "gain_vs_internal_anchor"]))
    print("Best strict nested grouped min fold gain:", float(screen_df.loc[0, "min_fold_gain"]))

    # --------------------------------------------------------
    # Build final full-train OOF features and test features
    # only if nested audit confirms real signal.
    # --------------------------------------------------------

    best = screen_df.iloc[0]
    best_gain = float(best["gain_vs_internal_anchor"])
    best_min_fold_gain = float(best["min_fold_gain"])
    best_lam = float(best["lambda"])

    if best_gain <= 0 or best_min_fold_gain <= 0:
        print("\nNested audit did not confirm a positive stable gain. Do not use grouped residual submission.")
    else:
        print("\nNested audit confirms positive grouped residual signal.")
        print("Now building full OOF/test grouped features for saved candidate.")

        X_full_oof, X_test_group = build_full_oof_and_test_features(
            train_meta=train_meta,
            test_meta=test_meta,
            train_keys_by_feature=train_keys_by_feature,
            test_keys_by_feature=test_keys_by_feature,
            resid=resid_left,
            feature_specs=feature_specs,
            inner_splits=5,
            seed=RANDOM_STATE + 2800,
        )

        final_model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=ridge_alphas)
        )

        final_model.fit(X_full_oof, resid_left)
        final_meta_test = final_model.predict(X_test_group)

        final_oof = clip100(internal_anchor_oof + best_lam * nested_meta_oof)
        final_test = clip100(internal_anchor_test + best_lam * final_meta_test)

        name = f"group_resid_nested_on_{internal_anchor_name}_lambda_{safe_token(best_lam)}"

        oof_path = RESULTS_DIR / f"oof_{name}.csv"
        testpred_path = RESULTS_DIR / f"testpred_{name}.csv"
        sub_path = Path(f"submission_{name}.csv")

        pd.DataFrame({
            TARGET_COL: y,
            f"OOF_{name}": final_oof,
        }).to_csv(oof_path, index=False)

        pd.DataFrame({
            ID_COL: get_test_ids_safe().values,
            f"TESTPRED_{name}": final_test,
        }).to_csv(testpred_path, index=False)

        pd.DataFrame({
            ID_COL: get_test_ids_safe().values,
            TARGET_COL: final_test,
        }).to_csv(sub_path, index=False)

        saved = pd.DataFrame([{
            "candidate": name,
            "lambda": best_lam,
            "oof_mse_clipped": mse_clip(y, final_oof),
            "gain_vs_internal_anchor": internal_anchor_mse - mse_clip(y, final_oof),
            "min_fold_gain": best_min_fold_gain,
            "submission_path": str(sub_path),
            "testpred_path": str(testpred_path),
            "test_mean": float(np.mean(final_test)),
            "test_std": float(np.std(final_test)),
            "test_min": float(np.min(final_test)),
            "test_max": float(np.max(final_test)),
        }])

        saved_path = RESULTS_DIR / "group_residual_nested_meta_saved_candidate.csv"
        saved.to_csv(saved_path, index=False)

        print("\nSaved strict nested grouped residual candidate:")
        try:
            display(saved)
        except NameError:
            print(saved.to_string(index=False))

        print("Saved candidate list:", saved_path)
        print("Candidate file:", sub_path)

        # Coefficients for interpretation only.
        ridge = final_model.named_steps["ridgecv"]
        coef_df = pd.DataFrame({
            "feature": [f[0] for f in feature_specs],
            "coef_scaled_space": ridge.coef_,
        }).assign(abs_coef=lambda d: d["coef_scaled_space"].abs())

        coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
        coef_path = RESULTS_DIR / "group_residual_nested_meta_top_coefficients.csv"
        coef_df.to_csv(coef_path, index=False)

        print("\nTop strict nested grouped residual meta features:")
        try:
            display(coef_df.head(30))
        except NameError:
            print(coef_df.head(30).to_string(index=False))

        print("Coefficient table saved to:", coef_path)

    print("\nTotal elapsed minutes:", round((time.time() - overall_start) / 60, 2))

Internal 25B anchor: oldnew_internal_oldw_0p025_neww_0p875
Internal 25B anchor OOF MSE: 85.64738488718324
Residual-left std: 9.25440618958025

Nested grouped audit feature count: 64
Group specs:
   ('SCHOOL',)
   ('SCHOOL', 'SUBGROUP_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME')
   ('SCHOOL', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('DISTRICT',)
   ('DISTRICT', 'SUBGROUP_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME')
   ('DISTRICT', 'ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('COUNTY',)
   ('COUNTY', 'SUBGROUP_NAME')
   ('COUNTY', 'ASSESSMENT_NAME')
   ('SUBGROUP_NAME',)
   ('ASSESSMENT_NAME',)
   ('ASSESSMENT_NAME', 'SUBGROUP_NAME')
   ('SUBGROUP_NAME', 'N_STUDENTS_BIN')
   ('ASSESSMENT_NAME', 'N_STUDENTS_BIN')

Nested grouped residual audit: outer fold 1/5
Fold before MSE: 86.81741151120198
Fold after raw nested meta MSE: 81.98623487551272
Fold raw gain: 4.8311766356892605
Elapsed minutes: 0.14

Nested grouped residual audit: outer fold 2/5
Fold before MSE: 84.71751538100786
Fold after raw nested me

,fold,mse_before,mse_after_raw_meta,gain_raw_meta,elapsed_sec
0,1,86.817412,81.986235,4.831177,8.564025
1,2,84.717515,79.826294,4.891221,8.369281
2,3,85.375399,80.594340,4.781059,7.687655
3,4,85.305619,80.843871,4.461749,7.783906
4,5,86.020939,81.733959,4.286980,7.551215



Top strict nested grouped residual lambda candidates:


,lambda,oof_mse_clipped,gain_vs_internal_anchor,min_fold_gain,mean_fold_gain,max_fold_gain,fold_mse_std
0,1.175,80.892639,4.754746,4.289962,4.754744,5.056076,0.913218
1,1.180,80.892721,4.754664,4.286859,4.754662,5.057837,0.914379
2,1.170,80.892726,4.754659,4.292897,4.754657,5.054151,0.912063
3,1.185,80.892974,4.754411,4.283575,4.754409,5.059435,0.915549
4,1.165,80.892981,4.754404,4.295673,4.754403,5.052061,0.910911
5,1.190,80.893396,4.753989,4.280110,4.753987,5.060871,0.916729
6,1.160,80.893402,4.753983,4.298294,4.753982,5.049805,0.909760
7,1.195,80.893989,4.753396,4.276464,4.753395,5.062147,0.917919
8,1.155,80.893993,4.753392,4.300735,4.753390,5.047383,0.908616
9,1.200,80.894752,4.752633,4.272641,4.752631,5.063257,0.919116



Strict nested grouped screen saved to: model_results/group_residual_nested_meta_lambda_screen.csv
Internal anchor OOF MSE: 85.64738488718324
Best strict nested grouped OOF MSE: 80.89263904946873
Best strict nested grouped gain: 4.754745837714509
Best strict nested grouped min fold gain: 4.289962241156971

Nested audit confirms positive grouped residual signal.
Now building full OOF/test grouped features for saved candidate.

Saved strict nested grouped residual candidate:


,candidate,lambda,oof_mse_clipped,gain_vs_internal_anchor,min_fold_gain,submission_path,testpred_path,test_mean,test_std,test_min,test_max
0,group_resid_nested_on_oldnew_internal_oldw_0p0...,1.175,80.892639,4.754746,4.289962,submission_group_resid_nested_on_oldnew_intern...,model_results/testpred_group_resid_nested_on_o...,54.10565,24.709315,0.0,100.0


Saved candidate list: model_results/group_residual_nested_meta_saved_candidate.csv
Candidate file: submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv

Top strict nested grouped residual meta features:


,feature,coef_scaled_space,abs_coef
0,grp_SCHOOL__SUBGROUP_NAME__a100,58.158787,58.158787
1,grp_SCHOOL__a100,57.613281,57.613281
2,grp_SCHOOL__SUBGROUP_NAME__a300,-42.132343,42.132343
3,grp_SCHOOL__a300,-37.478886,37.478886
4,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a100,34.236969,34.236969
5,grp_SCHOOL__ASSESSMENT_NAME__a20,-26.756861,26.756861
6,grp_SCHOOL__a20,-25.956554,25.956554
7,grp_SCHOOL__SUBGROUP_NAME__a20,-18.420261,18.420261
8,grp_SUBGROUP_NAME__N_STUDENTS_BIN__a5,-17.465593,17.465593
9,grp_SCHOOL__ASSESSMENT_NAME__a300,11.704391,11.704391


Coefficient table saved to: model_results/group_residual_nested_meta_top_coefficients.csv

Total elapsed minutes: 0.85


### 26B strict nested grouped residual submission — public check failed

Submitted file:

`submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv`

Public leaderboard MSE:

**83.131**

Local strict nested OOF diagnostics had looked strong:

- internal 25B anchor OOF MSE: **85.647385**
- strict nested grouped residual OOF MSE: **80.892639**
- local OOF gain: **4.754746**
- all five outer folds had positive gains
- minimum outer-fold gain: about **4.29**

However, the public leaderboard result is much worse than the current submitted anchor:

`submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv`

with public MSE **77.277**.

Interpretation:

The 26B nested audit reduced leakage risk relative to 26A, but the public result shows that the direct high-cardinality grouped residual correction is not robust to the public test distribution. The grouped residual signal may still describe real training-set structure, but it is too public/private-sensitive in this form. Do not submit 26A, do not submit further small 26B lambda variants, and do not continue optimizing direct school/group residual maps for Kaggle slots.

Decision:

Demote the grouped residual meta-correction branch to diagnostics/backlog. Continue from the 77.277 residual-stack public anchor and pivot to safer residual diversity models that avoid direct high-cardinality grouped residual maps.

In [52]:
# 27A. Log failed 26B public result and protect current public anchor

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

tracker_path = RESULTS_DIR / "submission_tracker.csv"

new_row = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "file": "submission_group_resid_nested_on_oldnew_internal_oldw_0p025_neww_0p875_lambda_1p175.csv",
    "source": "current notebook",
    "model_family": "strict nested grouped residual meta-correction",
    "local_oof_mse": 80.89263904946873,
    "public_mse": 83.131,
    "current_public_anchor_file": "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    "current_public_anchor_mse": 77.277,
    "decision": "failed public check; do not continue direct high-cardinality grouped residual submissions",
    "notes": (
        "Local nested grouped OOF gain was large and fold-stable, but public MSE was much worse "
        "than the 77.277 residual-stack anchor. Treat direct group residual maps as public-fragile."
    ),
}

if tracker_path.exists():
    tracker = pd.read_csv(tracker_path)
else:
    tracker = pd.DataFrame()

# Remove older duplicate row for the same file, then append the corrected public result.
if len(tracker) > 0 and "file" in tracker.columns:
    tracker = tracker[tracker["file"] != new_row["file"]].copy()

tracker = pd.concat([tracker, pd.DataFrame([new_row])], ignore_index=True)
tracker.to_csv(tracker_path, index=False)

print("Saved tracker:", tracker_path)
display(tracker.tail(10))

Saved tracker: model_results/submission_tracker.csv


,submission_file,source,model_family,public_mse,notes,file,private_mse,timestamp,local_oof_mse,current_public_anchor_file,current_public_anchor_mse,decision
0,submission_blend_auto_et_lgbm_long_oof_weighte...,current notebook,automatic OOF blend,NaN,Automatic post-long-run OOF blend; OOF MSE=88....,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,current notebook,OOF blend,80.662,Automatic long-run blend: 0.319 ExtraTrees saf...,submission_blend_auto_et_lgbm_long_oof_weighte...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,77.277,New public anchor. Expanded residual stack on ...,submission_residstack_lgbm_on_best80p662_blend...,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,current notebook,strict nested grouped residual meta-correction,83.131,Local nested grouped OOF gain was large and fo...,submission_group_resid_nested_on_oldnew_intern...,NaN,2026-05-09 03:34:53,80.892639,submission_residstack_lgbm_on_best80p662_blend...,77.277,failed public check; do not continue direct hi...


### 27B fresh restart after invalid zero-residual run

The previous 27 overnight no-group residual suite is discarded.

Reason:
The run finished suspiciously fast and produced impossible diagnostics, including zero OOF MSE candidates. This indicates that the 27B anchor loader likely selected the true training target instead of the real OOF prediction for the submitted 77.277 residual-stack anchor.

Decision:
Restart the 27 branch from a corrected 27B. The corrected 27B must explicitly verify that the anchor OOF prediction has MSE near the known residual-stack OOF value, about **87.136**, and must reject any target-like column with MSE near zero.

Only the bad 27 branch artifacts are archived. Older RF, ExtraTrees, LightGBM, blend, and residual-stack artifacts are kept.

In [56]:
# 27B fresh restart: archive bad resid27 artifacts and rebuild feature matrix from verified 77.277 anchor

from pathlib import Path
import glob
import shutil
import gc

import numpy as np
import pandas as pd
from scipy import sparse

RESULTS_DIR = Path("model_results")
RESULTS_DIR.mkdir(exist_ok=True)

ID_COL = "ASSESSMENT_ID"
PRED_COL = "PERCENT_PROFICIENT"
EXPECTED_ANCHOR_OOF_MSE = 87.136049

# ---------------------------------------------------------------------
# 1. Archive bad resid27 files so the next run cannot resume from them
# ---------------------------------------------------------------------

BAD_DIR = RESULTS_DIR / "bad_resid27_zero_residual_run"
BAD_DIR.mkdir(exist_ok=True)

bad_patterns = [
    RESULTS_DIR / "resid27_nogroup_*",
    RESULTS_DIR / "oofcorr_resid27_nogroup_*.csv",
    RESULTS_DIR / "testcorr_resid27_nogroup_*.csv",
    RESULTS_DIR / "oofpred_resid27_nogroup_*.csv",
    RESULTS_DIR / "testpred_resid27_nogroup_*.csv",
    RESULTS_DIR / "resid27_overnight_*.csv",
    Path("submission_resid27_nogroup_*.csv"),
]

moved = []

for pattern in bad_patterns:
    for path_str in glob.glob(str(pattern)):
        path = Path(path_str)
        if not path.exists() or not path.is_file():
            continue

        dest = BAD_DIR / path.name
        k = 1
        while dest.exists():
            dest = BAD_DIR / f"{path.stem}_dup{k}{path.suffix}"
            k += 1

        shutil.move(str(path), str(dest))
        moved.append((str(path), str(dest)))

print(f"Archived {len(moved)} old/bad resid27 files.")
if moved:
    display(pd.DataFrame(moved, columns=["from", "to"]).head(40))

# ---------------------------------------------------------------------
# 2. Basic checks and helpers
# ---------------------------------------------------------------------

required_objects = [
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "train_ids",
    "test_ids",
]

missing = [x for x in required_objects if x not in globals()]
if missing:
    raise RuntimeError(
        "Missing required objects. Rerun preprocessing/setup cells first: "
        + ", ".join(missing)
    )

y_arr = np.asarray(y_train, dtype=np.float64).reshape(-1)
n_train = len(y_arr)
n_test = X_test_proc_model.shape[0]

def mse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))

def get_id_series(id_obj, expected_len, label):
    if isinstance(id_obj, pd.DataFrame):
        if ID_COL in id_obj.columns:
            vals = id_obj[ID_COL].to_numpy()
        else:
            vals = id_obj.iloc[:, 0].to_numpy()
    elif isinstance(id_obj, pd.Series):
        vals = id_obj.to_numpy()
    else:
        vals = np.asarray(id_obj).reshape(-1)

    if len(vals) != expected_len:
        raise ValueError(f"{label} length mismatch: got {len(vals)}, expected {expected_len}")

    return pd.Series(vals, name=ID_COL).astype(str).reset_index(drop=True)

train_id_values = get_id_series(train_ids, n_train, "train_ids")
test_id_values = get_id_series(test_ids, n_test, "test_ids")

def read_aligned_numeric_column(path, expected_ids, col):
    df = pd.read_csv(path)

    if col not in df.columns:
        raise ValueError(f"Column {col} not found in {path}")

    vals = pd.to_numeric(df[col], errors="coerce")

    if ID_COL in df.columns:
        tmp = pd.DataFrame({
            ID_COL: df[ID_COL].astype(str),
            "_pred": vals,
        })

        if tmp[ID_COL].duplicated().any():
            raise ValueError(f"Duplicate {ID_COL} values in {path}")

        aligned = tmp.set_index(ID_COL).reindex(expected_ids)["_pred"]

        if aligned.isna().any():
            raise ValueError(
                f"{path} column {col} failed ID alignment; missing {int(aligned.isna().sum())} rows"
            )

        return aligned.to_numpy(dtype=float)

    if len(vals) != len(expected_ids):
        raise ValueError(f"{path} has no ID column and length does not match expected IDs")

    if vals.isna().any():
        raise ValueError(f"{path} column {col} contains missing/non-numeric values")

    return vals.to_numpy(dtype=float)

def scan_numeric_prediction_columns(path, expected_ids, y_true=None):
    df = pd.read_csv(path)
    rows = []

    for col in df.columns:
        if col == ID_COL:
            continue

        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() < 0.95:
            continue

        try:
            pred = read_aligned_numeric_column(path, expected_ids, col)

            row = {
                "path": str(path),
                "column": col,
                "n": len(pred),
                "mean": float(np.mean(pred)),
                "std": float(np.std(pred)),
                "min": float(np.min(pred)),
                "max": float(np.max(pred)),
                "missing": int(np.isnan(pred).sum()),
                "outside_0_100": int(((pred < 0) | (pred > 100)).sum()),
            }

            if y_true is not None:
                mse = mse_np(y_true, pred)
                row["mse_vs_y"] = mse
                row["target_like"] = bool(mse < 1e-8)
                row["abs_from_expected_anchor"] = abs(mse - EXPECTED_ANCHOR_OOF_MSE)

            rows.append(row)

        except Exception as e:
            rows.append({
                "path": str(path),
                "column": col,
                "error": repr(e),
            })

    return rows

# ---------------------------------------------------------------------
# 3. Find and verify the real submitted 77.277 residual-stack OOF anchor
# ---------------------------------------------------------------------

anchor_oof_patterns = [
    RESULTS_DIR / "oofpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    RESULTS_DIR / "oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    Path("oofpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    Path("oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    RESULTS_DIR / "*oof*residstack*lgbm*best80p662*1p135*.csv",
    RESULTS_DIR / "*oof*residstack*lgbm*lambda_1p135*.csv",
    Path("*oof*residstack*lgbm*best80p662*1p135*.csv"),
]

anchor_oof_files = []

for pattern in anchor_oof_patterns:
    for p in glob.glob(str(pattern)):
        p = Path(p)
        if p.exists() and p.is_file() and p not in anchor_oof_files:
            anchor_oof_files.append(p)

if not anchor_oof_files:
    raise FileNotFoundError(
        "Could not find the OOF prediction file for the submitted 77.277 residual-stack anchor."
    )

scan_rows = []
for p in anchor_oof_files:
    scan_rows.extend(scan_numeric_prediction_columns(p, train_id_values, y_true=y_arr))

scan_df = pd.DataFrame(scan_rows)

print("\nCandidate OOF columns for the 77.277 anchor:")
display(
    scan_df
    .sort_values(["target_like", "abs_from_expected_anchor"], ascending=[True, True])
    .head(50)
)

valid_anchor_rows = scan_df[
    (scan_df["missing"] == 0)
    & (scan_df["target_like"] == False)
    & (scan_df["mse_vs_y"].between(85.0, 89.5))
].copy()

if valid_anchor_rows.empty:
    raise RuntimeError(
        "No valid 77.277-anchor OOF column found. "
        "The correct OOF MSE should be around 87.136, not 0. Paste the scan table."
    )

best_anchor_row = valid_anchor_rows.sort_values("abs_from_expected_anchor").iloc[0]

anchor27_oof_path = Path(best_anchor_row["path"])
anchor27_oof_col = best_anchor_row["column"]

anchor27_oof = read_aligned_numeric_column(anchor27_oof_path, train_id_values, anchor27_oof_col)
anchor27_oof = np.clip(anchor27_oof, 0, 100)
anchor27_oof_mse = mse_np(y_arr, anchor27_oof)

print("\nSelected verified 77.277-anchor OOF:")
print("path:", anchor27_oof_path)
print("column:", anchor27_oof_col)
print("OOF MSE:", anchor27_oof_mse)

if not (86.5 <= anchor27_oof_mse <= 87.8):
    raise RuntimeError(
        f"Selected anchor OOF MSE is {anchor27_oof_mse}, not close enough to expected 87.136. Stop."
    )

# ---------------------------------------------------------------------
# 4. Load matching submitted test predictions
# ---------------------------------------------------------------------

anchor_test_patterns = [
    Path("submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv"),
    RESULTS_DIR / "testpred_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    RESULTS_DIR / "*testpred*residstack*lgbm*best80p662*1p135*.csv",
    RESULTS_DIR / "*testpred*residstack*lgbm*lambda_1p135*.csv",
    Path("submission*residstack*lgbm*best80p662*1p135*.csv"),
]

anchor_test_files = []

for pattern in anchor_test_patterns:
    for p in glob.glob(str(pattern)):
        p = Path(p)
        if p.exists() and p.is_file() and p not in anchor_test_files:
            anchor_test_files.append(p)

if not anchor_test_files:
    raise FileNotFoundError(
        "Could not find the matching test/submission file for the submitted 77.277 anchor."
    )

test_scan_rows = []
for p in anchor_test_files:
    test_scan_rows.extend(scan_numeric_prediction_columns(p, test_id_values, y_true=None))

test_scan_df = pd.DataFrame(test_scan_rows)

print("\nCandidate test columns for the 77.277 anchor:")
display(test_scan_df)

valid_test_rows = test_scan_df[
    (test_scan_df["n"] == n_test)
    & (test_scan_df["missing"] == 0)
    & (test_scan_df["outside_0_100"] == 0)
].copy()

if valid_test_rows.empty:
    raise RuntimeError("No valid matching test prediction found. Paste the test scan table.")

valid_test_rows["is_exact_submission"] = valid_test_rows["path"].str.contains(
    "submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv",
    regex=False
)

best_test_row = valid_test_rows.sort_values("is_exact_submission", ascending=False).iloc[0]

anchor27_test_path = Path(best_test_row["path"])
anchor27_test_col = best_test_row["column"]

anchor27_test = read_aligned_numeric_column(anchor27_test_path, test_id_values, anchor27_test_col)
anchor27_test = np.clip(anchor27_test, 0, 100)

print("\nSelected verified 77.277-anchor test/submission prediction:")
print("path:", anchor27_test_path)
print("column:", anchor27_test_col)
print(pd.Series(anchor27_test).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# ---------------------------------------------------------------------
# 5. Load saved component predictions as residual-model meta-features
# ---------------------------------------------------------------------

component_paths = {
    "rf500": (
        RESULTS_DIR / "oof_rf_500_base.csv",
        RESULTS_DIR / "testpred_rf_500_base_folds.csv",
    ),
    "et_safe": (
        RESULTS_DIR / "oof_extratrees_safe_base.csv",
        RESULTS_DIR / "testpred_extratrees_safe_base_foldavg.csv",
    ),
    "lgbm_12k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_foldavg.csv",
    ),
    "lgbm_long80k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv",
    ),
    "lgbm_long100k": (
        RESULTS_DIR / "oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv",
        RESULTS_DIR / "testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv",
    ),
    "lgbm_div02_3fold": (
        RESULTS_DIR / "oof_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof.csv",
        RESULTS_DIR / "testpred_lgbm_div02_l127_child50_l2_15_lr02_3fold_oof_foldavg.csv",
    ),
}

def auto_load_oof_component(path, expected_ids, y_true, label):
    rows = scan_numeric_prediction_columns(path, expected_ids, y_true=y_true)
    df = pd.DataFrame(rows)

    valid = df[
        (df["missing"] == 0)
        & (df["target_like"] == False)
        & (df["mse_vs_y"] > 1.0)
    ].copy()

    if valid.empty:
        raise RuntimeError(f"No valid prediction column found for {label}: {path}")

    # Usually there is one prediction column. If more than one, choose the best non-target-like MSE.
    row = valid.sort_values("mse_vs_y").iloc[0]
    pred = read_aligned_numeric_column(path, expected_ids, row["column"])

    print(f"Loaded {label}: {path} | column={row['column']} | OOF MSE={row['mse_vs_y']:.6f}")
    return pred

def auto_load_test_component(path, expected_ids, label):
    rows = scan_numeric_prediction_columns(path, expected_ids, y_true=None)
    df = pd.DataFrame(rows)

    valid = df[
        (df["n"] == len(expected_ids))
        & (df["missing"] == 0)
    ].copy()

    if valid.empty:
        raise RuntimeError(f"No valid test prediction column found for {label}: {path}")

    # Prefer standard submission column if present.
    valid["preferred"] = (valid["column"] == PRED_COL).astype(int)
    row = valid.sort_values("preferred", ascending=False).iloc[0]
    pred = read_aligned_numeric_column(path, expected_ids, row["column"])

    print(f"Loaded {label}: {path} | column={row['column']}")
    return pred

component_oof = {}
component_test = {}

print("\nLoading component prediction artifacts:")

for name, (oof_path, test_path) in component_paths.items():
    if Path(oof_path).exists() and Path(test_path).exists():
        component_oof[name] = auto_load_oof_component(oof_path, train_id_values, y_arr, f"{name} OOF")
        component_test[name] = auto_load_test_component(test_path, test_id_values, f"{name} test")
    else:
        print(f"Skipping missing component: {name}")

# ---------------------------------------------------------------------
# 6. Build corrected augmented residual-model matrix
# ---------------------------------------------------------------------

def to_csr_float32(X):
    if sparse.issparse(X):
        return X.tocsr().astype(np.float32)
    if isinstance(X, pd.DataFrame):
        arr = X.to_numpy()
    else:
        arr = np.asarray(X)
    return sparse.csr_matrix(arr.astype(np.float32))

X_base_train_27B = to_csr_float32(X_train_proc_model)
X_base_test_27B = to_csr_float32(X_test_proc_model)

extra_train = []
extra_test = []
extra_names_27B = []

def add_extra_feature(name, train_values, test_values):
    train_values = np.asarray(train_values, dtype=np.float32).reshape(-1)
    test_values = np.asarray(test_values, dtype=np.float32).reshape(-1)

    if len(train_values) != n_train:
        raise ValueError(f"{name} train length mismatch")
    if len(test_values) != n_test:
        raise ValueError(f"{name} test length mismatch")

    extra_names_27B.append(name)
    extra_train.append(train_values)
    extra_test.append(test_values)

add_extra_feature("anchor77_residstack_pred", anchor27_oof, anchor27_test)

for name in sorted(component_oof):
    tr = np.clip(component_oof[name], 0, 100)
    te = np.clip(component_test[name], 0, 100)

    add_extra_feature(f"pred_{name}", tr, te)
    add_extra_feature(f"diff_{name}_minus_anchor77", tr - anchor27_oof, te - anchor27_test)

if len(component_oof) >= 2:
    comp_names_sorted = sorted(component_oof)

    comp_train_mat = np.column_stack([
        np.clip(component_oof[n], 0, 100) for n in comp_names_sorted
    ])

    comp_test_mat = np.column_stack([
        np.clip(component_test[n], 0, 100) for n in comp_names_sorted
    ])

    add_extra_feature("component_mean", comp_train_mat.mean(axis=1), comp_test_mat.mean(axis=1))
    add_extra_feature("component_std", comp_train_mat.std(axis=1), comp_test_mat.std(axis=1))
    add_extra_feature("component_min", comp_train_mat.min(axis=1), comp_test_mat.min(axis=1))
    add_extra_feature("component_max", comp_train_mat.max(axis=1), comp_test_mat.max(axis=1))
    add_extra_feature(
        "component_range",
        comp_train_mat.max(axis=1) - comp_train_mat.min(axis=1),
        comp_test_mat.max(axis=1) - comp_test_mat.min(axis=1),
    )

extra_train_mat_27B = np.column_stack(extra_train).astype(np.float32)
extra_test_mat_27B = np.column_stack(extra_test).astype(np.float32)

X_aug_train_27B = sparse.hstack(
    [X_base_train_27B, sparse.csr_matrix(extra_train_mat_27B)],
    format="csr"
).astype(np.float32)

X_aug_test_27B = sparse.hstack(
    [X_base_test_27B, sparse.csr_matrix(extra_test_mat_27B)],
    format="csr"
).astype(np.float32)

resid_target_27B = (y_arr - anchor27_oof).astype(np.float32)

print("\nCorrected 27B complete.")
print("Verified anchor OOF MSE:", anchor27_oof_mse)
print("Residual target summary:")
print(pd.Series(resid_target_27B).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print()
print("Extra feature count:", len(extra_names_27B))
print("Extra feature names:", extra_names_27B)
print("Augmented train shape:", X_aug_train_27B.shape)
print("Augmented test shape:", X_aug_test_27B.shape)

# ---------------------------------------------------------------------
# 7. Compatibility aliases for the old 27C residual-training cell
# ---------------------------------------------------------------------
# These aliases let the old 27C cell use the corrected 27B objects.

anchor_oof = anchor27_oof
anchor_test = anchor27_test
anchor_oof_mse = anchor27_oof_mse

X_aug_train = X_aug_train_27B
X_aug_test = X_aug_test_27B
resid_target_27 = resid_target_27B
extra_names = extra_names_27B

print("\nCompatibility aliases set:")
print("anchor_oof, anchor_test, anchor_oof_mse")
print("X_aug_train, X_aug_test, resid_target_27, extra_names")

gc.collect()

Archived 63 old/bad resid27 files.


,from,to
0,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
1,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
2,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
3,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
4,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
5,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
6,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
7,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...
8,model_results/resid27_nogroup_huber_l95_child1...,model_results/bad_resid27_zero_residual_run/re...
9,model_results/resid27_nogroup_l2_smooth_l63_ch...,model_results/bad_resid27_zero_residual_run/re...



Candidate OOF columns for the 77.277 anchor:


,path,column,n,mean,std,min,max,missing,outside_0_100,mse_vs_y,target_like,abs_from_expected_anchor
1,model_results/oof_residstack_lgbm_on_best80p66...,OOF_residstack_lgbm_on_best80p662_blend_5fold_...,144921,54.186484,24.786756,0.0,100.0,0,0,87.136049,False,2.514173e-07
0,model_results/oof_residstack_lgbm_on_best80p66...,PERCENT_PROFICIENT,144921,54.183521,26.429834,0.0,100.0,0,0,0.000000,True,8.713605e+01



Selected verified 77.277-anchor OOF:
path: model_results/oof_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
column: OOF_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135
OOF MSE: 87.13604874858268

Candidate test columns for the 77.277 anchor:


,path,column,n,mean,std,min,max,missing,outside_0_100
0,submission_residstack_lgbm_on_best80p662_blend...,PERCENT_PROFICIENT,48307,54.097842,24.676033,0.18864,100.0,0,0
1,model_results/testpred_residstack_lgbm_on_best...,TESTPRED_residstack_lgbm_on_best80p662_blend_5...,48307,54.097842,24.676033,0.18864,100.0,0,0



Selected verified 77.277-anchor test/submission prediction:
path: submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv
column: PERCENT_PROFICIENT
count    48307.000000
mean        54.097842
std         24.676288
min          0.188640
1%           7.488321
5%          15.889755
50%         52.544583
95%         95.256640
99%         99.462163
max        100.000000
dtype: float64

Loading component prediction artifacts:
Loaded rf500 OOF: model_results/oof_rf_500_base.csv | column=rf_500_base_oof_pred | OOF MSE=122.328024
Loaded rf500 test: model_results/testpred_rf_500_base_folds.csv | column=rf_500_base_fold1_pred
Loaded et_safe OOF: model_results/oof_extratrees_safe_base.csv | column=oof_pred | OOF MSE=110.749294
Loaded et_safe test: model_results/testpred_extratrees_safe_base_foldavg.csv | column=PERCENT_PROFICIENT
Loaded lgbm_12k OOF: model_results/oof_lgbm_t03_base_5fold_oof.csv | column=OOF_lgbm_t03_base_5fold_oof | OOF MSE=98.779406
Loaded lgbm_12k test:

98

In [57]:
# 27B.1 patch: replace RF test feature with fold-average, then rebuild augmented matrices

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import gc

rf_test_path = Path("model_results/testpred_rf_500_base_folds.csv")

if not rf_test_path.exists():
    raise FileNotFoundError(rf_test_path)

rf_test_df = pd.read_csv(rf_test_path)

rf_num_cols = [
    c for c in rf_test_df.columns
    if c != ID_COL and pd.api.types.is_numeric_dtype(rf_test_df[c])
]

print("RF test numeric columns:")
print(rf_num_cols)

avg_like_cols = [
    c for c in rf_num_cols
    if any(tok in c.lower() for tok in ["avg", "mean", "foldavg", "fold_avg"])
]

fold_cols = [
    c for c in rf_num_cols
    if "fold" in c.lower()
]

if avg_like_cols:
    rf_used_cols = [avg_like_cols[0]]
    rf_test_fixed_method = f"existing average column: {avg_like_cols[0]}"
elif len(fold_cols) >= 2:
    rf_used_cols = fold_cols
    rf_test_fixed_method = f"mean of {len(fold_cols)} fold columns"
else:
    rf_used_cols = rf_num_cols
    rf_test_fixed_method = f"mean of all {len(rf_num_cols)} numeric prediction columns"

if len(rf_used_cols) == 0:
    raise RuntimeError("Could not identify RF test prediction columns.")

tmp = rf_test_df[[ID_COL] + rf_used_cols].copy()
tmp[ID_COL] = tmp[ID_COL].astype(str)

if tmp[ID_COL].duplicated().any():
    raise RuntimeError("Duplicate ASSESSMENT_ID in RF test file.")

aligned_rf = tmp.set_index(ID_COL).reindex(test_id_values)

if aligned_rf[rf_used_cols].isna().any().any():
    raise RuntimeError("RF test predictions failed ID alignment.")

rf500_test_fixed = aligned_rf[rf_used_cols].mean(axis=1).to_numpy(dtype=np.float64)

component_test["rf500"] = rf500_test_fixed

print("\nRF test fixed using:", rf_test_fixed_method)
print(pd.Series(rf500_test_fixed).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# Rebuild augmented features using corrected component_test["rf500"]

extra_train = []
extra_test = []
extra_names_27B = []

def add_extra_feature_27B(name, train_values, test_values):
    train_values = np.asarray(train_values, dtype=np.float32).reshape(-1)
    test_values = np.asarray(test_values, dtype=np.float32).reshape(-1)

    if len(train_values) != n_train:
        raise ValueError(f"{name} train length mismatch")
    if len(test_values) != n_test:
        raise ValueError(f"{name} test length mismatch")

    extra_names_27B.append(name)
    extra_train.append(train_values)
    extra_test.append(test_values)

add_extra_feature_27B("anchor77_residstack_pred", anchor27_oof, anchor27_test)

for name in sorted(component_oof):
    tr = np.clip(component_oof[name], 0, 100)
    te = np.clip(component_test[name], 0, 100)

    add_extra_feature_27B(f"pred_{name}", tr, te)
    add_extra_feature_27B(f"diff_{name}_minus_anchor77", tr - anchor27_oof, te - anchor27_test)

if len(component_oof) >= 2:
    comp_names_sorted = sorted(component_oof)

    comp_train_mat = np.column_stack([
        np.clip(component_oof[n], 0, 100) for n in comp_names_sorted
    ])

    comp_test_mat = np.column_stack([
        np.clip(component_test[n], 0, 100) for n in comp_names_sorted
    ])

    add_extra_feature_27B("component_mean", comp_train_mat.mean(axis=1), comp_test_mat.mean(axis=1))
    add_extra_feature_27B("component_std", comp_train_mat.std(axis=1), comp_test_mat.std(axis=1))
    add_extra_feature_27B("component_min", comp_train_mat.min(axis=1), comp_test_mat.min(axis=1))
    add_extra_feature_27B("component_max", comp_train_mat.max(axis=1), comp_test_mat.max(axis=1))
    add_extra_feature_27B(
        "component_range",
        comp_train_mat.max(axis=1) - comp_train_mat.min(axis=1),
        comp_test_mat.max(axis=1) - comp_test_mat.min(axis=1),
    )

extra_train_mat_27B = np.column_stack(extra_train).astype(np.float32)
extra_test_mat_27B = np.column_stack(extra_test).astype(np.float32)

X_aug_train_27B = sparse.hstack(
    [X_base_train_27B, sparse.csr_matrix(extra_train_mat_27B)],
    format="csr"
).astype(np.float32)

X_aug_test_27B = sparse.hstack(
    [X_base_test_27B, sparse.csr_matrix(extra_test_mat_27B)],
    format="csr"
).astype(np.float32)

# Reset compatibility aliases for 27C
X_aug_train = X_aug_train_27B
X_aug_test = X_aug_test_27B
resid_target_27 = resid_target_27B
extra_names = extra_names_27B
anchor_oof = anchor27_oof
anchor_test = anchor27_test
anchor_oof_mse = anchor27_oof_mse

print("\n27B.1 patch complete.")
print("Verified anchor OOF MSE:", anchor_oof_mse)
print("Residual target std:", float(np.std(resid_target_27)))
print("Extra feature count:", len(extra_names))
print("Augmented train shape:", X_aug_train.shape)
print("Augmented test shape:", X_aug_test.shape)

gc.collect()

RF test numeric columns:
['rf_500_base_fold1_pred', 'rf_500_base_fold2_pred', 'rf_500_base_fold3_pred', 'rf_500_base_fold4_pred', 'rf_500_base_fold5_pred', 'rf_500_base_foldavg_pred']

RF test fixed using: existing average column: rf_500_base_foldavg_pred
count    48307.000000
mean        54.056767
std         22.817699
min          0.322400
1%           9.781608
5%          19.065240
50%         52.255200
95%         92.641520
99%         98.827408
max         99.995200
dtype: float64

27B.1 patch complete.
Verified anchor OOF MSE: 87.13604874858268
Residual target std: 9.33466911315918
Extra feature count: 18
Augmented train shape: (144921, 180)
Augmented test shape: (48307, 180)


0

# This above modelling direction has been stopped

In [59]:
# ============================================================
# 28A. New branch diagnostic:
# bounded/proportion target + hierarchical key feasibility
# ============================================================
#
# Purpose:
# - Do not train a model yet.
# - Check whether PERCENT_PROFICIENT behaves like a bounded proportion.
# - Inspect N_STUDENTS as a reliability/noise proxy.
# - Confirm which raw grouping columns are available.
# - Measure train/test coverage and repetition for candidate hierarchical keys.
#
# This sets up the next branch:
# cross-fitted statistical/target-encoding features + bounded target modeling.
# ============================================================

import numpy as np
import pandas as pd

TARGET_COL = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

# ------------------------------------------------------------
# 1. Locate raw merged train/test frames
# ------------------------------------------------------------

if "train_full" in globals() and "test_full" in globals():
    diag_train = train_full.copy()
    diag_test = test_full.copy()
    source_used = "existing train_full/test_full"

elif all(name in globals() for name in [
    "scores_training", "scores_test", "school_covariates", "district_covariates"
]):
    diag_train = scores_training.merge(
        school_covariates, on="SCHOOL", how="left", validate="many_to_one"
    )
    diag_test = scores_test.merge(
        school_covariates, on="SCHOOL", how="left", validate="many_to_one"
    )

    diag_train = diag_train.merge(
        district_covariates, on="DISTRICT", how="left", validate="many_to_one"
    )
    diag_test = diag_test.merge(
        district_covariates, on="DISTRICT", how="left", validate="many_to_one"
    )

    source_used = "rebuilt from scores + school/district covariates"

else:
    raise ValueError(
        "Could not find train_full/test_full or the raw score/covariate tables. "
        "Please rerun the raw loading and merge cells first."
    )

print("Source used:", source_used)
print("diag_train shape:", diag_train.shape)
print("diag_test shape: ", diag_test.shape)

# ------------------------------------------------------------
# 2. Basic alignment checks
# ------------------------------------------------------------

print("\n--- Alignment checks ---")

if "y_train" in globals():
    print("len(y_train):", len(y_train))
    print("diag_train rows match y_train:", len(diag_train) == len(y_train))
else:
    print("y_train not found in globals; using target column from diag_train.")

if "X_train_proc_model" in globals():
    print("X_train_proc_model rows:", X_train_proc_model.shape[0])
    print("diag_train rows match X_train_proc_model:", len(diag_train) == X_train_proc_model.shape[0])

if "X_test_proc_model" in globals():
    print("X_test_proc_model rows:", X_test_proc_model.shape[0])
    print("diag_test rows match X_test_proc_model:", len(diag_test) == X_test_proc_model.shape[0])

if "test_ids" in globals():
    print("len(test_ids):", len(test_ids))
    print("diag_test rows match test_ids:", len(diag_test) == len(test_ids))

# ------------------------------------------------------------
# 3. Raw column availability
# ------------------------------------------------------------

important_cols = [
    ID_COL,
    "SCHOOL",
    "DISTRICT",
    "COUNTY",
    "REGION",
    "DISTRICT_TYPE",
    "SUBGROUP_NAME",
    "ASSESSMENT_NAME",
    "N_STUDENTS",
    TARGET_COL,
]

availability = []
for col in important_cols:
    availability.append({
        "column": col,
        "in_train": col in diag_train.columns,
        "in_test": col in diag_test.columns,
        "train_missing_pct": (
            float(diag_train[col].isna().mean() * 100) if col in diag_train.columns else np.nan
        ),
        "test_missing_pct": (
            float(diag_test[col].isna().mean() * 100) if col in diag_test.columns else np.nan
        ),
        "train_unique": (
            int(diag_train[col].nunique(dropna=False)) if col in diag_train.columns else np.nan
        ),
        "test_unique": (
            int(diag_test[col].nunique(dropna=False)) if col in diag_test.columns else np.nan
        ),
    })

availability_df = pd.DataFrame(availability)

print("\n--- Important raw column availability ---")
display(availability_df)

# ------------------------------------------------------------
# 4. Target bounded/proportion diagnostics
# ------------------------------------------------------------

print("\n--- Target diagnostics ---")

if "y_train" in globals():
    y_diag = pd.Series(y_train).astype(float).reset_index(drop=True)
else:
    y_diag = pd.Series(diag_train[TARGET_COL]).astype(float).reset_index(drop=True)

print(y_diag.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\nTarget boundary / discreteness checks:")
print("share target == 0:   ", float((y_diag == 0).mean()))
print("share target == 100: ", float((y_diag == 100).mean()))
print("share target < 0:    ", float((y_diag < 0).mean()))
print("share target > 100:  ", float((y_diag > 100).mean()))
print("share integer-like:  ", float(np.isclose(y_diag, np.round(y_diag), atol=1e-9).mean()))
print("number of unique target values:", int(y_diag.nunique()))

# ------------------------------------------------------------
# 5. N_STUDENTS reliability / rounded-count plausibility
# ------------------------------------------------------------

print("\n--- N_STUDENTS diagnostics ---")

if "N_STUDENTS" in diag_train.columns:
    n_train = pd.Series(diag_train["N_STUDENTS"]).astype(float).reset_index(drop=True)

    print(n_train.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    valid_n = n_train.notna() & (n_train > 0) & y_diag.notna()
    n_valid = n_train[valid_n].to_numpy(dtype=float)
    y_valid = y_diag[valid_n].to_numpy(dtype=float)

    # Check whether the reported percent is compatible with an integer number proficient,
    # allowing for whole-percent reporting.
    k_hat = np.rint((y_valid / 100.0) * n_valid)
    pct_from_count = 100.0 * k_hat / n_valid
    pct_from_count_rounded = np.rint(pct_from_count)

    exact_count_diff = np.abs(y_valid - pct_from_count)
    rounded_count_diff = np.abs(y_valid - pct_from_count_rounded)

    print("\nRounded-count plausibility:")
    print("valid rows checked:", int(valid_n.sum()))
    print("median abs diff vs nearest exact count percent:", float(np.median(exact_count_diff)))
    print("share compatible with rounded whole-percent count:", float((rounded_count_diff <= 1e-9).mean()))
    print("share within <= 0.5 percentage point of nearest count percent:", float((exact_count_diff <= 0.5).mean()))
    print("share within <= 1.0 percentage point of nearest count percent:", float((exact_count_diff <= 1.0).mean()))

    if "N_STUDENTS" in diag_test.columns:
        n_test = pd.Series(diag_test["N_STUDENTS"]).astype(float)
        print("\nTest N_STUDENTS summary:")
        print(n_test.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

else:
    print("N_STUDENTS not found. Bounded target modeling is still possible, but reliability-aware diagnostics are limited.")

# ------------------------------------------------------------
# 6. Candidate hierarchical key coverage
# ------------------------------------------------------------

print("\n--- Candidate hierarchical key coverage ---")

def make_key(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return df[cols[0]].astype("string").fillna("<NA>").astype(str)
    tmp = df.loc[:, list(cols)].astype("string").fillna("<NA>").astype(str)
    return tmp.agg(" || ".join, axis=1)

candidate_keys = [
    ("SCHOOL",),
    ("DISTRICT",),
    ("COUNTY",),
    ("REGION",),
    ("DISTRICT_TYPE",),
    ("ASSESSMENT_NAME",),
    ("SUBGROUP_NAME",),
    ("SCHOOL", "ASSESSMENT_NAME"),
    ("SCHOOL", "SUBGROUP_NAME"),
    ("ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME"),
    ("DISTRICT", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME"),
    ("COUNTY", "SUBGROUP_NAME"),
    ("REGION", "ASSESSMENT_NAME"),
    ("DISTRICT_TYPE", "ASSESSMENT_NAME"),
    ("SCHOOL", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("DISTRICT", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
    ("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"),
]

if "N_STUDENTS" in diag_train.columns and "N_STUDENTS" in diag_test.columns:
    # Small, stable bins for interaction diagnostics only.
    train_bins = pd.cut(
        pd.Series(diag_train["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")

    test_bins = pd.cut(
        pd.Series(diag_test["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")

    diag_train["_N_STUDENTS_BIN_DIAG"] = train_bins
    diag_test["_N_STUDENTS_BIN_DIAG"] = test_bins

    candidate_keys += [
        ("ASSESSMENT_NAME", "_N_STUDENTS_BIN_DIAG"),
        ("SUBGROUP_NAME", "_N_STUDENTS_BIN_DIAG"),
        ("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_DIAG"),
    ]

coverage_rows = []

for cols in candidate_keys:
    cols = tuple(cols)

    if not all(c in diag_train.columns for c in cols):
        continue
    if not all(c in diag_test.columns for c in cols):
        continue

    tr_key = make_key(diag_train, cols)
    te_key = make_key(diag_test, cols)

    vc = tr_key.value_counts(dropna=False)
    seen_set = set(vc.index)

    test_seen = te_key.isin(seen_set)

    coverage_rows.append({
        "key": " x ".join(cols).replace("_N_STUDENTS_BIN_DIAG", "N_STUDENTS_BIN"),
        "train_groups": int(vc.shape[0]),
        "test_groups": int(te_key.nunique(dropna=False)),
        "test_row_coverage": float(test_seen.mean()),
        "median_train_count": float(vc.median()),
        "mean_train_count": float(vc.mean()),
        "p90_train_count": float(vc.quantile(0.90)),
        "p99_train_count": float(vc.quantile(0.99)),
        "singleton_group_share": float((vc == 1).mean()),
        "max_train_count": int(vc.max()),
    })

coverage_df = pd.DataFrame(coverage_rows)

if len(coverage_df) > 0:
    coverage_df = coverage_df.sort_values(
        ["test_row_coverage", "median_train_count", "train_groups"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    display(coverage_df)

    recommended_start = coverage_df[
        (coverage_df["test_row_coverage"] >= 0.80) &
        (coverage_df["median_train_count"] >= 2)
    ].copy()

    print("\nKeys that look initially usable for cross-fitted statistical features:")
    if len(recommended_start) == 0:
        print("No keys passed the simple initial screen. We may need lower thresholds or fallback-only encodings.")
    else:
        display(recommended_start[[
            "key",
            "test_row_coverage",
            "train_groups",
            "median_train_count",
            "singleton_group_share",
            "p90_train_count",
            "p99_train_count"
        ]])

else:
    print("No candidate keys could be evaluated.")

# Save diagnostics for later cells.
encoding_key_diagnostics = coverage_df
raw_column_availability_diagnostics = availability_df

print("\nDone. Next step will depend on these diagnostics.")

Source used: existing train_full/test_full
diag_train shape: (144921, 62)
diag_test shape:  (48307, 61)

--- Alignment checks ---
len(y_train): 144921
diag_train rows match y_train: True
X_train_proc_model rows: 144921
diag_train rows match X_train_proc_model: True
X_test_proc_model rows: 48307
diag_test rows match X_test_proc_model: True
len(test_ids): 48307
diag_test rows match test_ids: True

--- Important raw column availability ---


,column,in_train,in_test,train_missing_pct,test_missing_pct,train_unique,test_unique
0,ASSESSMENT_ID,True,True,0.0,0.0,144921,48307.0
1,SCHOOL,True,True,0.0,0.0,4469,4448.0
2,DISTRICT,True,True,0.0,0.0,710,707.0
3,COUNTY,True,True,0.0,0.0,62,62.0
4,REGION,True,True,0.0,0.0,10,10.0
5,DISTRICT_TYPE,True,True,0.0,0.0,7,7.0
6,SUBGROUP_NAME,True,True,0.0,0.0,5,5.0
7,ASSESSMENT_NAME,True,True,0.0,0.0,32,32.0
8,N_STUDENTS,True,True,0.0,0.0,736,594.0
9,PERCENT_PROFICIENT,True,False,0.0,NaN,101,NaN



--- Target diagnostics ---
count    144921.000000
mean         54.183521
std          26.429925
min           0.000000
1%            0.000000
5%           13.000000
25%          33.000000
50%          53.000000
75%          76.000000
95%          98.000000
99%         100.000000
max         100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Target boundary / discreteness checks:
share target == 0:    0.01107499948247666
share target == 100:  0.04582496670599844
share target < 0:     0.0
share target > 100:   0.0
share integer-like:   1.0
number of unique target values: 101

--- N_STUDENTS diagnostics ---
count    144921.000000
mean         57.664748
std          67.211682
min           5.000000
1%            5.000000
5%            8.000000
25%          21.000000
50%          38.000000
75%          69.000000
95%         173.000000
99%         336.000000
max        1683.000000
Name: N_STUDENTS, dtype: float64

Rounded-count plausibility:
valid rows checked: 144921
median abs diff vs n

,key,train_groups,test_groups,test_row_coverage,median_train_count,mean_train_count,p90_train_count,p99_train_count,singleton_group_share,max_train_count
0,SUBGROUP_NAME,5,5,1.000000,29110.0,28984.200000,33771.8,36417.08,0.000000,36711
1,DISTRICT_TYPE,7,7,1.000000,13986.0,20703.000000,42308.8,43986.58,0.000000,44173
2,REGION,10,10,1.000000,9424.0,14492.100000,21815.9,51955.19,0.000000,55304
3,ASSESSMENT_NAME,32,32,1.000000,4532.0,4528.781250,8296.4,8389.21,0.000000,8392
4,SUBGROUP_NAME x N_STUDENTS_BIN,30,30,1.000000,3874.5,4830.700000,10516.9,13959.36,0.000000,13964
5,ASSESSMENT_NAME x SUBGROUP_NAME,132,132,1.000000,1004.5,1097.886364,1771.3,1837.76,0.000000,1883
6,COUNTY,62,62,1.000000,958.0,2337.435484,7086.5,14432.88,0.000000,17112
7,ASSESSMENT_NAME x N_STUDENTS_BIN,191,188,1.000000,470.0,758.748691,1786.0,3972.70,0.010471,3985
8,DISTRICT_TYPE x ASSESSMENT_NAME,222,221,1.000000,438.5,652.797297,1523.2,2507.06,0.009009,2545
9,REGION x ASSESSMENT_NAME,315,314,1.000000,295.0,460.066667,960.8,3118.04,0.000000,3140



Keys that look initially usable for cross-fitted statistical features:


,key,test_row_coverage,train_groups,median_train_count,singleton_group_share,p90_train_count,p99_train_count
0,SUBGROUP_NAME,1.000000,5,29110.0,0.000000,33771.8,36417.08
1,DISTRICT_TYPE,1.000000,7,13986.0,0.000000,42308.8,43986.58
2,REGION,1.000000,10,9424.0,0.000000,21815.9,51955.19
3,ASSESSMENT_NAME,1.000000,32,4532.0,0.000000,8296.4,8389.21
4,SUBGROUP_NAME x N_STUDENTS_BIN,1.000000,30,3874.5,0.000000,10516.9,13959.36
5,ASSESSMENT_NAME x SUBGROUP_NAME,1.000000,132,1004.5,0.000000,1771.3,1837.76
6,COUNTY,1.000000,62,958.0,0.000000,7086.5,14432.88
7,ASSESSMENT_NAME x N_STUDENTS_BIN,1.000000,191,470.0,0.010471,1786.0,3972.70
8,DISTRICT_TYPE x ASSESSMENT_NAME,1.000000,222,438.5,0.009009,1523.2,2507.06
9,REGION x ASSESSMENT_NAME,1.000000,315,295.0,0.000000,960.8,3118.04



Done. Next step will depend on these diagnostics.


In [60]:
# ============================================================
# 28B. Leakage-safe target/statistical encoding branch:
# holdout construction diagnostic
# ============================================================
#
# What this does:
# - Creates a fresh train/validation screen split by row index.
# - Builds target/statistical encoding features using ONLY the screen-training rows.
# - For screen-training rows, encodings are internally OOF/cross-fitted.
# - For screen-validation rows, encodings are mapped from screen-training only.
# - Adds both row-wise group mean and N_STUDENTS-weighted rounded-count group mean.
# - Creates augmented matrices for the next LightGBM holdout screen.
#
# This cell does NOT train a model.
# ============================================================

import os
import gc
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from scipy import sparse

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
TARGET_COL = "PERCENT_PROFICIENT"

required_objects = [
    "train_full",
    "test_full",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_required = [name for name in required_objects if name not in globals()]
if missing_required:
    raise ValueError(f"Missing required objects: {missing_required}")

raw_train_te = train_full.reset_index(drop=True).copy()
raw_test_te = test_full.reset_index(drop=True).copy()
y_arr_te = np.asarray(y_train, dtype=np.float32).reshape(-1)

if len(raw_train_te) != len(y_arr_te):
    raise ValueError("train_full and y_train are not aligned.")
if X_train_proc_model.shape[0] != len(y_arr_te):
    raise ValueError("X_train_proc_model and y_train are not aligned.")
if X_test_proc_model.shape[0] != len(raw_test_te):
    raise ValueError("X_test_proc_model and test_full are not aligned.")

def add_n_students_bin(df):
    out = df.copy()
    out["_N_STUDENTS_BIN_TE"] = pd.cut(
        pd.Series(out["N_STUDENTS"]).astype(float),
        bins=[-np.inf, 5, 10, 20, 50, 100, np.inf],
        labels=["<=5", "6-10", "11-20", "21-50", "51-100", ">100"]
    ).astype("string").fillna("<NA>")
    return out

raw_train_te = add_n_students_bin(raw_train_te)
raw_test_te = add_n_students_bin(raw_test_te)

candidate_key_specs = [
    # cols, alpha for row-wise mean, alpha_n for student-count-weighted mean
    (("ASSESSMENT_NAME",), 20.0, 200.0),
    (("SUBGROUP_NAME",), 20.0, 200.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME"), 20.0, 200.0),
    (("ASSESSMENT_NAME", "N_STUDENTS_BIN_TE"), 30.0, 250.0),
    (("ASSESSMENT_NAME", "SUBGROUP_NAME", "_N_STUDENTS_BIN_TE"), 50.0, 300.0),

    (("COUNTY",), 30.0, 250.0),
    (("COUNTY", "SUBGROUP_NAME"), 40.0, 300.0),
    (("COUNTY", "ASSESSMENT_NAME"), 60.0, 400.0),
    (("COUNTY", "ASSESSMENT_NAME", "SUBGROUP_NAME"), 100.0, 600.0),

    (("REGION", "ASSESSMENT_NAME"), 40.0, 300.0),
    (("DISTRICT_TYPE", "ASSESSMENT_NAME"), 40.0, 300.0),

    (("DISTRICT",), 60.0, 400.0),
    (("DISTRICT", "SUBGROUP_NAME"), 90.0, 500.0),
    (("DISTRICT", "ASSESSMENT_NAME"), 140.0, 700.0),

    (("SCHOOL",), 100.0, 600.0),
    (("SCHOOL", "SUBGROUP_NAME"), 140.0, 800.0),
    (("SCHOOL", "ASSESSMENT_NAME"), 220.0, 1000.0),
]

# Fix accidental name mismatch in one candidate.
candidate_key_specs = [
    (tuple("_N_STUDENTS_BIN_TE" if c == "N_STUDENTS_BIN_TE" else c for c in cols), alpha, alpha_n)
    for cols, alpha, alpha_n in candidate_key_specs
]

te_key_specs = []
for cols, alpha, alpha_n in candidate_key_specs:
    if all(c in raw_train_te.columns for c in cols) and all(c in raw_test_te.columns for c in cols):
        te_key_specs.append((cols, alpha, alpha_n))

if len(te_key_specs) == 0:
    raise ValueError("No usable target-encoding key specs found.")

def safe_key_name(cols):
    name = "__".join(cols)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def make_group_key(df, cols):
    cols = tuple(cols)
    if len(cols) == 1:
        return df[cols[0]].astype("string").fillna("<NA>").astype(str).reset_index(drop=True)
    return (
        df.loc[:, list(cols)]
        .astype("string")
        .fillna("<NA>")
        .astype(str)
        .agg(" || ".join, axis=1)
        .reset_index(drop=True)
    )

def fit_group_stats(keys, y, n_students, alpha, alpha_n):
    yy = np.asarray(y, dtype=np.float64)
    nn = np.asarray(n_students, dtype=np.float64)

    good_n = np.isfinite(nn) & (nn > 0)
    if not good_n.all():
        fill_n = np.nanmedian(nn[good_n]) if good_n.any() else 1.0
        nn = np.where(good_n, nn, fill_n)

    yy_clip = np.clip(yy, 0.0, 100.0)
    proficient_counts = np.rint((yy_clip / 100.0) * nn)
    proficient_counts = np.clip(proficient_counts, 0.0, nn)

    global_mean = float(np.mean(yy))
    global_std = float(np.std(yy, ddof=0))
    global_wmean = float(100.0 * proficient_counts.sum() / max(nn.sum(), 1.0))

    tmp = pd.DataFrame({
        "key": pd.Series(keys).astype(str).to_numpy(),
        "y": yy,
        "n": nn,
        "k": proficient_counts,
    })

    stats = tmp.groupby("key", sort=False).agg(
        cnt=("y", "size"),
        sum_y=("y", "sum"),
        std_y=("y", "std"),
        sum_n=("n", "sum"),
        sum_k=("k", "sum"),
    )

    stats["mean_s"] = (stats["sum_y"] + alpha * global_mean) / (stats["cnt"] + alpha)
    stats["wmean_s"] = 100.0 * (
        stats["sum_k"] + alpha_n * (global_wmean / 100.0)
    ) / (stats["sum_n"] + alpha_n)

    stats["std_y"] = stats["std_y"].fillna(global_std)
    stats["log_count"] = np.log1p(stats["cnt"].astype(float))
    stats["log_students"] = np.log1p(stats["sum_n"].astype(float))

    defaults = {
        "global_mean": global_mean,
        "global_wmean": global_wmean,
        "global_std": global_std,
    }

    return stats, defaults

def apply_group_stats(keys_apply, stats, defaults, prefix):
    kk = pd.Series(keys_apply).astype(str).reset_index(drop=True)

    out = pd.DataFrame(index=np.arange(len(kk)))
    out[f"{prefix}_mean"] = kk.map(stats["mean_s"]).fillna(defaults["global_mean"]).astype(np.float32)
    out[f"{prefix}_wmean"] = kk.map(stats["wmean_s"]).fillna(defaults["global_wmean"]).astype(np.float32)
    out[f"{prefix}_log_count"] = kk.map(stats["log_count"]).fillna(0.0).astype(np.float32)
    out[f"{prefix}_std"] = kk.map(stats["std_y"]).fillna(defaults["global_std"]).astype(np.float32)

    return out

def build_te_oof_and_apply(raw_fit, y_fit, raw_apply, key_specs, n_splits=5, random_state=9890):
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply = raw_apply.reset_index(drop=True)
    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)

    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = []
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key(raw_fit, cols)
        key_apply_all = make_group_key(raw_apply, cols)

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_va = apply_group_stats(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        apply_df = apply_group_stats(
            key_apply_all,
            stats_full,
            defaults_full,
            prefix,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))
        apply_parts.append(apply_df[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)
        apply_seen = key_apply_all.isin(set(fit_counts.index))

        summary_rows.append({
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "apply_groups": int(key_apply_all.nunique(dropna=False)),
            "apply_row_coverage": float(apply_seen.mean()),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        })

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = pd.concat(apply_parts, axis=1)
    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

def take_rows(X, idx):
    if sparse.issparse(X):
        return X[idx]
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx]
    return X[idx]

def to_float32_matrix(X):
    if sparse.issparse(X):
        return X.astype(np.float32).tocsr()
    if isinstance(X, pd.DataFrame):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)

def append_features(X_base, add_df):
    add = add_df.to_numpy(dtype=np.float32)
    if sparse.issparse(X_base):
        return sparse.hstack([X_base, sparse.csr_matrix(add)], format="csr")
    return np.hstack([X_base, add])

# Fresh screen split.
all_idx_te = np.arange(len(y_arr_te))
tr_idx_te_screen, val_idx_te_screen = train_test_split(
    all_idx_te,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)

raw_fit_screen = raw_train_te.iloc[tr_idx_te_screen].reset_index(drop=True)
raw_val_screen = raw_train_te.iloc[val_idx_te_screen].reset_index(drop=True)
y_fit_screen = y_arr_te[tr_idx_te_screen]
y_val_screen = y_arr_te[val_idx_te_screen]

te_tr_screen, te_val_screen, te_screen_key_summary = build_te_oof_and_apply(
    raw_fit=raw_fit_screen,
    y_fit=y_fit_screen,
    raw_apply=raw_val_screen,
    key_specs=te_key_specs,
    n_splits=5,
    random_state=RANDOM_STATE,
)

X_tr_base_screen = to_float32_matrix(take_rows(X_train_proc_model, tr_idx_te_screen))
X_val_base_screen = to_float32_matrix(take_rows(X_train_proc_model, val_idx_te_screen))

X_tr_te_screen = append_features(X_tr_base_screen, te_tr_screen)
X_val_te_screen = append_features(X_val_base_screen, te_val_screen)

finite_ok = (
    np.isfinite(te_tr_screen.to_numpy(dtype=np.float32)).all()
    and np.isfinite(te_val_screen.to_numpy(dtype=np.float32)).all()
)

shape_ok = (
    te_tr_screen.shape[0] == len(tr_idx_te_screen)
    and te_val_screen.shape[0] == len(val_idx_te_screen)
    and X_tr_te_screen.shape[0] == len(tr_idx_te_screen)
    and X_val_te_screen.shape[0] == len(val_idx_te_screen)
)

feature_count_ok = te_tr_screen.shape[1] > 0
column_ok = te_tr_screen.columns.is_unique and te_val_screen.columns.is_unique

TE_DIAGNOSTIC_PASS = bool(finite_ok and shape_ok and feature_count_ok and column_ok)

print("TE_DIAGNOSTIC_PASS =", TE_DIAGNOSTIC_PASS)
print("Screen train rows:", len(tr_idx_te_screen))
print("Screen valid rows:", len(val_idx_te_screen))
print("Base screen shapes:", X_tr_base_screen.shape, X_val_base_screen.shape)
print("Augmented TE screen shapes:", X_tr_te_screen.shape, X_val_te_screen.shape)
print("TE feature count:", te_tr_screen.shape[1])
print("Finite features:", finite_ok)
print("Unique TE columns:", column_ok)

print("\nTarget-encoding key summary:")
display(te_screen_key_summary.sort_values(
    ["apply_row_coverage", "median_fit_count"],
    ascending=[False, False]
).reset_index(drop=True))

print("\nTE train feature summary, first 10 columns:")
display(te_tr_screen.iloc[:, :10].describe().T)

gc.collect()

TE_DIAGNOSTIC_PASS = True
Screen train rows: 115936
Screen valid rows: 28985
Base screen shapes: (115936, 162) (28985, 162)
Augmented TE screen shapes: (115936, 230) (28985, 230)
TE feature count: 68
Finite features: True
Unique TE columns: True

Target-encoding key summary:


,key,alpha,alpha_n,fit_groups,apply_groups,apply_row_coverage,median_fit_count,singleton_share,features_added
0,SUBGROUP_NAME,20.0,200.0,5,5,1.000000,23233.0,0.000000,4
1,ASSESSMENT_NAME,20.0,200.0,32,32,1.000000,3601.0,0.000000,4
2,ASSESSMENT_NAME x SUBGROUP_NAME,20.0,200.0,132,132,1.000000,801.0,0.000000,4
3,COUNTY,30.0,250.0,62,62,1.000000,756.5,0.000000,4
4,DISTRICT_TYPE x ASSESSMENT_NAME,40.0,300.0,222,220,1.000000,354.5,0.009009,4
5,REGION x ASSESSMENT_NAME,40.0,300.0,315,314,1.000000,235.0,0.003175,4
6,COUNTY x SUBGROUP_NAME,40.0,300.0,310,309,1.000000,155.5,0.000000,4
7,DISTRICT,60.0,400.0,710,709,1.000000,78.0,0.000000,4
8,ASSESSMENT_NAME x N_STUDENTS_BIN,30.0,250.0,190,190,0.999965,372.5,0.005263,4
9,ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN,50.0,300.0,788,761,0.999965,109.0,0.013959,4



TE train feature summary, first 10 columns:


,count,mean,std,min,25%,50%,75%,max
te_ASSESSMENT_NAME_a20_n200_mean,115936.0,54.177391,12.351819,34.510574,43.745338,52.818825,65.268974,86.532150
te_ASSESSMENT_NAME_a20_n200_wmean,115936.0,54.782864,12.578279,31.266104,44.283440,53.665527,62.845219,84.903481
te_ASSESSMENT_NAME_a20_n200_log_count,115936.0,8.170941,0.494378,2.833213,7.962764,8.215007,8.561975,8.594524
te_ASSESSMENT_NAME_a20_n200_std,115936.0,23.093555,2.983797,2.500000,20.966339,22.582537,24.001675,30.780113
te_SUBGROUP_NAME_a20_n200_mean,115936.0,54.198841,4.719690,47.752342,52.528522,53.656490,54.151058,63.541710
te_SUBGROUP_NAME_a20_n200_wmean,115936.0,58.028111,7.377084,47.665009,55.763378,57.394348,57.541130,72.618271
te_SUBGROUP_NAME_a20_n200_log_count,115936.0,9.839026,0.148617,9.657715,9.686512,9.831508,10.059722,10.066923
te_SUBGROUP_NAME_a20_n200_std,115936.0,25.977516,0.553061,24.855640,25.627665,26.247187,26.384045,26.495983
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_mean,115936.0,54.151497,13.043442,28.668512,43.414429,53.396889,64.705765,86.532150
te_ASSESSMENT_NAME_SUBGROUP_NAME_a20_n200_wmean,115936.0,55.334736,14.385706,26.434465,44.359715,53.848438,64.508759,90.540169


0

In [61]:
# ============================================================
# 28C. Holdout screen:
# base LightGBM vs base + leakage-safe target/stat encodings
# ============================================================
#
# What this does:
# - Trains one base LightGBM on the screen split.
# - Trains one augmented LightGBM on the same split using the TE features from 28B.
# - Compares validation MSE directly.
#
# This cell does NOT create a Kaggle submission.
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

if not globals().get("TE_DIAGNOSTIC_PASS", False):
    raise ValueError("28B did not pass. Do not run 28C until TE_DIAGNOSTIC_PASS is True.")

os.makedirs("model_results", exist_ok=True)

def fit_lgbm_holdout_screen(X_tr_mat, y_tr_vec, X_val_mat, y_val_vec, label):
    params = {
        "objective": "regression",
        "metric": "l2",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "verbosity": -1,
        "force_col_wise": True,

        # Same general family as the successful long LightGBM,
        # but shorter for a screen.
        "n_estimators": 20000,
        "learning_rate": 0.03,
        "num_leaves": 95,
        "min_child_samples": 60,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "max_depth": -1,
    }

    model = lgb.LGBMRegressor(**params)

    t0 = time.time()
    model.fit(
        X_tr_mat,
        y_tr_vec,
        eval_set=[(X_val_mat, y_val_vec)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )
    elapsed = time.time() - t0

    best_iter = int(model.best_iteration_ or params["n_estimators"])

    pred_tr = np.clip(
        model.predict(X_tr_mat, num_iteration=best_iter),
        0,
        100,
    )
    pred_val = np.clip(
        model.predict(X_val_mat, num_iteration=best_iter),
        0,
        100,
    )

    result = {
        "label": label,
        "train_mse_clipped": float(mean_squared_error(y_tr_vec, pred_tr)),
        "valid_mse_clipped": float(mean_squared_error(y_val_vec, pred_val)),
        "best_iteration": best_iter,
        "elapsed_seconds": float(elapsed),
        "n_features": int(X_tr_mat.shape[1]),
        "pred_valid_mean": float(np.mean(pred_val)),
        "pred_valid_std": float(np.std(pred_val)),
        "pred_valid_min": float(np.min(pred_val)),
        "pred_valid_max": float(np.max(pred_val)),
    }

    return model, result, pred_val

screen_results = []

print("Training base LightGBM screen model...")
base_model_te_screen, base_result_te_screen, pred_val_base_te_screen = fit_lgbm_holdout_screen(
    X_tr_base_screen,
    y_fit_screen,
    X_val_base_screen,
    y_val_screen,
    label="base_lgbm_screen",
)
screen_results.append(base_result_te_screen)

gc.collect()

print("\nTraining base + target/stat encoding LightGBM screen model...")
aug_model_te_screen, aug_result_te_screen, pred_val_aug_te_screen = fit_lgbm_holdout_screen(
    X_tr_te_screen,
    y_fit_screen,
    X_val_te_screen,
    y_val_screen,
    label="base_plus_te_lgbm_screen",
)
screen_results.append(aug_result_te_screen)

lgbm_te_holdout_results = pd.DataFrame(screen_results)

base_mse = float(lgbm_te_holdout_results.loc[
    lgbm_te_holdout_results["label"] == "base_lgbm_screen",
    "valid_mse_clipped"
].iloc[0])

aug_mse = float(lgbm_te_holdout_results.loc[
    lgbm_te_holdout_results["label"] == "base_plus_te_lgbm_screen",
    "valid_mse_clipped"
].iloc[0])

te_holdout_gain = base_mse - aug_mse
TE_HOLDOUT_PASS = bool(te_holdout_gain >= 1.0)

lgbm_te_holdout_results["valid_gain_vs_base"] = base_mse - lgbm_te_holdout_results["valid_mse_clipped"]

print("\nHoldout screen results:")
display(lgbm_te_holdout_results)

print("\nBase valid MSE:       ", base_mse)
print("Base + TE valid MSE:  ", aug_mse)
print("TE holdout gain:      ", te_holdout_gain)
print("TE_HOLDOUT_PASS =     ", TE_HOLDOUT_PASS)

lgbm_te_holdout_results.to_csv(
    "model_results/lgbm_te_holdout_screen_results.csv",
    index=False,
)

# Keep predictions/results, but release heavy model objects unless you want feature importance later.
del base_model_te_screen, aug_model_te_screen
gc.collect()

Training base LightGBM screen model...
[1000]	valid_0's l2: 144.669
[2000]	valid_0's l2: 129.649
[3000]	valid_0's l2: 121.526
[4000]	valid_0's l2: 115.833
[5000]	valid_0's l2: 112.05
[6000]	valid_0's l2: 109.168
[7000]	valid_0's l2: 106.872
[8000]	valid_0's l2: 105.021
[9000]	valid_0's l2: 103.557
[10000]	valid_0's l2: 102.335
[11000]	valid_0's l2: 101.414
[12000]	valid_0's l2: 100.64
[13000]	valid_0's l2: 99.9944
[14000]	valid_0's l2: 99.4335
[15000]	valid_0's l2: 99.0222
[16000]	valid_0's l2: 98.6226
[17000]	valid_0's l2: 98.3013
[18000]	valid_0's l2: 98.0536
[19000]	valid_0's l2: 97.8065
[20000]	valid_0's l2: 97.6178

Training base + target/stat encoding LightGBM screen model...
[1000]	valid_0's l2: 90.631
[2000]	valid_0's l2: 89.6559
[3000]	valid_0's l2: 89.2743
[4000]	valid_0's l2: 89.1099
[5000]	valid_0's l2: 89.1189

Holdout screen results:


,label,train_mse_clipped,valid_mse_clipped,best_iteration,elapsed_seconds,n_features,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,valid_gain_vs_base
0,base_lgbm_screen,9.283496,96.849902,19992,354.543891,162,54.109896,24.564745,0.0,100.0,0.00000
1,base_plus_te_lgbm_screen,12.467271,88.808052,4868,157.827635,230,55.502333,25.017372,0.0,100.0,8.04185



Base valid MSE:        96.84990207455618
Base + TE valid MSE:   88.80805200212511
TE holdout gain:       8.041850072431075
TE_HOLDOUT_PASS =      True


11834

In [62]:
# ============================================================
# 28D. Preflight for full OOF target/stat-encoding LightGBM
# ============================================================
#
# Purpose:
# - Confirm the 28B/28C objects are still available.
# - Confirm row alignment.
# - Confirm the full OOF run will add the expected TE features.
# - Estimate the model artifact names before launching the heavy cell.
#
# Run 28E only if TE_OOF_PREFLIGHT_PASS = True.
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse

os.makedirs("model_results", exist_ok=True)

required_28d = [
    "TE_HOLDOUT_PASS",
    "raw_train_te",
    "raw_test_te",
    "te_key_specs",
    "build_te_oof_and_apply",
    "fit_group_stats",
    "apply_group_stats",
    "make_group_key",
    "safe_key_name",
    "append_features",
    "to_float32_matrix",
    "take_rows",
    "X_train_proc_model",
    "X_test_proc_model",
    "y_train",
    "test_ids",
]

missing_28d = [name for name in required_28d if name not in globals()]

alignment_ok_28d = True
alignment_notes_28d = []

if len(np.asarray(y_train).reshape(-1)) != raw_train_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("y_train length does not match raw_train_te rows.")

if X_train_proc_model.shape[0] != raw_train_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("X_train_proc_model rows do not match raw_train_te rows.")

if X_test_proc_model.shape[0] != raw_test_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("X_test_proc_model rows do not match raw_test_te rows.")

if len(test_ids) != raw_test_te.shape[0]:
    alignment_ok_28d = False
    alignment_notes_28d.append("test_ids length does not match raw_test_te rows.")

holdout_ok_28d = bool(globals().get("TE_HOLDOUT_PASS", False))
key_ok_28d = ("te_key_specs" in globals()) and (len(te_key_specs) > 0)

expected_te_feature_count_28d = int(len(te_key_specs) * 4) if key_ok_28d else 0
expected_aug_feature_count_28d = int(X_train_proc_model.shape[1] + expected_te_feature_count_28d)

TE_OOF_ARTIFACT_NAME = "lgbm_te_base_5fold_oof_v1"

TE_OOF_PREFLIGHT_PASS = bool(
    len(missing_28d) == 0
    and holdout_ok_28d
    and alignment_ok_28d
    and key_ok_28d
    and expected_te_feature_count_28d > 0
)

print("TE_OOF_PREFLIGHT_PASS =", TE_OOF_PREFLIGHT_PASS)
print("Missing required objects:", missing_28d)
print("TE_HOLDOUT_PASS:", holdout_ok_28d)
print("Alignment OK:", alignment_ok_28d)
print("Alignment notes:", alignment_notes_28d)
print("Number of TE key specs:", len(te_key_specs) if key_ok_28d else 0)
print("Expected TE feature count:", expected_te_feature_count_28d)
print("Base feature count:", X_train_proc_model.shape[1])
print("Expected augmented feature count:", expected_aug_feature_count_28d)
print("Artifact name:", TE_OOF_ARTIFACT_NAME)

print("\nTE keys to be used:")
for cols, alpha, alpha_n in te_key_specs:
    print(" -", " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
          "| alpha =", alpha, "| alpha_n =", alpha_n)

TE_OOF_PREFLIGHT_PASS = True
Missing required objects: []
TE_HOLDOUT_PASS: True
Alignment OK: True
Alignment notes: []
Number of TE key specs: 17
Expected TE feature count: 68
Base feature count: 162
Expected augmented feature count: 230
Artifact name: lgbm_te_base_5fold_oof_v1

TE keys to be used:
 - ASSESSMENT_NAME | alpha = 20.0 | alpha_n = 200.0
 - SUBGROUP_NAME | alpha = 20.0 | alpha_n = 200.0
 - ASSESSMENT_NAME x SUBGROUP_NAME | alpha = 20.0 | alpha_n = 200.0
 - ASSESSMENT_NAME x N_STUDENTS_BIN | alpha = 30.0 | alpha_n = 250.0
 - ASSESSMENT_NAME x SUBGROUP_NAME x N_STUDENTS_BIN | alpha = 50.0 | alpha_n = 300.0
 - COUNTY | alpha = 30.0 | alpha_n = 250.0
 - COUNTY x SUBGROUP_NAME | alpha = 40.0 | alpha_n = 300.0
 - COUNTY x ASSESSMENT_NAME | alpha = 60.0 | alpha_n = 400.0
 - COUNTY x ASSESSMENT_NAME x SUBGROUP_NAME | alpha = 100.0 | alpha_n = 600.0
 - REGION x ASSESSMENT_NAME | alpha = 40.0 | alpha_n = 300.0
 - DISTRICT_TYPE x ASSESSMENT_NAME | alpha = 40.0 | alpha_n = 300.0
 - DIS

In [63]:
# ============================================================
# 28E. Full 5-fold OOF artifact:
# LightGBM on base + leakage-safe target/stat encodings
# ============================================================
#
# What this does:
# - For each outer fold:
#     1. Uses only the outer-training rows to build TE mappings.
#     2. Builds inner-OOF TE features for the outer-training rows.
#     3. Builds validation TE features from outer-training mappings only.
#     4. Builds test TE features from outer-training mappings only.
#     5. Trains LightGBM and saves fold predictions.
# - Saves:
#     model_results/oof_lgbm_te_base_5fold_oof_v1.csv
#     model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
#     submission_lgbm_te_base_5fold_oof_v1_foldavg.csv
#
# This is a serious OOF artifact, not just a holdout screen.
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

if not globals().get("TE_OOF_PREFLIGHT_PASS", False):
    raise ValueError("28D did not pass. Do not run 28E until TE_OOF_PREFLIGHT_PASS is True.")

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)
ARTIFACT = globals().get("TE_OOF_ARTIFACT_NAME", "lgbm_te_base_5fold_oof_v1")

os.makedirs("model_results", exist_ok=True)

n_train_28e = X_train_proc_model.shape[0]
n_test_28e = X_test_proc_model.shape[0]
y_arr_28e = np.asarray(y_train, dtype=np.float32).reshape(-1)

metrics_path_28e = f"model_results/{ARTIFACT}_fold_metrics.csv"
oof_raw_path_28e = f"model_results/{ARTIFACT}_oof_raw.npy"
oof_clip_path_28e = f"model_results/{ARTIFACT}_oof_clipped.npy"

oof_csv_path_28e = f"model_results/oof_{ARTIFACT}.csv"
test_csv_path_28e = f"model_results/testpred_{ARTIFACT}_foldavg.csv"
submission_path_28e = f"submission_{ARTIFACT}_foldavg.csv"

def build_te_oof_and_apply_many(raw_fit, y_fit, raw_apply_dict, key_specs, n_splits=5, random_state=9890):
    """
    Build leakage-safe inner-OOF TE features for raw_fit,
    plus mapping-applied TE features for each raw_apply frame.

    raw_fit: rows used to train the outer-fold model
    raw_apply_dict: {"valid": raw_valid, "test": raw_test}
    """
    raw_fit = raw_fit.reset_index(drop=True)
    raw_apply_dict = {
        name: df.reset_index(drop=True)
        for name, df in raw_apply_dict.items()
    }

    y_fit = np.asarray(y_fit, dtype=np.float32).reshape(-1)
    n_fit = pd.Series(raw_fit["N_STUDENTS"]).astype(float).to_numpy()
    n_rows_fit = len(raw_fit)

    kf_inner = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_parts = []
    apply_parts = {name: [] for name in raw_apply_dict.keys()}
    summary_rows = []

    for cols, alpha, alpha_n in key_specs:
        cols = tuple(cols)
        prefix = f"te_{safe_key_name(cols)}_a{int(alpha)}_n{int(alpha_n)}"

        key_fit_all = make_group_key(raw_fit, cols)
        key_apply_all = {
            name: make_group_key(df, cols)
            for name, df in raw_apply_dict.items()
        }

        feature_cols = [
            f"{prefix}_mean",
            f"{prefix}_wmean",
            f"{prefix}_log_count",
            f"{prefix}_std",
        ]

        oof_arr = np.zeros((n_rows_fit, len(feature_cols)), dtype=np.float32)

        for inner_tr_idx, inner_va_idx in kf_inner.split(np.arange(n_rows_fit)):
            stats_fold, defaults_fold = fit_group_stats(
                key_fit_all.iloc[inner_tr_idx],
                y_fit[inner_tr_idx],
                n_fit[inner_tr_idx],
                alpha=alpha,
                alpha_n=alpha_n,
            )

            enc_inner_va = apply_group_stats(
                key_fit_all.iloc[inner_va_idx],
                stats_fold,
                defaults_fold,
                prefix,
            )

            oof_arr[inner_va_idx, :] = enc_inner_va[feature_cols].to_numpy(dtype=np.float32)

        stats_full, defaults_full = fit_group_stats(
            key_fit_all,
            y_fit,
            n_fit,
            alpha=alpha,
            alpha_n=alpha_n,
        )

        oof_parts.append(pd.DataFrame(oof_arr, columns=feature_cols))

        for name, key_apply in key_apply_all.items():
            enc_apply = apply_group_stats(
                key_apply,
                stats_full,
                defaults_full,
                prefix,
            )
            apply_parts[name].append(enc_apply[feature_cols])

        fit_counts = key_fit_all.value_counts(dropna=False)

        row = {
            "key": " x ".join(cols).replace("_N_STUDENTS_BIN_TE", "N_STUDENTS_BIN"),
            "alpha": alpha,
            "alpha_n": alpha_n,
            "fit_groups": int(fit_counts.shape[0]),
            "median_fit_count": float(fit_counts.median()),
            "singleton_share": float((fit_counts == 1).mean()),
            "features_added": len(feature_cols),
        }

        for name, key_apply in key_apply_all.items():
            row[f"{name}_groups"] = int(key_apply.nunique(dropna=False))
            row[f"{name}_row_coverage"] = float(key_apply.isin(set(fit_counts.index)).mean())

        summary_rows.append(row)

    oof_features = pd.concat(oof_parts, axis=1)
    apply_features = {
        name: pd.concat(parts, axis=1)
        for name, parts in apply_parts.items()
    }

    summary = pd.DataFrame(summary_rows)

    return oof_features, apply_features, summary

def fold_test_pred_path_28e(fold_num):
    return f"model_results/{ARTIFACT}_fold{fold_num}_test_pred.npy"

# Set up fold indices once so resume checks are stable.
outer_kf_28e = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_indices_28e = list(outer_kf_28e.split(np.arange(n_train_28e)))

if os.path.exists(oof_raw_path_28e):
    oof_pred_raw_28e = np.load(oof_raw_path_28e)
else:
    oof_pred_raw_28e = np.full(n_train_28e, np.nan, dtype=np.float32)

if os.path.exists(oof_clip_path_28e):
    oof_pred_clip_28e = np.load(oof_clip_path_28e)
else:
    oof_pred_clip_28e = np.full(n_train_28e, np.nan, dtype=np.float32)

if os.path.exists(metrics_path_28e):
    fold_metrics_28e = pd.read_csv(metrics_path_28e)
else:
    fold_metrics_28e = pd.DataFrame()

done_folds_28e = set()
if len(fold_metrics_28e) > 0 and "fold" in fold_metrics_28e.columns:
    done_folds_28e = set(fold_metrics_28e["fold"].astype(int).tolist())

params_28e = {
    "objective": "regression",
    "metric": "l2",
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "verbosity": -1,
    "force_col_wise": True,

    "n_estimators": 20000,
    "learning_rate": 0.03,
    "num_leaves": 95,
    "min_child_samples": 60,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "max_depth": -1,
}

print("Artifact:", ARTIFACT)
print("Rows:", n_train_28e, "train |", n_test_28e, "test")
print("Base features:", X_train_proc_model.shape[1])
print("TE features:", len(te_key_specs) * 4)
print("Augmented features expected:", X_train_proc_model.shape[1] + len(te_key_specs) * 4)
print("Already completed folds:", sorted(done_folds_28e))

for fold_num, (tr_idx, va_idx) in enumerate(fold_indices_28e, start=1):
    fold_test_path = fold_test_pred_path_28e(fold_num)

    if (
        fold_num in done_folds_28e
        and os.path.exists(fold_test_path)
        and np.isfinite(oof_pred_clip_28e[va_idx]).all()
    ):
        print(f"\nFold {fold_num} already complete. Skipping.")
        continue

    print(f"\n========== Fold {fold_num}/5 ==========")
    t0_fold = time.time()

    raw_fit_fold = raw_train_te.iloc[tr_idx].reset_index(drop=True)
    raw_val_fold = raw_train_te.iloc[va_idx].reset_index(drop=True)

    y_fit_fold = y_arr_28e[tr_idx]
    y_val_fold = y_arr_28e[va_idx]

    print("Building TE features for fold...")
    te_fit_fold, apply_dict_fold, te_key_summary_fold = build_te_oof_and_apply_many(
        raw_fit=raw_fit_fold,
        y_fit=y_fit_fold,
        raw_apply_dict={
            "valid": raw_val_fold,
            "test": raw_test_te,
        },
        key_specs=te_key_specs,
        n_splits=5,
        random_state=RANDOM_STATE + 100 * fold_num,
    )

    finite_te_ok = (
        np.isfinite(te_fit_fold.to_numpy(dtype=np.float32)).all()
        and np.isfinite(apply_dict_fold["valid"].to_numpy(dtype=np.float32)).all()
        and np.isfinite(apply_dict_fold["test"].to_numpy(dtype=np.float32)).all()
    )

    if not finite_te_ok:
        raise ValueError(f"Non-finite TE features detected in fold {fold_num}.")

    X_fit_base_fold = to_float32_matrix(take_rows(X_train_proc_model, tr_idx))
    X_val_base_fold = to_float32_matrix(take_rows(X_train_proc_model, va_idx))
    X_test_base_28e = to_float32_matrix(X_test_proc_model)

    X_fit_fold = append_features(X_fit_base_fold, te_fit_fold)
    X_val_fold = append_features(X_val_base_fold, apply_dict_fold["valid"])
    X_test_fold = append_features(X_test_base_28e, apply_dict_fold["test"])

    print("Fold train shape:", X_fit_fold.shape)
    print("Fold valid shape:", X_val_fold.shape)
    print("Fold test shape: ", X_test_fold.shape)

    model_fold = lgb.LGBMRegressor(**params_28e)

    print("Training LightGBM...")
    model_fold.fit(
        X_fit_fold,
        y_fit_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(stopping_rounds=1000, verbose=False),
            lgb.log_evaluation(period=1000),
        ],
    )

    best_iter_fold = int(model_fold.best_iteration_ or params_28e["n_estimators"])

    pred_fit_raw = model_fold.predict(X_fit_fold, num_iteration=best_iter_fold)
    pred_val_raw = model_fold.predict(X_val_fold, num_iteration=best_iter_fold)
    pred_test_raw = model_fold.predict(X_test_fold, num_iteration=best_iter_fold)

    pred_fit_clip = np.clip(pred_fit_raw, 0, 100).astype(np.float32)
    pred_val_clip = np.clip(pred_val_raw, 0, 100).astype(np.float32)
    pred_test_clip = np.clip(pred_test_raw, 0, 100).astype(np.float32)

    oof_pred_raw_28e[va_idx] = pred_val_raw.astype(np.float32)
    oof_pred_clip_28e[va_idx] = pred_val_clip

    np.save(oof_raw_path_28e, oof_pred_raw_28e)
    np.save(oof_clip_path_28e, oof_pred_clip_28e)
    np.save(fold_test_path, pred_test_clip)

    fold_elapsed = time.time() - t0_fold

    fold_row = {
        "artifact": ARTIFACT,
        "fold": fold_num,
        "train_rows": int(len(tr_idx)),
        "valid_rows": int(len(va_idx)),
        "n_features": int(X_fit_fold.shape[1]),
        "best_iteration": best_iter_fold,
        "train_mse_raw": float(mean_squared_error(y_fit_fold, pred_fit_raw)),
        "train_mse_clipped": float(mean_squared_error(y_fit_fold, pred_fit_clip)),
        "valid_mse_raw": float(mean_squared_error(y_val_fold, pred_val_raw)),
        "valid_mse_clipped": float(mean_squared_error(y_val_fold, pred_val_clip)),
        "pred_valid_mean": float(np.mean(pred_val_clip)),
        "pred_valid_std": float(np.std(pred_val_clip)),
        "pred_valid_min": float(np.min(pred_val_clip)),
        "pred_valid_max": float(np.max(pred_val_clip)),
        "elapsed_seconds": float(fold_elapsed),
    }

    fold_metrics_28e = pd.concat(
        [fold_metrics_28e[fold_metrics_28e.get("fold", pd.Series(dtype=int)) != fold_num], pd.DataFrame([fold_row])],
        ignore_index=True,
    ).sort_values("fold").reset_index(drop=True)

    fold_metrics_28e.to_csv(metrics_path_28e, index=False)

    print("Fold result:")
    print(pd.DataFrame([fold_row]).T)

    # Free memory aggressively.
    del model_fold
    del raw_fit_fold, raw_val_fold
    del te_fit_fold, apply_dict_fold, te_key_summary_fold
    del X_fit_base_fold, X_val_base_fold, X_test_base_28e
    del X_fit_fold, X_val_fold, X_test_fold
    del pred_fit_raw, pred_val_raw, pred_test_raw
    del pred_fit_clip, pred_val_clip, pred_test_clip
    gc.collect()

# Finalize only if all folds are complete.
TE_OOF_RUN_PASS = bool(np.isfinite(oof_pred_clip_28e).all())

print("\nTE_OOF_RUN_PASS =", TE_OOF_RUN_PASS)

if TE_OOF_RUN_PASS:
    overall_oof_mse_raw_28e = float(mean_squared_error(y_arr_28e, oof_pred_raw_28e))
    overall_oof_mse_clip_28e = float(mean_squared_error(y_arr_28e, oof_pred_clip_28e))

    print("Overall OOF MSE raw:    ", overall_oof_mse_raw_28e)
    print("Overall OOF MSE clipped:", overall_oof_mse_clip_28e)

    print("\nFold metrics:")
    display(fold_metrics_28e)

    fold_test_preds = []
    for fold_num in range(1, 6):
        fold_path = fold_test_pred_path_28e(fold_num)
        if not os.path.exists(fold_path):
            raise FileNotFoundError(f"Missing fold test prediction file: {fold_path}")
        fold_test_preds.append(np.load(fold_path).astype(np.float32))

    test_pred_28e = np.mean(np.vstack(fold_test_preds), axis=0)
    test_pred_28e = np.clip(test_pred_28e, 0, 100).astype(np.float32)

    oof_df_28e = pd.DataFrame({
        "row_index": np.arange(n_train_28e),
        "PERCENT_PROFICIENT": y_arr_28e,
        "pred_raw": oof_pred_raw_28e,
        "pred_clipped": oof_pred_clip_28e,
    })
    oof_df_28e.to_csv(oof_csv_path_28e, index=False)

    testpred_df_28e = pd.DataFrame({
        "ASSESSMENT_ID": np.asarray(test_ids),
        "PERCENT_PROFICIENT": test_pred_28e,
    })
    testpred_df_28e.to_csv(test_csv_path_28e, index=False)

    submission_28e = testpred_df_28e[["ASSESSMENT_ID", "PERCENT_PROFICIENT"]].copy()
    submission_28e.to_csv(submission_path_28e, index=False)

    print("\nSaved files:")
    print(" -", metrics_path_28e)
    print(" -", oof_raw_path_28e)
    print(" -", oof_clip_path_28e)
    print(" -", oof_csv_path_28e)
    print(" -", test_csv_path_28e)
    print(" -", submission_path_28e)

    print("\nSubmission prediction summary:")
    print(submission_28e["PERCENT_PROFICIENT"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    print("\nAnchor comparisons:")
    print("Best pure long100k LGBM OOF:          93.726214")
    print("Best pre-residual blend OOF:          88.966368")
    print("Current 77.277 residual-stack OOF:    87.136049")
    print("This TE OOF clipped MSE:             ", overall_oof_mse_clip_28e)
    print("Gain vs pure long100k LGBM:          ", 93.726214 - overall_oof_mse_clip_28e)
    print("Gain vs pre-residual blend:          ", 88.966368 - overall_oof_mse_clip_28e)
    print("Gain vs current residual-stack OOF:  ", 87.136049 - overall_oof_mse_clip_28e)

else:
    print("OOF predictions are incomplete. Rerun this same cell to continue/checkpoint.")

Artifact: lgbm_te_base_5fold_oof_v1
Rows: 144921 train | 48307 test
Base features: 162
TE features: 68
Augmented features expected: 230
Already completed folds: []

========== Fold 1/5 ==========
Building TE features for fold...
Fold train shape: (115936, 230)
Fold valid shape: (28985, 230)
Fold test shape:  (48307, 230)
Training LightGBM...
[1000]	valid_0's l2: 85.3052
[2000]	valid_0's l2: 84.3444
[3000]	valid_0's l2: 84.0699
[4000]	valid_0's l2: 84.0502
[5000]	valid_0's l2: 83.9653
[6000]	valid_0's l2: 84.0682
Fold result:
                                           0
artifact           lgbm_te_base_5fold_oof_v1
fold                                       1
train_rows                            115936
valid_rows                             28985
n_features                               230
best_iteration                          5033
train_mse_raw                      11.666033
train_mse_clipped                  11.650144
valid_mse_raw                      83.961103
valid_mse_clipped  

,artifact,fold,train_rows,valid_rows,n_features,best_iteration,train_mse_raw,train_mse_clipped,valid_mse_raw,valid_mse_clipped,pred_valid_mean,pred_valid_std,pred_valid_min,pred_valid_max,elapsed_seconds
0,lgbm_te_base_5fold_oof_v1,1,115936,28985,230,5033,11.666033,11.650144,83.961103,83.846176,54.934315,24.891148,0.0,100.0,281.356708
1,lgbm_te_base_5fold_oof_v1,2,115937,28984,230,5379,10.535665,10.520914,79.906313,79.800461,53.856457,25.086536,0.0,100.0,298.724768
2,lgbm_te_base_5fold_oof_v1,3,115937,28984,230,5629,9.836355,9.822016,84.624473,84.527496,54.604282,25.013144,0.0,100.0,277.812274
3,lgbm_te_base_5fold_oof_v1,4,115937,28984,230,4379,14.653784,14.633527,83.625740,83.476036,54.854794,24.772558,0.0,100.0,224.089123
4,lgbm_te_base_5fold_oof_v1,5,115937,28984,230,4240,15.136295,15.119184,82.946000,82.887871,54.425667,24.821636,0.0,100.0,219.875332



Saved files:
 - model_results/lgbm_te_base_5fold_oof_v1_fold_metrics.csv
 - model_results/lgbm_te_base_5fold_oof_v1_oof_raw.npy
 - model_results/lgbm_te_base_5fold_oof_v1_oof_clipped.npy
 - model_results/oof_lgbm_te_base_5fold_oof_v1.csv
 - model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv
 - submission_lgbm_te_base_5fold_oof_v1_foldavg.csv

Submission prediction summary:
count    48307.000000
mean        54.500694
std         24.779079
min          0.000000
1%           8.264243
5%          16.623557
25%         34.708241
50%         52.084106
75%         75.007858
95%         95.584126
99%         99.703726
max        100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Anchor comparisons:
Best pure long100k LGBM OOF:          93.726214
Best pre-residual blend OOF:          88.966368
Current 77.277 residual-stack OOF:    87.136049
This TE OOF clipped MSE:              82.90760803222656
Gain vs pure long100k LGBM:           10.818605967773436
Gain vs pre-residual blend:   

### 28E checkpoint: target/statistical encoding LightGBM OOF breakthrough

The leakage-safe cross-fitted target/statistical encoding branch completed successfully.

Artifact: `lgbm_te_base_5fold_oof_v1`

Base features: 162  
Target/statistical encoding features: 68  
Total features: 230  

OOF clipped MSE: **82.907608**

This beats:
- best pure long100k LightGBM OOF, 93.726214, by about 10.82 MSE points
- pre-residual ET + long100k blend OOF, 88.966368, by about 6.06 MSE points
- current 77.277 public residual-stack anchor OOF, 87.136049, by about 4.23 MSE points

Interpretation: cross-fitted hierarchical target/statistical encodings add major new signal. This branch should now be treated as a primary modeling direction, not a side experiment.

In [64]:
# ============================================================
# 28F. Preflight for blending the new TE OOF artifact
# ============================================================
#
# Purpose:
# - Load the new target/statistical-encoding LightGBM OOF artifact.
# - Load existing saved OOF/test artifacts if available.
# - Infer prediction columns safely.
# - Confirm shapes, finiteness, and component OOF MSEs.
#
# Run 28G only if BLEND28F_PASS = True.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

RANDOM_STATE = globals().get("RANDOM_STATE", 9890)

if "y_train" not in globals():
    raise ValueError("y_train is missing.")
if "test_ids" not in globals():
    raise ValueError("test_ids is missing.")

y_blend_28f = np.asarray(y_train, dtype=np.float64).reshape(-1)
test_ids_28f = np.asarray(test_ids)

n_train_28f = len(y_blend_28f)
n_test_28f = len(test_ids_28f)

def _existing(paths):
    return [p for p in paths if p is not None and os.path.exists(p)]

def _unique_keep_order(paths):
    out = []
    seen = set()
    for p in paths:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out

def infer_oof_prediction(path, y_ref):
    df = pd.read_csv(path)

    if "row_index" in df.columns and len(df) == len(y_ref):
        df = df.sort_values("row_index").reset_index(drop=True)

    if len(df) != len(y_ref):
        raise ValueError(f"OOF row count mismatch for {path}: {len(df)} vs {len(y_ref)}")

    numeric_cols = []
    for col in df.columns:
        if col.lower() in ["row_index", "assessment_id", "fold"]:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)

    candidates = []
    for col in numeric_cols:
        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64)
        if len(arr) != len(y_ref):
            continue
        if not np.isfinite(arr).all():
            continue

        arr_clip = np.clip(arr, 0, 100)
        mse = float(mean_squared_error(y_ref, arr_clip))

        # Exclude the true target column if present.
        if mse < 1e-8:
            continue

        # Exclude obvious index-like columns.
        if np.nanstd(arr_clip) < 1e-8:
            continue

        candidates.append((mse, col, arr_clip))

    if len(candidates) == 0:
        raise ValueError(f"Could not infer OOF prediction column for {path}. Columns: {list(df.columns)}")

    candidates = sorted(candidates, key=lambda x: x[0])
    best_mse, best_col, best_arr = candidates[0]
    return best_arr.astype(np.float32), best_col, best_mse, list(df.columns)

def infer_test_prediction(path, test_ids_ref):
    df = pd.read_csv(path)

    if "ASSESSMENT_ID" in df.columns:
        # Align to current test_ids if needed.
        if len(df) == len(test_ids_ref) and np.array_equal(df["ASSESSMENT_ID"].to_numpy(), test_ids_ref):
            aligned = df.copy()
        else:
            key_df = pd.DataFrame({"ASSESSMENT_ID": test_ids_ref})
            aligned = key_df.merge(df, on="ASSESSMENT_ID", how="left")
            if len(aligned) != len(test_ids_ref):
                raise ValueError(f"Test merge row mismatch for {path}")
    else:
        aligned = df.copy()

    if len(aligned) != len(test_ids_ref):
        raise ValueError(f"Test row count mismatch for {path}: {len(aligned)} vs {len(test_ids_ref)}")

    priority_cols = [
        "PERCENT_PROFICIENT",
        "pred_clipped",
        "prediction",
        "pred",
        "test_pred",
        "foldavg",
        "fold_avg",
        "mean_pred",
    ]

    for col in priority_cols:
        if col in aligned.columns and pd.api.types.is_numeric_dtype(aligned[col]):
            arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
            if np.isfinite(arr).all():
                return np.clip(arr, 0, 100).astype(np.float32), col, list(aligned.columns)

    numeric_cols = []
    for col in aligned.columns:
        if col.lower() in ["assessment_id", "row_index", "fold"]:
            continue
        if pd.api.types.is_numeric_dtype(aligned[col]):
            numeric_cols.append(col)

    pred_like_cols = [
        c for c in numeric_cols
        if ("pred" in c.lower()) or ("fold" in c.lower())
    ]

    if len(pred_like_cols) >= 2:
        arr = aligned[pred_like_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(arr).all():
            return np.clip(arr.mean(axis=1), 0, 100).astype(np.float32), "mean_of_pred_like_cols", list(aligned.columns)

    if len(numeric_cols) == 1:
        col = numeric_cols[0]
        arr = pd.to_numeric(aligned[col], errors="coerce").to_numpy(dtype=np.float64)
        if np.isfinite(arr).all():
            return np.clip(arr, 0, 100).astype(np.float32), col, list(aligned.columns)

    raise ValueError(f"Could not infer test prediction column for {path}. Columns: {list(aligned.columns)}")

def load_component(name, oof_candidates, test_candidates, y_ref, test_ids_ref):
    oof_candidates = _unique_keep_order(_existing(oof_candidates))
    test_candidates = _unique_keep_order(_existing(test_candidates))

    if len(oof_candidates) == 0:
        return None, f"{name}: no OOF file found"
    if len(test_candidates) == 0:
        return None, f"{name}: no test/submission file found"

    best_oof = None
    oof_errors = []

    for oof_path in oof_candidates:
        try:
            pred_oof, oof_col, oof_mse, oof_cols = infer_oof_prediction(oof_path, y_ref)
            if best_oof is None or oof_mse < best_oof["oof_mse"]:
                best_oof = {
                    "path": oof_path,
                    "pred": pred_oof,
                    "col": oof_col,
                    "oof_mse": oof_mse,
                    "columns": oof_cols,
                }
        except Exception as e:
            oof_errors.append((oof_path, str(e)))

    if best_oof is None:
        return None, f"{name}: OOF files found but none loaded. Errors: {oof_errors[:3]}"

    best_test = None
    test_errors = []

    for test_path in test_candidates:
        try:
            pred_test, test_col, test_cols = infer_test_prediction(test_path, test_ids_ref)
            best_test = {
                "path": test_path,
                "pred": pred_test,
                "col": test_col,
                "columns": test_cols,
            }
            break
        except Exception as e:
            test_errors.append((test_path, str(e)))

    if best_test is None:
        return None, f"{name}: test files found but none loaded. Errors: {test_errors[:3]}"

    comp = {
        "name": name,
        "oof_path": best_oof["path"],
        "test_path": best_test["path"],
        "oof_col": best_oof["col"],
        "test_col": best_test["col"],
        "oof_pred": best_oof["pred"],
        "test_pred": best_test["pred"],
        "oof_mse": best_oof["oof_mse"],
    }

    return comp, f"{name}: loaded"

# Known core artifacts.
artifact_specs_28f = [
    {
        "name": "rf500",
        "oof": ["model_results/oof_rf_500_base.csv"],
        "test": [
            "submission_rf_500_base_oof_foldavg.csv",
            "model_results/testpred_rf_500_base_folds.csv",
        ],
    },
    {
        "name": "extratrees_safe",
        "oof": ["model_results/oof_extratrees_safe_base.csv"],
        "test": ["model_results/testpred_extratrees_safe_base_foldavg.csv"],
    },
    {
        "name": "lgbm_12k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_foldavg.csv"],
    },
    {
        "name": "lgbm_long80k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof_long80k_lr03.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_long80k_lr03_foldavg.csv"],
    },
    {
        "name": "lgbm_long100k",
        "oof": ["model_results/oof_lgbm_t03_base_5fold_oof_long100k_lr02.csv"],
        "test": ["model_results/testpred_lgbm_t03_base_5fold_oof_long100k_lr02_foldavg.csv"],
    },
    {
        "name": "current_best_pre_resid_blend_80p662",
        "oof": ["model_results/oof_blend_auto_et_lgbm_long_oof_weighted.csv"],
        "test": [
            "model_results/testpred_blend_auto_et_lgbm_long_oof_weighted.csv",
            "submission_blend_auto_et_lgbm_long_oof_weighted.csv",
        ],
    },
    {
        "name": "lgbm_te_v1",
        "oof": ["model_results/oof_lgbm_te_base_5fold_oof_v1.csv"],
        "test": ["model_results/testpred_lgbm_te_base_5fold_oof_v1_foldavg.csv"],
    },
]

# Try to include the current 77.277 residual-stack anchor if an OOF file exists.
resid_oof_candidates_28f = _unique_keep_order(
    glob.glob("model_results/*resid*stack*oof*.csv") +
    glob.glob("model_results/oof*resid*stack*.csv") +
    glob.glob("model_results/*residstack*.csv")
)

resid_test_candidates_28f = _unique_keep_order(
    glob.glob("submission_residstack_lgbm_on_best80p662_blend_5fold_expanded_lambda_1p135.csv") +
    glob.glob("model_results/testpred*resid*stack*1p135*.csv") +
    glob.glob("model_results/*resid*stack*test*.csv")
)

artifact_specs_28f.append({
    "name": "residstack_best77_if_oof_available",
    "oof": resid_oof_candidates_28f,
    "test": resid_test_candidates_28f,
})

components_28f = []
load_messages_28f = []

for spec in artifact_specs_28f:
    comp, msg = load_component(
        name=spec["name"],
        oof_candidates=spec["oof"],
        test_candidates=spec["test"],
        y_ref=y_blend_28f,
        test_ids_ref=test_ids_28f,
    )
    load_messages_28f.append(msg)
    if comp is not None:
        components_28f.append(comp)

print("Load messages:")
for msg in load_messages_28f:
    print(" -", msg)

have_te_28f = any(c["name"] == "lgbm_te_v1" for c in components_28f)

shape_ok_28f = all(
    len(c["oof_pred"]) == n_train_28f and len(c["test_pred"]) == n_test_28f
    for c in components_28f
)

finite_ok_28f = all(
    np.isfinite(c["oof_pred"]).all() and np.isfinite(c["test_pred"]).all()
    for c in components_28f
)

BLEND28F_PASS = bool(have_te_28f and len(components_28f) >= 1 and shape_ok_28f and finite_ok_28f)

summary_28f = pd.DataFrame([
    {
        "name": c["name"],
        "oof_mse": c["oof_mse"],
        "oof_col": c["oof_col"],
        "test_col": c["test_col"],
        "oof_path": c["oof_path"],
        "test_path": c["test_path"],
        "oof_mean": float(np.mean(c["oof_pred"])),
        "oof_std": float(np.std(c["oof_pred"])),
        "test_mean": float(np.mean(c["test_pred"])),
        "test_std": float(np.std(c["test_pred"])),
    }
    for c in components_28f
]).sort_values("oof_mse").reset_index(drop=True)

print("\nBLEND28F_PASS =", BLEND28F_PASS)
print("Loaded components:", len(components_28f))
print("Have TE component:", have_te_28f)
print("Shape OK:", shape_ok_28f)
print("Finite OK:", finite_ok_28f)

display(summary_28f)

blend_component_names_28f = [c["name"] for c in components_28f]
blend_oof_mat_28f = np.column_stack([c["oof_pred"] for c in components_28f]).astype(np.float32)
blend_test_mat_28f = np.column_stack([c["test_pred"] for c in components_28f]).astype(np.float32)
blend_y_28f = y_blend_28f.astype(np.float32)
blend_component_summary_28f = summary_28f.copy()

if len(components_28f) >= 2:
    resid_mat_28f = blend_y_28f.reshape(-1, 1) - blend_oof_mat_28f
    resid_corr_28f = pd.DataFrame(
        resid_mat_28f,
        columns=blend_component_names_28f
    ).corr()
    print("\nResidual correlation matrix:")
    display(resid_corr_28f)

Load messages:
 - rf500: loaded
 - extratrees_safe: loaded
 - lgbm_12k: loaded
 - lgbm_long80k: loaded
 - lgbm_long100k: loaded
 - current_best_pre_resid_blend_80p662: no OOF file found
 - lgbm_te_v1: loaded
 - residstack_best77_if_oof_available: loaded

BLEND28F_PASS = True
Loaded components: 7
Have TE component: True
Shape OK: True
Finite OK: True


,name,oof_mse,oof_col,test_col,oof_path,test_path,oof_mean,oof_std,test_mean,test_std
0,lgbm_te_v1,82.907614,pred_raw,PERCENT_PROFICIENT,model_results/oof_lgbm_te_base_5fold_oof_v1.csv,model_results/testpred_lgbm_te_base_5fold_oof_...,54.535107,24.920244,54.500694,24.778822
1,residstack_best77_if_oof_available,85.647385,OOF_residstack_oldnew_expanded_oldw_0p025_neww...,PERCENT_PROFICIENT,model_results/oof_residstack_oldnew_expanded_o...,submission_residstack_lgbm_on_best80p662_blend...,54.241405,25.043358,54.097843,24.676031
2,lgbm_long100k,93.726214,OOF_lgbm_t03_base_5fold_oof_long100k_lr02,TESTPRED_lgbm_t03_base_5fold_oof_long100k_lr02,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...,54.153145,24.734667,54.074207,24.605724
3,lgbm_long80k,94.163485,OOF_lgbm_t03_base_5fold_oof_long80k_lr03,TESTPRED_lgbm_t03_base_5fold_oof_long80k_lr03,model_results/oof_lgbm_t03_base_5fold_oof_long...,model_results/testpred_lgbm_t03_base_5fold_oof...,54.147018,24.743267,54.067600,24.607933
4,lgbm_12k,98.779406,OOF_lgbm_t03_base_5fold_oof,TESTPRED_lgbm_t03_base_5fold_oof,model_results/oof_lgbm_t03_base_5fold_oof.csv,model_results/testpred_lgbm_t03_base_5fold_oof...,54.152809,24.283770,54.086674,24.207260
5,extratrees_safe,110.749294,oof_pred,PERCENT_PROFICIENT,model_results/oof_extratrees_safe_base.csv,model_results/testpred_extratrees_safe_base_fo...,54.127365,23.857744,54.040169,23.782902
6,rf500,122.328024,rf_500_base_oof_pred,PERCENT_PROFICIENT,model_results/oof_rf_500_base.csv,submission_rf_500_base_oof_foldavg.csv,54.134144,22.839348,54.056767,22.817463



Residual correlation matrix:


,rf500,extratrees_safe,lgbm_12k,lgbm_long80k,lgbm_long100k,lgbm_te_v1,residstack_best77_if_oof_available
rf500,1.000000,0.929709,0.842448,0.791132,0.792623,0.759557,0.834595
extratrees_safe,0.929709,1.000000,0.802689,0.771684,0.773277,0.789169,0.873778
lgbm_12k,0.842448,0.802689,1.000000,0.972530,0.965221,0.797072,0.935210
lgbm_long80k,0.791132,0.771684,0.972530,1.000000,0.991213,0.787838,0.955306
lgbm_long100k,0.792623,0.773277,0.965221,0.991213,1.000000,0.789823,0.958859
lgbm_te_v1,0.759557,0.789169,0.797072,0.787838,0.789823,1.000000,0.837032
residstack_best77_if_oof_available,0.834595,0.873778,0.935210,0.955306,0.958859,0.837032,1.000000


### public leaderboard checkpoint: target/statistical encoding breakthrough

Submitted file:

`submission_lgbm_te_base_5fold_oof_v1_foldavg.csv`

Public leaderboard MSE:

**70.997**

Artifact:

`lgbm_te_base_5fold_oof_v1`

Model:

LightGBM on the 162-feature base matrix plus 68 leakage-safe cross-fitted target/statistical encoding features, for a total of 230 features.

OOF clipped MSE:

**82.907608**

Fold validation MSEs:

- Fold 1: 83.846176
- Fold 2: 79.800461
- Fold 3: 84.527496
- Fold 4: 83.476036
- Fold 5: 82.887871

Interpretation:

This is the strongest public result so far and a major validation of the new hierarchical target/statistical encoding direction. The previous best public score was 77.277 from the residual-stack branch. The new public score, 70.997, beats it by about 6.28 MSE points.

This branch should now be treated as a primary modeling direction. Future work should include repaired blending with verified artifacts, residual stacking on the TE model, and model-family expansion using the same target/statistical encoding representation.

In [66]:
# ============================================================
# 28G REPAIRED. Safe convex blend with TE artifact
# Excludes residual-stack artifacts unless explicitly verified later.
# ============================================================
#
# Why this replaces the previous 28G:
# - Previous 28G may have mixed a residual-stack OOF file with a different
#   residual-stack test/submission file.
# - This repaired cell uses only verified non-residual components:
#     rf500
#     extratrees_safe
#     lgbm_12k
#     lgbm_long80k
#     lgbm_long100k
#     lgbm_te_v1
#
# Output submission:
#   submission_blend_te_verified_noresid_weighted.csv
#
# Paste the output before submitting.
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

if "components_28f" not in globals():
    raise ValueError("components_28f is missing. Re-run 28F first, then run this repaired 28G.")

if "test_ids_28f" not in globals():
    raise ValueError("test_ids_28f is missing. Re-run 28F first, then run this repaired 28G.")

if "y_train" not in globals():
    raise ValueError("y_train is missing.")

os.makedirs("model_results", exist_ok=True)

allowed_safe_components_28g = {
    "rf500",
    "extratrees_safe",
    "lgbm_12k",
    "lgbm_long80k",
    "lgbm_long100k",
    "lgbm_te_v1",
}

safe_components_28g = [
    c for c in components_28f
    if c["name"] in allowed_safe_components_28g
]

safe_names_28g = [c["name"] for c in safe_components_28g]

missing_required_28g = [
    name for name in ["lgbm_te_v1"]
    if name not in safe_names_28g
]

SAFE_BLEND28G_PREFLIGHT_PASS = bool(
    len(safe_components_28g) >= 2
    and len(missing_required_28g) == 0
)

print("SAFE_BLEND28G_PREFLIGHT_PASS =", SAFE_BLEND28G_PREFLIGHT_PASS)
print("Allowed safe components loaded:", safe_names_28g)
print("Missing required components:", missing_required_28g)

if not SAFE_BLEND28G_PREFLIGHT_PASS:
    raise ValueError("Safe blend preflight failed. Do not continue.")

y_safe_28g = np.asarray(y_train, dtype=np.float64).reshape(-1)
test_ids_safe_28g = np.asarray(test_ids_28f)

P_safe_28g = np.column_stack([
    np.asarray(c["oof_pred"], dtype=np.float64)
    for c in safe_components_28g
])

T_safe_28g = np.column_stack([
    np.asarray(c["test_pred"], dtype=np.float64)
    for c in safe_components_28g
])

# Since all component predictions are clipped to [0, 100],
# convex combinations remain inside [0, 100].
n_safe_28g, k_safe_28g = P_safe_28g.shape

shape_ok_safe_28g = (
    P_safe_28g.shape[0] == len(y_safe_28g)
    and T_safe_28g.shape[0] == len(test_ids_safe_28g)
)

finite_ok_safe_28g = (
    np.isfinite(P_safe_28g).all()
    and np.isfinite(T_safe_28g).all()
)

if not shape_ok_safe_28g or not finite_ok_safe_28g:
    raise ValueError("Shape or finite check failed.")

component_mses_safe_28g = np.array([
    float(mean_squared_error(y_safe_28g, P_safe_28g[:, i]))
    for i in range(k_safe_28g)
])

component_summary_safe_28g = pd.DataFrame({
    "component": safe_names_28g,
    "component_oof_mse": component_mses_safe_28g,
    "oof_mean": P_safe_28g.mean(axis=0),
    "oof_std": P_safe_28g.std(axis=0),
    "test_mean": T_safe_28g.mean(axis=0),
    "test_std": T_safe_28g.std(axis=0),
}).sort_values("component_oof_mse").reset_index(drop=True)

print("\nSafe component summary:")
display(component_summary_safe_28g)

A_safe_28g = (P_safe_28g.T @ P_safe_28g) / n_safe_28g
b_safe_28g = (P_safe_28g.T @ y_safe_28g) / n_safe_28g
c_safe_28g = float((y_safe_28g @ y_safe_28g) / n_safe_28g)

def mse_for_weights_safe_28g(w):
    w = np.asarray(w, dtype=np.float64)
    return float(w @ A_safe_28g @ w - 2.0 * (w @ b_safe_28g) + c_safe_28g)

def mse_for_weight_matrix_safe_28g(W):
    W = np.asarray(W, dtype=np.float64)
    return (
        np.einsum("ij,jk,ik->i", W, A_safe_28g, W)
        - 2.0 * (W @ b_safe_28g)
        + c_safe_28g
    )

candidate_rows_safe_28g = []

def add_candidate_safe_28g(label, w):
    w = np.asarray(w, dtype=np.float64)
    w = np.maximum(w, 0)
    if w.sum() <= 0:
        return
    w = w / w.sum()
    row = {
        "label": label,
        "mse": mse_for_weights_safe_28g(w),
        "weights": w,
    }
    candidate_rows_safe_28g.append(row)

# Pure models.
for i, name in enumerate(safe_names_28g):
    w = np.zeros(k_safe_28g)
    w[i] = 1.0
    add_candidate_safe_28g(f"pure_{name}", w)

# Pairwise fine grid.
grid_safe_28g = np.linspace(0.0, 1.0, 1001)

for i in range(k_safe_28g):
    for j in range(i + 1, k_safe_28g):
        W = np.zeros((len(grid_safe_28g), k_safe_28g), dtype=np.float64)
        W[:, i] = grid_safe_28g
        W[:, j] = 1.0 - grid_safe_28g

        mses = mse_for_weight_matrix_safe_28g(W)
        best_idx = int(np.argmin(mses))

        add_candidate_safe_28g(
            f"pair_{safe_names_28g[i]}__{safe_names_28g[j]}",
            W[best_idx],
        )

# Random simplex searches.
rng_safe_28g = np.random.default_rng(globals().get("RANDOM_STATE", 9890) + 2809)

# Uniform search.
W_uniform_safe = rng_safe_28g.dirichlet(np.ones(k_safe_28g), size=50000)
mses_uniform_safe = mse_for_weight_matrix_safe_28g(W_uniform_safe)
add_candidate_safe_28g(
    "random_dirichlet_uniform_50k",
    W_uniform_safe[int(np.argmin(mses_uniform_safe))]
)

# TE-focused search.
te_idx_safe_28g = safe_names_28g.index("lgbm_te_v1")

alpha_te_safe = np.ones(k_safe_28g) * 0.25
alpha_te_safe[te_idx_safe_28g] = 10.0

# Also allow mass on the best older LGBM artifacts.
order_safe_28g = np.argsort(component_mses_safe_28g)
for idx in order_safe_28g[:min(3, k_safe_28g)]:
    alpha_te_safe[idx] = max(alpha_te_safe[idx], 2.5)

W_te_safe = rng_safe_28g.dirichlet(alpha_te_safe, size=75000)
mses_te_safe = mse_for_weight_matrix_safe_28g(W_te_safe)
add_candidate_safe_28g(
    "random_dirichlet_te_focused_75k",
    W_te_safe[int(np.argmin(mses_te_safe))]
)

# Top-components focused search.
alpha_top_safe = np.ones(k_safe_28g) * 0.20
for rank, idx in enumerate(order_safe_28g[:min(4, k_safe_28g)]):
    alpha_top_safe[idx] = 6.0 / (rank + 1)

W_top_safe = rng_safe_28g.dirichlet(alpha_top_safe, size=75000)
mses_top_safe = mse_for_weight_matrix_safe_28g(W_top_safe)
add_candidate_safe_28g(
    "random_dirichlet_top_focused_75k",
    W_top_safe[int(np.argmin(mses_top_safe))]
)

best_candidate_safe_28g = min(candidate_rows_safe_28g, key=lambda d: d["mse"])
best_w_safe_28g = best_candidate_safe_28g["weights"]

blend_oof_safe_28g = np.clip(P_safe_28g @ best_w_safe_28g, 0, 100).astype(np.float32)
blend_test_safe_28g = np.clip(T_safe_28g @ best_w_safe_28g, 0, 100).astype(np.float32)

best_mse_safe_28g = float(mean_squared_error(y_safe_28g, blend_oof_safe_28g))
pure_te_mse_safe_28g = float(mean_squared_error(y_safe_28g, P_safe_28g[:, te_idx_safe_28g]))
gain_vs_te_safe_28g = pure_te_mse_safe_28g - best_mse_safe_28g

weight_table_safe_28g = pd.DataFrame({
    "component": safe_names_28g,
    "weight": best_w_safe_28g,
    "component_oof_mse": component_mses_safe_28g,
}).sort_values("weight", ascending=False).reset_index(drop=True)

screen_rows_safe_28g = []
for cand in candidate_rows_safe_28g:
    row = {
        "label": cand["label"],
        "mse": cand["mse"],
    }
    for name, w_val in zip(safe_names_28g, cand["weights"]):
        row[f"w_{name}"] = w_val
    screen_rows_safe_28g.append(row)

blend_screen_safe_28g = (
    pd.DataFrame(screen_rows_safe_28g)
    .sort_values("mse")
    .reset_index(drop=True)
)

blend_oof_path_safe_28g = "model_results/oof_blend_te_verified_noresid_weighted.csv"
blend_test_path_safe_28g = "model_results/testpred_blend_te_verified_noresid_weighted.csv"
blend_submission_path_safe_28g = "submission_blend_te_verified_noresid_weighted.csv"
blend_screen_path_safe_28g = "model_results/blend_te_verified_noresid_weight_screen.csv"
blend_weight_path_safe_28g = "model_results/blend_te_verified_noresid_best_weights.csv"

pd.DataFrame({
    "row_index": np.arange(len(y_safe_28g)),
    "PERCENT_PROFICIENT": y_safe_28g,
    "pred_clipped": blend_oof_safe_28g,
}).to_csv(blend_oof_path_safe_28g, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_safe_28g,
    "PERCENT_PROFICIENT": blend_test_safe_28g,
}).to_csv(blend_test_path_safe_28g, index=False)

pd.DataFrame({
    "ASSESSMENT_ID": test_ids_safe_28g,
    "PERCENT_PROFICIENT": blend_test_safe_28g,
}).to_csv(blend_submission_path_safe_28g, index=False)

blend_screen_safe_28g.to_csv(blend_screen_path_safe_28g, index=False)
weight_table_safe_28g.to_csv(blend_weight_path_safe_28g, index=False)

SAFE_BLEND28G_PASS = True
SAFE_BLEND28G_SUBMISSION_WORTHY = bool(gain_vs_te_safe_28g >= 1.0)

print("\nSAFE_BLEND28G_PASS =", SAFE_BLEND28G_PASS)
print("Best candidate label:", best_candidate_safe_28g["label"])
print("Best safe blend OOF MSE:", best_mse_safe_28g)
print("Pure TE OOF MSE:", pure_te_mse_safe_28g)
print("Gain vs pure TE OOF:", gain_vs_te_safe_28g)
print("Current pure TE public MSE: 70.997")
print("SAFE_BLEND28G_SUBMISSION_WORTHY =", SAFE_BLEND28G_SUBMISSION_WORTHY)

print("\nBest safe weights:")
display(weight_table_safe_28g)

print("\nTop 20 safe blend candidates:")
display(blend_screen_safe_28g.head(20))

print("\nSaved files:")
print(" -", blend_oof_path_safe_28g)
print(" -", blend_test_path_safe_28g)
print(" -", blend_submission_path_safe_28g)
print(" -", blend_screen_path_safe_28g)
print(" -", blend_weight_path_safe_28g)

print("\nSubmission prediction summary:")
print(pd.Series(blend_test_safe_28g).describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

if SAFE_BLEND28G_SUBMISSION_WORTHY:
    print("\nCandidate to consider after review:")
    print(blend_submission_path_safe_28g)
else:
    print("\nSafe blend did not improve enough over pure TE. Keep it in backlog.")

SAFE_BLEND28G_PREFLIGHT_PASS = True
Allowed safe components loaded: ['rf500', 'extratrees_safe', 'lgbm_12k', 'lgbm_long80k', 'lgbm_long100k', 'lgbm_te_v1']
Missing required components: []

Safe component summary:


,component,component_oof_mse,oof_mean,oof_std,test_mean,test_std
0,lgbm_te_v1,82.907614,54.535106,24.920245,54.500693,24.778823
1,lgbm_long100k,93.726214,54.153144,24.734667,54.074208,24.605725
2,lgbm_long80k,94.163485,54.147019,24.743266,54.067601,24.607933
3,lgbm_12k,98.779406,54.152808,24.283770,54.086671,24.207260
4,extratrees_safe,110.749294,54.127367,23.857745,54.040173,23.782901
5,rf500,122.328024,54.134143,22.839348,54.056767,22.817463



SAFE_BLEND28G_PASS = True
Best candidate label: random_dirichlet_te_focused_75k
Best safe blend OOF MSE: 78.07151033226157
Pure TE OOF MSE: 82.90761366243048
Gain vs pure TE OOF: 4.8361033301689105
Current pure TE public MSE: 70.997
SAFE_BLEND28G_SUBMISSION_WORTHY = True

Best safe weights:


,component,weight,component_oof_mse
0,lgbm_te_v1,0.624592,82.907614
1,lgbm_long100k,0.207687,93.726214
2,lgbm_long80k,0.127595,94.163485
3,extratrees_safe,0.040027,110.749294
4,lgbm_12k,0.000082,98.779406
5,rf500,0.000017,122.328024



Top 20 safe blend candidates:


,label,mse,w_rf500,w_extratrees_safe,w_lgbm_12k,w_lgbm_long80k,w_lgbm_long100k,w_lgbm_te_v1
0,random_dirichlet_te_focused_75k,78.071510,0.000017,0.040027,0.000082,0.127595,0.207687,0.624592
1,random_dirichlet_top_focused_75k,78.085494,0.000026,0.034714,0.002192,0.130861,0.196293,0.635913
2,random_dirichlet_uniform_50k,78.129569,0.001443,0.029954,0.011069,0.100940,0.249669,0.606925
3,pair_lgbm_long100k__lgbm_te_v1,78.158831,0.000000,0.000000,0.000000,0.000000,0.356000,0.644000
4,pair_lgbm_long80k__lgbm_te_v1,78.206523,0.000000,0.000000,0.000000,0.352000,0.000000,0.648000
5,pair_lgbm_12k__lgbm_te_v1,79.778222,0.000000,0.000000,0.289000,0.000000,0.000000,0.711000
6,pair_extratrees_safe__lgbm_te_v1,81.633548,0.000000,0.173000,0.000000,0.000000,0.000000,0.827000
7,pair_rf500__lgbm_te_v1,82.103802,0.124000,0.000000,0.000000,0.000000,0.000000,0.876000
8,pure_lgbm_te_v1,82.907614,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
9,pair_extratrees_safe__lgbm_long100k,88.966368,0.000000,0.319000,0.000000,0.000000,0.681000,0.000000



Saved files:
 - model_results/oof_blend_te_verified_noresid_weighted.csv
 - model_results/testpred_blend_te_verified_noresid_weighted.csv
 - submission_blend_te_verified_noresid_weighted.csv
 - model_results/blend_te_verified_noresid_weight_screen.csv
 - model_results/blend_te_verified_noresid_best_weights.csv

Submission prediction summary:
count    48307.000000
mean        54.338383
std         24.551657
min          0.201202
1%           7.931219
5%          16.555257
25%         34.791862
50%         52.371319
75%         74.537514
95%         95.197914
99%         99.362563
max         99.999191
dtype: float64

Candidate to consider after review:
submission_blend_te_verified_noresid_weighted.csv
